In [1]:
# -*- coding: utf-8 -*-
"""R57b 路B ML 因子 (ml_xgb_wf35_pathB) 平台推理管线 — 多文件包成员。
35 特征平台侧现算 (权威实现对应 scripts/mine_r57b_mltrain.py 的 FEATURES 表,
公式逐一对齐各批次挖掘脚本/提交 notebook, 缓存因子为带符号值的已按要求乘 sign):
  L1 deal_number 系 (R45/R46/R52 notebook 口径: bar1m volume/deal_number/amount
     当日累计 -> 组内差分; close/盘口快照不差分);
  L2 在榜基底 4 (R38/R37/R20 notebook 公式, 含 MAD-winsorize 截面 z 与 tilt);
  L3 波动族 (R47/R52/R54/R55 挖掘口径 + factorlib daily_return 系 + 1d Parkinson/GK);
  L4 历史强因子 (R17-R20 notebook 公式; 注意两套组内 corr 约定: R45 系 min_n->NaN,
     R17/R20 系 n<30 或零方差 -> 0.0)。
模型: XGBoost walk-forward 年度 fold (models/model_YYYY.json); 预测年 Y 用
  model_Y (Y<=2020->2020, 2021->2021, 2022->2022, 2023->2023, >=2024->2024)。
  优先 xgboost.Booster, 无 xgboost 时纯 numpy 向量化树遍历兜底 (79 号 notebook 同思路,
  已向量化, 无 Python 行循环)。
输出: 逐日截面 plain z 后的特征面板 fillna(0) -> 模型 -> SIGN=+1 ->
  bigalpha_2026_instruments inner merge。特征 NaN 处理与训练严格一致
  (inner join 35 特征 -> 逐日 z (std=0 -> NaN) -> fillna(0))。
"""
import json
import os

import numpy as np
import pandas as pd

SIGN = 1

FEATURE_COLS = [
    "deals_absret_corr", "deals_ac1", "bigdeal_ratio", "updn_asym_15m",
    "updn_asym_ma5", "ret_tail3", "resid_rv5_20", "z_retrange30_volac1",
    "retmax15_cashq", "retmax15_cffps", "vwapdevchg5_cffps", "gk5", "park5",
    "vol_ts_5_20", "rv5_tsz20", "vwap_disp", "imb_x_ret_std", "exec_int_mean",
    "exec_int_std", "vollead_corr_ma5", "turn", "rv_skew_ma5", "rv_skew_ma5_15m",
    "upvol_asym_ma5", "kurt_ma5", "n_reversals_30m", "ret_max_15m",
    "absret_ac1_ma5", "vwap_dev_chg5", "gap_freq_20", "b_pvsign_resid_rank_top1",
    "down_vol_share_15", "nm_vol_ret_corr", "nm_n_reversals", "upvol_asym_ma5_15m",
    # R62 扩展 48
    "depth_mean", "depth_std", "depth_skew", "book_slope_mean", "book_slope_std",
    "slope_asym_std", "osize_asym_mean", "osize_asym_std", "imb3_std", "imb1_std",
    "imb1_skew", "imb_x_ret_mean", "mp_dev_mean", "mp_dev_std", "spread_mean",
    "spread_std", "imb3_ac1", "mp_dev_ac1", "orders_ret_corr", "osize_ret_corr",
    "spread_am", "imb3_am", "depth_am",
    "vollead_corr", "deallead_corr", "retlead_corr", "absretlead_corr",
    "mkt_corr", "mkt_corr_am", "amt_center", "deal_center",
    "deals_total", "deals_mean", "deals_std", "deals_skew", "deals_cv",
    "deals_updn_asym",
    "fill_asym_mean", "exec_x_ret", "deals_orders_corr",
    "deals_tail1_share", "vol_tail1_share", "ret_tail1", "tail1_deal_size",
    "tail3_deals_share",
    "vol_top_third", "vol_bot_third", "vwap_skew",
    # R74 19
    "bigdeal_ratio_d5",
    "deal_amt_mean_d5",
    "deal_amt_skew",
    "deal_size_skew_d5",
    "deal_size_std_d5",
    "deals_absret_corr_d5",
    "deals_ac1_d5",
    "deals_conc_d5",
    "deals_cv_d5",
    "deals_kurt_d5",
    "deals_mean_d5",
    "deals_morning_share_d5",
    "deals_pm_am_d5",
    "deals_skew_d5",
    "deals_std_d5",
    "deals_tail_share_d5",
    "deals_total_d5",
    "deals_x_ret_d15",
    "deals_x_ret_d5",
    # R72 29
    "deals_open1_share",
    "vol_open1_share",
    "deals_open5_share",
    "vol_open5_share",
    "open1_deal_size",
    "exec_int_med",
    "exec_int_am",
    "exec_int_pm",
    "ret_1430_1456",
    "depth_mean_b15m",
    "depth_std_b15m",
    "depth_skew_b15m",
    "book_slope_mean_b15m",
    "book_slope_std_b15m",
    "slope_asym_std_b15m",
    "osize_asym_mean_b15m",
    "osize_asym_std_b15m",
    "imb3_std_b15m",
    "imb1_std_b15m",
    "imb1_skew_b15m",
    "imb_x_ret_mean_b15m",
    "imb_x_ret_std_b15m",
    "mp_dev_mean_b15m",
    "mp_dev_std_b15m",
    "spread_mean_b15m",
    "spread_std_b15m",
    "imb3_ac1_b15m",
    "mp_dev_ac1_b15m",
    "orders_ret_corr_b15m",
    "osize_ret_corr_b15m",
    # R70 tail2 15
    "t3_amt_share",
    "dsize_t3",
    "t3_dsize_vs_day",
    "t3_sgnvol",
    "t3_ret_vs_day",
    "t3_vol_conc",
    "t3_vs_pre_ret",
    "pre_vol_share",
    "t3_dn_vs_pre_dn",
    "t3_imb1_mean",
    "t3_spread_mean",
    "ret_1455",
    "vol_1455_share",
    "ret_t3_std",
    "dsize_t3_max",
    # R67 tech 21
    "rsi_12",
    "bias_20",
    "cci_14",
    "macd_diff_12_26_9",
    "macd_dea_12_26_9",
    "macd_hist_12_26_9",
    "kdj_k_9_3_3",
    "kdj_d_9_3_3",
    "atr_14",
    "ema_20",
    "sma_20",
    "change_ratio",
    "netflow_amount_main",
    "netflow_amount_rate_main",
    "net_active_buy_amount_main",
    "pe_ttm",
    "ps_ttm",
    "pb",
    "momentum_5",
    "reversal_5",
    "volatility_5",
]

MORNING = (575, 630)  # 09:35-10:30

# ---------------------------------------------------------------- 数据访问

def _iter_month_ranges(q_start, q_end):
    q_start = pd.Timestamp(q_start)
    q_end = pd.Timestamp(q_end)
    cur = pd.Timestamp(year=q_start.year, month=q_start.month, day=1)
    while cur <= q_end:
        nxt = cur + pd.offsets.MonthBegin(1)
        cs = max(q_start, cur)
        ce = min(q_end, nxt - pd.Timedelta(seconds=1))
        if cs <= ce:
            yield cs, ce
        cur = nxt


def _dai_query_monthly(sql, q_start, q_end):
    """按月分片查询, 避免一次性加载全量 bar1m 导致 OOM。"""
    import dai
    frames = []
    for cs, ce in _iter_month_ranges(q_start, q_end):
        part = dai.query(
            sql,
            filters={"date": [cs.strftime("%Y-%m-%d %H:%M:%S"),
                              ce.strftime("%Y-%m-%d %H:%M:%S")]},
            compression=True,
        ).df()
        if len(part):
            frames.append(part)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _load_bar1m(datasources, start_date, end_date, buf_days=100):
    """平台 bar1m: volume/deal_number/amount 当日累计 -> 组内差分 (R45 结论);
    close/open/盘口字段为快照, 不差分 (R47 结论)。"""
    bar1m = datasources["bar1m"]
    q_start = pd.to_datetime(start_date) - pd.Timedelta(days=buf_days)
    q_end = pd.to_datetime(end_date)
    sql = (f"SELECT date, instrument::STRING AS instrument, open, high, low, close, "
           f"pre_close, volume, amount, deal_number, bid_price1, ask_price1, "
           f"bid_price2, bid_price3, ask_price2, ask_price3, "
           f"bid_volume1, bid_volume2, bid_volume3, "
           f"ask_volume1, ask_volume2, ask_volume3, "
           f"bid_num_orders1, bid_num_orders2, bid_num_orders3, "
           f"ask_num_orders1, ask_num_orders2, ask_num_orders3 "
           f"FROM {bar1m} WHERE close > 0 ORDER BY instrument, date")
    df = _dai_query_monthly(sql, q_start, q_end)
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    df["td"] = df["date"].dt.normalize()
    df["mins"] = df["date"].dt.hour * 60 + df["date"].dt.minute
    gk = ["instrument", "td"]
    for col, out in (("volume", "vol_min"), ("amount", "amt_min"),
                     ("deal_number", "dn_min")):
        df[out] = df[col] - df.groupby(gk, sort=False)[col].shift(1)
        df[out] = df[out].fillna(df[col]).clip(lower=0)
    return df


def _load_factorlib(start_date, end_date, buf_days=100):
    q_start = pd.to_datetime(start_date) - pd.Timedelta(days=buf_days)
    q_end = pd.to_datetime(end_date)
    sql = ("SELECT date, instrument, daily_return, turn, "
            "rsi_12, bias_20, cci_14, macd_diff_12_26_9, macd_dea_12_26_9, macd_hist_12_26_9, kdj_k_9_3_3, kdj_d_9_3_3, atr_14, ema_20, sma_20, change_ratio, netflow_amount_main, netflow_amount_rate_main, net_active_buy_amount_main, pe_ttm, ps_ttm, pb, momentum_5, reversal_5, volatility_5 FROM bigalpha_2026_factorlib")
    df = _dai_query_monthly(sql, q_start, q_end)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    return df.sort_values(["instrument", "date"]).reset_index(drop=True)


def _fin_grid_end(end_date):
    """财务 ffill 的 canonical 窗口右端 (R58 移植发现, 见 _load_fin 注释):
    缓存构建用 grid 2019-01-01..2024-12-31 + fin 2018-07..2024-12-31;
    verify_f8 证明右端取 2023-12-31 即可逐值复现缓存 (≤2023-12-31),
    生产期 (>2023-12-31) 用实际 end_date 向右自然延展。"""
    return max(pd.to_datetime(end_date), pd.Timestamp("2023-12-31"))


def _load_fin(datasources, start_date, end_date):
    """财务 PIT, 固定 canonical 窗口 2018-07-01 .. _fin_grid_end(end_date)。
    注意 (R58 移植发现): mine_r26_tilt2.fin_components 的 grid.merge+ffill 结果
    依赖合并后行序 (当前 pandas 下左合并不保序, 乱序 ffill 成为缓存构建的一部分),
    只有用与缓存构建相同的 canonical 输入窗口才能复现缓存值。"""
    fin = datasources["financial"]
    q_start = pd.Timestamp("2018-07-01")
    q_end = _fin_grid_end(end_date)
    sql = (f"SELECT date, instrument::STRING AS instrument, category, "
           f"net_cffoa, net_profit, net_cfffa, latest_shares "
           f"FROM {fin} WHERE shift=0")
    df = _dai_query_monthly(sql, q_start, q_end)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    return df


def _stk(start_date, end_date):
    import dai
    stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                    filters={"date": [str(pd.to_datetime(start_date)),
                                      str(pd.to_datetime(end_date))]}).df()
    stk["date"] = pd.to_datetime(stk["date"]).dt.normalize()
    stk["instrument"] = stk["instrument"].astype(str)
    return stk


# ---------------------------------------------------------------- 通用算子

def _grp_corr_nan(m, x, y, min_n=10):
    """组内 pearson; 样本<min_n 或零方差 -> NaN (R45 系约定)。"""
    t = m[["instrument", "td", x, y]].dropna(subset=[x, y])
    t = t.assign(xx=t[x] * t[x], yy=t[y] * t[y], xy=t[x] * t[y])
    gb = t.groupby(["instrument", "td"], sort=False)
    s = gb[[x, y, "xx", "yy", "xy"]].sum()
    n = gb.size()
    num = n * s["xy"] - s[x] * s[y]
    den = np.sqrt((n * s["xx"] - s[x] ** 2) * (n * s["yy"] - s[y] ** 2))
    return (num / den.replace(0, np.nan)).where(n >= min_n)


def _grp_corr_zero(m, x, y, min_n=30):
    """组内 pearson; n<min_n 或零方差 -> 0.0 (R17/R20 系约定)。"""
    t = m[["instrument", "td", x, y]].dropna(subset=[x, y]).copy()
    t["xy"] = t[x] * t[y]
    t["xx"] = t[x] ** 2
    t["yy"] = t[y] ** 2
    a = t.groupby(["instrument", "td"], sort=False).agg(
        n=(x, "size"), sx=(x, "sum"), sy=(y, "sum"),
        sxy=("xy", "sum"), sxx=("xx", "sum"), syy=("yy", "sum"))
    cov = a["sxy"] - a["sx"] * a["sy"] / a["n"]
    vx = a["sxx"] - a["sx"] ** 2 / a["n"]
    vy = a["syy"] - a["sy"] ** 2 / a["n"]
    out = cov / np.sqrt(vx.clip(lower=0) * vy.clip(lower=0)).replace(0, np.nan)
    out[(a["n"] < min_n) | (vx <= 0) | (vy <= 0)] = 0.0
    return out.fillna(0.0)


def _winsorize_mad(s, n=3.0):
    med = s.median()
    mad = (s - med).abs().median()
    if mad == 0 or mad != mad:
        return s
    return s.clip(med - n * 1.4826 * mad, med + n * 1.4826 * mad)


def _xs_z(df, col):
    """逐日 MAD-winsorize + z (mine_r26_tilt2 口径)。"""
    def _proc(g):
        g = g.copy()
        g[col] = _winsorize_mad(g[col])
        std = g[col].std()
        if std and std > 0:
            g[col] = (g[col] - g[col].mean()) / std
        return g
    out = df.dropna(subset=[col]).replace([np.inf, -np.inf], np.nan).dropna(subset=[col])
    return out.groupby("date", group_keys=False).apply(_proc)


def _to_nm(df, n):
    """1m -> %n 采样, 加 ret/absret/dn/deal_size/volume/amount (R45/R46 口径)。"""
    m = df[df["date"].dt.minute % n == 0].copy()
    gk = ["instrument", "td"]
    prev_c = m.groupby(gk, sort=False)["close"].shift(1)
    m["ret"] = (m["close"] / prev_c - 1.0).fillna(0.0)
    m["absret"] = m["ret"].abs()
    m["volume"] = m["vol_min"]
    m["amount"] = m["amt_min"]
    dn = m["dn_min"].astype(float)
    m["dn"] = dn.where(dn > 0)
    m["deal_size"] = (m["volume"] / dn.replace(0, np.nan)).where(dn > 0)
    return m


def _updn_asym(m):
    """涨/跌 bar 均笔(或均量) 不对称 (ret=0 剔除), 列名 dn 或 volume 自适应。"""
    gk = ["instrument", "td"]
    up = m["ret"] > 0
    dwn = m["ret"] < 0
    vu = m.assign(vu_up=m["_x"] * up, cnt_up=up.astype(float),
                  vu_dn=m["_x"] * dwn, cnt_dn=dwn.astype(float))
    va = vu.groupby(gk, sort=False).agg(
        vu_up=("vu_up", "sum"), cnt_up=("cnt_up", "sum"),
        vu_dn=("vu_dn", "sum"), cnt_dn=("cnt_dn", "sum"))
    mu = va["vu_up"] / va["cnt_up"].replace(0, np.nan)
    md = va["vu_dn"] / va["cnt_dn"].replace(0, np.nan)
    return ((mu - md) / (mu + md).replace(0, np.nan))


def _roll5(s, mp=3):
    return s.rolling(5, min_periods=mp).mean()


# ---------------------------------------------------------------- 特征计算

def _features(df1m, fl, fin, stk_grid):
    """返回 dict[name] -> [date, instrument, factor] (与缓存一致的带符号值)。"""
    F = {}
    m5 = _to_nm(df1m, 5)
    m15 = _to_nm(df1m, 15)
    m30 = _to_nm(df1m, 30)

    def day(s, name):
        d = s.rename(name).reset_index()
        d.columns = ["instrument", "date", name] if d.shape[1] == 3 else d.columns
        return d

    # ---- L1 deal_number 系 ----
    F["deals_absret_corr"] = -_grp_corr_nan(m5, "dn", "absret", 10)
    m5["dn_lag"] = m5.groupby(["instrument", "td"], sort=False)["dn"].shift(1)
    F["deals_ac1"] = -_grp_corr_nan(m5.dropna(subset=["dn", "dn_lag"]),
                                    "dn", "dn_lag", 10)
    q = m5.groupby(["instrument", "td"], sort=False)["deal_size"].agg(["max", "median"])
    F["bigdeal_ratio"] = q["max"] / q["median"].replace(0, np.nan)

    m15["_x"] = m15["dn"]
    F["updn_asym_15m"] = -_updn_asym(m15.dropna(subset=["dn"]))
    m5["_x"] = m5["dn"]
    a5 = _updn_asym(m5.dropna(subset=["dn"])).rename("v").reset_index()
    a5 = a5.sort_values(["instrument", "td"])
    a5["v"] = a5.groupby("instrument", group_keys=False)["v"].transform(_roll5)
    F["updn_asym_ma5"] = -a5.set_index(["instrument", "td"])["v"]

    # ret_tail3 (1m)
    g1 = ["instrument", "td"]
    df1m["seq"] = df1m.groupby(g1, sort=False).cumcount()
    df1m["nbar"] = df1m.groupby(g1, sort=False)["seq"].transform("max") + 1
    df1m["close_l3"] = df1m.groupby(g1, sort=False)["close"].shift(3)
    last = df1m[df1m["seq"] == df1m["nbar"] - 1]
    F["ret_tail3"] = -(last["close"] / last["close_l3"] - 1.0) \
        .set_axis(pd.MultiIndex.from_frame(last[["instrument", "td"]]))

    # ---- L2 在榜基底 ----
    # z_retrange30_volac1: (z_mad(retrange30)+z_mad(vol_ac1_30m))/2, sign -1
    a = m30.groupby(["instrument", "td"], sort=False).agg(
        rmax=("ret", "max"), rmin=("ret", "min"))
    rr = (a["rmax"] - a["rmin"]).rename("v").reset_index() \
        .rename(columns={"td": "date"})
    m30["vol_l1"] = m30.groupby(g1, sort=False)["volume"].shift(1)
    t = m30.dropna(subset=["vol_l1"]).copy()
    t["xy"] = t["volume"] * t["vol_l1"]
    t["xx"] = t["volume"] ** 2
    t["yy"] = t["vol_l1"] ** 2
    a2 = t.groupby(g1, sort=False).agg(
        n=("volume", "size"), sx=("volume", "sum"), sy=("vol_l1", "sum"),
        sxy=("xy", "sum"), sxx=("xx", "sum"), syy=("yy", "sum"))
    cov = a2["sxy"] - a2["sx"] * a2["sy"] / a2["n"]
    vx = a2["sxx"] - a2["sx"] ** 2 / a2["n"]
    vy = a2["syy"] - a2["sy"] ** 2 / a2["n"]
    va = (cov / np.sqrt(vx.clip(lower=0) * vy.clip(lower=0)).replace(0, np.nan))
    va[(a2["n"] < 5) | (vx <= 0) | (vy <= 0)] = np.nan
    va = va.rename("v").reset_index().rename(columns={"td": "date"})
    za = _xs_z(rr, "v").rename(columns={"v": "za"})
    zb = _xs_z(va.dropna(subset=["v"]), "v").rename(columns={"v": "zb"})
    mm = za.merge(zb, on=["date", "instrument"], how="inner")
    F["z_retrange30_volac1"] = -((mm["za"] + mm["zb"]) / 2.0) \
        .set_axis(pd.MultiIndex.from_frame(mm[["date", "instrument"]]))

    # ret_max_15m (raw, 也供 tilt 用)
    rmax15 = m15.groupby(["instrument", "td"], sort=False)["ret"].max()
    F["ret_max_15m"] = rmax15

    # fin cashq / cffps (r13 PIT 口径, ffill 到池网格)
    fin_z = {}
    if fin is not None and len(fin):
        grid = stk_grid[["date", "instrument"]].drop_duplicates()

        def _fin_ratio(num, den):
            a = fin[fin["category"] == "ttm"][["date", "instrument", num]] \
                .rename(columns={num: "vn"})
            b = fin[fin["category"] == "ttm"][["date", "instrument", den]] \
                .rename(columns={den: "vd"})
            m = a.merge(b, on=["date", "instrument"])
            m["fv"] = m["vn"] / (m["vd"] + 1e-8)
            m = m[["date", "instrument", "fv"]].dropna()
            g = grid.merge(m, on=["date", "instrument"], how="left")
            g["fv"] = g.groupby("instrument", group_keys=False)["fv"].ffill()
            return g.dropna(subset=["fv"])

        def _fin_ps(num):
            a = fin[fin["category"] == "ttm"][["date", "instrument", num]] \
                .rename(columns={num: "vn"})
            b = fin[fin["category"] == "lf"][["date", "instrument", "latest_shares"]] \
                .rename(columns={"latest_shares": "sh"})
            m = a.merge(b, on=["date", "instrument"])
            m["fv"] = m["vn"] / (m["sh"] + 1e-8)
            m = m[["date", "instrument", "fv"]].dropna()
            g = grid.merge(m, on=["date", "instrument"], how="left")
            g["fv"] = g.groupby("instrument", group_keys=False)["fv"].ffill()
            return g.dropna(subset=["fv"])

        fin_z["cashq"] = _fin_ratio("net_cffoa", "net_profit")
        fin_z["cffps"] = _fin_ps("net_cfffa")

    rm = rmax15.rename("v").reset_index().rename(columns={"td": "date"})
    zrm = _xs_z(rm, "v").rename(columns={"v": "zb"})
    for fid, fin_name, w, fsign in (("retmax15_cashq", "cashq", 1.0, 1),
                                    ("retmax15_cffps", "cffps", 1.0, -1)):
        zf = _xs_z(fin_z[fin_name], "fv").rename(columns={"fv": "zf"})
        mm2 = zrm.merge(zf, on=["date", "instrument"], how="inner")
        # cache = -[zb * (1+w*(fsign*zf).clip(-2,2))] (cffps FIN_SIGN=-1, 见 r37 #05)
        F[fid] = -(mm2["zb"] * (1.0 + w * (fsign * mm2["zf"]).clip(-2.0, 2.0))) \
            .set_axis(pd.MultiIndex.from_frame(mm2[["date", "instrument"]]))

    # vwapdevchg5_cffps: -z(vwap_dev_chg5) * (1+0.5*clip(-z(cffps),-2,2)), sign +1
    am5 = m5.groupby(["instrument", "td"], sort=False).agg(
        amt=("amount", "sum"), vol=("volume", "sum"), close_d=("close", "last"))
    dev = (am5["close_d"] / (am5["amt"] / am5["vol"].replace(0, np.nan)) - 1.0) \
        .rename("v").reset_index().rename(columns={"td": "date"})
    dev = dev.sort_values(["instrument", "date"])
    dev["v"] = dev["v"] - dev.groupby("instrument", group_keys=False)["v"].shift(5)
    F["vwap_dev_chg5"] = -dev.set_index(["date", "instrument"])["v"]  # sign -1
    zdev = _xs_z(dev.dropna(subset=["v"]), "v").rename(columns={"v": "zb"})
    zcf = _xs_z(fin_z["cffps"], "fv").rename(columns={"fv": "zf"})
    mm3 = zdev.merge(zcf, on=["date", "instrument"], how="inner")
    F["vwapdevchg5_cffps"] = ((-mm3["zb"])
                              * (1.0 + 0.5 * (-mm3["zf"]).clip(-2.0, 2.0))) \
        .set_axis(pd.MultiIndex.from_frame(mm3[["date", "instrument"]]))

    # ---- L3 波动族 ----
    # vwap_disp (raw, R54 口径)
    hl = m5.groupby(["instrument", "td"], sort=False).agg(
        vol=("volume", "sum"), amt=("amount", "sum"))
    vwap_d = (hl["amt"] / hl["vol"].replace(0, np.nan)).rename("vwap")
    m5j = m5.join(vwap_d, on=["instrument", "td"])
    dv = m5j["close"] - m5j["vwap"]
    m5j["wv2"] = m5j["volume"] * dv ** 2
    w2 = m5j.groupby(["instrument", "td"], sort=False).agg(
        wv2=("wv2", "sum"), vol=("volume", "sum"))
    w2 = w2.join(vwap_d, on=["instrument", "td"])
    F["vwap_disp"] = (np.sqrt(w2["wv2"] / w2["vol"].replace(0, np.nan))
                      / w2["vwap"].replace(0, np.nan))

    # imb_x_ret_std (sign -1, R47 口径)
    bv = m5["bid_volume1"] + m5["bid_volume2"] + m5["bid_volume3"]
    av = m5["ask_volume1"] + m5["ask_volume2"] + m5["ask_volume3"]
    ok_vol = (bv + av) > 0
    m5["imb3"] = ((bv - av) / (bv + av)).where(ok_vol)
    m5["imb_x_ret"] = m5["imb3"] * m5["ret"]
    F["imb_x_ret_std"] = -m5.groupby(["instrument", "td"], sort=False)[
        "imb_x_ret"].std()

    # exec_int (raw, R52 口径)
    bno = m5["bid_num_orders1"] + m5["bid_num_orders2"] + m5["bid_num_orders3"]
    ano = m5["ask_num_orders1"] + m5["ask_num_orders2"] + m5["ask_num_orders3"]
    no_total = (bno + ano).where((bno + ano) > 0)
    m5["exec_int"] = (m5["dn_min"].astype(float) / no_total).where(no_total > 0)
    F["exec_int_mean"] = m5.groupby(["instrument", "td"], sort=False)["exec_int"].mean()
    F["exec_int_std"] = m5.groupby(["instrument", "td"], sort=False)["exec_int"].std()

    # vollead_corr_ma5 (sign -1): corr(vol_t, |ret_t+1|) 5m, roll5
    m5["absret_lead"] = m5.groupby(["instrument", "td"], sort=False)["absret"].shift(-1)
    vl = _grp_corr_nan(m5, "volume", "absret_lead", 10).rename("v").reset_index()
    vl = vl.sort_values(["instrument", "td"])
    vl["v"] = vl.groupby("instrument", group_keys=False)["v"].transform(_roll5)
    F["vollead_corr_ma5"] = -vl.set_index(["instrument", "td"])["v"]

    # turn (raw factorlib)
    F["turn"] = fl.set_index(["date", "instrument"])["turn"]

    # R67 技术指标/资金流/估值 (raw, main 统一逐日 z)
    F["rsi_12"] = fl.set_index(["date", "instrument"])["rsi_12"]
    F["bias_20"] = fl.set_index(["date", "instrument"])["bias_20"]
    F["cci_14"] = fl.set_index(["date", "instrument"])["cci_14"]
    F["macd_diff_12_26_9"] = fl.set_index(["date", "instrument"])["macd_diff_12_26_9"]
    F["macd_dea_12_26_9"] = fl.set_index(["date", "instrument"])["macd_dea_12_26_9"]
    F["macd_hist_12_26_9"] = fl.set_index(["date", "instrument"])["macd_hist_12_26_9"]
    F["kdj_k_9_3_3"] = fl.set_index(["date", "instrument"])["kdj_k_9_3_3"]
    F["kdj_d_9_3_3"] = fl.set_index(["date", "instrument"])["kdj_d_9_3_3"]
    F["atr_14"] = fl.set_index(["date", "instrument"])["atr_14"]
    F["ema_20"] = fl.set_index(["date", "instrument"])["ema_20"]
    F["sma_20"] = fl.set_index(["date", "instrument"])["sma_20"]
    F["change_ratio"] = fl.set_index(["date", "instrument"])["change_ratio"]
    F["netflow_amount_main"] = fl.set_index(["date", "instrument"])["netflow_amount_main"]
    F["netflow_amount_rate_main"] = fl.set_index(["date", "instrument"])["netflow_amount_rate_main"]
    F["net_active_buy_amount_main"] = fl.set_index(["date", "instrument"])["net_active_buy_amount_main"]
    F["pe_ttm"] = fl.set_index(["date", "instrument"])["pe_ttm"]
    F["ps_ttm"] = fl.set_index(["date", "instrument"])["ps_ttm"]
    F["pb"] = fl.set_index(["date", "instrument"])["pb"]
    F["momentum_5"] = fl.set_index(["date", "instrument"])["momentum_5"]
    F["reversal_5"] = fl.set_index(["date", "instrument"])["reversal_5"]
    F["volatility_5"] = fl.set_index(["date", "instrument"])["volatility_5"]

    # resid_rv5_20 (sign -1, R52 口径; factorlib daily_return)
    P = fl.sort_values(["instrument", "date"]).reset_index(drop=True)
    mkt = P.groupby("date")["daily_return"].mean().rename("mkt")
    P = P.merge(mkt, on="date", how="left")
    g = P.groupby("instrument", group_keys=False)
    xy = P["daily_return"] * P["mkt"]
    y2 = P["mkt"] ** 2
    ex = g["daily_return"].transform(lambda s: s.rolling(60, min_periods=45).mean())
    ey = g["mkt"].transform(lambda s: s.rolling(60, min_periods=45).mean())
    exy = xy.groupby(P["instrument"], group_keys=False).transform(
        lambda s: s.rolling(60, min_periods=45).mean())
    ey2 = y2.groupby(P["instrument"], group_keys=False).transform(
        lambda s: s.rolling(60, min_periods=45).mean())
    beta60 = (exy - ex * ey) / (ey2 - ey ** 2).replace(0, np.nan)
    resid = P["daily_return"] - beta60 * P["mkt"]
    gr = resid.groupby(P["instrument"], group_keys=False)
    rv5 = gr.transform(lambda s: s.rolling(5, min_periods=4).std())
    rv20 = gr.transform(lambda s: s.rolling(20, min_periods=15).std())
    P["resid_rv5_20"] = -(rv5 / rv20.replace(0, np.nan))
    F["resid_rv5_20"] = P.set_index(["date", "instrument"])["resid_rv5_20"]

    # vol_ts_5_20 / rv5_tsz20 (raw, factorlib daily_return)
    rv5r = g["daily_return"].transform(lambda s: s.rolling(5, min_periods=4).std())
    rv20r = g["daily_return"].transform(lambda s: s.rolling(20, min_periods=15).std())
    P["vol_ts_5_20"] = rv5r / rv20r.replace(0, np.nan)
    g5 = rv5r.groupby(P["instrument"], group_keys=False)
    ma20 = g5.transform(lambda s: s.rolling(20, min_periods=15).mean())
    sd20 = g5.transform(lambda s: s.rolling(20, min_periods=15).std())
    P["rv5_tsz20"] = (rv5r - ma20) / sd20.replace(0, np.nan)
    F["vol_ts_5_20"] = P.set_index(["date", "instrument"])["vol_ts_5_20"]
    F["rv5_tsz20"] = P.set_index(["date", "instrument"])["rv5_tsz20"]

    # ---- L4 历史强因子 ----
    # rv_skew (r2 矩偏度, r19b 口径) ma5, 5m/15m
    for m, name in ((m5, "rv_skew_ma5"), (m15, "rv_skew_ma5_15m")):
        m["r2"] = m["ret"] ** 2
        m["r4"] = m["r2"] ** 2
        m["r6"] = m["r2"] ** 3
        a = m.groupby(["instrument", "td"], sort=False).agg(
            n=("r2", "size"), s2=("r2", "sum"), s4=("r4", "sum"), s6=("r6", "sum"))
        m1, m2_, m3 = a["s2"] / a["n"], a["s4"] / a["n"], a["s6"] / a["n"]
        var = (m2_ - m1 ** 2).clip(lower=0)
        sk = ((m3 - 3 * m1 * m2_ + 2 * m1 ** 3) / var.pow(1.5).replace(0, np.nan)) \
            .rename("v").reset_index()
        sk = sk.sort_values(["instrument", "td"])
        sk["v"] = sk.groupby("instrument", group_keys=False)["v"].transform(_roll5)
        F[name] = -sk.set_index(["instrument", "td"])["v"]  # sign -1

    # upvol_asym_ma5 / upvol_asym_ma5_15m (volume 版, sign -1 缓存为 -raw)
    for m, name in ((m5, "upvol_asym_ma5"), (m15, "upvol_asym_ma5_15m")):
        m["_x"] = m["volume"]
        ua = _updn_asym(m).rename("v").reset_index()
        ua = ua.sort_values(["instrument", "td"])
        ua["v"] = ua.groupby("instrument", group_keys=False)["v"].transform(_roll5)
        F[name] = -ua.set_index(["instrument", "td"])["v"]

    # kurt_ma5 (r19c 口径: 5m ret 超额峰度, 全 bar; roll5; sign -1)
    m5["r1"] = m5["ret"]
    m5["s2_"] = m5["r1"] ** 2
    m5["s3_"] = m5["r1"] ** 3
    m5["s4_"] = m5["r1"] ** 4
    a = m5.groupby(["instrument", "td"], sort=False).agg(
        n=("r1", "size"), s1=("r1", "sum"), s2=("s2_", "sum"),
        s3=("s3_", "sum"), s4=("s4_", "sum"))
    n = a["n"]
    m1 = a["s1"] / n
    m2v = a["s2"] / n - m1 ** 2
    m4 = (a["s4"] / n - 4 * m1 * a["s3"] / n
          + 6 * m1 ** 2 * a["s2"] / n - 3 * m1 ** 4)
    kd = (m4 / m2v.clip(lower=0) ** 2 - 3.0)
    kd[m2v <= 0] = np.nan
    kd = kd.rename("v").reset_index()
    kd = kd.sort_values(["instrument", "td"])
    kd["v"] = kd.groupby("instrument", group_keys=False)["v"].transform(_roll5)
    F["kurt_ma5"] = -kd.set_index(["instrument", "td"])["v"]  # sign -1

    # n_reversals_30m (B_N_REV_30M, sign -1)
    s = np.sign(m30["ret"]).replace(0, np.nan)
    sff = s.groupby([m30["instrument"], m30["td"]], sort=False).ffill()
    prev_s = sff.groupby([m30["instrument"], m30["td"]], sort=False).shift(1)
    rev = ((sff * prev_s) < 0).astype(float)
    F["n_reversals_30m"] = -rev.groupby([m30["instrument"], m30["td"]],
                                        sort=False).sum()

    # absret_ac1_ma5 (R17 系 fill-0 corr min30, roll5, sign -1)
    m5["absret_l1"] = m5.groupby(["instrument", "td"], sort=False)["absret"].shift(1)
    ac = _grp_corr_zero(m5.dropna(subset=["absret_l1"]), "absret", "absret_l1", 30) \
        .rename("v").reset_index()
    ac = ac.sort_values(["instrument", "td"])
    ac["v"] = ac.groupby("instrument", group_keys=False)["v"].transform(_roll5)
    F["absret_ac1_ma5"] = -ac.set_index(["instrument", "td"])["v"]  # sign -1

    # gap_freq_20 (sign -1): |open/pre_close-1|>0.01 的 20 日均 (min10)
    # down_vol_share_15 (sign +1)
    # park5 / gk5 (raw)
    dy = df1m.groupby(["instrument", "td"], sort=False).agg(
        open=("open", "first"), high=("high", "max"), low=("low", "min"),
        close=("close", "last"))
    dy = dy.reset_index().rename(columns={"td": "date"})
    dy = dy.sort_values(["instrument", "date"]).reset_index(drop=True)
    dy["pre_close"] = dy.groupby("instrument", group_keys=False)["close"].shift(1)
    # 注: 平台 pre_close 官方昨收与 shift(close) 在非除权日一致;
    # r17 系缓存经实测与 shift(close) 口径最接近 (gap 0.99995/down_vol 0.99998,
    # 优于 csv pre_close 比例修正的 0.993/0.977), 故用 shift 口径。
    dy["gap"] = dy["open"] / dy["pre_close"] - 1.0
    is_gap = (dy["gap"].abs() > 0.01).astype(float).where(dy["gap"].notna())
    dy["ret"] = dy["close"] / dy["pre_close"] - 1.0
    dy["ret_dn2"] = np.minimum(dy["ret"], 0.0) ** 2
    dy["ret2"] = dy["ret"] ** 2
    gdy = dy.groupby("instrument", group_keys=False)
    F["gap_freq_20"] = -is_gap.groupby(dy["instrument"], group_keys=False).transform(
        lambda s: s.rolling(20, min_periods=10).mean()) \
        .set_axis(pd.MultiIndex.from_frame(dy[["date", "instrument"]]))
    dn15 = gdy["ret_dn2"].transform(lambda s: s.rolling(15, min_periods=7).sum())
    tt15 = gdy["ret2"].transform(lambda s: s.rolling(15, min_periods=7).sum())
    F["down_vol_share_15"] = (dn15 / tt15.replace(0, np.nan)) \
        .set_axis(pd.MultiIndex.from_frame(dy[["date", "instrument"]]))
    ln_hl2 = np.log(dy["high"] / dy["low"]) ** 2
    ln_co2 = np.log(dy["close"] / dy["open"]) ** 2
    m5_hl = ln_hl2.groupby(dy["instrument"], group_keys=False).transform(
        lambda s: s.rolling(5, min_periods=4).mean())
    m5_co = ln_co2.groupby(dy["instrument"], group_keys=False).transform(
        lambda s: s.rolling(5, min_periods=4).mean())
    F["park5"] = np.sqrt(m5_hl / (4.0 * np.log(2))) \
        .set_axis(pd.MultiIndex.from_frame(dy[["date", "instrument"]]))
    F["gk5"] = np.sqrt((m5_hl * 0.5 - m5_co * (2 * np.log(2) - 1)).clip(lower=0)) \
        .set_axis(pd.MultiIndex.from_frame(dy[["date", "instrument"]]))

    # b_pvsign_resid_rank_top1 (sign -1; 池内残差, R20 notebook 口径)
    prev_c1 = df1m.groupby(g1, sort=False)["close"].shift(1)
    df1m["ret1"] = df1m["close"] / prev_c1 - 1.0
    ok = df1m["ret1"].notna() & (df1m["vol_min"] >= 0)
    t1 = df1m[ok].copy()
    t1["sv"] = np.sign(t1["ret1"]) * t1["vol_min"]
    pr = t1.groupby(g1, sort=False).agg(sv=("sv", "sum"), vv=("vol_min", "sum"))
    pr["pressure"] = -(pr["sv"] / (pr["vv"] + 1e-8))
    prev_v5 = m5.groupby(g1, sort=False)["volume"].shift(1)
    m5["dv"] = (m5["volume"] / prev_v5.replace(0, np.nan) - 1.0).fillna(0.0)
    pvs = _grp_corr_zero(m5, "dv", "ret", 30).rename("pvsign").reset_index()
    day = pvs.merge(pr[["pressure"]].reset_index(), on=g1, how="inner")
    day = day.rename(columns={"td": "date"})
    day = day.merge(stk_grid[["date", "instrument"]].drop_duplicates(),
                    on=["date", "instrument"], how="inner")

    def _resid(g):
        y = g["pvsign"].to_numpy(dtype=float)
        x = g["pressure"].rank().to_numpy(dtype=float)
        okk = ~np.isnan(y) & ~np.isnan(x)
        res = np.full_like(y, np.nan)
        if okk.sum() >= 31:
            A = np.column_stack([np.ones(okk.sum()), x[okk]])
            beta, *_ = np.linalg.lstsq(A, y[okk], rcond=None)
            res[okk] = y[okk] - A @ beta
        return pd.Series(res, index=g.index)

    day["pvsign_resid"] = day.groupby("date", group_keys=False).apply(_resid)
    F["b_pvsign_resid_rank_top1"] = -day.set_index(["date", "instrument"])[
        "pvsign_resid"]

    # nm_vol_ret_corr / nm_n_reversals (【5m】口径, recompute_nm3.factors_from_5m:
    # corr(5m volume, |5m ret|) 与 5m close.diff 非零符号变号次数, n<30->0.0, sign -1)
    F["nm_vol_ret_corr"] = -_grp_corr_zero(m5, "volume", "absret", 30)
    s1 = np.sign((m5["close"] - m5.groupby(g1, sort=False)["close"].shift(1))
                 .fillna(0.0))
    regime = s1.replace(0, np.nan).groupby([m5["instrument"], m5["td"]],
                                           sort=False).ffill()
    prev_regime = regime.groupby([m5["instrument"], m5["td"]],
                                 sort=False).shift(1)
    nrev = (regime.notna() & prev_regime.notna()
            & (regime != prev_regime)).astype(float)
    nrev_d = nrev.groupby([m5["instrument"], m5["td"]], sort=False).sum()
    nbar5 = m5.groupby(g1, sort=False)["close"].size()
    F["nm_n_reversals"] = -nrev_d.where(nbar5 >= 30, 0.0)


    # ================= R62 扩展特征 (raw 无 sign, 与挖掘缓存一致) =================
    # ---- 盘口动态 (R47 bookdyn 口径, 5m) ----
    mid = (m5["bid_price1"] + m5["ask_price1"]) / 2
    ok_book = (m5["ask_price1"] > m5["bid_price1"]) & (m5["bid_price1"] > 0) \
        & (mid > 0)
    m5["spread"] = ((m5["ask_price1"] - m5["bid_price1"]) / mid).where(ok_book)
    bv = m5["bid_volume1"] + m5["bid_volume2"] + m5["bid_volume3"]
    av = m5["ask_volume1"] + m5["ask_volume2"] + m5["ask_volume3"]
    ok_vol = (bv + av) > 0
    m5["imb1"] = ((m5["bid_volume1"] - m5["ask_volume1"])
                  / (m5["bid_volume1"] + m5["ask_volume1"]).replace(0, np.nan))
    m5["imb3"] = ((bv - av) / (bv + av)).where(ok_vol)
    m5["depth"] = np.log((bv + av).where(ok_vol))
    bno = m5["bid_num_orders1"] + m5["bid_num_orders2"] + m5["bid_num_orders3"]
    ano = m5["ask_num_orders1"] + m5["ask_num_orders2"] + m5["ask_num_orders3"]
    m5["ono"] = (bno + ano).where((bno + ano) > 0)
    _bs = bv / bno.replace(0, np.nan)
    _as = av / ano.replace(0, np.nan)
    m5["osize_asym"] = ((_bs - _as) / (_bs + _as)).where((_bs > 0) & (_as > 0))
    m5["slope_asym"] = (
        ((m5["bid_volume1"] - m5["bid_volume3"])
         / (m5["bid_volume1"] + m5["bid_volume3"]).replace(0, np.nan))
        - ((m5["ask_volume1"] - m5["ask_volume3"])
           / (m5["ask_volume1"] + m5["ask_volume3"]).replace(0, np.nan)))
    m5["book_slope"] = (((m5["bid_price1"] - m5["bid_price3"])
                         + (m5["ask_price3"] - m5["ask_price1"]))
                        / mid).where(ok_book)
    mp_den = (m5["bid_volume1"] + m5["ask_volume1"]).replace(0, np.nan)
    _mp = (m5["bid_price1"] * m5["ask_volume1"]
           + m5["ask_price1"] * m5["bid_volume1"]) / mp_den
    m5["mp_dev"] = (_mp - m5["close"]) / m5["close"].replace(0, np.nan)
    m5["imb_x_ret"] = m5["imb3"] * m5["ret"]
    bk = m5.groupby(g1, sort=False).agg(
        depth_mean=("depth", "mean"), depth_std=("depth", "std"),
        depth_skew=("depth", "skew"),
        book_slope_mean=("book_slope", "mean"),
        book_slope_std=("book_slope", "std"),
        slope_asym_std=("slope_asym", "std"),
        osize_asym_mean=("osize_asym", "mean"),
        osize_asym_std=("osize_asym", "std"),
        imb3_std=("imb3", "std"), imb1_std=("imb1", "std"),
        imb1_skew=("imb1", "skew"),
        imb_x_ret_mean=("imb_x_ret", "mean"),
        mp_dev_mean=("mp_dev", "mean"), mp_dev_std=("mp_dev", "std"),
        spread_mean=("spread", "mean"), spread_std=("spread", "std"))
    for _c in bk.columns:
        F[_c] = bk[_c]
    m5["imb3_lag"] = m5.groupby(g1, sort=False)["imb3"].shift(1)
    F["imb3_ac1"] = _grp_corr_nan(m5.dropna(subset=["imb3", "imb3_lag"]),
                                  "imb3", "imb3_lag", 10)
    m5["mp_dev_lag"] = m5.groupby(g1, sort=False)["mp_dev"].shift(1)
    F["mp_dev_ac1"] = _grp_corr_nan(m5.dropna(subset=["mp_dev", "mp_dev_lag"]),
                                    "mp_dev", "mp_dev_lag", 10)
    F["orders_ret_corr"] = _grp_corr_nan(m5, "ono", "absret", 10)
    F["osize_ret_corr"] = _grp_corr_nan(m5, "osize_asym", "ret", 10)
    _am = (m5["mins"] >= MORNING[0]) & (m5["mins"] <= MORNING[1])
    _amb = m5[_am].groupby(g1, sort=False).agg(
        spread_am=("spread", "mean"), imb3_am=("imb3", "mean"),
        depth_am=("depth", "mean"))
    for _c in _amb.columns:
        F[_c] = _amb[_c]

    # ---- lead-lag + 市场联动 + 时间重心 (R55 口径, 5m) ----
    m5["absret_lead"] = m5.groupby(g1, sort=False)["absret"].shift(-1)
    m5["vol_lead"] = m5.groupby(g1, sort=False)["volume"].shift(-1)
    m5["dn_lead"] = m5.groupby(g1, sort=False)["dn"].shift(-1)
    F["vollead_corr"] = _grp_corr_nan(m5, "volume", "absret_lead", 10)
    F["deallead_corr"] = _grp_corr_nan(m5, "dn", "absret_lead", 10)
    F["retlead_corr"] = _grp_corr_nan(m5, "ret", "vol_lead", 10)
    F["absretlead_corr"] = _grp_corr_nan(m5, "absret", "dn_lead", 10)
    _mkt = m5.groupby(["td", "mins"], sort=False)["ret"].median().rename("mkt_ret")
    m5 = m5.join(_mkt, on=["td", "mins"])
    F["mkt_corr"] = _grp_corr_nan(m5, "ret", "mkt_ret", 10)
    F["mkt_corr_am"] = _grp_corr_nan(m5[_am], "ret", "mkt_ret", 5)
    m5["amt_x_mins"] = m5["amount"] * m5["mins"]
    m5["dn_x_mins"] = m5["dn"] * m5["mins"]
    _c = m5.groupby(g1, sort=False).agg(
        amt_sum=("amount", "sum"), amt_x=("amt_x_mins", "sum"),
        dn_sum=("dn", "sum"), dn_x=("dn_x_mins", "sum"))
    F["amt_center"] = _c["amt_x"] / _c["amt_sum"].replace(0, np.nan)
    F["deal_center"] = _c["dn_x"] / _c["dn_sum"].replace(0, np.nan)

    # ---- deal 15m (R46 口径) ----
    d15 = m15.groupby(g1, sort=False).agg(
        deals_total=("dn", "sum"), deals_mean=("dn", "mean"),
        deals_std=("dn", "std"), deals_skew=("dn", "skew"))
    d15["deals_cv"] = d15["deals_std"] / d15["deals_mean"].replace(0, np.nan)
    _sub = m15.dropna(subset=["dn"])
    _up = _sub["ret"] > 0
    _dn = _sub["ret"] < 0
    _va = _sub.assign(dn_up=_sub["dn"] * _up, cnt_up=_up.astype(float),
                      dn_dn=_sub["dn"] * _dn, cnt_dn=_dn.astype(float)) \
        .groupby(g1, sort=False).agg(
            dn_up=("dn_up", "sum"), cnt_up=("cnt_up", "sum"),
            dn_dn=("dn_dn", "sum"), cnt_dn=("cnt_dn", "sum"))
    _mu = _va["dn_up"] / _va["cnt_up"].replace(0, np.nan)
    _md = _va["dn_dn"] / _va["cnt_dn"].replace(0, np.nan)
    d15["deals_updn_asym"] = (_mu - _md) / (_mu + _md).replace(0, np.nan)
    for _c in d15.columns:
        F[_c] = d15[_c]

    # ---- exec 补充 (R52 口径, 5m; dn>0 过滤版 exec_int) ----
    m5["no_total"] = no_total
    _exi = (m5["dn"] / no_total.replace(0, np.nan)).where(no_total > 0)
    _eb = m5["dn"] / bno.replace(0, np.nan)
    _ea = m5["dn"] / ano.replace(0, np.nan)
    m5["fill_asym"] = ((_eb - _ea) / (_eb + _ea)).where((_eb > 0) & (_ea > 0))
    m5["exec_sgn"] = _exi * np.sign(m5["ret"])
    _ex = m5.groupby(g1, sort=False).agg(
        fill_asym_mean=("fill_asym", "mean"),
        exec_sgn_sum=("exec_sgn", "sum"), n_bars=("ret", "size"))
    F["fill_asym_mean"] = _ex["fill_asym_mean"]
    F["exec_x_ret"] = _ex["exec_sgn_sum"] / _ex["n_bars"].replace(0, np.nan)
    F["deals_orders_corr"] = _grp_corr_nan(m5, "dn", "no_total", 10)

    # ---- 尾盘竞价补充 (R52 口径, 1m; ret_tail3 已有) ----
    df1m["close_l1"] = df1m.groupby(g1, sort=False)["close"].shift(1)
    df1m["seq"] = df1m.groupby(g1, sort=False).cumcount()
    df1m["nbar"] = df1m.groupby(g1, sort=False)["seq"].transform("max") + 1
    _tot = df1m.groupby(g1, sort=False).agg(
        dn_total=("dn_min", "sum"), vol_total=("vol_min", "sum"))
    _last = df1m[df1m["seq"] == df1m["nbar"] - 1].set_index(g1)
    _t3 = df1m[df1m["seq"] >= df1m["nbar"] - 3] \
        .groupby(g1, sort=False)["dn_min"].sum()
    _dn1 = _last["dn_min"].astype(float)
    F["deals_tail1_share"] = (_dn1 / _tot["dn_total"].replace(0, np.nan))
    F["vol_tail1_share"] = (_last["vol_min"]
                            / _tot["vol_total"].replace(0, np.nan))
    F["ret_tail1"] = (_last["close"] / _last["close_l1"] - 1.0).fillna(0.0)
    F["tail1_deal_size"] = (_last["vol_min"]
                            / _dn1.replace(0, np.nan)).where(_dn1 > 0)
    F["tail3_deals_share"] = (_t3 / _tot["dn_total"].replace(0, np.nan))

    # ---- 量价价格分布 (R54 口径, 5m; vwap_disp 已有) ----
    _hl = m5.groupby(g1, sort=False)["close"].agg(["max", "min"])
    m5 = m5.join(_hl.rename(columns={"max": "h", "min": "l"}), on=g1)
    _rng = (m5["h"] - m5["l"]).replace(0, np.nan)
    _top = (m5["close"] >= m5["l"] + 2.0 / 3.0 * _rng).astype(float) \
        .where(_rng.notna())
    _bot = (m5["close"] <= m5["l"] + 1.0 / 3.0 * _rng).astype(float) \
        .where(_rng.notna())
    m5["vol_top"] = m5["volume"] * _top
    m5["vol_bot"] = m5["volume"] * _bot
    _a = m5.groupby(g1, sort=False).agg(
        vol=("volume", "sum"), amt=("amount", "sum"),
        vol_top=("vol_top", "sum"), vol_bot=("vol_bot", "sum"))
    F["vol_top_third"] = _a["vol_top"] / _a["vol"].replace(0, np.nan)
    F["vol_bot_third"] = _a["vol_bot"] / _a["vol"].replace(0, np.nan)
    _vwap = (_a["amt"] / _a["vol"].replace(0, np.nan)).rename("vwap")
    m5 = m5.join(_vwap, on=g1)
    _dev = m5["close"] - m5["vwap"]
    m5["wv2"] = m5["volume"] * _dev ** 2
    m5["wv3"] = m5["volume"] * _dev ** 3
    _m = m5.groupby(g1, sort=False).agg(wv2=("wv2", "sum"), wv3=("wv3", "sum"),
                                        vol=("volume", "sum"))
    _m["vw_std"] = np.sqrt(_m["wv2"] / _m["vol"].replace(0, np.nan))
    F["vwap_skew"] = (_m["wv3"] / _m["vol"].replace(0, np.nan)) \
        / _m["vw_std"].replace(0, np.nan) ** 3


    # ---- R70 尾盘竞价扩展 (tail2, mine_r70 口径, 1m) ----
    _tot = df1m.groupby(g1, sort=False).agg(
        dn_total=("dn_min", "sum"), vol_total=("vol_min", "sum"),
        amt_total=("amt_min", "sum"), day_first=("close", "first"),
        day_last=("close", "last"))
    _tot["day_ret"] = _tot["day_last"] / _tot["day_first"] - 1.0
    _t3b = df1m[df1m["seq"] >= df1m["nbar"] - 3]
    _t3g = _t3b.groupby(g1, sort=False).agg(
        dn_t3=("dn_min", "sum"), vol_t3=("vol_min", "sum"),
        amt_t3=("amt_min", "sum"), close_last=("close", "last"),
        close_l3=("close_l1", "first"), dsize_t3_max=("dn_min", "max"),
        ret_t3_std=("ret1", "std"), vol_t3_max=("vol_min", "max"))
    _t3g["ret_t3"] = _t3g["close_last"] / _t3g["close_l3"] - 1.0
    _t3g["dsize_t3"] = _t3g["vol_t3"] / _t3g["dn_t3"].replace(0, np.nan)
    _t3g["t3_amt_share"] = _t3g["amt_t3"] / _tot["amt_total"].replace(0, np.nan)
    _t3g["t3_dsize_vs_day"] = _t3g["dsize_t3"] / (
        _tot["vol_total"] / _tot["dn_total"].replace(0, np.nan)) \
        .replace(0, np.nan)
    _t3g["t3_ret_vs_day"] = _t3g["ret_t3"] - _tot["day_ret"]
    _t3g["t3_vol_conc"] = _t3g["vol_t3_max"] / _t3g["vol_t3"].replace(0, np.nan)
    _sv = _t3b.assign(sv=_t3b["vol_min"] * np.sign(_t3b["ret1"])) \
        .groupby(g1, sort=False)["sv"].sum()
    _t3g["t3_sgnvol"] = _sv / _t3g["vol_t3"].replace(0, np.nan)
    _pre = df1m[(df1m["mins"] >= 870) & (df1m["mins"] <= 896)]
    _preg = _pre.groupby(g1, sort=False).agg(
        pre_first=("close", "first"), pre_last=("close", "last"),
        pre_vol=("vol_min", "sum"), pre_dn=("dn_min", "sum"))
    _preg["pre_ret"] = _preg["pre_last"] / _preg["pre_first"] - 1.0
    _t3g["t3_vs_pre_ret"] = _t3g["ret_t3"] - _preg["pre_ret"]
    _t3g["pre_vol_share"] = _preg["pre_vol"] \
        / _tot["vol_total"].replace(0, np.nan)
    _t3g["t3_dn_vs_pre_dn"] = (_t3g["dn_t3"] / 3.0) \
        / (_preg["pre_dn"] / 27.0).replace(0, np.nan)
    _bv1 = _t3b["bid_volume1"].astype(float)
    _av1 = _t3b["ask_volume1"].astype(float)
    _t3g["t3_imb1_mean"] = ((_bv1 - _av1) / (_bv1 + _av1).replace(0, np.nan)) \
        .groupby([_t3b["instrument"], _t3b["td"]], sort=False).mean()
    _spr = ((_t3b["ask_price1"] - _t3b["bid_price1"])
            / ((_t3b["ask_price1"] + _t3b["bid_price1"]) / 2)) \
        .where(_t3b["ask_price1"] > _t3b["bid_price1"])
    _t3g["t3_spread_mean"] = _spr.groupby(
        [_t3b["instrument"], _t3b["td"]], sort=False).mean()
    _b895 = df1m[df1m["mins"] == 895]
    _t3g["ret_1455"] = _b895.groupby(g1, sort=False)["ret1"].first()
    _t3g["vol_1455_share"] = (_b895.groupby(g1, sort=False)["vol_min"].first()
                              / _tot["vol_total"].replace(0, np.nan))
    for _c in ["t3_amt_share", "dsize_t3", "t3_dsize_vs_day", "t3_sgnvol",
               "t3_ret_vs_day", "t3_vol_conc", "t3_vs_pre_ret",
               "pre_vol_share", "t3_dn_vs_pre_dn", "t3_imb1_mean",
               "t3_spread_mean", "ret_1455", "vol_1455_share",
               "ret_t3_std", "dsize_t3_max"]:
        F[_c] = _t3g[_c]


    # ================= R72 扩展 (open/exec_v2/ret1430/book15m) =================
    # ---- 开盘 (R50 口径, 1m 首根/前5根) ----
    _o1 = df1m[df1m["seq"] == 0].set_index(g1)
    _o5 = df1m[df1m["seq"] < 5].groupby(g1, sort=False).agg(
        dn_o5=("dn_min", "sum"), vol_o5=("vol_min", "sum"))
    F["deals_open1_share"] = (_o1["dn_min"].astype(float)
                              / _tot["dn_total"].replace(0, np.nan))
    F["vol_open1_share"] = (_o1["vol_min"] / _tot["vol_total"].replace(0, np.nan))
    F["deals_open5_share"] = (_o5["dn_o5"] / _tot["dn_total"].replace(0, np.nan))
    F["vol_open5_share"] = (_o5["vol_o5"]
                            / _tot["vol_total"].replace(0, np.nan))
    _dn1 = _o1["dn_min"].astype(float)
    F["open1_deal_size"] = (_o1["vol_min"] / _dn1.replace(0, np.nan)) \
        .where(_dn1 > 0)

    # ---- exec 变体 (R53 口径, 5m) ----
    F["exec_int_med"] = m5.groupby(g1, sort=False)["exec_int"].median()
    F["exec_int_am"] = m5[_am].groupby(g1, sort=False)["exec_int"].mean()
    F["exec_int_pm"] = m5[m5["mins"] >= 780] \
        .groupby(g1, sort=False)["exec_int"].mean()

    # ---- ret_1430_1456 (R53 口径, 实际窗口 mins 870-876 忠实移植) ----
    _w = df1m[(df1m["mins"] >= 870) & (df1m["mins"] <= 876)] \
        .groupby(g1, sort=False)["close"].agg(["first", "last"])
    F["ret_1430_1456"] = _w["last"] / _w["first"] - 1.0

    # ---- 盘口 15m (R47 同公式换 m15 帧, _b15m 后缀) ----
    mid15 = (m15["bid_price1"] + m15["ask_price1"]) / 2
    ok_book15 = (m15["ask_price1"] > m15["bid_price1"]) \
        & (m15["bid_price1"] > 0) & (mid15 > 0)
    m15["spread"] = ((m15["ask_price1"] - m15["bid_price1"]) / mid15) \
        .where(ok_book15)
    bv15 = m15["bid_volume1"] + m15["bid_volume2"] + m15["bid_volume3"]
    av15 = m15["ask_volume1"] + m15["ask_volume2"] + m15["ask_volume3"]
    ok_vol15 = (bv15 + av15) > 0
    m15["imb1"] = ((m15["bid_volume1"] - m15["ask_volume1"])
                   / (m15["bid_volume1"] + m15["ask_volume1"]).replace(0, np.nan))
    m15["imb3"] = ((bv15 - av15) / (bv15 + av15)).where(ok_vol15)
    m15["depth"] = np.log((bv15 + av15).where(ok_vol15))
    bno15 = (m15["bid_num_orders1"] + m15["bid_num_orders2"]
             + m15["bid_num_orders3"])
    ano15 = (m15["ask_num_orders1"] + m15["ask_num_orders2"]
             + m15["ask_num_orders3"])
    m15["ono"] = (bno15 + ano15).where((bno15 + ano15) > 0)
    _bs15 = bv15 / bno15.replace(0, np.nan)
    _as15 = av15 / ano15.replace(0, np.nan)
    m15["osize_asym"] = ((_bs15 - _as15) / (_bs15 + _as15)) \
        .where((_bs15 > 0) & (_as15 > 0))
    m15["slope_asym"] = (
        ((m15["bid_volume1"] - m15["bid_volume3"])
         / (m15["bid_volume1"] + m15["bid_volume3"]).replace(0, np.nan))
        - ((m15["ask_volume1"] - m15["ask_volume3"])
           / (m15["ask_volume1"] + m15["ask_volume3"]).replace(0, np.nan)))
    m15["book_slope"] = (((m15["bid_price1"] - m15["bid_price3"])
                          + (m15["ask_price3"] - m15["ask_price1"]))
                         / mid15).where(ok_book15)
    mp_den15 = (m15["bid_volume1"] + m15["ask_volume1"]).replace(0, np.nan)
    _mp15 = (m15["bid_price1"] * m15["ask_volume1"]
             + m15["ask_price1"] * m15["bid_volume1"]) / mp_den15
    m15["mp_dev"] = (_mp15 - m15["close"]) / m15["close"].replace(0, np.nan)
    m15["imb_x_ret"] = m15["imb3"] * m15["ret"]
    bk15 = m15.groupby(g1, sort=False).agg(
        depth_mean_b15m=("depth", "mean"), depth_std_b15m=("depth", "std"),
        depth_skew_b15m=("depth", "skew"),
        book_slope_mean_b15m=("book_slope", "mean"),
        book_slope_std_b15m=("book_slope", "std"),
        slope_asym_std_b15m=("slope_asym", "std"),
        osize_asym_mean_b15m=("osize_asym", "mean"),
        osize_asym_std_b15m=("osize_asym", "std"),
        imb3_std_b15m=("imb3", "std"), imb1_std_b15m=("imb1", "std"),
        imb1_skew_b15m=("imb1", "skew"),
        imb_x_ret_mean_b15m=("imb_x_ret", "mean"),
        imb_x_ret_std_b15m=("imb_x_ret", "std"),
        mp_dev_mean_b15m=("mp_dev", "mean"),
        mp_dev_std_b15m=("mp_dev", "std"),
        spread_mean_b15m=("spread", "mean"),
        spread_std_b15m=("spread", "std"))
    for _c in bk15.columns:
        F[_c] = bk15[_c]
    m15["imb3_lag"] = m15.groupby(g1, sort=False)["imb3"].shift(1)
    F["imb3_ac1_b15m"] = _grp_corr_nan(m15.dropna(subset=["imb3", "imb3_lag"]),
                                       "imb3", "imb3_lag", 10)
    m15["mp_dev_lag"] = m15.groupby(g1, sort=False)["mp_dev"].shift(1)
    F["mp_dev_ac1_b15m"] = _grp_corr_nan(
        m15.dropna(subset=["mp_dev", "mp_dev_lag"]), "mp_dev", "mp_dev_lag", 10)
    F["orders_ret_corr_b15m"] = _grp_corr_nan(m15, "ono", "absret", 10)
    F["osize_ret_corr_b15m"] = _grp_corr_nan(m15, "osize_asym", "ret", 10)


    # ================= R74 misc (deal 5m 分布 + deal_amt_skew + deals_x_ret_d15) =================
    # ---- deal 5m (mine_r45 口径, m5 帧已有 dn/ret/absret/deal_size) ----
    m5["deal_amt"] = (m5["amount"] / m5["dn"].replace(0, np.nan)) \
        .where(m5["dn"] > 0)
    _d5 = m5.groupby(g1, sort=False).agg(
        deals_total_d5=("dn", "sum"), deals_mean_d5=("dn", "mean"),
        deals_std_d5=("dn", "std"), deals_skew_d5=("dn", "skew"),
        deal_size_std_d5=("deal_size", "std"),
        deal_size_skew_d5=("deal_size", "skew"),
        deal_amt_mean_d5=("deal_amt", "mean"),
        deal_amt_skew=("deal_amt", "skew"))
    _d5["deals_cv_d5"] = _d5["deals_std_d5"] \
        / _d5["deals_mean_d5"].replace(0, np.nan)
    # 峰度 (向量化四阶矩, mine_r45 grp_kurt 口径)
    _t = m5[["instrument", "td", "dn"]].dropna()
    _t = _t.assign(x2=_t["dn"] ** 2, x3=_t["dn"] ** 3, x4=_t["dn"] ** 4)
    _gb = _t.groupby(["instrument", "td"], sort=False)
    _s = _gb[["dn", "x2", "x3", "x4"]].sum()
    _n = _gb.size()
    _ex, _ex2 = _s["dn"] / _n, _s["x2"] / _n
    _ex3, _ex4 = _s["x3"] / _n, _s["x4"] / _n
    _m2 = _ex2 - _ex ** 2
    _m4 = _ex4 - 4 * _ex * _ex3 + 6 * _ex ** 2 * _ex2 - 3 * _ex ** 4
    _d5["deals_kurt_d5"] = (_m4 / _m2.replace(0, np.nan) ** 2 - 3).where(_n >= 10)
    m5["dn_rank"] = m5.groupby(g1, sort=False)["dn"].rank(pct=True)
    _top = m5[m5["dn_rank"] >= 0.8].groupby(g1, sort=False)["dn"].sum()
    _d5["deals_conc_d5"] = _top / _d5["deals_total_d5"].replace(0, np.nan)
    m5["dn_lag"] = m5.groupby(g1, sort=False)["dn"].shift(1)
    _d5["deals_ac1_d5"] = _grp_corr_nan(m5.dropna(subset=["dn", "dn_lag"]),
                                       "dn", "dn_lag", 10)
    _d5["deals_absret_corr_d5"] = _grp_corr_nan(m5, "dn", "absret", 10)
    _am5 = (m5["mins"] >= MORNING[0]) & (m5["mins"] <= MORNING[1])
    _d5["deals_morning_share_d5"] = (
        m5[_am5].groupby(g1, sort=False)["dn"].sum()
        / _d5["deals_total_d5"].replace(0, np.nan))
    _d5["deals_tail_share_d5"] = (
        m5[m5["mins"] >= 870].groupby(g1, sort=False)["dn"].sum()
        / _d5["deals_total_d5"].replace(0, np.nan))
    _pm = m5[m5["mins"] >= 780].groupby(g1, sort=False)["dn"].sum()
    _am_ = m5[m5["mins"] < 690].groupby(g1, sort=False)["dn"].sum()
    _d5["deals_pm_am_d5"] = _pm / _am_.replace(0, np.nan)
    _sgn = m5.assign(sgn=m5["dn"] * np.sign(m5["ret"])) \
        .groupby(g1, sort=False)["sgn"].sum()
    _d5["deals_x_ret_d5"] = _sgn / _d5["deals_total_d5"].replace(0, np.nan)
    _q = m5.groupby(g1, sort=False)["deal_size"].agg(["max", "median"])
    _d5["bigdeal_ratio_d5"] = _q["max"] / _q["median"].replace(0, np.nan)
    for _c in _d5.columns:
        F[_c] = _d5[_c]

    # ---- deals_x_ret 15m (R45 口径换 m15) ----
    _sgn15 = m15.assign(sgn=m15["dn"] * np.sign(m15["ret"])) \
        .groupby(g1, sort=False)["sgn"].sum()
    _tot15 = m15.groupby(g1, sort=False)["dn"].sum()
    F["deals_x_ret_d15"] = _sgn15 / _tot15.replace(0, np.nan)

    out = {}
    for name, s in F.items():
        d = s.rename("factor").reset_index()
        d = d.rename(columns={"td": "date"})
        d["date"] = pd.to_datetime(d["date"]).dt.normalize()
        out[name] = d[["date", "instrument", "factor"]] \
            .replace([np.inf, -np.inf], np.nan)
    return out


# ---------------------------------------------------------------- 模型推理

def _model_year(y):
    if y <= 2020:
        return 2020
    if y == 2021:
        return 2021
    if y == 2022:
        return 2022
    if y == 2023:
        return 2023
    return 2024



MLP_WEIGHTS = json.loads('''{"2020": {"W1": [[-0.16139447689056396, -0.1695140153169632, 0.11903125047683716, -0.13889870047569275, 0.1913885474205017, -0.1614505797624588, 0.1413324475288391, -0.04111098870635033, 0.09627785533666611, 0.24953612685203552, 0.15464359521865845, -0.06869512051343918, 0.0174558125436306, 0.18437756597995758, 0.025584746152162552, -0.16215713322162628, -0.1535426527261734, -0.12204067409038544, -0.043758004903793335, 0.0637284591794014, 0.01662161946296692], [0.13848547637462616, -0.09044051170349121, 0.07652225345373154, 0.15375052392482758, -0.059459611773490906, -0.20952428877353668, 0.15539997816085815, -0.16322003304958344, 0.08141662925481796, 0.13704699277877808, -0.10636359453201294, -0.12852221727371216, -0.22477959096431732, -0.039641495794057846, -0.1567697674036026, -0.10461526364088058, -0.24351416528224945, -0.08622834831476212, 0.11094451695680618, -0.021874170750379562, -0.09359250217676163], [0.20420493185520172, -0.16779419779777527, 0.07228301465511322, 0.13818243145942688, 0.16168734431266785, -0.13144934177398682, -0.1598171442747116, -0.026013750582933426, 0.09811569005250931, 0.08417165279388428, 0.17026054859161377, 0.014893144369125366, 0.08429684489965439, -0.08295381814241409, -0.07764624804258347, -0.1401057094335556, -0.032329536974430084, -0.09643810987472534, -0.07529080659151077, -0.12229544669389725, 0.02451382763683796], [-0.1500692516565323, -0.05154667794704437, -0.17811627686023712, 0.1460266411304474, 0.12720148265361786, -0.0751873105764389, -0.046929921954870224, -0.1355232298374176, 0.040308598428964615, 0.0977882519364357, -0.1729588359594345, -0.13765399158000946, -0.1832236647605896, -0.12817378342151642, 0.15104642510414124, 0.1827290952205658, -0.06718959659337997, 0.17368386685848236, 0.0546656958758831, 0.030852532014250755, -0.12880772352218628], [0.16969218850135803, -0.2016555368900299, 0.05160298943519592, 0.18245086073875427, 0.1799452006816864, 0.10762417316436768, -0.09811606258153915, 0.07973749935626984, -0.10171152651309967, -0.10854766517877579, -0.0050357915461063385, 0.24466687440872192, 0.027575619518756866, 0.2492038458585739, -0.1358490288257599, 0.070965975522995, 0.01827961578965187, 0.13476498425006866, 0.02130218595266342, 0.06115836650133133, 0.1190791130065918], [0.21027342975139618, 0.15289044380187988, -0.18035739660263062, -0.07708830386400223, 0.16242942214012146, 0.18240046501159668, 0.19073258340358734, 0.08771941065788269, -0.11688189953565598, 0.027115831151604652, 0.16572266817092896, 0.19587434828281403, 0.04282223805785179, 0.11400975286960602, -0.037225332111120224, -0.1536388397216797, -0.04876872897148132, 0.032317016273736954, -0.061948370188474655, -0.15277129411697388, -0.056619349867105484], [-0.16584040224552155, 0.2162519097328186, -0.15428493916988373, 0.09760705381631851, -0.22521866858005524, -0.16864503920078278, 0.07918841391801834, -0.2029932290315628, -0.09140583127737045, 0.06484539806842804, -0.10408849269151688, -0.13954834640026093, 0.12225545942783356, 0.08089665323495865, 0.018834998831152916, -0.043439626693725586, 0.021799057722091675, -0.03756445273756981, 0.17813998460769653, -0.13597463071346283, 0.1724497377872467], [0.11235053092241287, -0.11741269379854202, 0.11605805903673172, 0.08979683369398117, -0.03904542699456215, -0.1007474735379219, -0.093959741294384, -0.04422086477279663, -0.15826930105686188, 0.2204732596874237, -0.13157372176647186, -0.029931863769888878, 0.060676511377096176, -0.08440514653921127, 0.034547723829746246, -0.07440110296010971, -0.1851801574230194, 0.11428862065076828, 0.19329999387264252, -0.1297593116760254, 0.12194640189409256], [0.08462569117546082, 0.08726093173027039, -0.1485908031463623, -0.21401461958885193, -0.24175970256328583, -0.20350733399391174, 0.025909483432769775, 0.03389149159193039, 0.1648305356502533, -0.12524022161960602, -0.20688654482364655, 0.09870979934930801, 0.07426632195711136, -0.087708979845047, -0.003995790611952543, 0.040828313678503036, -0.09812229126691818, -0.1700601875782013, -0.09073035418987274, -0.19809521734714508, -0.18758328258991241], [0.19217732548713684, 0.0564957819879055, -0.03688976541161537, 0.06450409442186356, 0.13467055559158325, -0.16868992149829865, -0.09038837999105453, -0.0055417330004274845, 0.0618654228746891, -0.08880899846553802, -0.07304544746875763, 0.11018974334001541, -0.07598641514778137, 0.05773443356156349, -0.019373588263988495, 0.15524376928806305, -0.18878714740276337, 0.16080069541931152, 0.19646939635276794, -0.005231162067502737, 0.17195381224155426], [-0.17320269346237183, 0.16679105162620544, -0.11003007739782333, -0.132375106215477, -0.16910825669765472, -0.13077297806739807, -0.05459907650947571, 0.10995178669691086, 0.01023025345057249, -0.19208694994449615, -0.026614535599946976, 0.10356774926185608, -0.09059494733810425, 0.09102637320756912, 0.07962866872549057, 0.13425947725772858, 0.08162126690149307, -0.015621887519955635, -0.0715218335390091, -0.14032573997974396, 0.13603754341602325], [-0.011417297646403313, 0.18865777552127838, -0.018736068159341812, -0.18856631219387054, 0.0011010102462023497, -0.0895833969116211, -0.1482902616262436, -0.09610773622989655, 0.09911534190177917, 0.1016009971499443, -0.2339574247598648, 0.10313785076141357, 0.13752706348896027, 0.07240841537714005, 0.11741515249013901, -0.05550292879343033, -0.10835041850805283, -0.11111672967672348, -0.01854918897151947, 0.15117770433425903, 0.15989643335342407], [-0.04605155438184738, 0.156622514128685, 0.00498930225148797, -0.15631163120269775, -0.05649644508957863, -0.08308681100606918, -0.12377262860536575, -0.07869058847427368, 0.1636204570531845, 0.03892458602786064, -0.07278284430503845, -0.1296875923871994, -0.11368557810783386, 0.00820175465196371, -0.18458864092826843, 0.05980394408106804, -0.11295849829912186, -0.0804758220911026, -0.03759121894836426, 0.13728322088718414, 0.0411650650203228], [-0.05719867721199989, -0.17313909530639648, -0.2175295352935791, -0.17601771652698517, -0.08324827998876572, -0.022150559350848198, -0.12268521636724472, 0.14788329601287842, 0.07251401245594025, 0.1573847085237503, -0.03726116940379143, -0.15554575622081757, -0.20847631990909576, -0.09277703613042831, -0.1363566368818283, -0.1332557648420334, -0.06570165604352951, 0.14672712981700897, 0.10464094579219818, -0.1895519345998764, -0.1754310131072998], [-0.12206873297691345, -0.02334248088300228, 0.049701347947120667, 0.051526665687561035, 0.13588550686836243, 0.0870896652340889, -0.08416935056447983, -0.11506079137325287, 0.02839849330484867, 0.08563976734876633, -0.018581470474600792, -0.09193048626184464, 0.11100611090660095, 0.19073179364204407, 0.0445229634642601, -0.008711304515600204, -0.004985966719686985, -0.06845876574516296, -0.0037718620151281357, -0.13969862461090088, -0.18096239864826202], [-0.007319135591387749, 0.017896192148327827, -0.05504226312041283, 0.17973770201206207, -0.1162547916173935, -0.039940305054187775, -0.09224200248718262, -0.0006376146920956671, 0.07953948527574539, -0.07474727183580399, -0.02884194813668728, 0.10627931356430054, 0.08514759689569473, -0.029359061270952225, -0.18899589776992798, -0.049502819776535034, -0.1433437615633011, -0.1045488715171814, 0.09213659167289734, -0.1008683517575264, 0.18190477788448334], [-0.11063600331544876, -0.18189799785614014, -0.1522918939590454, -0.1636466383934021, -0.011972226202487946, 0.011159085668623447, -0.045624375343322754, -0.03409752622246742, -0.21608081459999084, -0.013666385784745216, -0.09033478796482086, -0.020101334899663925, -0.11678119748830795, -0.1644129455089569, -0.1564137190580368, 0.036249853670597076, -0.08472540974617004, -0.05856035277247429, -0.14079895615577698, 0.07515788823366165, 0.16115935146808624], [-0.02830464020371437, 0.11786386370658875, 0.2138601541519165, -0.15515518188476562, 0.06599258631467819, 0.14885307848453522, -0.20229171216487885, 0.026415150612592697, 0.13659663498401642, -0.13097181916236877, -0.021179620176553726, -0.13199052214622498, 0.06756696850061417, 0.03786744177341461, -0.13436393439769745, 0.0694681853055954, 0.1776622235774994, -0.1136954054236412, 0.0043839444406330585, 0.06255125254392624, -0.0760069489479065], [-0.09692174196243286, -0.07200310379266739, 0.02987257018685341, 0.03306393697857857, 0.1017010509967804, -0.1864805817604065, 0.017604179680347443, -0.16645313799381256, 0.06253106892108917, -0.014993352815508842, -0.11082158982753754, 0.011584688909351826, -0.11577193439006805, -0.1304454654455185, 0.05396813526749611, 0.08153089880943298, 0.08478257060050964, 0.16560186445713043, 0.19573891162872314, -0.06667999923229218, -0.12136436253786087], [-0.03977058082818985, -0.05898962542414665, 0.12974970042705536, -0.014927096664905548, -0.09145162254571915, -0.08379290997982025, -0.033001579344272614, 0.061396148055791855, -0.03997638449072838, -0.014112061820924282, 0.09503908455371857, 0.14821766316890717, -0.16541291773319244, 0.034311093389987946, -0.1569322794675827, -0.03248445689678192, -0.18497641384601593, 0.06379345804452896, -0.09221202880144119, -0.07007034122943878, -0.09684226661920547], [-0.16590368747711182, -0.06931109726428986, 0.03974003344774246, -0.1398182064294815, 0.11673959344625473, 0.14134690165519714, 0.04583683982491493, -0.02066759765148163, -0.1281718760728836, -0.11763592809438705, 0.08060458302497864, -0.1698586642742157, -0.19253268837928772, 0.09033544361591339, 0.1900874674320221, 0.008993664756417274, -0.23279893398284912, 0.0662248507142067, -0.1663055121898651, 0.11469920724630356, -0.140024334192276], [0.10635779052972794, 0.15165133774280548, -0.23852913081645966, -0.007462844718247652, 0.042429033666849136, 0.19559365510940552, -0.14210425317287445, 0.16436855494976044, -0.009632335044443607, -0.042955853044986725, -0.1455378532409668, -0.011901107616722584, -0.05943295732140541, -0.15380531549453735, 0.17394429445266724, 0.019901175051927567, 0.062078624963760376, 0.07562918961048126, 0.04639357328414917, 0.135797381401062, -0.15598836541175842], [0.03730035573244095, -0.08084573596715927, -0.11570026725530624, 0.07563530653715134, 0.11343930661678314, 0.1691281497478485, 0.03307604417204857, -0.009324222803115845, 0.00522577716037631, -0.13445070385932922, 0.1316903829574585, 0.014911722391843796, 0.1628091037273407, -0.16877394914627075, 0.12721647322177887, -0.15231449902057648, -0.15830393135547638, 0.13095098733901978, -0.13917475938796997, -0.18791234493255615, 0.1616109311580658], [0.17875829339027405, 0.0767691507935524, 0.05523049086332321, -0.12274901568889618, 0.12155547738075256, 0.004142592661082745, -0.15139180421829224, -0.1880609542131424, 0.14521300792694092, -0.1260504126548767, 0.10191966593265533, -0.13160857558250427, -0.16812321543693542, -0.0013325531035661697, 0.18739919364452362, 0.048460762947797775, 0.08701525628566742, -0.16641056537628174, -0.06042089685797691, -0.11774259060621262, -0.006915743462741375], [-0.12828023731708527, 0.10734222084283829, 0.1284053921699524, -0.00803589541465044, 0.07615610212087631, -0.21760906279087067, 0.16785828769207, 0.2061031311750412, -0.17510025203227997, -0.17296193540096283, -0.012274793349206448, -0.028365444391965866, -0.19092373549938202, -0.1064426451921463, 0.18975721299648285, 0.08094408363103867, 0.04086637124419212, -0.158382385969162, 0.1433413028717041, -0.044273264706134796, -0.0003005049657076597], [-0.05674964189529419, 0.19244162738323212, 0.01106597576290369, 0.16845928132534027, -0.04782849922776222, -0.11141076683998108, 0.05100403353571892, 0.022174591198563576, 0.0450299046933651, 0.12112367898225784, -0.13131646811962128, -0.03125070780515671, -0.22464247047901154, -0.04579581692814827, -0.008044229820370674, -0.01700863242149353, 0.057101111859083176, 0.13705694675445557, -0.16898714005947113, -0.1250603049993515, -0.10719883441925049], [-0.054991621524095535, -0.11798407882452011, -0.06321823596954346, -0.21358709037303925, -0.1527462750673294, -0.1819288432598114, -0.09820885956287384, -0.033850789070129395, 0.046362392604351044, 0.07476849853992462, 0.09347806870937347, -0.0279021468013525, 0.032594822347164154, 0.03245726227760315, 0.1402161717414856, 0.11285074800252914, -0.10304097831249237, 0.04328078404068947, -0.009786969050765038, -0.17148245871067047, 0.1514476239681244], [-0.1311577707529068, -0.11410076171159744, 0.05072800815105438, 0.043262965977191925, 0.107939213514328, -0.12258969247341156, 0.009323581121861935, 0.029065856710076332, 0.21959280967712402, 0.061048783361911774, 0.10202644765377045, 0.11921989917755127, 0.1621803194284439, 0.17131613194942474, 0.15986227989196777, 0.02771776169538498, -0.08303727954626083, -0.14890773594379425, -0.05920155346393585, -0.07197205722332001, 0.028908921405673027], [-0.13121630251407623, 0.1022864356637001, 0.15257465839385986, 0.120778888463974, 0.06138690561056137, 0.19447891414165497, 0.12981735169887543, -0.02844550833106041, 0.13586631417274475, 0.010222041048109531, 0.18261872231960297, 0.055101171135902405, -0.08913935720920563, -0.06216176599264145, 0.0877310037612915, 0.18053923547267914, -0.04190180450677872, 0.17199650406837463, 0.0780748575925827, -0.08674445748329163, 0.2017514556646347], [0.1899697631597519, 0.1051088273525238, -0.032860349863767624, -0.01098294835537672, 0.043197501450777054, 0.18260960280895233, 0.15607525408267975, -0.032851964235305786, 0.18435324728488922, 0.1754840761423111, -0.16399049758911133, 0.1167740598320961, -0.21528740227222443, 0.06511330604553223, 0.040432464331388474, -0.005346883554011583, 0.01195525098592043, 0.161596417427063, 0.0467013455927372, -0.12812840938568115, 0.02153809741139412], [0.16665774583816528, 0.13207615911960602, -0.15782803297042847, -0.007607625797390938, -0.014263243414461613, -0.12999655306339264, 0.18608112633228302, 0.07003182172775269, -0.05718493461608887, 0.08685103803873062, -0.014384042471647263, -0.0879199355840683, -0.1686539500951767, -0.10373648256063461, -0.06820520013570786, 0.015123979188501835, 0.09219293296337128, 0.10999767482280731, 0.017731858417391777, -0.19124963879585266, 0.03917207941412926], [-0.018001697957515717, 0.15088185667991638, 0.15443803369998932, -0.11564154177904129, 0.13295777142047882, -0.022221770137548447, 0.17692604660987854, -0.02048185095191002, 0.047913119196891785, -0.16649146378040314, 0.13322705030441284, -0.10083774477243423, -0.1293826699256897, -0.12396284192800522, -0.03702111542224884, 0.18201109766960144, -0.09487254172563553, -0.09658131003379822, -0.10696112364530563, -0.07703724503517151, 0.017840944230556488], [0.02797730639576912, -0.06808429956436157, 0.13029389083385468, 0.008591223508119583, -0.15923020243644714, -0.0988704264163971, -0.1429789662361145, 0.09375458210706711, 0.1342434138059616, -0.0844615250825882, 0.18914563953876495, -0.200400248169899, 0.13605017960071564, 0.13134710490703583, -0.10436270385980606, 0.17799244821071625, 0.17968937754631042, 0.06194768473505974, -0.113181471824646, 0.22200900316238403, -0.055910225957632065], [-0.16443268954753876, 0.087399922311306, 0.23412692546844482, 0.1659359335899353, 0.12589877843856812, 0.04734807834029198, -0.0064237224869430065, 0.022359061986207962, -0.11037147045135498, -0.05645843967795372, -0.1914132982492447, -0.12223529070615768, -0.12369108200073242, -0.13693858683109283, -0.13026165962219238, -0.03357703983783722, -0.1815810352563858, -0.103950135409832, 0.13092486560344696, -0.038108568638563156, 0.11328978836536407], [0.015874341130256653, 0.1613755226135254, 0.2019045501947403, -0.10382375121116638, -0.08183962851762772, 0.13960182666778564, -0.11280824989080429, 0.12429285794496536, -0.030918780714273453, 0.03521978482604027, -0.20241010189056396, 0.11272729188203812, -0.02980024367570877, -0.11259745806455612, 0.12148062884807587, -0.17705902457237244, -0.15149755775928497, -0.10569009184837341, 0.1449531614780426, -0.061003319919109344, 0.006744917947798967], [0.17126309871673584, 0.19125746190547943, 0.06977368146181107, 0.16284014284610748, 0.21740193665027618, -0.07084689289331436, -0.17132391035556793, -0.1143636554479599, 0.08293715119361877, 0.1772785633802414, -0.16139504313468933, 0.054498668760061264, -0.15023522078990936, -0.0881466194987297, -0.1953132450580597, 0.06302740424871445, 0.20626994967460632, -0.08594906330108643, -0.16816988587379456, 0.1310385763645172, 0.11260714381933212], [-0.013076422736048698, 0.08222238719463348, 0.06973008066415787, 0.010766226798295975, 0.13021962344646454, 0.18916131556034088, -0.10579542070627213, -0.14014838635921478, 0.12562017142772675, -0.12775477766990662, 0.07753901183605194, -0.17147797346115112, -0.24896158277988434, -0.048762742429971695, -0.11954661458730698, 0.13120335340499878, 0.04503268003463745, -0.03562478721141815, 0.09809362888336182, 0.011290726251900196, 0.05723730847239494], [-0.07757148146629333, -0.0061510419473052025, -0.11971858143806458, 0.0642135888338089, 0.10717941075563431, -0.005271372850984335, 0.16518644988536835, 0.09622559696435928, -0.11057820171117783, 0.1707507222890854, -0.025932757183909416, 0.010380410589277744, 0.18678073585033417, 0.061507698148489, -0.08514003455638885, 0.034944940358400345, 0.08293543756008148, 0.1311209350824356, -0.20288129150867462, -0.1695765107870102, 0.01023049745708704], [-0.08913722634315491, -0.17311632633209229, -0.1214059442281723, 0.12748906016349792, -0.1482347697019577, 0.14188185334205627, 0.019696863368153572, 0.04161893576383591, -0.0369885116815567, 0.18298552930355072, 0.07687126845121384, -0.04331187158823013, -0.09377509355545044, 0.10169941186904907, -0.0651969388127327, 0.11882414668798447, 0.1438048779964447, 0.12980049848556519, -0.14662474393844604, -0.09378406405448914, -0.06225866824388504], [-0.0564899779856205, 0.19707924127578735, -0.07168885320425034, -0.06862413138151169, 0.20137862861156464, 0.051539096981287, 0.06756720691919327, 0.008980626240372658, 0.07035861164331436, 0.0706285685300827, -0.09305506944656372, 0.1312386840581894, -0.016001658514142036, 0.04100402817130089, -0.2483242005109787, -0.14635910093784332, -0.0023276458960026503, -0.07074201107025146, 0.05400795489549637, 0.05345165729522705, -0.08696606010198593], [-0.19840307533740997, -0.11845467239618301, -0.17696025967597961, -0.07857625186443329, 0.1594696193933487, -0.09508366137742996, 0.1959926337003708, -0.08358639478683472, 0.14042842388153076, -0.08606148511171341, 0.22225575149059296, -0.1292317658662796, 0.008890763856470585, -0.03792532533407211, -0.09182443469762802, 0.10560502856969833, -0.1446678340435028, -0.11016920953989029, -0.11892247945070267, 0.15448249876499176, -0.09741461277008057], [-0.16721726953983307, 0.14859206974506378, 0.039719533175230026, 0.04229249060153961, -0.0972784236073494, 0.21363747119903564, 0.11960906535387039, -0.20667994022369385, -0.1347142606973648, -0.19961343705654144, -0.04270222410559654, 0.11477691680192947, -0.12435601651668549, 0.14782267808914185, 0.0477573499083519, -0.05242334306240082, 0.13180820643901825, -0.12206532806158066, 0.18924129009246826, -0.04268249496817589, 0.16251124441623688], [0.0306197889149189, -0.04977112635970116, 0.10222537815570831, 0.014723449945449829, 0.12944567203521729, -0.010481573641300201, -0.047556035220623016, -0.04186997562646866, 0.031375594437122345, 0.23641014099121094, -0.11366152763366699, -0.15892283618450165, -0.18988975882530212, 0.11247597634792328, -0.16298554837703705, 0.02877708524465561, -0.1486719697713852, -0.007633207831531763, 0.17572979629039764, -0.07363519072532654, 0.037935394793748856], [-0.01617998257279396, -0.06857119500637054, -0.13776572048664093, 0.011190099641680717, 0.057700615376234055, 0.1722162961959839, -0.0489235557615757, 0.043251145631074905, -0.06603921204805374, -0.1430296152830124, 0.01673109270632267, 0.1675989180803299, 0.2031342089176178, -0.03317316249012947, -0.2146545648574829, -0.08835455030202866, -0.08285863697528839, -0.1824694573879242, -0.020576557144522667, -0.0375388003885746, -0.027117781341075897], [0.12275214493274689, -0.1243038922548294, -0.05436435714364052, 0.184473916888237, -0.16445516049861908, -0.05506095290184021, -0.0517924502491951, 0.15076880156993866, -0.005664335563778877, 0.18848252296447754, 0.16412462294101715, -0.0431794710457325, 0.06392844766378403, 0.014767174609005451, 0.14053191244602203, 0.053139131516218185, -0.11985304951667786, -0.12916380167007446, 0.1391117423772812, 0.021831201389431953, 0.174761101603508], [0.003681380534544587, -0.052985742688179016, -0.10941355675458908, 0.1046973392367363, -0.1285303235054016, -0.04843858629465103, -0.152394101023674, 0.02535262331366539, 0.03758245334029198, -0.0058389827609062195, -0.18174348771572113, 0.1527063101530075, 0.07297392189502716, 0.15278086066246033, 0.1675158143043518, 0.10898372530937195, 0.006944472901523113, 0.12868832051753998, -0.04341515526175499, 0.17948690056800842, 0.11940724402666092], [-0.18930377066135406, -0.07049889117479324, 0.17161141335964203, 0.18109437823295593, 0.06622689962387085, 0.17251239717006683, -0.13887763023376465, -0.15710464119911194, -0.14302285015583038, -0.09075963497161865, -0.22282816469669342, 0.18691299855709076, -0.0216506477445364, 0.04897942394018173, 0.06900259107351303, 0.19476404786109924, 0.1889752447605133, 0.03427702188491821, -0.17718720436096191, 0.16076965630054474, -0.03546689450740814], [0.06902393698692322, -0.09289764612913132, -0.11282531917095184, 0.15267537534236908, -0.10610803216695786, 0.09230469912290573, -0.16421544551849365, -0.13541041314601898, -0.04898855462670326, 0.09207166731357574, -0.076230488717556, -0.1852884590625763, -0.12112756818532944, 0.10372921824455261, 0.03982546553015709, -0.08857380598783493, -0.14317354559898376, -0.01921510323882103, -0.03430885821580887, 0.05757011100649834, -0.07029575109481812], [0.004412153270095587, 0.07729311287403107, 0.1938726305961609, -0.2219022810459137, -0.14877814054489136, 0.14668408036231995, -0.10722851753234863, -0.13079985976219177, 0.13237030804157257, 0.18185581266880035, -0.23788177967071533, -0.16179941594600677, 0.13761626183986664, 0.05037396401166916, 0.16037091612815857, 0.010904739610850811, 0.002022867789492011, 0.07529088109731674, 0.01602797955274582, 0.0509103499352932, -0.1938253790140152], [-0.24094529449939728, 0.06852887570858002, 0.18342286348342896, 0.04608563333749771, -0.17714959383010864, 0.17275382578372955, 0.15399038791656494, -0.1875075250864029, 0.1652964949607849, -0.1583653837442398, 0.00828560534864664, -0.017059508711099625, -0.008238270878791809, -0.016108373180031776, -0.0769772157073021, 0.1106673926115036, -0.24653233587741852, 0.05933193862438202, 0.16791106760501862, 0.06276863813400269, -0.09128215909004211], [0.16191652417182922, -0.04913010820746422, 0.1783522218465805, -0.014388974756002426, 0.11369722336530685, 0.18883243203163147, 0.07623475044965744, -0.158692866563797, -0.10033220797777176, 0.16343045234680176, 0.01706567034125328, 0.09318847954273224, 0.08153383433818817, -0.1400795727968216, 0.037956271320581436, -0.09586744010448456, -0.11938260495662689, 0.04280301183462143, -0.11188973486423492, 0.14760330319404602, 0.20815934240818024], [0.049545932561159134, 0.10620097070932388, -0.12935680150985718, -0.10359735786914825, 0.1862286478281021, 0.12900157272815704, 0.023415260016918182, 0.0443374328315258, 0.10898036509752274, 0.0860092043876648, -0.007613653317093849, 0.062192801386117935, 0.04148178920149803, -0.1807677000761032, -0.19771845638751984, 0.1656806617975235, 0.21853549778461456, 0.08388320356607437, 0.2040715515613556, -0.03459206968545914, -0.21543492376804352], [-0.02803656831383705, 0.17179177701473236, 0.21048595011234283, -0.04493982717394829, 0.08231987059116364, -0.1807951033115387, -0.03572475537657738, 0.19608476758003235, 0.10213525593280792, 0.04733143001794815, 0.03521218150854111, -0.255357027053833, -0.1181812658905983, -0.03589121252298355, 0.11307662725448608, 0.12908251583576202, -0.07642757147550583, 0.05901111289858818, -0.2064877599477768, -0.09098739922046661, -0.14539965987205505], [-0.07386332750320435, -0.01232829224318266, -0.08784884214401245, 0.05817073583602905, -0.006515213288366795, 0.16211335361003876, -0.05632738769054413, -0.12858286499977112, -0.028976518660783768, 0.07991241663694382, 0.2399565726518631, -0.08818022161722183, 0.02008187398314476, -0.05664864927530289, -0.1659294217824936, -0.012105497531592846, -0.23559798300266266, -0.03813357651233673, -0.054925937205553055, 0.03206411749124527, -0.14164942502975464], [-0.06250932067632675, -0.10881604254245758, -0.0013011062983423471, 0.07434574514627457, -0.08539571613073349, -0.1796514391899109, -0.04398811236023903, 0.05772359296679497, 0.04081302136182785, -0.08168268203735352, -0.09365951269865036, -0.01889091357588768, -0.0182852391153574, -0.1164616271853447, -0.04544435068964958, -0.01001069601625204, -0.22879397869110107, 0.1404632031917572, 0.07641631364822388, -0.026105176657438278, -0.13265061378479004], [0.10471848398447037, -0.031152987852692604, 0.024481069296598434, 0.12335596978664398, -0.14925791323184967, -0.1915445625782013, -0.09598084539175034, 0.0641489252448082, 0.10425335168838501, 0.20608054101467133, 0.05514879152178764, -0.05373515188694, -0.1494014859199524, -0.07575087249279022, -0.1963905245065689, 0.07413311302661896, -0.04887905716896057, 0.0680999606847763, 0.11934293061494827, -0.15288619697093964, 0.19374209642410278], [0.12992820143699646, -0.1565103679895401, 0.19226102530956268, 0.2537500560283661, -0.21196028590202332, 0.03362731263041496, 0.18101775646209717, 0.11706748604774475, 0.07355749607086182, 0.06537606567144394, 0.20496493577957153, -0.17874795198440552, -0.21080321073532104, 0.06605511158704758, -0.025658270344138145, -0.06994766741991043, -0.14091207087039948, -0.082720085978508, 0.09581492096185684, 0.16606442630290985, 0.035153523087501526], [0.023393936455249786, -0.16853031516075134, -0.11291860044002533, -0.18092882633209229, 0.007433192804455757, -0.06159914657473564, 0.11551544070243835, 0.17125838994979858, -0.1364881843328476, 0.0961550623178482, 0.1017538383603096, -0.10591570287942886, -0.16371546685695648, 0.15792998671531677, -0.18191710114479065, 0.14263302087783813, 0.05739878490567207, 0.03442312031984329, -0.18354986608028412, 0.09601377695798874, -0.16290663182735443], [-0.14321723580360413, 0.13468517363071442, -0.06347719579935074, -0.13971777260303497, -0.19214168190956116, 0.0002576620609033853, -0.0997508242726326, -0.07423495501279831, 0.058434683829545975, 0.13363489508628845, -0.18862833082675934, -0.07411635667085648, 0.0845426544547081, -0.01599469780921936, -0.007337275892496109, 0.05150323361158371, -0.01765524409711361, -0.02548188529908657, -0.1138642281293869, -0.04505510628223419, -0.07827184349298477], [-0.11952870339155197, 0.19844895601272583, 0.17250369489192963, 0.041108664125204086, 0.06455475836992264, -0.19234305620193481, 0.09620452672243118, -0.10646319389343262, 0.003225969849154353, -0.06604688614606857, -0.07527758181095123, -0.048503439873456955, 0.07910405099391937, 0.11109557002782822, 0.1841031014919281, -0.057280637323856354, 0.1414201557636261, 0.07665099948644638, 0.056785378605127335, -0.1418224722146988, -0.01990373060107231], [-0.13841485977172852, -0.12911632657051086, -0.043589260429143906, 0.10317016392946243, -0.03306511417031288, -0.07708223164081573, 0.06171497702598572, -0.0640525072813034, -0.1698162704706192, 0.1557144671678543, -0.19171753525733948, -0.04922594502568245, -0.139322429895401, -0.08014319837093353, 0.040785498917102814, -0.015587715432047844, -0.06924745440483093, -0.04638226330280304, -0.13943356275558472, -0.1589745581150055, 0.035041507333517075], [-0.12936608493328094, -0.15267840027809143, -0.02010984532535076, 0.06521697342395782, 0.16772466897964478, -0.14958766102790833, 0.17226091027259827, 0.05126909166574478, -0.03204408288002014, 0.061866190284490585, -0.026965398341417313, -0.17150458693504333, -0.031914450228214264, 0.11845719069242477, 0.020210832357406616, -0.05422964692115784, -0.14454403519630432, 0.1600235551595688, 0.17750567197799683, -0.0227240938693285, -0.18650831282138824], [0.0793166533112526, -0.11383962631225586, -0.03224403038620949, -0.09765627980232239, 0.07908275723457336, 0.027333436533808708, 0.1955205649137497, 0.08875582367181778, -0.07732924818992615, -0.16448482871055603, -0.026330722495913506, 0.1269129067659378, 0.10775841027498245, 0.18150459229946136, 0.07450450211763382, 0.03559069335460663, 0.08035282790660858, 0.1631547510623932, 0.13212361931800842, -0.12154970318078995, -0.053202901035547256], [0.08832642436027527, -0.10703250765800476, -0.0920846164226532, 0.16680453717708588, -0.1172933429479599, 0.09067077189683914, 0.07483828812837601, 0.14051394164562225, -0.07408466190099716, -0.15970921516418457, -0.10281641036272049, 0.12483067065477371, 0.01406190823763609, -0.09795688837766647, 0.0789683386683464, 0.12758928537368774, -0.17529433965682983, 0.05597211420536041, -0.046661119908094406, -0.013021937571465969, 0.07817807048559189], [0.20972271263599396, 0.17867325246334076, -0.16152365505695343, -0.036333177238702774, 0.07339322566986084, 0.07326815277338028, -0.06703357398509979, -0.07154043763875961, 0.14784082770347595, 0.1766122728586197, 0.07219284027814865, 0.13003602623939514, -0.009760580025613308, -0.04595969244837761, 0.047811079770326614, -0.022890819236636162, 0.10729891806840897, -0.044951941817998886, 0.004264104180037975, -0.041810669004917145, 0.1854473501443863], [-0.007198403123766184, 0.0443560890853405, 0.10708228498697281, 0.07984326034784317, -0.0990465059876442, 0.12789243459701538, 0.11027009785175323, 0.2658717930316925, -0.017590174451470375, 0.07444719970226288, -0.05709761381149292, 0.08872431516647339, -0.0621396005153656, 0.017646582797169685, 0.10197827965021133, -0.21951600909233093, -0.1289108395576477, -0.14844419062137604, 0.08780551701784134, -0.000595343706663698, 0.10743936151266098], [0.025378134101629257, -0.059521131217479706, 0.07734686881303787, -0.07985668629407883, 0.13602741062641144, -0.061931315809488297, -0.005025102291256189, -0.03526129573583603, 0.07428579032421112, -0.10952646285295486, 0.009987161494791508, 0.06221903860569, 0.055109113454818726, 0.007306309416890144, 0.08179304748773575, 0.12457018345594406, 0.03149227052927017, 0.1598232388496399, 0.17361097037792206, 0.08098206669092178, 0.1415901929140091], [-0.001373794162645936, 0.14822132885456085, 0.24633489549160004, -0.10193949192762375, -0.10893678665161133, 0.066067174077034, -0.15386609733104706, -0.17289653420448303, -0.06651773303747177, 0.20914752781391144, -0.21401233971118927, -0.04027203097939491, -0.01923811063170433, -0.08268290013074875, 0.04028566554188728, -0.16587170958518982, 0.03987885266542435, 0.22499960660934448, 0.17694152891635895, -0.02038874477148056, -0.13666120171546936], [0.009924052283167839, -0.13418486714363098, 0.04860072210431099, -0.020312795415520668, 0.058424871414899826, 0.00010575357737252489, 0.07318907976150513, -0.11688995361328125, -0.03652466833591461, -0.15350386500358582, -0.10000316798686981, -0.05387838929891586, 0.010833141393959522, 0.061371173709630966, 0.05187288299202919, 0.19549186527729034, -0.007709338329732418, -0.006986046209931374, -0.16746175289154053, -0.18792903423309326, -0.14771629869937897], [-0.06363730132579803, -0.001453168224543333, -0.22828392684459686, -0.04229586571455002, 0.09845340251922607, 0.031487684696912766, -0.055916182696819305, 0.14780157804489136, -0.10090454667806625, -0.1688462644815445, -0.19752003252506256, 0.10849252343177795, -0.18880628049373627, -0.10262663662433624, -0.09365163743495941, 0.20529374480247498, 0.12154925614595413, -0.04847319424152374, 0.1265399158000946, -0.18482187390327454, -0.16001449525356293], [-0.03480423241853714, 0.1757701188325882, -0.08782565593719482, -0.028353244066238403, 0.021437233313918114, -0.12604698538780212, -0.07469294965267181, -0.015561815351247787, 0.11233017593622208, 0.15212513506412506, -0.047793108969926834, -0.15336208045482635, 0.22756265103816986, 0.004735115449875593, 0.10310948640108109, -0.012283760122954845, -0.14203643798828125, 0.015499980188906193, 0.15499047935009003, 0.15267346799373627, 0.0027472025249153376], [-0.12643615901470184, -0.10956864058971405, 0.11676923930644989, -0.0077514443546533585, 0.19367040693759918, 0.17919804155826569, 0.1735212802886963, -0.0329856239259243, -0.15065905451774597, 0.06671877205371857, 0.027347620576620102, -0.10337761789560318, -0.14103540778160095, -0.14962930977344513, -0.1931435912847519, 0.16422785818576813, -0.2322208434343338, 0.09387877583503723, 0.18750301003456116, 0.06170035898685455, 0.06513910740613937], [-0.024510914459824562, 0.02468247339129448, -0.02961617149412632, 0.1355876326560974, -0.14813852310180664, -0.011667846702039242, -0.06322864443063736, 0.02498744986951351, -0.1727665364742279, 0.10142415761947632, -0.09598688036203384, -0.06055128201842308, 0.002841213485226035, 0.13938462734222412, -0.139089435338974, -0.13442666828632355, -0.14020086824893951, -0.030288342386484146, -0.16007503867149353, -0.1195249855518341, 0.16786932945251465], [0.04779628664255142, 0.018775654956698418, 0.05969592183828354, -0.0015629995614290237, -0.06681565940380096, 0.1098172515630722, -0.09798912703990936, -0.10131081193685532, 0.1576995551586151, -0.1035454124212265, 0.10425940901041031, 0.05029759183526039, 0.08722631633281708, 0.03704888001084328, -0.16362906992435455, 0.15692180395126343, 0.12675431370735168, -0.07744657248258591, -0.10759248584508896, -0.10952412337064743, 0.2133827656507492], [-0.0680975466966629, 0.07194232195615768, 0.08208328485488892, 0.07396858930587769, -0.17559537291526794, 0.16102859377861023, 0.1460946798324585, -0.01014020387083292, -0.15680482983589172, 0.12598666548728943, 0.13909079134464264, -0.15176160633563995, 0.17739613354206085, 0.09227771311998367, 0.044342778623104095, -0.14936912059783936, 0.17112377285957336, 0.15690255165100098, -0.18331855535507202, 0.0036506003234535456, -0.08349102735519409], [0.10332811623811722, -0.012245416641235352, -0.1744977980852127, 0.15714043378829956, -0.007973444648087025, -0.11546027660369873, -0.054797906428575516, -0.06729050725698471, 0.08153927326202393, -0.04865161329507828, -0.11623696982860565, 0.05785378813743591, 0.09170546382665634, 0.0457080714404583, -0.0365200974047184, -0.02010240964591503, -0.1173776164650917, -0.1263027787208557, 0.018234847113490105, 0.12533681094646454, -0.07015980035066605], [0.04213510453701019, 0.08962935209274292, -0.1062847450375557, 0.10884815454483032, 0.02386210672557354, -0.12138721346855164, -0.07450783252716064, 0.09861589968204498, 0.018682874739170074, -0.09806019067764282, -0.14741083979606628, 0.02947370335459709, 0.14593349397182465, -0.12945722043514252, -0.06839589029550552, 0.04859786853194237, -0.18415100872516632, 0.14086183905601501, -0.0274494718760252, 0.18435141444206238, -0.011572036892175674], [-0.037460215389728546, 0.1285301148891449, -0.038352422416210175, -0.15820302069187164, -0.12499594688415527, -0.24306567013263702, -0.10007579624652863, -0.026029376313090324, -0.05953590199351311, 0.06136653572320938, 0.1054009422659874, 0.09212534874677658, -0.10013902187347412, 0.10906494408845901, -0.1509292721748352, 0.06802792847156525, -0.15855856239795685, -0.06330490857362747, -0.108364038169384, 0.02632814645767212, 0.17270374298095703], [-0.16135792434215546, 0.08807043731212616, -0.12445008754730225, -0.03315198794007301, -0.1328846961259842, 0.14978983998298645, 0.20934301614761353, 0.013215410523116589, 0.18887419998645782, -0.014936068095266819, 0.09098193049430847, -0.0865471139550209, -0.05378280207514763, -0.03131815791130066, 0.1986042559146881, -0.08128451555967331, -0.1036088690161705, 0.15648409724235535, -0.0936422199010849, 0.03902252018451691, 0.07321719080209732], [-0.040121618658304214, 0.030746161937713623, 0.13051719963550568, 0.1575809121131897, -0.18765421211719513, -0.09134188294410706, -0.1301969438791275, -0.15625609457492828, -0.11811122298240662, 0.13216446340084076, 0.14487089216709137, -0.0678296610713005, -0.17664669454097748, -0.04264645650982857, 0.03183520957827568, -0.01666787825524807, -0.19323326647281647, 0.154405415058136, -0.13036592304706573, -0.051486462354660034, -0.05750978738069534], [0.002991973189637065, 0.1826649010181427, -0.08861617743968964, 0.027603140100836754, -0.03454132378101349, 0.1625114232301712, 0.12853172421455383, 0.05823305994272232, 0.016394108533859253, -0.01506317500025034, -0.08098147064447403, 0.1572377234697342, 0.0038191708736121655, 0.11957797408103943, -0.1475689858198166, -0.15644188225269318, -0.14196854829788208, -0.1170140728354454, -0.0829562321305275, -0.04254882410168648, 0.13333803415298462], [-0.1253986805677414, 0.11410903930664062, 0.055741503834724426, -0.08560533821582794, 0.006616006605327129, 0.15030229091644287, -0.14856311678886414, -0.07061630487442017, 0.059356532990932465, 0.05569007620215416, 0.08473359048366547, -0.020794011652469635, 0.23549406230449677, 0.22452442348003387, -0.22106683254241943, -0.012625976465642452, -0.09379956126213074, -0.0780300498008728, 0.15226690471172333, -0.030398590490221977, -0.2208501696586609], [-0.02639644965529442, -0.008089258335530758, 0.08164577186107635, -0.06042526662349701, -0.039553653448820114, 0.2247801572084427, 0.04290168359875679, -0.18824267387390137, -0.10993672162294388, 0.01977563090622425, -0.16252446174621582, 0.09157628566026688, -0.11951707303524017, 0.030605433508753777, 0.10588862746953964, 0.1155167669057846, -0.009885878302156925, -0.028628697618842125, 0.03427295386791229, 0.0263729989528656, -0.15283922851085663], [-0.07221753150224686, 0.08397993445396423, -0.043261606246232986, -0.0601092092692852, 0.07862403243780136, -0.059332799166440964, -0.06660065799951553, 0.03967675939202309, -0.02383795566856861, 0.13330039381980896, -0.047883398830890656, -0.1782330423593521, 0.21309785544872284, 0.06638214737176895, -0.019012829288840294, -0.02328268624842167, 0.041268859058618546, 0.1118982657790184, 0.03210296854376793, -0.057266708463430405, 0.001598235685378313], [0.17261181771755219, 0.02328478731215, 0.0579957515001297, -0.17827460169792175, 0.015919268131256104, -0.1431751251220703, -0.21333006024360657, 0.04415804520249367, 0.10837303847074509, -0.10146664828062057, -0.057914864271879196, 0.10978077352046967, 0.084743931889534, -0.15014462172985077, 0.18637989461421967, -0.14411917328834534, 0.2242973893880844, -0.10204320400953293, 0.03563424199819565, 0.1476556956768036, 0.04560791328549385], [0.2120448648929596, -0.06825652718544006, -0.15481004118919373, 0.16820891201496124, 0.19069406390190125, 0.025433694943785667, -0.15743950009346008, 0.11005344241857529, -0.03532571345567703, -0.14327527582645416, 0.1375928372144699, 0.10754178464412689, 0.20602980256080627, 0.013430982828140259, 0.10671819746494293, -0.13466639816761017, 0.05940180644392967, 0.16568870842456818, -0.15248312056064606, -0.055028289556503296, 0.06007712334394455], [0.03417650982737541, 0.06217639148235321, -0.14110946655273438, 0.11504723131656647, 0.22495660185813904, 0.09334558993577957, -0.041011910885572433, 0.15965864062309265, 0.00875299796462059, 0.026278208941221237, -0.005462460685521364, -0.1928134709596634, 0.09236497431993484, 0.0939609482884407, -0.05160705745220184, -0.1858065277338028, 0.2159961611032486, -0.14306165277957916, 0.19442814588546753, -0.13931968808174133, -0.009780149906873703], [0.03244808688759804, -0.16080747544765472, 0.027068937197327614, 0.0384942851960659, 0.1866421103477478, -0.12085412442684174, -0.10975228250026703, 0.07660768926143646, -0.07039006054401398, 0.12687960267066956, -0.051329270005226135, 0.015105321072041988, 0.0617818608880043, 0.12174264341592789, 0.11248471587896347, 0.08044003695249557, -0.03415456786751747, -0.028836894780397415, 0.08761321753263474, 0.1659046858549118, -0.06470461934804916], [0.11327794194221497, 0.14904864132404327, -0.2645960748195648, -0.05366632342338562, 0.17621029913425446, 0.17348195612430573, 0.12328499555587769, -0.02337508089840412, 0.13320596516132355, 0.06322374194860458, -0.04778359457850456, 0.17792485654354095, -0.16488030552864075, -0.027569012716412544, 0.06489022076129913, -0.1430361270904541, -0.02657388523221016, 0.16111639142036438, 0.0017974061192944646, -0.021959103643894196, -0.19131986796855927], [-0.045935459434986115, 0.11520461738109589, 0.10868076980113983, -0.07125069946050644, -0.16248354315757751, 0.037308983504772186, -0.1002340242266655, -0.12732121348381042, -0.18408536911010742, -0.13240328431129456, -0.1232820525765419, -0.019694197922945023, 0.2037617266178131, 0.16005709767341614, -0.003389523597434163, 0.028470775112509727, 0.08887114375829697, 0.0650026798248291, -0.16149967908859253, -0.0021462554577738047, -0.030602574348449707], [-0.021694805473089218, -0.10088588297367096, 0.09001677483320236, -0.13704513013362885, -0.10169939696788788, 0.022450122982263565, 0.14486528933048248, -0.14670297503471375, -0.13666073977947235, 0.03136962652206421, -0.05497058108448982, -0.18974842131137848, 0.004552186466753483, 0.16579675674438477, 0.08961883187294006, 0.001530289533548057, 0.14877374470233917, 0.15029148757457733, -0.09889262914657593, -0.1205824539065361, 0.06921080499887466], [0.034662239253520966, 0.16796328127384186, 0.03788125887513161, 0.027876535430550575, -0.0007198266685009003, -0.07882096618413925, -0.08584754914045334, -0.05048822984099388, -0.1169157326221466, -0.05436887592077255, -0.18822848796844482, 0.09641794115304947, 0.0225922130048275, -0.11869119107723236, -0.05243263393640518, 0.045058030635118484, 0.24363134801387787, 0.05865626782178879, 0.16311481595039368, -0.08834799379110336, 0.013783174566924572], [-0.15475527942180634, -0.060343094170093536, 0.07264906167984009, 0.20903314650058746, -0.06910315901041031, -0.02626502513885498, 0.19007527828216553, -0.1306384652853012, 0.03600464388728142, -0.13744626939296722, 0.014129320159554482, 0.03505181893706322, -0.15121178328990936, 0.1616797298192978, -0.027414267882704735, 0.01921791024506092, 0.09157729148864746, 0.017013035714626312, 0.042204637080430984, 0.002359718084335327, -0.013744720257818699], [0.07982297241687775, 0.13460592925548553, 0.014717599377036095, 0.1835559904575348, 0.07387696951627731, -0.1076025441288948, 0.06930144876241684, 0.1976792961359024, -0.0035193488001823425, 0.0765049085021019, 0.07114201039075851, -0.1775149405002594, -0.054136715829372406, -0.06203353404998779, 0.1894710212945938, -0.16022400557994843, -0.17931318283081055, 0.1152893677353859, 0.04356931895017624, 0.07451435923576355, 0.06581659615039825], [0.04152010381221771, 0.073484867811203, -0.05461222678422928, 0.02714138850569725, 0.06853815913200378, -0.07771333307027817, 0.20540432631969452, 0.15203553438186646, 0.13749447464942932, -0.022375917062163353, 0.0401756726205349, 0.04450178146362305, 0.12465689331293106, -0.09179257601499557, 0.1286858320236206, -0.11248493194580078, -0.19420674443244934, 0.09299135953187943, 0.16004925966262817, 0.012145920656621456, -0.1813747137784958], [-0.08967453241348267, -0.07575233280658722, 0.1356476992368698, -0.1507328301668167, 0.12744691967964172, -0.05982193350791931, 0.18269409239292145, -0.08702048659324646, -0.1276317983865738, -0.027154531329870224, -0.05397125706076622, 0.06505700945854187, 0.24249719083309174, -0.05858088284730911, -0.0014623492024838924, 0.08553913980722427, 0.1468568742275238, -0.09929460287094116, 0.07776713371276855, 0.029406437650322914, -0.1623034030199051]], "b1": [-0.16960479319095612, 0.2144639790058136, 0.19470053911209106, -0.2036464959383011, 0.23685775697231293, 0.1782929003238678, -0.06725400686264038, -0.046849966049194336, 0.14970923960208893, 0.07638909667730331, -0.14976412057876587, -0.00804917886853218, 0.03870541229844093, -0.09662380814552307, -0.1820957362651825, 0.06043266877532005, -0.029400281608104706, 0.17976514995098114, 0.04076414927840233, 0.15889067947864532, -0.12449478358030319, 0.15753480792045593, 0.24404433369636536, 0.07086770981550217, 0.05298830196261406, -0.09707793593406677, 0.16155238449573517, 0.02192290499806404, -0.1262894719839096, -0.022653594613075256, -0.08687321096658707, -0.030127158388495445, 0.10233300179243088, 0.11738578230142593, 0.2359592765569687, 0.11823718249797821, 0.23791280388832092, 0.03130047023296356, -0.09236177802085876, 0.18230308592319489, -0.16053810715675354, -0.10217247903347015, -0.08014243096113205, 0.05283340811729431, -0.08194372802972794, 0.07294964045286179, -0.050833478569984436, -0.11462672054767609, -0.019467821344733238, 0.088483065366745, 0.017288845032453537, 0.03598640486598015, -0.0645960196852684, 0.11462925374507904, 0.017966896295547485, -0.031962499022483826, 0.008278806693851948, -0.053112514317035675, 0.09132533520460129, 0.01647482067346573, -0.1308964341878891, 0.09980791062116623, 0.14514921605587006, 0.20759189128875732, -0.03870392590761185, -0.15518569946289062, -0.21548868715763092, 0.15740320086479187, -0.11766969412565231, -0.11398795247077942, -0.006477247923612595, -0.055271659046411514, -0.013556609861552715, -0.21969516575336456, -0.14404109120368958, 0.1348859965801239, 0.2305358648300171, 0.12591134011745453, 0.04118478670716286, -0.07020634412765503, -0.016778774559497833, 0.205311581492424, 0.053816989064216614, 0.03234677016735077, 0.17453213036060333, 0.08624254167079926, 0.1889631152153015, -0.10956187546253204, 0.07925473153591156, 0.11349757015705109, 0.13872230052947998, -0.0623118057847023, -0.03317505866289139, -0.20086199045181274, 0.20770464837551117, 0.21192175149917603], "W2": [[0.059096332639455795, 0.011587217450141907, 0.005858595483005047, -0.07234637439250946, -0.019428297877311707, -0.03883178532123566, -0.06152159348130226, -0.017954207956790924, 0.07009362429380417, 0.0739155188202858, -0.1016298159956932, 0.07655952870845795, -0.05315830186009407, 0.02849949523806572, 0.06469497084617615, 0.018076296895742416, -0.03612218797206879, -0.07448235154151917, 0.09970054030418396, 0.0023883385583758354, 0.0954500138759613, -0.05439957603812218, 0.06797145307064056, 0.07029052078723907, -0.02656949870288372, 0.025087742134928703, 0.059518493711948395, 0.08121174573898315, -0.04014196991920471, 0.0595841221511364, -0.07416068017482758, 0.038743071258068085, -0.034074947237968445, 0.011372244916856289, 0.07095380872488022, 0.04659390449523926, 0.01935698464512825, 0.07906358689069748, -0.013526204973459244, -0.0705283135175705, 0.02260661870241165, 0.0778278112411499, -0.07716122269630432, 0.02202950417995453, -0.00886513851583004, 0.09138284623622894, -0.10519956797361374, -0.006643285043537617, 0.023976672440767288, 0.07447493821382523, 0.04424980282783508, 0.048211462795734406, 0.07168411463499069, -0.061099573969841, -0.04600776359438896, 0.0028721927665174007, -0.04149860516190529, 0.021673787385225296, 0.07642163336277008, -0.028725679963827133, -0.064604751765728, -0.06327792257070541, -0.013565361499786377, -0.04841946065425873, 0.011232571676373482, -0.11006499081850052, -0.0495121031999588, 0.011996564455330372, 0.011706026270985603, -0.04049272835254669, 0.041581619530916214, 0.0393904484808445, -0.03782619908452034, -0.08840540796518326, -0.0474187433719635, 0.04866426810622215, -0.00016903544019442052, 0.06163991615176201, -0.07453049719333649, -0.009721881709992886, -0.028264658525586128, 0.08423715829849243, -0.03968862444162369, 0.002249687910079956, 0.10165665298700333, 0.025985516607761383, -0.025875123217701912, -0.03870202973484993, -0.024291682988405228, -0.004586613271385431, 0.048572342842817307, 0.11192994564771652, -0.040365397930145264, -0.08988882601261139, 0.05804820358753204, 0.02370329201221466], [0.04238656163215637, -0.04819776117801666, 0.09329408407211304, 0.025067338719964027, -0.03259509056806564, 0.04642441123723984, 0.0493193045258522, -0.021993620321154594, -0.010770988650619984, 0.015063643455505371, 0.07461148500442505, -0.02481864020228386, -0.030549651011824608, 0.06465630978345871, -0.04861558973789215, 0.0017833173042163253, 0.10508229583501816, 0.008701398968696594, -0.03218071535229683, -0.00045177689753472805, 0.06789309531450272, 0.07851254940032959, 0.048811472952365875, -0.0831567719578743, -5.8570545661496e-05, -0.0023143577855080366, -0.06835600733757019, 0.0425284244120121, 0.0830121710896492, -0.04364216700196266, 0.002778782742097974, 0.06541784107685089, 0.076125368475914, 0.021771132946014404, -0.012025924399495125, -0.05664006993174553, -0.08975465595722198, -0.021611128002405167, -0.044930990785360336, 0.04476798698306084, 0.07258116453886032, 0.08023101836442947, -0.05485749617218971, -0.019764648750424385, 0.026979854330420494, 0.045213375240564346, -0.027568552643060684, 0.06423984467983246, -0.03393248841166496, -0.03197867423295975, -0.1010625958442688, 0.058927785605192184, 0.03525687754154205, -0.02904883772134781, -0.015152276493608952, 0.03218367323279381, -0.0034520432818681, -0.0308445543050766, -0.017486657947301865, 0.0006340354448184371, 0.07064566761255264, -0.038749810308218, -0.05118773505091667, -0.032547350972890854, 0.08116195350885391, 0.10808271914720535, -0.01153042633086443, -0.07647881656885147, 0.07703307271003723, 0.11993654817342758, 0.07596045732498169, -0.03209909796714783, 0.05396217480301857, 0.04704548045992851, 0.015633178874850273, -0.005435279104858637, 0.016794348135590553, 0.034046437591314316, 0.059013377875089645, -0.04245878383517265, 0.012734145857393742, -0.02735796384513378, 0.06723557412624359, 0.05552903562784195, 0.10069839656352997, 0.04206143319606781, 0.08193231374025345, -0.037401698529720306, 0.13283999264240265, -0.008785728365182877, 0.03419655188918114, 0.018898988142609596, 0.045492615550756454, -0.05181119218468666, 0.07113116979598999, 0.00045945713645778596], [0.06894576549530029, 0.04879297316074371, -0.014630154706537724, -0.06127696856856346, 0.02532763034105301, -0.0390690341591835, -0.07068319618701935, 0.016095228493213654, 0.040121063590049744, -0.015936264768242836, -0.02175627276301384, 0.12073265016078949, 0.11120538413524628, -0.12084949016571045, 0.07074368000030518, 0.013049590401351452, 0.05469944328069687, -0.04890068247914314, -0.050878699868917465, 0.00477082934230566, 0.0022940211929380894, -0.07982981204986572, 0.019066834822297096, 0.044156670570373535, 0.10578469932079315, 0.043031059205532074, 0.08153978735208511, 0.04858548939228058, -0.10793662071228027, -0.05560704693198204, -0.099776990711689, -0.08799385279417038, -0.06468545645475388, 0.08698848634958267, -0.0343647338449955, 0.10206780582666397, -0.015629855915904045, 0.09239698946475983, 0.041098400950431824, -0.06329852342605591, -0.03197378292679787, -0.019121825695037842, -0.07273602485656738, -0.024448173120617867, -0.07386842370033264, 0.017380021512508392, 0.05579753592610359, -0.03343184292316437, 0.11897414922714233, -0.04570562392473221, -0.05906190723180771, 0.0953667014837265, 0.020796574652194977, 0.06815429776906967, -0.0327492356300354, 0.10493016242980957, -0.038015954196453094, 0.058737676590681076, 0.0715399831533432, -0.03313083201646805, -0.012335998937487602, 0.023245370015501976, -0.028281468898057938, 0.012318077497184277, 0.05045495927333832, -0.1310964673757553, -0.06186673045158386, 0.006735046394169331, 0.011233614757657051, -0.08726384490728378, 0.09229518473148346, -0.01803261786699295, 0.04175974056124687, -0.10221643000841141, 0.021056199446320534, 0.010750546120107174, 0.09614000469446182, 0.014167636632919312, 0.0057046012952923775, -0.05106041952967644, 0.026095667853951454, -0.020649438723921776, 0.03621118888258934, 0.09639947861433029, 0.04007028788328171, 0.06492193788290024, 0.03662591800093651, -0.06619620323181152, -0.047533705830574036, 0.06899483501911163, 0.07607415318489075, 0.13218270242214203, 0.07148455083370209, 0.033032551407814026, 0.05483347550034523, 0.11421415954828262], [0.07126162946224213, 0.07544705271720886, 0.027964159846305847, -0.004315175116062164, 0.018209373578429222, 0.04140394553542137, 0.042809467762708664, 0.005491032265126705, 0.045628871768713, 0.07853571325540543, -0.0008367843693122268, -0.013284324668347836, -0.038432348519563675, -0.02212969958782196, -0.05216691642999649, 0.08969523012638092, -0.051159657537937164, 0.02970738522708416, 0.10138361901044846, 0.021542832255363464, 0.012050281278789043, 0.08485481888055801, -0.03189875930547714, 0.03587581217288971, -0.022021161392331123, 0.06721401959657669, -0.01473671942949295, 0.10180725157260895, -0.008730268105864525, -0.042967788875103, -0.0800260603427887, -0.064214326441288, 0.09183791279792786, -0.05858798697590828, 0.024220505729317665, -0.007332047447562218, -0.054498378187417984, 0.06509759277105331, -0.08994532376527786, -0.04921005293726921, -0.07124577462673187, 0.04189527779817581, 0.08833374083042145, 0.058990441262722015, -0.07538419216871262, -0.055621106177568436, -0.03001372143626213, -0.07217088341712952, 0.005125317722558975, 0.030199173837900162, -0.06409969180822372, -0.04263441637158394, 0.050627242773771286, 0.039795778691768646, 0.060961704701185226, 0.0978480875492096, 0.09983078390359879, -0.021992789581418037, 0.10208695381879807, 0.027910614386200905, -0.07526849955320358, 0.04063164442777634, 0.059131477028131485, -0.020842647179961205, -0.047123294323682785, 0.010980337858200073, 0.06375223398208618, 0.031166089698672295, -0.07104413956403732, 0.019336622208356857, -0.05319371074438095, -0.026797886937856674, 0.022239090874791145, -0.06472528725862503, -0.09112709015607834, 0.07211317867040634, 0.01931571774184704, 0.07256337255239487, -0.05543980747461319, 0.007421540096402168, -0.030049260705709457, -0.05998842790722847, -0.026210639625787735, -0.015473968349397182, -0.07095947861671448, -0.03269176185131073, 0.05632183700799942, -0.07151754945516586, 0.0010094087338075042, 0.007567192427814007, 0.08624681085348129, -0.04489511996507645, 0.0005580016877502203, 0.07753647863864899, -0.019548803567886353, 0.041390951722860336], [0.00980381853878498, -0.0077073280699551105, 0.07498177140951157, -0.09643327444791794, 0.07848677784204483, -0.08122502267360687, 0.012238065712153912, 0.015586290508508682, -0.07014769315719604, -0.015402915887534618, -0.11419892311096191, 0.08057419210672379, -0.038516394793987274, -0.08120622485876083, 0.052263688296079636, 0.08019183576107025, 0.05406208708882332, 0.03261725977063179, 0.037130050361156464, 0.04689595475792885, -0.006284498143941164, -0.012654668651521206, 0.10311540961265564, 0.05572253838181496, 0.04057899862527847, -0.10059860348701477, 0.097169890999794, -0.016698181629180908, -0.057355016469955444, 0.08080633729696274, -0.04608849808573723, 0.005535766948014498, -0.05883868411183357, 0.06755897402763367, -0.0854993537068367, 0.08425239473581314, 0.05106402188539505, 0.08923858404159546, -0.05472908169031143, 0.060018390417099, -0.011232320219278336, 0.08173256367444992, -0.011166802607476711, -0.0043474710546433926, -0.09233308583498001, 0.012512138113379478, 0.08886230736970901, -0.03930748626589775, -0.052148912101984024, 0.00019285790040157735, -0.07671090215444565, 0.0824401006102562, -0.04386265203356743, 0.0837821513414383, -0.057403285056352615, 0.005407045595347881, 0.0752173364162445, 0.037059981375932693, 0.01926039345562458, 0.06637825816869736, 0.056316375732421875, -0.04117748141288757, -0.02322450838983059, -0.011498814448714256, -0.09960852563381195, -0.09498949348926544, -0.020462483167648315, -0.024862507358193398, 0.041244424879550934, -0.03886093571782112, -0.07978636026382446, 0.03041025996208191, -0.0354253388941288, -0.05150759592652321, 0.00860671978443861, -0.06359434127807617, -0.06608811765909195, 0.10224056243896484, 0.07670336961746216, 0.04039718583226204, 0.03018827550113201, 0.07144753634929657, 0.04154862090945244, -0.034142278134822845, 0.008541200309991837, -0.08533742278814316, 0.07335502654314041, -0.01458797138184309, 0.07020006328821182, -0.04612893983721733, -0.015851818025112152, 0.09928414970636368, 0.00047516560880467296, -0.0266706682741642, 0.021470021456480026, 0.009180906228721142], [0.06725846230983734, -0.00728461891412735, -0.007079371716827154, 0.05435354262590408, -0.09391164779663086, -0.010230698622763157, 0.014564482495188713, 0.0031776963733136654, 0.0069857407361269, -0.10642290115356445, -0.003747603390365839, -0.05364398658275604, -0.07519706338644028, 0.13840094208717346, -0.059688881039619446, 0.056346528232097626, 0.11108581721782684, -0.051672037690877914, 0.01370679959654808, -0.048989858478307724, -0.007534539792686701, -0.11262768507003784, 0.027351325377821922, 0.01736670918762684, 0.07375656068325043, 0.07940421253442764, 0.05476885288953781, -0.06293213367462158, 0.01957479491829872, -0.07450197637081146, 0.004233892075717449, -0.03668212890625, -0.11352904886007309, -0.037031739950180054, -0.08859805017709732, -0.08054795116186142, -0.06257829815149307, -0.028732972219586372, 0.0030801899265497923, 0.06786270439624786, -0.05180518329143524, 0.06584978103637695, 0.004834877327084541, 0.0064759631641209126, -0.04365846887230873, -0.09067121893167496, 0.0818178653717041, 0.04046595096588135, -0.026914119720458984, 0.05019119009375572, -0.031641844660043716, 0.051889996975660324, -0.12394503504037857, -0.09392785280942917, -0.019265195354819298, -0.06721022725105286, -0.10075566917657852, 0.06425122171640396, 0.058698005974292755, -0.09095362573862076, 0.11778997629880905, -0.03445655480027199, 0.06772032380104065, -0.06442440301179886, 0.019037004560232162, 0.04301592335104942, -0.06474515050649643, -0.04242793470621109, 0.009585962630808353, 0.05358538031578064, -0.061957381665706635, -0.03660838305950165, 0.015130999498069286, -0.027824323624372482, -0.011096150614321232, -0.051652390509843826, -0.12522338330745697, 0.09591071307659149, 0.004384973552078009, -0.019633159041404724, 0.044752296060323715, 0.003861104603856802, 0.027137866243720055, 0.03518853336572647, 0.015277822501957417, -0.0450638122856617, 0.06584445387125015, 0.02089625783264637, 0.08990322053432465, -0.11526069790124893, 0.07709745317697525, -0.035051409155130386, 0.040715090930461884, 0.04234996438026428, -0.01880980283021927, -0.0063966731540858746], [-0.010104028508067131, -0.029718240723013878, -0.09808008372783661, 0.05821773782372475, 0.0278262160718441, -0.049051836133003235, 0.001722270273603499, 0.04335639625787735, -0.039630599319934845, -0.1143803521990776, 0.08766388893127441, -0.07988707721233368, -0.08412394672632217, 0.07741490006446838, 0.0628058910369873, 0.012432850897312164, -0.00021549533994402736, 0.012572933919727802, 0.01019689068198204, 0.07202714681625366, -0.02315317653119564, -0.006037982180714607, -0.014832506887614727, -0.05002128705382347, -0.07189010083675385, -0.002663033315911889, 0.04687214642763138, -0.03601585701107979, -0.00018538028234615922, -0.09118612855672836, -0.0009868284687399864, -0.04938700050115585, -0.002333826618269086, -0.061083097010850906, -0.12298471480607986, 0.008253972046077251, 0.02985091507434845, -0.09479102492332458, -0.08940651267766953, -0.04685860499739647, -0.04645386338233948, -0.0400075800716877, -0.02826033905148506, 0.07038936764001846, 0.022695142775774002, 0.006354254670441151, -0.11194934695959091, 0.07243968546390533, -0.051381733268499374, 0.022930758073925972, -0.08520630747079849, -0.030794227495789528, -0.08641742914915085, -0.09479687362909317, 0.0376281701028347, -0.1083817407488823, 0.05778610333800316, -0.0029053566977381706, -0.008414710871875286, 0.07450179010629654, 0.0469016432762146, -0.10759436339139938, 0.08213338255882263, -0.003341824049130082, -0.05096287652850151, 0.01659487374126911, 0.10367888957262039, 0.008806913159787655, -0.00014955198275856674, 0.07972141355276108, -0.07216592133045197, 0.06571989506483078, -0.08356563001871109, 0.03596464917063713, -0.08149847388267517, -0.08760722726583481, -0.07714217901229858, 0.029359120875597, -0.08552626520395279, 0.06401369720697403, 0.020707618445158005, -0.055196359753608704, -0.08362603932619095, -0.10647737979888916, -0.11494173109531403, -0.10058861970901489, -0.052273839712142944, 0.0025993632152676582, -0.05194064602255821, 0.03936976194381714, 0.05666071176528931, 0.006237208377569914, 0.06233753263950348, 0.0014401394873857498, 0.056397221982479095, -0.015928147360682487], [0.004173832945525646, -0.014187987893819809, -0.05172164738178253, 0.03300591558218002, -0.0358085036277771, -0.02548227645456791, -0.01920197904109955, -0.043341271579265594, -0.047703005373477936, 0.04525641351938248, -0.025869298726320267, -0.02022169902920723, 0.059447064995765686, -0.03382991999387741, 0.002836568746715784, 0.040852710604667664, 0.04290764406323433, 0.05280669406056404, -0.04308747872710228, 0.08081291615962982, -0.03330112621188164, 0.032745394855737686, -0.043742693960666656, -0.04317359998822212, -0.052742261439561844, -0.018006639555096626, -0.03156498074531555, -0.002129734493792057, -0.03333533927798271, 0.0466720275580883, 0.002433840651065111, 0.012831159867346287, 0.04919632896780968, -0.027970077469944954, 0.011925223283469677, 0.06312037259340286, -0.03476109728217125, 0.044470880180597305, -0.002571897814050317, 0.013597607612609863, 0.020983364433050156, 0.051234856247901917, -0.02015792950987816, 0.02750535123050213, 0.00837080180644989, -0.05497712641954422, -0.010614336468279362, 0.027666447684168816, 0.039738550782203674, -0.042473673820495605, 0.024782143533229828, 0.010445398278534412, 0.014204914681613445, 0.045306265354156494, -0.006420462857931852, -0.04956039413809776, 0.07895716279745102, 0.05731537193059921, -0.02524583786725998, -0.008473623543977737, 0.020157670602202415, -0.05870220437645912, 0.04756295308470726, 0.0070228190161287785, 0.039971064776182175, 0.002652845811098814, -0.007010705303400755, -0.07072392851114273, -0.013565054163336754, 0.027743682265281677, -0.03999367356300354, -0.000447331607574597, 0.022056829184293747, 0.006431337911635637, 0.004014281090348959, 0.06375350803136826, -0.029668282717466354, -0.012756303884088993, -0.02486334554851055, -0.023316072300076485, 0.024933679029345512, -0.008170858956873417, -0.04831438511610031, 0.023958342149853706, 0.0061126104556024075, -0.03244601935148239, -0.033235128968954086, -0.022673670202493668, 0.04425305500626564, 0.012685888446867466, -0.059708643704652786, -0.0018794899806380272, -0.017894547432661057, 0.013647285290062428, -0.029054606333374977, 0.08636584132909775], [0.08123384416103363, 0.06519041210412979, -0.013687915168702602, 0.08183449506759644, 0.002963816747069359, -0.06674303859472275, 0.09697876870632172, -0.0032286420464515686, -0.016966726630926132, -0.0765264630317688, -0.07785888016223907, 0.06165637448430061, 0.09667065739631653, -0.09996330738067627, -0.09954921901226044, 0.046847593039274216, 0.06847550719976425, -0.07204137742519379, -0.07951268553733826, 0.09542615711688995, 0.05252604931592941, 0.05690474063158035, 0.05242018401622772, -0.07963740825653076, 0.07964622229337692, 0.0392700619995594, 0.041196513921022415, 0.11991079896688461, 0.011706672608852386, 0.09058797359466553, 0.009031635709106922, 0.02005389705300331, -0.05015478655695915, 0.08429016172885895, 0.06576276570558548, 0.006449740845710039, 0.09253266453742981, -0.026871494948863983, 0.04394954442977905, 0.002080341102555394, 0.04362645372748375, -0.06653639674186707, 0.006618367973715067, -0.02168697491288185, -0.061263557523489, -0.0905131846666336, -0.10323793441057205, -0.023965248838067055, 0.03709833323955536, 0.03774632513523102, 0.009409463964402676, 0.06598719209432602, 0.05246168002486229, 0.03550182282924652, 0.10132771730422974, 0.05181590095162392, 0.046089448034763336, -0.08465062081813812, 0.026060866191983223, 0.013149283826351166, 0.042032379657030106, -0.019852066412568092, 0.10379553586244583, 0.025089288130402565, 0.005536564625799656, -0.02800082415342331, -0.0926208347082138, 0.07642513513565063, 0.0007034523878246546, -0.012669750489294529, -0.028792256489396095, 0.06414844840765, 0.03358988091349602, -0.12621550261974335, 0.008301406167447567, 0.0648449957370758, 0.09604091942310333, 0.025983359664678574, 0.017181342467665672, 0.010893851518630981, -0.079560287296772, -0.020370762795209885, 0.04124648869037628, -0.09448525309562683, -0.0653475821018219, -0.10615769028663635, -0.01177099160850048, 0.018492765724658966, -0.08653093129396439, 0.07883580029010773, -0.058413147926330566, -0.0782034769654274, 0.01891508139669895, 0.047149792313575745, 0.035442106425762177, -0.048978354781866074], [0.1005019024014473, 0.06952718645334244, -0.08217819780111313, 0.0007166247814893723, 0.050800830125808716, 0.0024281018413603306, -0.06866246461868286, -0.010249398648738861, 0.017871957272291183, 0.042125310748815536, -0.021248646080493927, -0.027763815596699715, 0.07537224888801575, 0.02728373557329178, -0.05756690353155136, -0.03918353468179703, 0.037355851382017136, 0.09414041042327881, 0.05694621056318283, 0.001171351526863873, 0.02057616040110588, -0.011050269939005375, 0.016968490555882454, 0.08527848869562149, -0.030546482652425766, -0.12209638208150864, -0.018093284219503403, 0.005678481422364712, 0.06949318200349808, -0.08282102644443512, -0.11378371715545654, -0.0689479187130928, -0.04185838624835014, -0.044651348143815994, 0.06881013512611389, 0.10203129798173904, 0.09366181492805481, 0.09355082362890244, -0.008147800341248512, -0.0353727862238884, 0.06757831573486328, -0.09450284391641617, -0.03229302912950516, -0.036724742501974106, -0.1220654547214508, -0.05107056722044945, -0.022165557369589806, 0.046795982867479324, 0.09126985818147659, 0.007005937397480011, 0.034385520964860916, -0.030088672414422035, 0.040891531854867935, -0.05844166502356529, 0.02205784060060978, -0.05807805806398392, -0.03506186977028847, 0.09570198506116867, 0.02921062335371971, 0.02341863140463829, -0.06802498549222946, 0.07872703671455383, 0.09927317500114441, -0.07655113935470581, -0.06750782579183578, -0.1295413076877594, 0.022460363805294037, 0.0008334338781423867, -0.056157633662223816, 0.002104030456393957, 0.03829718381166458, 0.08276648074388504, -0.052463971078395844, -0.09947217255830765, 0.0722183808684349, -0.0029782564379274845, -0.044129982590675354, 0.0928105041384697, 0.07188858836889267, -0.047062456607818604, 0.09438099712133408, 0.04437266290187836, 0.04611368477344513, 0.016188105568289757, 0.048461832106113434, -0.0646689236164093, -0.008469749242067337, -0.06386500597000122, -0.009168025106191635, 0.07864775508642197, -0.02620801143348217, -0.042264144867658615, -0.057264428585767746, -0.12105675786733627, -0.06737638264894485, -0.034310679882764816], [0.006707675289362669, -0.025276800617575645, 0.06721466779708862, -0.037167131900787354, -0.05685838311910629, 0.002141730859875679, -0.06993235647678375, 0.029393520206212997, -0.04712006822228432, 0.07335492223501205, -0.006577720399945974, 0.012376976199448109, -0.04345152527093887, 0.002917524194344878, -0.028536267578601837, -0.06239793822169304, -0.05716817453503609, 0.08812898397445679, 0.014711751602590084, -0.023473622277379036, -0.053629450500011444, -0.06982483714818954, 0.024884648621082306, 0.07963423430919647, 0.07396572828292847, -0.03674769029021263, 0.08820697665214539, 0.0949493795633316, 0.0523536279797554, -0.03281761333346367, -0.041943613439798355, 0.016125984489917755, 0.050368960946798325, 0.04610492289066315, 0.027477869763970375, 0.10442057996988297, 0.05226531997323036, 0.08418469876050949, -0.02825777418911457, -0.003710504388436675, 0.07455536723136902, 0.017277279868721962, -0.02897767350077629, -0.019703051075339317, -0.02788214199244976, 0.061136092990636826, -0.009356457740068436, 0.005997651722282171, -0.004350949078798294, 0.09207464754581451, -0.01910487562417984, -0.003948557656258345, -0.07101668417453766, -0.06035231053829193, 0.0223414096981287, -0.06181814894080162, 0.07049586623907089, -0.006451746448874474, -0.026338787749409676, -0.08079259842634201, 0.003735871519893408, 0.08748314529657364, -0.07274258881807327, -0.006317648570984602, 0.03929411247372627, -0.10715421289205551, -0.03345879167318344, -0.06430433690547943, -0.05296972021460533, 0.0542323924601078, -0.018550530076026917, 0.05623583123087883, 0.022481173276901245, 0.0009204215020872653, 0.018030351027846336, 0.026845835149288177, 0.06968038529157639, 0.0972273051738739, -0.03551686927676201, 0.044093161821365356, -0.055196840316057205, -0.07079989463090897, 0.0013145201373845339, 0.021397186443209648, 0.015561698004603386, -0.10411324352025986, 0.016266612336039543, -0.1092633306980133, 0.09953264892101288, 0.03831364959478378, 0.10406172275543213, -0.024361908435821533, -0.046089351177215576, -0.06642371416091919, 0.06648190319538116, 0.03878441080451012], [0.08267167955636978, -0.04298524931073189, -0.0227559432387352, 0.045214228332042694, 0.04251740127801895, -0.02175114117562771, -0.07749588787555695, 0.036908071488142014, -0.06397268176078796, -0.0494772233068943, 0.07793182879686356, -0.023242471739649773, -0.10800763219594955, 0.09659160673618317, 0.05275823548436165, 0.02713153138756752, 0.042915601283311844, -0.10695812851190567, -0.10450001806020737, 0.012831815518438816, -0.09744136780500412, -0.08636775612831116, -0.0858236625790596, -0.04700920730829239, 0.04210200905799866, -0.029802145436406136, 0.019754519686102867, -0.0583808533847332, 0.014093535020947456, 0.07518593966960907, 0.026345524936914444, -0.10717178136110306, -0.05921519920229912, -0.12087994068861008, 0.008521397598087788, 0.01772947423160076, -0.02487914450466633, -0.07470984756946564, -0.004582100547850132, -0.04157889261841774, 0.04358362779021263, -0.004153553396463394, -0.06038201227784157, 0.00811232440173626, 0.0059445989318192005, 0.055979836732149124, -0.04558408260345459, -0.0153923649340868, -0.08474156260490417, -0.12113817781209946, -0.08362863957881927, -0.0014093086356297135, -0.018465369939804077, 0.0041931322775781155, -0.07021821290254593, 0.019551021978259087, -0.06175190210342407, 0.08528146892786026, -0.01961745135486126, -0.08860055357217789, -0.0009633463923819363, -0.10380057990550995, -0.08591444790363312, 0.043619588017463684, 0.005581832490861416, 0.08365701884031296, -0.04360121861100197, -0.029589595273137093, 0.08373230695724487, 0.11347932368516922, -0.07700792700052261, -0.11423173546791077, 0.06742110103368759, 0.09182220697402954, 0.04816238582134247, 0.08168356120586395, -0.07640127837657928, -0.060613323003053665, 0.04033544287085533, 0.005640274379402399, -0.06530320644378662, 0.016353745013475418, 0.07287613302469254, 0.05833880603313446, 0.04087236523628235, 0.10181351751089096, -0.00035407469840720296, 0.08207270503044128, -0.0350966677069664, -0.03271244838833809, -0.07897422462701797, -0.07896117866039276, -0.08031083643436432, -0.023466261103749275, 0.06597485393285751, -0.05991119146347046], [-0.005865802057087421, -0.061567556113004684, -0.02663719654083252, 0.06456811726093292, 0.010787272825837135, 0.029824277386069298, 0.05167989432811737, 0.07021058350801468, -0.09939655661582947, 0.10915467143058777, 0.03883075341582298, 0.053963445127010345, 0.009820621460676193, -0.015423103235661983, -0.05130460113286972, -0.0450432114303112, 0.07127999514341354, -0.04802185297012329, -0.07201492786407471, -0.030653774738311768, -0.00863633118569851, 0.08879408985376358, 0.04090198501944542, 0.03773348033428192, -0.019144542515277863, -0.05610021948814392, -0.00628979317843914, 0.004566496703773737, 0.06179993227124214, -0.08503703773021698, 0.016386572271585464, 0.016731303185224533, 0.011008035391569138, 0.0011471848702058196, -0.014126396737992764, 0.05583938956260681, -0.09636668115854263, 0.0393759086728096, -0.04303198680281639, -0.06759554892778397, 0.04752038046717644, -0.02957344427704811, 0.055075183510780334, 0.04362065717577934, 0.06198997423052788, 0.03218089044094086, -0.010863891802728176, 0.07783177495002747, -0.0006433464004658163, -0.017289992421865463, -0.019405361264944077, 0.054102908819913864, -0.07546226680278778, -0.020106282085180283, -0.014580149203538895, 0.09000276774168015, 0.020847272127866745, -0.06167445704340935, -0.01353929378092289, -0.1041688621044159, -0.03641568124294281, -0.006518618203699589, -0.03211820870637894, 0.004433099180459976, 0.10085678845643997, 0.07169707864522934, 0.07702544331550598, 0.031437043100595474, 0.029422514140605927, -0.013574269600212574, 0.04529936611652374, 0.003919774200767279, 0.00815572775900364, 0.06558774411678314, -0.013807659037411213, 0.024636365473270416, 0.0009465414914302528, -0.09623488038778305, -0.0633348897099495, -0.028455683961510658, -0.04049209505319595, -0.1008714884519577, 0.008245020173490047, 0.04022691026329994, 0.0819447711110115, 0.08436111360788345, 0.06389298290014267, 0.11094138771295547, -0.011378871276974678, 0.0032653789967298508, 0.029310131445527077, -0.018906977027654648, -0.01295528095215559, 0.0032468209974467754, 0.017435183748602867, -0.03371386602520943], [-0.05262429267168045, 0.03941265866160393, 0.08729153126478195, 0.06033683568239212, 0.0034081635531038046, 0.0780024528503418, -0.09511402249336243, 0.07288115471601486, 0.09839261323213577, 0.03585256636142731, -0.1066611036658287, 0.09295717626810074, 0.03243698179721832, 0.005259972531348467, 0.044481031596660614, 0.01002453826367855, 0.044288020581007004, -0.08780872821807861, 0.04025417938828468, 0.0024683501105755568, 0.05156826227903366, -0.036761440336704254, -0.06612378358840942, -0.012814491987228394, 0.01266053132712841, 0.049578141421079636, -0.07728473842144012, 0.1022406667470932, -0.004950800444930792, 0.03098536841571331, -0.10607461631298065, -0.054552800953388214, 0.004084871616214514, 0.08916672319173813, 0.10648015141487122, 0.05732988566160202, -0.06958220154047012, -0.046787161380052567, -0.08457887172698975, 0.08637035638093948, 0.003992385696619749, 0.03931993618607521, 0.02093290351331234, 0.09744775295257568, 0.042755644768476486, -0.015397584065794945, -0.0792885273694992, -0.08819594234228134, -0.0750422403216362, 0.019335605204105377, -0.05791301652789116, -0.09607084840536118, -0.0756477564573288, -0.05525222048163414, 0.06253648549318314, -0.08521304279565811, 0.08161415159702301, -0.044972024857997894, 0.0461006723344326, 0.0991133600473404, -0.0622054822742939, -0.041654571890830994, 0.030984418466687202, 0.037268634885549545, -0.07151150703430176, 0.07224645465612411, 0.08720780164003372, 0.013387894257903099, 0.08796167373657227, -0.041394781321287155, 0.007251304108649492, -0.05575289577245712, -0.030892647802829742, 0.03181634470820427, -0.017265744507312775, -0.04245758801698685, 0.07865098118782043, -0.06293521821498871, -0.013099721632897854, -0.03616765886545181, -0.06690894067287445, 0.004519967827945948, -0.049333736300468445, 0.08918310701847076, -0.09920945763587952, 0.03294965997338295, 0.03456198796629906, 0.07579010725021362, 0.07499442994594574, -0.04245173558592796, 0.02064639888703823, -0.03875933960080147, -0.0518847294151783, -0.05305420979857445, 0.04870443418622017, -0.014743327163159847], [-0.009609900414943695, 0.04151162505149841, -0.08203145116567612, -0.028401000425219536, 0.04107678309082985, -0.07295706868171692, -0.013254580087959766, 0.02977607399225235, -0.056747980415821075, -0.05383012816309929, 0.021409202367067337, -0.060038767755031586, 0.020601391792297363, 0.06641064584255219, 0.04784673452377319, 0.07049315422773361, 0.008174881339073181, -0.006573878228664398, -0.07259739190340042, 0.09481492638587952, 0.00020245934138074517, 0.01905798725783825, 0.08762849122285843, 0.06725037842988968, 0.08014993369579315, -0.04777572304010391, 0.06623031198978424, 0.08433197438716888, 0.029977809637784958, 0.09679913520812988, 0.02286268025636673, 0.04560326039791107, -0.021659553050994873, 0.11648821085691452, 0.07852595299482346, 0.0645274892449379, 0.05238945037126541, -0.06720245629549026, -0.0860048159956932, 0.05313961207866669, 0.03991566598415375, -0.07765654474496841, 0.09865691512823105, 0.06842021644115448, -0.07772935181856155, 0.0497945174574852, 0.07747430354356766, -0.0671195313334465, 0.03565819188952446, 0.1000959649682045, -0.02988908626139164, 0.03977904096245766, 0.06865495443344116, 0.10633508116006851, -0.024955814704298973, 0.042711302638053894, 0.020267371088266373, -0.01943235844373703, 0.10928303748369217, -0.013686001300811768, -0.07603172212839127, 0.07645810395479202, -0.033694181591272354, -0.04250429943203926, -0.01885511539876461, -0.027711840346455574, -0.049645066261291504, 0.07649459689855576, -0.028220323845744133, -0.03512606769800186, -0.07377415150403976, 0.0017710962565615773, 0.06160588935017586, -0.026235191151499748, 0.013240114785730839, 0.017785569652915, -0.07678625732660294, 0.018996430560946465, 0.01125542726367712, 0.07318993657827377, 0.04124685004353523, 0.04473935440182686, 0.045385804027318954, -0.03449658304452896, 0.0799773558974266, -0.002485868986696005, 0.024454420432448387, 0.0426676869392395, -0.05934836342930794, 0.00414642971009016, 0.08901627361774445, -0.03393726050853729, 0.05217111483216286, 0.08088289201259613, 0.05004791542887688, -0.04084751754999161], [-0.0007456630119122565, 0.00826812069863081, 0.014167595654726028, 0.0719640925526619, 0.011055836454033852, -0.03800667077302933, -0.03718288987874985, -0.0695076435804367, 0.11621904373168945, 0.009650676511228085, 0.013121819123625755, -0.04976188391447067, 0.01167283020913601, 0.04038208723068237, 0.01268224697560072, -0.04648328572511673, 0.10787292569875717, -0.050267502665519714, 0.006177659612149, -0.04714680090546608, 0.01845667138695717, -0.025203745812177658, -0.05550496280193329, 0.0053720478899776936, 0.03739907965064049, 0.0013415580615401268, 0.06135845184326172, 0.027461184188723564, 0.0055631850846111774, -0.015839003026485443, 0.025095630437135696, 0.026440424844622612, 0.007298912853002548, -0.08068593591451645, -0.05049264430999756, -0.08020171523094177, -0.08234421163797379, -0.021507641300559044, 0.028220172971487045, -0.04312686622142792, 0.03680101037025452, 0.04160376638174057, -0.02046223171055317, 0.0962921530008316, -0.017225146293640137, -0.02145133726298809, 0.07775049656629562, 0.004433496855199337, -0.01198531873524189, -0.11128977686166763, -0.06962992995977402, 0.022603735327720642, -0.04349441081285477, -0.016770116984844208, -0.041322432458400726, -0.059085212647914886, -0.09733276069164276, 0.11138075590133667, 0.01654670760035515, -0.09811863303184509, 0.08480275422334671, -0.04478081688284874, -0.01297060027718544, -0.04877964407205582, -0.04815536364912987, -0.012467538937926292, 0.02177317999303341, -0.0967351570725441, 0.04641968384385109, 0.04014743119478226, -0.04550023376941681, -0.029802629724144936, -0.005749656818807125, 0.039848119020462036, -0.02649921551346779, -0.029176803305745125, 0.023009026423096657, 0.04762653261423111, -0.017694756388664246, -0.07637961953878403, -0.034416571259498596, -0.020153841003775597, 0.06101122498512268, -0.08010873943567276, -0.05175900459289551, 0.05979583412408829, 0.057599760591983795, 0.04440857842564583, 0.06457023322582245, -0.10232267528772354, -0.048022303730249405, -0.04732465371489525, -0.03494100645184517, -0.006467109080404043, -0.08141545206308365, 0.017826875671744347], [0.062240395694971085, -0.02733645960688591, 0.0828387588262558, 0.021827876567840576, -0.09959624707698822, 0.014752442948520184, -0.06695547699928284, -0.04671406000852585, -0.09941086173057556, -0.014971740543842316, 0.047871194779872894, 0.01430375687777996, 0.06074508652091026, -0.05640174821019173, 0.08015316724777222, 0.005502157844603062, -0.060498252511024475, -0.037427134811878204, -0.04905945435166359, 0.010591818019747734, -0.006793518550693989, 0.008227763697504997, 0.02011028677225113, -0.01614750362932682, 0.06870710849761963, 0.026911422610282898, -0.040402013808488846, -0.006296043749898672, 0.06954853981733322, 0.06001061946153641, -0.060851410031318665, 0.03812006115913391, -0.028943872079253197, 0.02582596242427826, -0.0012268604477867484, 0.017884818837046623, 0.01316694263368845, -0.06273747980594635, -0.03999319300055504, 0.06372471153736115, -0.07701370865106583, 0.03601573780179024, -0.04977523908019066, 0.01932056061923504, -0.03741134703159332, 0.06571019440889359, 0.08743229508399963, -0.0380886010825634, -0.062171339988708496, -0.06722819805145264, 0.07794851809740067, -0.08723700046539307, 0.07376579940319061, -0.06744100898504257, -0.021519524976611137, 0.03307662159204483, -0.09089914709329605, -0.0016644411953166127, -0.07557918131351471, 0.04527214542031288, -0.05195841193199158, -0.07410445809364319, -0.06713560968637466, 0.06307820975780487, 0.041400086134672165, 0.0852527841925621, 0.014763531275093555, -0.06046067550778389, 0.0775889977812767, -0.017308909446001053, 0.012721330858767033, -0.04528597369790077, -0.04535382613539696, -0.04250628873705864, 0.010036855936050415, 0.038626037538051605, -0.08898089826107025, 0.05055990070104599, -0.07659876346588135, 0.004840330220758915, 0.07451537996530533, 0.0392826646566391, 0.04803198575973511, -0.030628647655248642, 0.060662902891635895, 0.08811032772064209, -0.07663033902645111, -0.0572238452732563, 0.05418916791677475, -0.010044969618320465, 0.05787736549973488, 0.006751065608114004, -0.08035078644752502, 0.016505395993590355, -0.052122440189123154, -0.04885748401284218], [0.04333601891994476, -0.048797935247421265, -0.017069006338715553, -0.06907107681035995, 0.07607455551624298, 0.027661023661494255, -0.00530257448554039, -0.08422749489545822, 0.05176498740911484, 0.0786944329738617, -0.09468904882669449, 0.0024016033858060837, 0.07099758833646774, -0.09453475475311279, 0.02729555405676365, -0.05061223730444908, -0.0002979637065436691, -0.08332312852144241, 0.005172311328351498, -0.03840078413486481, 0.05154721811413765, -0.0732504352927208, 0.014764669351279736, -0.0058509474620223045, -0.029069123789668083, -0.034509509801864624, 0.0387740433216095, 0.023281216621398926, -0.05548836290836334, 0.049101635813713074, -0.056623682379722595, 0.03257790952920914, -0.024881431832909584, 0.004082394763827324, 0.006526955869048834, -0.00593448244035244, -0.04409907013177872, 0.052468441426754, -0.02253985032439232, 0.030975742265582085, -0.05304624140262604, -0.013419423252344131, 0.06791005283594131, -0.03013068251311779, 0.022269831970334053, -0.023031074553728104, -0.013754368759691715, -0.041146304458379745, 0.013947316445410252, 0.07364410161972046, -0.03173504397273064, -0.0014039190718904138, 0.029033342376351357, 0.10559722036123276, -0.03297723829746246, 0.052551232278347015, -0.016249697655439377, 0.04149860888719559, 0.014528326690196991, 0.0008054601494222879, -0.03785727918148041, 0.04308916628360748, -0.012208899483084679, 0.010171191766858101, 0.04264218360185623, -0.012007427401840687, -0.03493359312415123, 0.05954805761575699, 0.024107391014695168, -0.02683337777853012, -0.02370580844581127, -0.04527508467435837, 0.003413410624489188, -0.042705684900283813, -0.00983070582151413, 0.023850664496421814, 0.06284927576780319, 0.0459400899708271, -0.015741193667054176, 0.06141817569732666, -0.010488993488252163, 0.0343291312456131, 0.02063748799264431, 0.06322675198316574, 0.041157208383083344, 0.03718413785099983, -0.029102221131324768, 0.03411053121089935, -0.04309213161468506, 0.07716786861419678, -0.00700587360188365, 0.013013815507292747, -0.032084546983242035, -0.04908972978591919, -0.040164485573768616, 0.04847340285778046], [0.05240674689412117, 0.023879338055849075, 0.012340621091425419, 0.03546227142214775, 0.08436528593301773, 0.02310764230787754, -0.04397386685013771, 0.030306335538625717, 0.015894010663032532, -0.07154467701911926, 0.017425596714019775, 0.0416211262345314, 0.09249657392501831, 0.021586060523986816, 0.06674337387084961, -0.00225373194552958, -0.009732704609632492, 0.01831935904920101, 0.00924474373459816, 0.10452251136302948, -0.03174911066889763, 0.05297749489545822, -0.004402799066156149, 0.0933413952589035, -0.05951008200645447, 0.024007553234696388, 0.0032852990552783012, 0.0722227469086647, -0.08936846256256104, -0.004645883571356535, -0.03048165887594223, 0.034280478954315186, 0.007342081051319838, 0.10151632875204086, -0.020310131832957268, 0.031452398747205734, 0.043569937348365784, 0.0075052883476018906, -0.007819127291440964, -0.03300253674387932, 0.05822460725903511, -0.017585551366209984, 0.025920359417796135, 0.06910208612680435, 0.027193687856197357, 0.04057396948337555, -0.07972782850265503, 0.00898254755884409, 0.05312405526638031, 0.0929059386253357, 0.005093489773571491, -0.0585143081843853, 0.07114982604980469, 0.10061086714267731, 0.019742971286177635, -0.0831824243068695, -0.02930130437016487, -0.05391808599233627, 0.06382188200950623, 0.004539082292467356, -0.018500033766031265, 0.09789744019508362, 0.03599253296852112, -0.05537103861570358, 0.02940770797431469, -0.10795681178569794, -0.013695845380425453, 0.04315081611275673, 0.07497470080852509, -0.0778157189488411, -0.021405110135674477, 0.09062258154153824, 0.010852127335965633, 0.04750339314341545, -0.00380157888866961, 0.002876745071262121, -0.02857930399477482, 0.02903088368475437, 0.029024142771959305, 0.0488206185400486, 0.067580446600914, 0.0576232373714447, -0.04965169355273247, 0.0186290442943573, -0.014091131277382374, -0.02841010130941868, -0.10136637091636658, -0.03216308355331421, -0.08626824617385864, -0.007385551929473877, 0.0010207887971773744, -0.030224749818444252, 0.0863390788435936, 0.06129743903875351, 0.06823860108852386, -0.001382643822580576], [0.10368899255990982, 0.10122082382440567, -0.05019738897681236, -0.022310525178909302, -0.03233889490365982, 0.10195530951023102, 0.09207335859537125, -0.059744786471128464, 0.0724419429898262, 0.012522704899311066, 0.03235619515180588, -0.01312015950679779, -0.0020848349668085575, -0.05156012251973152, 0.0265816580504179, 0.08742684870958328, -0.023508820682764053, 0.0769183412194252, -0.06528045237064362, 0.00849840510636568, 0.08563319593667984, -0.050405751913785934, -0.07325967401266098, -0.044691380113363266, 0.04258264601230621, 0.07357978075742722, -0.042629241943359375, 0.0626043975353241, -0.07190939784049988, -0.0233750082552433, 0.015388108789920807, -0.04825887084007263, 0.05117693543434143, -0.050792012363672256, -0.000671686080750078, 0.041916340589523315, 0.11750990897417068, -0.07615906745195389, -0.0890485942363739, 0.08977823704481125, 0.02123941481113434, -0.06065598502755165, 0.0841936320066452, 0.07886553555727005, 0.009688764810562134, -0.010853230953216553, 0.06063437834382057, 0.1018434688448906, -0.03176708146929741, 0.07867462188005447, 0.011627702042460442, -0.08246342837810516, 0.10035760700702667, 0.013438096269965172, -0.04256703704595566, 0.05114561691880226, -0.07651062309741974, 0.04000529646873474, 0.004497362766414881, -0.010437353514134884, 0.000883248751051724, 0.07825858145952225, -0.023987790569663048, 0.09254252910614014, -0.07336769998073578, -0.07619974762201309, -0.07575802505016327, 0.11088461428880692, 0.07007075101137161, -0.12158834934234619, -0.00625299708917737, 0.08155865222215652, -0.0666893720626831, 0.011933163739740849, -0.0285143181681633, -0.06897524744272232, 0.08740053325891495, 0.03669640049338341, -0.051669567823410034, -0.06545548886060715, -0.02228468470275402, -0.039122141897678375, 0.0371541753411293, -0.00601715873926878, 0.03368803486227989, -0.07295936346054077, -0.08019044995307922, -0.10490836203098297, -0.09188222885131836, -0.01446017436683178, 0.017726507037878036, -0.04979698359966278, -0.07818230986595154, 0.01202507596462965, 0.0909854918718338, -0.019160116091370583], [-0.07079741358757019, 0.04597334936261177, -0.011527917347848415, -0.003606137353926897, -0.10592910647392273, 0.06776847690343857, -0.10187514871358871, -0.054273732006549835, -0.046510666608810425, 0.04732969030737877, -0.0627061054110527, -0.08168689906597137, 0.05882181227207184, -0.04276132956147194, 0.03643583133816719, 0.04670635983347893, 0.09384574741125107, 0.05850658193230629, -0.048685964196920395, 0.0668191984295845, 0.057613641023635864, -0.04279602691531181, 0.003199094207957387, 0.006854129023849964, -0.03344376012682915, -0.013423824682831764, -0.04076852276921272, -0.016729308292269707, -0.07822055369615555, -0.06541670113801956, 0.024873100221157074, -0.07073357701301575, 0.018137415871024132, 0.017495358362793922, -0.004879283718764782, -0.05995767191052437, 0.07310561090707779, -0.06632237136363983, 0.027433006092905998, 0.0820036306977272, 0.086356520652771, -0.002721503609791398, -0.10310479998588562, -0.02060319483280182, 0.058442629873752594, -0.11080808192491531, -0.0613008514046669, 0.009971643798053265, -0.07463682442903519, -0.017050471156835556, -0.02465365268290043, 0.055672649294137955, -0.05785645917057991, -0.1006101667881012, -0.10035901516675949, -0.10432963818311691, -0.048036035150289536, 0.03183414414525032, 0.00934603065252304, -0.11242953687906265, -0.04148942977190018, -0.05304278805851936, -0.0028671687468886375, -0.02473721094429493, -0.04725842550396919, 0.0697166696190834, -0.011039859615266323, -0.07733170688152313, 0.05617035925388336, 0.09832172840833664, 0.08389516919851303, -0.06884497404098511, -0.09463310986757278, 0.1260044276714325, -0.05721508711576462, -0.060894425958395004, -0.03127164766192436, -0.07558887451887131, -0.0006870470242574811, -0.03672553598880768, 0.07682540267705917, -0.0035210319329053164, -0.09689810127019882, 0.05147712305188179, -0.03391096368432045, 0.0010698024416342378, -0.10003397613763809, -0.005139651708304882, 0.07789086550474167, -0.08384041488170624, -0.06310930103063583, -0.03465734049677849, -0.10660386085510254, -0.034492310136556625, -0.09779289364814758, -0.008972114883363247], [-0.00016881940246094018, -0.13885614275932312, -0.0944448783993721, 0.057906679809093475, 0.0031856221612542868, -0.06540781259536743, 0.003818150609731674, -0.10879353433847427, -0.05878506228327751, -0.08435684442520142, 0.01837623119354248, -0.05032746493816376, 0.05206659808754921, -0.10667973756790161, -0.025277353823184967, -0.05151531845331192, -0.06636788696050644, 0.04176781326532364, -0.048314955085515976, 0.037823636084795, -0.07437484711408615, -0.003118201158940792, -0.09671944379806519, -0.009766003116965294, -0.023483404889702797, -0.07023660093545914, 0.037095025181770325, 0.039639219641685486, -0.04464421421289444, -0.07260497659444809, -0.09987594932317734, 0.03823317214846611, 0.011667695827782154, -0.07148440927267075, 0.019176574423909187, -0.015054015442728996, -0.0009982624324038625, 0.04830503091216087, 0.04924777150154114, -0.07565834373235703, 0.013270999304950237, -0.05662134662270546, -0.1029411032795906, -0.03445851802825928, 0.032033830881118774, -0.025449248030781746, 0.04010361060500145, -0.027238400653004646, -0.10413963347673416, -0.07622972130775452, -0.03894985467195511, 0.029002375900745392, -0.009578472003340721, -0.005991295911371708, 0.005370695143938065, -0.021037230268120766, 0.0350705087184906, 0.06619612872600555, -0.100107342004776, -0.05662168934941292, -0.020475907251238823, -0.024166032671928406, -0.02109505608677864, -0.11962073296308517, 0.01064575370401144, -0.05685925856232643, 0.007515289355069399, -0.11309367418289185, 0.016825834289193153, -0.06410624831914902, -0.05441786348819733, -0.002494585234671831, -0.07382344454526901, 0.0550345778465271, 0.032006654888391495, -0.0920451283454895, -0.01980516128242016, -0.04698243737220764, -0.046300988644361496, -0.007175390142947435, -0.07987727224826813, 0.03160392493009567, -0.05909324809908867, -0.093718022108078, -0.07796911895275116, -0.010447210632264614, 0.0018023074371740222, 0.01223878562450409, -0.08356907963752747, 0.0062537710182368755, -0.09620103985071182, -0.08803108334541321, -0.015174738131463528, 0.00972759909927845, -0.06874626874923706, -0.09934598207473755], [0.08734748512506485, -0.1134619265794754, -0.04312220960855484, 0.02305370382964611, -0.023068662732839584, 0.05178162455558777, -0.09398294240236282, -0.02756418287754059, -0.042207084596157074, -0.10601721704006195, 0.04492522403597832, -0.07816534489393234, 0.06043288856744766, 0.06515336781740189, 0.08699867874383926, -0.05390143021941185, 0.022638356313109398, -0.09040805697441101, -0.0011702073970809579, -0.03998500481247902, -0.004038863815367222, -0.014728316105902195, 0.01555988285690546, 0.05572520196437836, 0.06968419253826141, -0.0340409129858017, 0.012356269173324108, -0.03311580792069435, -0.08093234896659851, -0.06999921053647995, -0.01785420812666416, -0.0640091821551323, 0.081907719373703, -0.01780843921005726, -0.046125054359436035, 0.042349886149168015, -0.018296459689736366, -0.07288661599159241, 0.056457243859767914, 0.08324272930622101, 0.08460239320993423, -0.04539363831281662, -0.024210693314671516, -0.015146837569773197, -0.0718686580657959, 0.05575990676879883, -0.023267295211553574, -0.051661167293787, -0.1156444400548935, -0.06018533557653427, 0.000336100667482242, -0.04828215762972832, -0.06469950824975967, -0.03897850960493088, -0.04412760213017464, 0.009035841561853886, -0.12143781036138535, 0.06647168844938278, 0.05725059658288956, -0.012428323738276958, 0.11567747592926025, -0.11497629433870316, 0.03950326144695282, 0.04214012250304222, -0.04112693667411804, -0.03707383945584297, 0.08027094602584839, -0.10167625546455383, -0.05962143465876579, 0.11934354901313782, -0.017558934167027473, -0.0601702556014061, 0.06886940449476242, 0.009460379369556904, 0.07381262630224228, 0.0318521149456501, -0.09981318563222885, 0.06978536397218704, 0.05300555005669594, 0.016605006530880928, -0.05228017270565033, -0.09743811935186386, -0.10074738413095474, -0.0004306477203499526, 0.0056473021395504475, 0.0005293907597661018, -0.04580044373869896, -0.010367823764681816, -0.061123866587877274, -0.022697824984788895, -0.07595174014568329, -0.07432374358177185, 0.07306607067584991, -0.09581863880157471, 0.07023288309574127, 0.023000409826636314], [0.06759358197450638, 0.0533679835498333, 0.050794817507267, 0.0876501277089119, -0.060910336673259735, 0.054636236280202866, 0.030738692730665207, 0.0381210558116436, 0.04505634307861328, -0.03986194357275963, 0.024076739326119423, -0.05151316151022911, -0.00190205208491534, 0.0348137728869915, 0.07007073611021042, 0.03128775209188461, 0.12468519061803818, -0.0348387211561203, -0.033669061958789825, -0.05809541791677475, 0.07709550112485886, -0.0193280428647995, 0.06970002502202988, -0.035301219671964645, -0.08875856548547745, 0.07558158785104752, -0.08010999113321304, 0.046309079974889755, -0.03378668799996376, 0.06131211668252945, -0.006672069430351257, -0.08680716156959534, 0.06855788826942444, -0.08798922598361969, -0.02637583389878273, 0.005712483078241348, -0.08652956038713455, -0.051509857177734375, 0.0702865794301033, -0.023188456892967224, -0.025488771498203278, -0.06750140339136124, 0.05182944983243942, -0.07124838978052139, 0.011245041154325008, 0.05216645449399948, -0.06950698792934418, 0.044521819800138474, -0.05538437142968178, -0.11465589702129364, 0.005225407425314188, 0.01648019626736641, -0.06653095781803131, -0.06972313672304153, -0.08172062039375305, 0.004633783362805843, -0.11906853318214417, 0.04466822370886803, -0.04644604027271271, 0.019390294328331947, 0.08696065843105316, -0.10847542434930801, 0.03195583075284958, 0.01698991097509861, -0.007059775292873383, 0.024319972842931747, -0.04011328145861626, -0.028787754476070404, 0.03910818323493004, -0.030356604605913162, -0.0671004056930542, -0.011661101132631302, -0.0536360964179039, 0.06967155635356903, -0.035265736281871796, -0.068098284304142, -0.11207561939954758, 0.021057352423667908, -0.027564844116568565, -0.06701024621725082, -0.008009269833564758, 0.017168095335364342, -0.02754337713122368, -0.04011731594800949, 0.06760860234498978, 0.012321066111326218, 0.049739036709070206, 0.06580659002065659, -0.08967548608779907, -0.1122257336974144, -0.04378996416926384, 0.02262072078883648, -0.06975666433572769, -0.018154790624976158, 0.01700681820511818, -0.0691823810338974], [-0.02336161956191063, -0.0005817865603603423, -0.07641519606113434, -0.02869272790849209, 0.05024503171443939, 0.0003346810699440539, -0.031131360679864883, -0.029316693544387817, -0.05865700915455818, -0.01449272595345974, -0.027544423937797546, -0.016177374869585037, -0.012283656746149063, -0.060437608510255814, -0.013251928612589836, -0.01260002888739109, 0.001739745493978262, 0.032261043787002563, 0.015157709829509258, 0.059179168194532394, -0.019383370876312256, 0.04246364161372185, -0.015861980617046356, 0.026652982458472252, -0.002219691639766097, -0.004734077490866184, 0.0010548185091465712, -0.0032187949400395155, -0.010923600755631924, -0.017699923366308212, -0.02016124129295349, -0.0061278752982616425, -0.029072321951389313, -0.01261063665151596, -0.0035567244049161673, 0.07558022439479828, -0.010611644014716148, -0.010533543303608894, -0.020463844761252403, 0.041019849479198456, -0.03137374296784401, -0.03442535549402237, -0.0424187108874321, -0.016250889748334885, -0.024799181148409843, 0.0012249513529241085, -0.005824776366353035, -0.00401014368981123, 0.0017484741983935237, 0.022811854258179665, 0.042798880487680435, -0.011263785883784294, -0.003076577093452215, 0.033837947994470596, -0.011963874101638794, -0.011860702186822891, -0.007284965366125107, 0.03379173204302788, -0.014933420345187187, -0.008629593066871166, -0.002297233557328582, 0.01821327395737171, 0.04453860968351364, -0.058388832956552505, 0.0009384991135448217, -0.03391111269593239, -0.02407931722700596, 0.0347321443259716, 0.02841622568666935, 0.0008588262717239559, -0.027236035093665123, -0.005498623009771109, 0.01314624585211277, -0.031128674745559692, -0.013764618895947933, 0.05713853985071182, -0.0008858492947183549, -0.024167710915207863, -0.014256568625569344, 0.01781894825398922, 0.03087206929922104, 0.02951434627175331, 0.05292729660868645, 6.228515121620148e-05, 0.009571789763867855, -0.033167943358421326, 0.053465358912944794, -0.023928508162498474, 0.038576457649469376, -0.06015932187438011, -0.04793615639209747, -0.0017998693510890007, -0.04709721356630325, -0.02853964827954769, 0.02033330872654915, 0.06971970945596695], [-0.09034819900989532, 0.06048062816262245, 0.05040162801742554, 0.05033073201775551, 0.042607009410858154, -0.03630879893898964, 0.07222042232751846, 0.07960889488458633, 0.026605846360325813, 0.07427863776683807, -0.024212663993239403, 0.05044931545853615, 0.04568250849843025, 0.08327298611402512, 0.009763555601239204, 0.09115073084831238, -0.06713605672121048, 0.03496021404862404, -0.02790580317378044, -0.09310109168291092, 0.08127673715353012, -0.02242298051714897, -0.042389340698719025, 0.0030107193160802126, -0.04863009601831436, -0.06993666291236877, -0.048721667379140854, -0.02100963704288006, -0.025448648259043694, -0.09035981446504593, -0.06393823027610779, 0.03472734987735748, 0.007330707274377346, -0.09440112859010696, 0.0570402555167675, 0.005915183108299971, -0.10029269754886627, 0.0380963459610939, 0.03921116143465042, -0.08191326260566711, 0.03063536249101162, 0.031897276639938354, 0.07892452925443649, 0.027987856417894363, 0.011638679541647434, 0.09779714047908783, 0.026988307014107704, 0.04378357157111168, -0.03604234755039215, -0.08389487117528915, 0.08227669447660446, 0.007639999035745859, 0.05919256433844566, 0.0362432561814785, 0.006587193813174963, -0.07047989219427109, 0.05047595500946045, 0.07095938920974731, 0.031040696427226067, 0.06309854239225388, -0.070771224796772, -0.06921999156475067, -0.04063967987895012, 0.04121122509241104, -0.006677137687802315, -0.029938366264104843, 0.05895110219717026, -0.05328657850623131, 0.06627672165632248, 0.07263396680355072, 0.02664928138256073, 0.03331876918673515, 0.023809675127267838, -0.03903230279684067, 0.02989310584962368, -0.07834449410438538, -0.0345541313290596, -0.028004392981529236, -0.05833993852138519, -0.039237044751644135, 0.020481418818235397, -0.09290853887796402, -0.06185891851782799, -0.03672818839550018, -0.042642638087272644, 0.05962000787258148, 0.01805300824344158, 0.048324014991521835, -0.006447577849030495, 0.06901442259550095, 0.0230629350990057, 0.04914639890193939, -0.03232860937714577, -0.005445219110697508, 0.06365016102790833, -0.02238762006163597], [0.047092340886592865, 0.07843203097581863, -0.08919095993041992, -0.08242038637399673, -0.01077105663716793, -0.011662392877042294, -0.014091527089476585, 0.05918974056839943, 0.06208464130759239, 0.03704795613884926, -0.004798718262463808, -0.06766506284475327, 0.10722547769546509, -0.06499803066253662, -0.026013564318418503, 0.09005459398031235, -0.038245681673288345, -0.06133536994457245, 0.10608629882335663, 0.05733723193407059, -0.019796350970864296, 0.024157574400305748, -0.01591145619750023, -0.014799732714891434, 0.011673259548842907, -0.06487518548965454, -0.021317310631275177, 0.08970320224761963, -0.08907460421323776, 0.0966997817158699, -0.11392316967248917, 0.048412978649139404, -0.07661157101392746, 0.019403178244829178, -0.049979936331510544, -0.009366032667458057, 0.09179964661598206, 0.08947549760341644, -0.04180734604597092, 0.10334134101867676, 0.08184392750263214, 0.06828167289495468, 0.014338972978293896, -0.041138190776109695, -0.0107763996347785, -0.05275474488735199, -0.027558021247386932, 0.07582785189151764, -0.07869941741228104, -0.017017070204019547, -0.04801279306411743, -0.03940592706203461, 0.04587244242429733, -0.004803508520126343, 0.11034266650676727, -0.008912372402846813, 0.04595065861940384, -0.05861013010144234, -0.014264584518969059, 0.09768126904964447, -0.07390405237674713, 0.005693457089364529, 0.07185681164264679, 0.11352375149726868, -0.0256419088691473, 0.03990047425031662, 0.002188749611377716, -0.033080264925956726, -0.08197712898254395, 0.03158470615744591, 0.07025613635778427, 0.058590568602085114, 0.043740227818489075, -0.056557491421699524, -0.003638196038082242, -0.07443805783987045, 0.0017153038643300533, 0.04663444682955742, 0.08927930891513824, -0.013399066403508186, 0.008229542523622513, -0.02522583305835724, -0.09158467501401901, 0.03757409378886223, 0.09640413522720337, -0.06759928166866302, -0.040979642421007156, 0.023319847881793976, -0.0022242243867367506, -0.0027012797072529793, -0.04817640781402588, -0.002095939125865698, 0.08747770637273788, -0.10082632303237915, -0.06279787421226501, 0.10750674456357956], [-0.042511679232120514, -0.03841278329491615, -0.0602131225168705, 0.017079032957553864, -0.022095609456300735, -0.03964075446128845, -0.014797760173678398, 0.08109495043754578, 0.10159192234277725, 0.056778840720653534, 0.010032562538981438, -0.04023682326078415, -0.03744015470147133, 0.04618613049387932, -0.06498638540506363, -0.030184004455804825, -0.05224994942545891, 0.010282720439136028, -0.06827935576438904, -0.049454864114522934, 0.0227658748626709, -0.07112303376197815, -0.02501300349831581, -0.01750493049621582, -0.05347869172692299, 0.05535779148340225, 0.1044638529419899, 0.06133784353733063, 0.052376292645931244, -0.05112508684396744, -0.05244140699505806, 0.07304111123085022, -0.031465236097574234, 0.11718757450580597, 0.02758401446044445, -0.0074112145230174065, 0.11167654395103455, -0.007719581481069326, -0.020792726427316666, 0.03115970268845558, 0.015165001153945923, 0.025388389825820923, 0.06607984751462936, -0.011897128075361252, -0.09821773320436478, 0.06046595796942711, 0.061211273074150085, 0.03308691456913948, -0.07474162429571152, -0.02804664708673954, 0.0004056384495925158, -0.08723758906126022, 0.0815269947052002, -0.022363336756825447, -0.042070597410202026, 0.0477827712893486, 0.06504104286432266, -0.04503998905420303, -0.012141312472522259, 0.04125089943408966, -0.0732397809624672, 0.05179288610816002, -0.06294016540050507, 0.10115362703800201, -0.09239576011896133, -0.10205502808094025, -0.11850394308567047, -0.006812484469264746, -0.045104291290044785, -0.03110285848379135, -0.03988825902342796, 0.08845915645360947, 0.10429017245769501, 0.03201255947351456, 0.07522336393594742, -0.05049474909901619, -8.273806452052668e-05, -0.05591197684407234, -0.0012075304985046387, 0.07501504570245743, 0.011701526120305061, 0.043853893876075745, 0.09721455723047256, 0.04833175241947174, -0.023264260962605476, -0.08833934366703033, 0.07187830656766891, 0.06815067678689957, 0.0030453200452029705, 0.029607050120830536, 0.0016921053174883127, 0.11615726351737976, 0.032470058649778366, -0.052367668598890305, 0.017216328531503677, 0.07558311522006989], [0.0552566833794117, -0.11123287677764893, 0.05262448266148567, 0.03193115442991257, -0.0084298150613904, 0.03007415682077408, 0.019572513177990913, -0.04289066419005394, 0.004593190737068653, -0.09256257861852646, 0.037858910858631134, -0.11369146406650543, -0.06461143493652344, 0.0022833177354186773, 0.050088606774806976, -0.03561890497803688, 0.10147488862276077, -0.10246604681015015, -0.011271276511251926, -0.02531668357551098, 0.07731050252914429, 0.00525258295238018, 0.0028225493151694536, -0.05361011251807213, -0.0411514937877655, -0.03784452751278877, -0.029243478551506996, 0.0664212629199028, -0.04121158644556999, 0.007865815423429012, 0.07612790167331696, -0.06301627308130264, -0.08625011891126633, -0.08229780942201614, 0.016125094145536423, 0.038542453199625015, -0.023663759231567383, -0.03391449898481369, 0.0488009974360466, -0.09711512178182602, 0.02354283630847931, 0.05423552915453911, -0.09541765600442886, 0.03642486408352852, -0.05011611431837082, 0.013844965025782585, -0.016883498057723045, -0.027374835684895515, -0.03653765097260475, -0.08291637152433395, -0.12736950814723969, -0.03851034492254257, -0.11162446439266205, -0.05179700627923012, -0.07103422284126282, -0.02531689964234829, 0.03544074669480324, 0.10025544464588165, 0.04478469491004944, 0.010507242754101753, 0.09648676216602325, -0.047954343259334564, 0.025652265176177025, -0.057761043310165405, -0.030336717143654823, -0.023846078664064407, 0.010258804075419903, 0.04383528232574463, -0.002797644818201661, 0.05513070523738861, -0.04169413447380066, -0.0022681234404444695, -0.07298408448696136, 0.12942196428775787, -0.12675337493419647, 0.04097827523946762, -0.052203696221113205, -0.031249750405550003, -0.092167429625988, -0.011450444348156452, -0.0998438224196434, -0.06277567148208618, -0.06650740653276443, 0.06219086796045303, -0.11303693801164627, 0.08453985303640366, 0.02092239260673523, 0.0808318629860878, -0.004336185287684202, 0.06470091640949249, -0.10265269875526428, -0.04432324320077896, -0.015967613086104393, -0.023453012108802795, -0.08910638093948364, -0.021804388612508774], [0.07845192402601242, 0.016255928203463554, -0.069291852414608, 0.07101509720087051, 0.047400519251823425, -0.10857587307691574, 0.024478554725646973, 0.057844143360853195, -0.08350717276334763, -0.10138896852731705, 0.0028251928742974997, 0.05436477065086365, -0.04758189991116524, -0.017563138157129288, -0.0156963262706995, 0.06711001694202423, 0.006921032443642616, 0.041311394423246384, -0.029831847175955772, -0.013857100158929825, -0.023886660113930702, 0.0666087195277214, -0.07275136560201645, -0.0700419545173645, -0.01106721255928278, 0.07227852195501328, 0.03528831899166107, -0.043151021003723145, 0.03939220681786537, -0.0854569599032402, 0.007500109262764454, -0.021852966398000717, -0.06525000929832458, -0.11231604963541031, -0.05229583755135536, -0.08141516149044037, 0.04497886076569557, 0.011469565331935883, -0.004844318609684706, -0.051302000880241394, 0.08346061408519745, 0.002676316536962986, 0.04023119434714317, -0.08392171561717987, 0.05665736272931099, -0.08887343108654022, -0.04278269410133362, -0.08237051963806152, -0.08296233415603638, 0.036273058503866196, 0.07226302474737167, 0.06296984851360321, -0.10247761756181717, -0.09692148119211197, -0.09152901917695999, 0.028898881748318672, -0.0662231370806694, 0.027073044329881668, 0.023964906111359596, -0.047770336270332336, 0.02433258295059204, -0.06667662411928177, -0.023350996896624565, -0.054757025092840195, -0.05513990670442581, -0.01082201674580574, 0.06024976819753647, 0.04460497200489044, -0.06798919290304184, -0.03461958095431328, -0.020267441868782043, -0.055976033210754395, -0.008260355331003666, 0.02372339926660061, 0.050042103976011276, 0.08253268897533417, 0.022284066304564476, 0.0009010634385049343, -0.018424933776259422, -0.07280155271291733, -0.10123922675848007, -0.019305190071463585, 0.026168860495090485, 0.07253339886665344, 0.07046708464622498, 0.06503205001354218, -0.09599277377128601, -0.005161229521036148, 0.07864771038293839, 0.053812410682439804, 0.01545946579426527, -0.03190566971898079, 0.030103789642453194, 0.02898789942264557, -0.025001440197229385, -0.06567005813121796], [-0.03860590606927872, -0.05901798978447914, -0.08530106395483017, 0.08976662158966064, -0.10409991443157196, 0.03176567330956459, -0.09235220402479172, -0.019264651462435722, -0.07591978460550308, -0.043445076793432236, 0.012410744093358517, 0.03719811514019966, 0.031115436926484108, -0.08141406625509262, 0.07969202101230621, 0.0025066849775612354, 0.033723339438438416, -0.05395956709980965, 0.04244423285126686, 0.011818045750260353, 0.027504393830895424, -0.018601832911372185, 0.07833521068096161, 0.07515653967857361, -0.04541756957769394, -0.044498492032289505, 0.01674543134868145, -0.11737905442714691, -0.047122225165367126, -0.07763268053531647, -0.01905755139887333, -0.04880644753575325, -0.051299285143613815, 0.007745532318949699, -0.02919994853436947, 0.04371648654341698, -0.03513016179203987, 0.009683548472821712, -0.08739044517278671, -0.003049588529393077, -0.10513710230588913, -0.002722355304285884, -0.07472075521945953, 0.02132328227162361, -0.006285817362368107, 0.007813658565282822, -0.01582772471010685, 0.015478822402656078, -0.09412882477045059, 0.05438833311200142, 0.0051209828816354275, -0.02496662363409996, -0.0006459443247877061, 0.012475240044295788, -0.11894932389259338, -0.015560939908027649, -0.10102743655443192, 0.0874217301607132, -0.052221156656742096, -0.03502259775996208, 0.02562669664621353, 0.052787020802497864, -0.00761415995657444, -0.04569368436932564, -0.025679253041744232, -0.021082567051053047, -0.04596957564353943, -0.10373629629611969, 0.08910497277975082, -0.024777885526418686, -0.06491295248270035, -0.016317011788487434, 0.008772323839366436, -0.00842958502471447, 0.06000073254108429, 0.060337960720062256, -0.02160545252263546, -0.0011419417569413781, 0.05570731312036514, -0.001300898613408208, -0.005247965455055237, -0.04537663608789444, -0.03281612694263458, 0.030764514580368996, -0.09344196319580078, 0.019868608564138412, 0.03886011615395546, 0.021970821544528008, 0.09350257366895676, -0.09435258060693741, 0.030992191284894943, 0.018771197646856308, 0.04251663386821747, 0.030565688386559486, -0.014792073518037796, -0.009886064566671848], [-0.0828869491815567, 0.0548832081258297, 0.032266903668642044, -0.07705385237932205, -0.09515804797410965, 0.036193691194057465, -0.05093717947602272, -0.006509054452180862, -0.07126471400260925, 0.05287519097328186, -0.030872493982315063, -0.046934209764003754, 0.0436164066195488, -0.05194823071360588, 0.03567526116967201, 0.04440678283572197, -0.0026260698214173317, -0.08730946481227875, 0.014692374505102634, 0.04343141242861748, -0.01696869172155857, 0.06025930866599083, 0.017733288928866386, -0.06906400620937347, -0.05733354017138481, -0.036035165190696716, 0.06228932365775108, 0.05054910108447075, 0.006361487787216902, 0.09203062206506729, 0.010625559836626053, -0.06969688832759857, -0.023454178124666214, 0.06635265797376633, 0.038508087396621704, 0.06731976568698883, -0.014458377845585346, 0.08393033593893051, 0.10213255137205124, -0.03133886307477951, -0.03845762833952904, -0.005436794366687536, -0.09422317147254944, -0.06630467623472214, 0.09824063628911972, 0.03848962485790253, -0.08626805990934372, -0.09178082644939423, 0.049725137650966644, 0.008023064583539963, 0.00026912300381809473, -0.07204993814229965, 0.03995126113295555, 0.010727934539318085, -0.10272886604070663, 0.050683218985795975, -0.052108034491539, -0.05892936885356903, -0.019831180572509766, 0.008033611811697483, -0.03339318558573723, -0.04541750252246857, -0.06539648771286011, 0.0787753239274025, -0.08599280565977097, -0.007403205614537001, -0.051334235817193985, -0.0473795123398304, 0.017602970823645592, 0.01572521962225437, -0.0910380631685257, 0.005730212200433016, -0.049491360783576965, 0.09949034452438354, -0.04844759777188301, -0.09961539506912231, 0.07164312899112701, -0.05571148544549942, 0.010067603550851345, -0.06438352167606354, -0.06490417569875717, 0.011220105923712254, 0.07160171121358871, -0.019461553543806076, 0.08436542004346848, 0.11596587300300598, 0.02668023109436035, -0.028595736250281334, 0.0888417512178421, -0.07069990783929825, -0.09288235008716583, -0.05328066274523735, -0.11326047033071518, 0.0695650652050972, 0.0258284080773592, -0.08153661340475082]], "b2": [0.015431580133736134, -0.07422678917646408, 0.02646571211516857, 0.05487901344895363, 0.038400135934352875, -0.11586776375770569, -0.05750410631299019, -0.029089123010635376, 0.03379625454545021, 0.025235917419195175, 0.008959529921412468, -0.03386750817298889, -0.0799836814403534, -0.026941116899251938, 0.11190573126077652, -0.08674664795398712, 0.045182887464761734, 0.10843447595834732, -0.030280090868473053, 0.10789526998996735, 0.07306451350450516, 0.033063799142837524, -0.006809012498706579, -0.02290238067507744, 0.05410091578960419, -0.00980293471366167, -0.009773422963917255, 0.12632331252098083, -0.019008373841643333, -0.09465357661247253, -0.06669393181800842, -0.03971089422702789], "W3": [[-0.11705661565065384, 0.04370298609137535, -0.1644536256790161, -0.03643213212490082, -0.21142715215682983, 0.06620347499847412, 0.1429108828306198, 0.0031924021895974874, -0.1514652669429779, -0.10714960098266602, -0.05668181553483009, 0.14063039422035217, 0.04320835322141647, -0.14055228233337402, -0.05949705094099045, 0.04510362818837166, 0.024543646723031998, -0.033045440912246704, -0.02145751379430294, -0.0781719759106636, 0.15573088824748993, 0.06686040014028549, 0.1723519265651703, 0.09250621497631073, -0.017333198338747025, 0.17483779788017273, -0.16266098618507385, -0.09905143827199936, 0.12668053805828094, 0.07976388931274414, 0.1199994683265686, 0.10020473599433899]], "b3": [0.14893239736557007]}, "2021": {"W1": [[-0.15542805194854736, -0.12569335103034973, 0.08442945033311844, -0.19236579537391663, 0.16866731643676758, -0.09929396212100983, 0.08171688765287399, -0.015799978747963905, 0.04997099190950394, 0.27855023741722107, 0.06662118434906006, 0.003034556983038783, -0.005928275175392628, 0.12363434582948685, 0.02404419332742691, -0.13350526988506317, -0.18281888961791992, -0.08950918912887573, -0.06363797932863235, 0.03011566773056984, -0.013163385912775993], [0.11728917807340622, -0.07500872015953064, 0.06405080109834671, 0.12343348562717438, -0.03774278610944748, -0.14683198928833008, 0.11092078685760498, -0.1343950778245926, 0.07178356498479843, 0.12744243443012238, -0.08228828012943268, -0.0829428881406784, -0.2307235300540924, -0.024975158274173737, -0.13182809948921204, -0.08808889985084534, -0.2723864018917084, -0.06614845991134644, 0.08751700818538666, -0.02637576498091221, -0.07994594424962997], [0.13897885382175446, -0.09152322262525558, 0.07347488403320312, 0.11638243496417999, 0.14079128205776215, -0.11365845054388046, -0.0961386188864708, -0.023037562146782875, 0.07734832912683487, 0.05588643252849579, 0.1334877461194992, 0.0007756967097520828, 0.007177604362368584, -0.009259235113859177, -0.05535397678613663, -0.07899453490972519, -0.03365816920995712, -0.0518772229552269, -0.05767164006829262, -0.07354411482810974, 0.024553466588258743], [-0.09043136984109879, -0.023300210013985634, -0.17861361801624298, 0.10940434783697128, 0.09435208141803741, -0.040400855243206024, -0.047681618481874466, -0.10657598078250885, 0.034291062504053116, 0.1496359407901764, -0.15746159851551056, -0.09003499150276184, -0.16320444643497467, -0.10246027261018753, 0.12142926454544067, 0.13155105710029602, -0.10133777558803558, 0.1345013529062271, 0.07463819533586502, 0.012895457446575165, -0.0985267236828804], [0.12846314907073975, -0.17143628001213074, 0.05949613079428673, 0.16968534886837006, 0.15452496707439423, 0.07356930524110794, -0.07500816136598587, 0.05266518518328667, -0.09613990783691406, -0.06232922151684761, -0.01621486432850361, 0.22345581650733948, -0.014599110931158066, 0.25890305638313293, -0.14002113044261932, 0.038395438343286514, 0.00436326302587986, 0.11045081168413162, 0.004501961637288332, 0.013575979508459568, 0.08796961605548859], [0.17105652391910553, 0.13194167613983154, -0.16254910826683044, -0.0754975751042366, 0.14258261024951935, 0.1655421257019043, 0.1675470620393753, 0.0996127501130104, -0.1014656126499176, 0.045389242470264435, 0.14516673982143402, 0.15899668633937836, 0.02558266930282116, 0.09567973017692566, -0.04196487367153168, -0.09930574893951416, -0.04904675856232643, 0.027712993323802948, -0.047404881566762924, -0.10977282375097275, -0.05051746964454651], [-0.116579569876194, 0.16920702159404755, -0.12614257633686066, 0.11349281668663025, -0.1272655874490738, -0.11393259465694427, 0.10466976463794708, -0.18056824803352356, -0.07226737588644028, -0.0005494683282449841, -0.07825874537229538, -0.10874028503894806, 0.07058531790971756, 0.08852285146713257, 0.03294964134693146, -0.022161109372973442, 0.050492458045482635, -0.011866787448525429, 0.15400607883930206, -0.07963911443948746, 0.12242631614208221], [0.07805997133255005, -0.08707690238952637, 0.11077046394348145, 0.05260476842522621, -0.023188183084130287, -0.07582267373800278, -0.07513415068387985, -0.020627634599804878, -0.13933518528938293, 0.1912979781627655, -0.09985432028770447, -0.011944274418056011, 0.03395161032676697, -0.06453397870063782, 0.046518824994564056, -0.06497424095869064, -0.1918167918920517, 0.08507934212684631, 0.14524509012699127, -0.0914110317826271, 0.09322122484445572], [0.05519755929708481, 0.04536540433764458, -0.13686904311180115, -0.20059749484062195, -0.21037514507770538, -0.17408572137355804, 0.014251298271119595, 0.029526082798838615, 0.13161562383174896, -0.08288154751062393, -0.17895600199699402, 0.08195990324020386, 0.04618079960346222, -0.07636120915412903, -0.015981009230017662, 0.033704839646816254, -0.09421911835670471, -0.16884787380695343, -0.10004682838916779, -0.17445030808448792, -0.1786043494939804], [0.10827627032995224, -0.0030560174491256475, -0.05640626698732376, 0.04168357327580452, 0.15115144848823547, -0.1282479614019394, -0.05085310339927673, -0.02294551022350788, 0.030698953196406364, -0.07557187229394913, -0.03938809409737587, 0.09453009814023972, -0.13393676280975342, 0.07942865043878555, -0.08523046970367432, 0.08854859322309494, -0.15252576768398285, 0.07521715760231018, 0.10967011749744415, -0.01606299728155136, 0.11656703799962997], [-0.15873442590236664, 0.12724143266677856, -0.05300268903374672, -0.15131117403507233, -0.15671347081661224, -0.12056247889995575, -0.04282300919294357, 0.07605226337909698, 0.011670786887407303, -0.16351038217544556, -0.024105532094836235, 0.17001688480377197, -0.07599551230669022, 0.07457414269447327, 0.0767756849527359, 0.05891653150320053, 0.02770353853702545, -0.04631798341870308, -0.06816336512565613, -0.1016111895442009, 0.11655151098966599], [-0.0159678366035223, 0.1349230408668518, 0.003275485010817647, -0.1426694542169571, 0.025766540318727493, -0.06939509510993958, -0.10371861606836319, -0.102255679666996, 0.08735692501068115, 0.08738736063241959, -0.15714390575885773, 0.03153010085225105, 0.13467156887054443, 0.07696356624364853, 0.12364998459815979, -0.05907668545842171, -0.14081206917762756, -0.11388426274061203, -0.02772204950451851, 0.1225941926240921, 0.14433428645133972], [-0.03030218929052353, 0.09691265970468521, 0.02146674320101738, -0.10549207776784897, -0.047250136733055115, -0.0425976924598217, -0.08651864528656006, -0.03613027557730675, 0.13569767773151398, 0.0038530267775058746, -0.039002638310194016, -0.09829474240541458, -0.09725338965654373, 0.008832985535264015, -0.10523929446935654, 0.046401798725128174, -0.1346624791622162, -0.06876910477876663, -0.027089321985840797, 0.11874710023403168, 0.04149411618709564], [-0.06671691685914993, -0.11166135966777802, -0.24987353384494781, -0.1639786958694458, -0.056063827127218246, -0.01771346665918827, -0.12086725234985352, 0.12830500304698944, 0.08095172792673111, 0.18299993872642517, -0.03373758867383003, -0.11884831637144089, -0.1852496713399887, -0.07667795568704605, -0.15433476865291595, -0.1259845793247223, -0.05446682870388031, 0.16558612883090973, 0.09142327308654785, -0.198794424533844, -0.15145497024059296], [-0.09642574936151505, -0.05269153043627739, 0.0875624567270279, 0.011842988431453705, 0.09171996265649796, 0.02785349078476429, -0.07921025902032852, -0.06703085452318192, 0.014311167411506176, 0.08253353834152222, -0.004453448578715324, -0.08485283702611923, 0.06914646923542023, 0.13829396665096283, 0.0034558987244963646, -0.007123625371605158, 0.043412234634160995, -0.09260330349206924, -0.0065611847676336765, -0.10884086042642593, -0.14178414642810822], [-0.01554875448346138, 0.005651875399053097, -0.028424497693777084, 0.14413043856620789, -0.10085759311914444, -0.033422499895095825, -0.0776873230934143, -0.013545088469982147, 0.060375746339559555, -0.044957295060157776, -0.009167558513581753, 0.08254729956388474, 0.05774867534637451, -0.01015664916485548, -0.1381329745054245, -0.044089701026678085, -0.13988833129405975, -0.0854487344622612, 0.06353117525577545, -0.07251620292663574, 0.15128053724765778], [-0.11605110764503479, -0.1753920316696167, -0.11615811288356781, -0.2141275405883789, -0.00993124395608902, 0.01085363794118166, -0.0344633050262928, -0.026806864887475967, -0.19408228993415833, 0.028847288340330124, -0.10082846879959106, 0.0403522290289402, -0.10598932206630707, -0.1413467973470688, -0.08471979200839996, 0.005533210933208466, -0.05442200228571892, -0.08207051455974579, -0.14129036664962769, 0.031046224758028984, 0.1292281448841095], [-0.04253675043582916, 0.09643958508968353, 0.2320776730775833, -0.1718778759241104, 0.009869927540421486, 0.141915962100029, -0.18359430134296417, 0.061614181846380234, 0.11135857552289963, -0.12472739815711975, 0.009070126339793205, -0.16291610896587372, 0.05143909528851509, 0.025073857977986336, -0.096263587474823, 0.046060677617788315, 0.19536855816841125, -0.0794161856174469, -0.011894239112734795, 0.02844877354800701, -0.059749722480773926], [-0.07143858820199966, -0.04934777691960335, 0.029887421056628227, 0.02603732794523239, 0.09659553319215775, -0.12186521291732788, 0.005692547187209129, -0.13221591711044312, 0.06452164798974991, -0.006448044441640377, -0.0852992981672287, -0.008039411157369614, -0.13534992933273315, -0.12527239322662354, 0.03701294586062431, 0.07526341825723648, 0.12201685458421707, 0.14037863910198212, 0.17774330079555511, -0.05658845603466034, -0.08799882233142853], [-0.02080707810819149, -0.05485177040100098, 0.10454057157039642, -0.0030383551493287086, -0.08597789704799652, -0.021774116903543472, -0.044042542576789856, 0.050688110291957855, -0.02391250617802143, -0.0411113016307354, 0.06779418885707855, 0.1074824184179306, -0.14546647667884827, 0.028921134769916534, -0.08070634305477142, -0.02967149391770363, -0.21284841001033783, 0.05687157064676285, -0.07031898945569992, -0.057865243405103683, -0.07140859216451645], [-0.11222920566797256, -0.03287035599350929, 0.047312118113040924, -0.17822226881980896, 0.1214858740568161, 0.1294591724872589, 0.044304538518190384, -0.03707673400640488, -0.07431115210056305, -0.017390217632055283, 0.03758801892399788, -0.12008939683437347, -0.13455328345298767, 0.05302335321903229, 0.1859760582447052, 0.004308165051043034, -0.2391676902770996, 0.05597773194313049, -0.1152123510837555, 0.052499961107969284, -0.07986607402563095], [0.060118429362773895, 0.10330574214458466, -0.18425007164478302, 0.026287388056516647, 0.019278516992926598, 0.13473938405513763, -0.06544312834739685, 0.052340514957904816, -0.018349938094615936, -0.10662630200386047, -0.09909308701753616, -0.020240094512701035, -0.053249672055244446, -0.10422385483980179, 0.16388960182666779, 0.01263718493282795, 0.005842560436576605, 0.05597912520170212, 0.03199268504977226, 0.1012977808713913, -0.11208613961935043], [0.011441478505730629, -0.06420726329088211, -0.08798767626285553, 0.05110378935933113, 0.10685797780752182, 0.1307925581932068, 0.0414116233587265, -0.040705084800720215, 0.01549931988120079, -0.0854092687368393, 0.1278819590806961, 0.01808171160519123, 0.1347154676914215, -0.12883000075817108, 0.10068149119615555, -0.12898896634578705, -0.1317741572856903, 0.11557750403881073, -0.1403602808713913, -0.14805683493614197, 0.15292029082775116], [0.1426960676908493, 0.08742654323577881, 0.03907365724444389, -0.10550620406866074, 0.08729220926761627, -0.015048468485474586, -0.12087416648864746, -0.18109838664531708, 0.13069523870944977, -0.06965256482362747, 0.09144887328147888, -0.10311664640903473, -0.18726606667041779, -0.028960488736629486, 0.152872696518898, 0.02811300940811634, 0.10805122554302216, -0.10621951520442963, -0.05818263813853264, -0.09021718055009842, 0.000213800638448447], [-0.11264342069625854, 0.11526079475879669, 0.10141893476247787, -0.04507632926106453, 0.06634451448917389, -0.15848365426063538, 0.10718714445829391, 0.19434909522533417, -0.1471499353647232, -0.09652625769376755, -0.04217462241649628, -0.0026492546312510967, -0.22188623249530792, -0.09136272966861725, 0.18238705396652222, 0.07485432177782059, 0.055847540497779846, -0.08994827419519424, 0.11874928325414658, -0.04987761378288269, -0.004014153033494949], [-0.10473816096782684, 0.1894761174917221, -0.08155161142349243, 0.1053260937333107, -0.03453044593334198, -0.09758635610342026, 0.11384710669517517, -0.034032952040433884, 0.023957788944244385, 0.15748830139636993, -0.10545842349529266, 0.06580109894275665, -0.19295455515384674, 0.02313944138586521, -0.053411684930324554, -0.024262884631752968, 0.02722230926156044, 0.1273510605096817, -0.19209428131580353, -0.13962127268314362, -0.11075703054666519], [-0.05181131884455681, -0.09033462405204773, -0.0591234527528286, -0.1659270077943802, -0.1422201246023178, -0.1343984603881836, -0.06488971412181854, -0.05190066993236542, 0.05314679443836212, 0.06476498395204544, 0.09035270661115646, -0.029417019337415695, 0.03294222056865692, 0.015135470777750015, 0.1306600570678711, 0.07076065987348557, -0.07949759811162949, 0.049982596188783646, -0.012693406082689762, -0.10129015147686005, 0.14551211893558502], [-0.13719645142555237, -0.07945553213357925, 0.012200943194329739, 0.02105480432510376, 0.10391651839017868, -0.09707903116941452, 0.013177672401070595, -0.0007708892226219177, 0.20416079461574554, 0.07345826178789139, 0.0998874232172966, 0.07390881329774857, 0.18193745613098145, 0.1648155152797699, 0.16363516449928284, 0.024783950299024582, -0.12970155477523804, -0.1105487197637558, -0.07608317583799362, -0.05909367650747299, 0.030699988827109337], [-0.12420811504125595, 0.08655892312526703, 0.14766308665275574, 0.08970143646001816, 0.04140160605311394, 0.16987775266170502, 0.1355723887681961, -0.03528478741645813, 0.11151117831468582, -0.0006966069922782481, 0.14473770558834076, 0.07159402221441269, -0.07986385375261307, 0.005034680478274822, 0.08337277173995972, 0.16158071160316467, -0.018589187413454056, 0.14471514523029327, 0.05622541159391403, -0.08481606841087341, 0.1698417216539383], [0.1741599440574646, 0.10134723782539368, -0.0020272170659154654, -0.009008076973259449, 0.05232742428779602, 0.16658282279968262, 0.14030800759792328, -0.03143756836652756, 0.16660358011722565, 0.17659294605255127, -0.15370818972587585, 0.08946328610181808, -0.22544586658477783, 0.044228848069906235, 0.06818793714046478, -0.010077192448079586, 0.029422590509057045, 0.15038186311721802, 0.03754277899861336, -0.14425325393676758, 0.01630743220448494], [0.15617235004901886, 0.1432771533727646, -0.20190444588661194, -0.014918754808604717, -0.0012431979412212968, -0.1263788789510727, 0.19896647334098816, 0.020841199904680252, -0.07321080565452576, 0.11338935047388077, -0.037129927426576614, -0.07179877161979675, -0.14412160217761993, -0.06392323970794678, -0.07104820013046265, 0.03383338078856468, 0.0012663573725149035, 0.11820800602436066, 0.017604365944862366, -0.20867188274860382, 0.017023680731654167], [-0.010826836340129375, 0.11252167820930481, 0.1719060093164444, -0.07352571934461594, 0.0825532078742981, -0.002661976497620344, 0.1691199392080307, -0.035503435879945755, 0.04384933412075043, -0.13677427172660828, 0.11048270761966705, -0.08981439471244812, -0.12400157004594803, -0.061700791120529175, -0.08231883496046066, 0.11813780665397644, -0.07712037116289139, -0.06797543913125992, -0.07762817293405533, -0.054028693586587906, 0.019406618550419807], [0.010783669538795948, -0.0579833909869194, 0.18910357356071472, -0.0649808719754219, -0.12041713297367096, -0.08588439226150513, -0.07985741645097733, 0.058772388845682144, 0.10562160611152649, -0.08910338580608368, 0.17958660423755646, -0.1979096531867981, 0.10632798820734024, 0.12292490154504776, -0.0763864815235138, 0.10699369758367538, 0.15579193830490112, 0.03678670525550842, -0.10059845447540283, 0.17521649599075317, -0.04836951941251755], [-0.15734414756298065, 0.08011556416749954, 0.21289511024951935, 0.15595602989196777, 0.11156929284334183, 0.025927700102329254, -0.004658645484596491, 0.024986878037452698, -0.10247241705656052, -0.044233955442905426, -0.17911869287490845, -0.08028412610292435, -0.15570983290672302, -0.11459892243146896, -0.08953617513179779, -0.02013903111219406, -0.19495883584022522, -0.0790097787976265, 0.11359609663486481, -0.03348613902926445, 0.10611338913440704], [0.012754783034324646, 0.11246118694543839, 0.1830788552761078, -0.12632878124713898, -0.03687882423400879, 0.12573964893817902, -0.055595800280570984, 0.07838446646928787, -0.04352172091603279, 0.06644399464130402, -0.15878309309482574, 0.0801401361823082, -0.05172329768538475, -0.0663222000002861, 0.15781818330287933, -0.1697571724653244, -0.20515675842761993, -0.10489597916603088, 0.11573360860347748, -0.05976059287786484, -0.007563203573226929], [0.16156376898288727, 0.1543836146593094, 0.047229405492544174, 0.14716380834579468, 0.14976461231708527, -0.0619126595556736, -0.15399542450904846, -0.099985770881176, 0.07271840423345566, 0.1326281875371933, -0.13210558891296387, 0.02007230743765831, -0.15823735296726227, -0.07355581969022751, -0.13301385939121246, 0.06301332265138626, 0.2470935881137848, -0.07794369012117386, -0.14343476295471191, 0.0985383614897728, 0.10153944045305252], [-0.020570555701851845, 0.09178567677736282, 0.05381402373313904, 0.010323083959519863, 0.10007774084806442, 0.1511434018611908, -0.11025716364383698, -0.13321445882320404, 0.11792902648448944, -0.1015789657831192, 0.06896911561489105, -0.12520825862884521, -0.2613491415977478, -0.06394493579864502, -0.10276967287063599, 0.11096546053886414, 0.07285086810588837, -0.009472664445638657, 0.08024056255817413, 0.0031043128110468388, 0.054861199110746384], [-0.07366462051868439, -0.014244739897549152, -0.12430901825428009, 0.06226985156536102, 0.08177293092012405, 0.01433649193495512, 0.1456093043088913, 0.08112572878599167, -0.09225557744503021, 0.14952950179576874, -0.0341525562107563, -0.018465537577867508, 0.1913709193468094, 0.05413694307208061, -0.021128416061401367, 0.01988881453871727, 0.11847604066133499, 0.11259063333272934, -0.18711043894290924, -0.15147589147090912, 0.014925784431397915], [-0.10626516491174698, -0.14640115201473236, -0.13197918236255646, 0.08773542940616608, -0.09801153093576431, 0.13331012427806854, 0.019927503541111946, 0.03765108063817024, -0.04568759351968765, 0.20444738864898682, 0.07123347371816635, 0.007859387435019016, -0.13059695065021515, 0.0829131230711937, -0.06902090460062027, 0.07802396267652512, 0.06229352578520775, 0.10695403814315796, -0.15403155982494354, -0.11807738244533539, -0.06920938193798065], [-0.0644698217511177, 0.1474008560180664, -0.062198348343372345, -0.08899359405040741, 0.15170374512672424, 0.04366335645318031, 0.04376521334052086, 0.0025966812390834093, 0.03397898003458977, 0.06688997894525528, -0.06050431355834007, 0.11964307725429535, -0.0216513741761446, 0.040797822177410126, -0.15044806897640228, -0.12109237909317017, -0.0017952022608369589, -0.049624003469944, 0.01829884573817253, 0.011395786888897419, -0.08488892763853073], [-0.1590452790260315, -0.10261905193328857, -0.11345958709716797, -0.11816497892141342, 0.13609468936920166, -0.05471070855855942, 0.17266207933425903, -0.08738065510988235, 0.10043781250715256, -0.026767311617732048, 0.16134066879749298, -0.08233042806386948, 0.025837397202849388, -0.03424069285392761, -0.013667043298482895, 0.0551566556096077, -0.10283022373914719, -0.10080989450216293, -0.09921137243509293, 0.08166474848985672, -0.07960774004459381], [-0.12227218598127365, 0.09129097312688828, 0.022663462907075882, 0.025272144004702568, -0.05957777053117752, 0.15912701189517975, 0.10511691868305206, -0.1766173541545868, -0.10627929121255875, -0.13476191461086273, -0.03798125684261322, 0.06771430373191833, -0.08754099160432816, 0.12548860907554626, 0.07804057002067566, -0.03739337623119354, 0.10426335036754608, -0.06806767731904984, 0.11463738977909088, -0.02973366342484951, 0.10707243531942368], [0.009892723523080349, -0.03754546120762825, 0.10609016567468643, -0.005983368959277868, 0.08074240386486053, 0.010363185778260231, 0.003883218392729759, -0.03593502938747406, -0.001637743436731398, 0.18591195344924927, -0.08701391518115997, -0.069204181432724, -0.19810402393341064, 0.12420912086963654, -0.10537541657686234, 0.0034361479338258505, -0.16116994619369507, -0.010354331694543362, 0.12278662621974945, -0.05969179794192314, -0.0016426339279860258], [-0.01349923387169838, -0.035160552710294724, -0.06741081923246384, 0.015394611284136772, 0.029059408232569695, 0.10216214507818222, -0.03227565810084343, 0.022471141070127487, -0.03528570011258125, -0.08188937604427338, 0.03286268562078476, 0.1019827276468277, 0.1340705007314682, -0.0032311691902577877, -0.11536355316638947, -0.07384325563907623, -0.06689220666885376, -0.10187321156263351, -0.023164181038737297, -0.013471168465912342, -0.009423211216926575], [0.13157491385936737, -0.12730905413627625, -0.09414421021938324, 0.16223743557929993, -0.10429199784994125, -0.05898335203528404, -0.05564091354608536, 0.1347316950559616, -0.007534028496593237, 0.16604512929916382, 0.1053178459405899, -0.03478023409843445, 0.05623773857951164, -0.010870435275137424, 0.13896669447422028, 0.053157564252614975, -0.11067172139883041, -0.12826797366142273, 0.14874793589115143, -0.0013695905217900872, 0.15905863046646118], [0.004573803395032883, -0.057613976299762726, -0.09498699754476547, 0.08262369781732559, -0.0919065847992897, -0.06621827930212021, -0.09722450375556946, -0.0012803140562027693, 0.018466683104634285, -0.020572969689965248, -0.1424974501132965, 0.1440516710281372, 0.04069230705499649, 0.10330058634281158, 0.1596001535654068, 0.07373298704624176, 0.004032400902360678, 0.05847940966486931, -0.03247338905930519, 0.12956875562667847, 0.0825347751379013], [-0.1519366204738617, -0.050301261246204376, 0.15810494124889374, 0.16040468215942383, 0.009111829102039337, 0.11516740918159485, -0.12174013257026672, -0.09514933079481125, -0.12764905393123627, -0.09817051887512207, -0.17712335288524628, 0.11438103020191193, -0.01733468659222126, 0.047171663492918015, 0.07235579937696457, 0.13191646337509155, 0.18250694870948792, 0.03114410676062107, -0.1424238383769989, 0.10307306051254272, -0.03906233608722687], [0.08620162308216095, -0.06065107882022858, -0.1506720632314682, 0.13045336306095123, -0.060193754732608795, 0.1218593642115593, -0.15113726258277893, -0.09961948543787003, -0.02110935188829899, 0.06384439766407013, -0.07743567228317261, -0.1855665147304535, -0.11046253889799118, 0.06603210419416428, 0.0375758595764637, -0.07727779448032379, -0.1287933886051178, -0.01314222626388073, 0.004990284331142902, 0.03717789053916931, -0.04539460316300392], [0.0035687703639268875, 0.05101224407553673, 0.14359243214130402, -0.20151659846305847, -0.12768973410129547, 0.14118994772434235, -0.07903757691383362, -0.09769706428050995, 0.09527051448822021, 0.15810894966125488, -0.17337758839130402, -0.16683495044708252, 0.15371979773044586, 0.06420893967151642, 0.1841648817062378, -0.022039275616407394, 0.06303142011165619, 0.051630280911922455, 0.011038065887987614, 0.03126020357012749, -0.16789692640304565], [-0.24153243005275726, 0.04524853825569153, 0.1466829776763916, 0.041248150169849396, -0.13873803615570068, 0.15327569842338562, 0.13967151939868927, -0.18249762058258057, 0.13833990693092346, -0.11338061094284058, 0.02252890355885029, -0.0020435028709471226, -0.02693183906376362, 0.00042052866774611175, -0.06286048144102097, 0.10048986971378326, -0.2640691101551056, 0.04197869449853897, 0.11285875737667084, 0.04895143583416939, -0.08593505620956421], [0.14053109288215637, -0.03940168768167496, 0.16566285490989685, -0.01627322845160961, 0.10697219520807266, 0.1764194369316101, 0.08801420032978058, -0.14945529401302338, -0.09393409639596939, 0.12941651046276093, 0.013837935402989388, 0.08980691432952881, 0.035229552537202835, -0.08687379211187363, 0.009452123194932938, -0.07965156435966492, -0.09544886648654938, 0.038106419146060944, -0.10742239654064178, 0.1352020502090454, 0.18539834022521973], [0.06527537852525711, 0.07392335683107376, -0.07661480456590652, -0.0475308895111084, 0.11873611807823181, 0.10193757712841034, 0.007832018658518791, 0.037787020206451416, 0.09275210648775101, 0.060483526438474655, -0.029668418690562248, -0.01575167290866375, 0.03902726620435715, -0.13342595100402832, -0.0962817370891571, 0.08316345512866974, 0.21933835744857788, 0.05678403377532959, 0.1785443127155304, -0.01698082871735096, -0.17358870804309845], [-0.02279849350452423, 0.13348472118377686, 0.156447172164917, -0.05002318695187569, 0.07686356455087662, -0.12105061113834381, -0.019539769738912582, 0.16827291250228882, 0.09701181948184967, 0.03221552446484566, -0.00014251812535803765, -0.2301509529352188, -0.11722330749034882, -0.030886633321642876, 0.14989922940731049, 0.09693365544080734, -0.08107583969831467, 0.05629919096827507, -0.1706385314464569, -0.06055564805865288, -0.10625507682561874], [-0.05453411489725113, -0.04133140295743942, -0.09477715939283371, 0.06878302246332169, -0.028175657615065575, 0.09467693418264389, -0.05068988353013992, -0.10888545960187912, -0.0022609152365475893, 0.05935443937778473, 0.18530401587486267, -0.04111141711473465, 0.023729659616947174, -0.008652890101075172, -0.1464061737060547, -0.013805163092911243, -0.21452099084854126, -0.04746168106794357, -0.04400249198079109, 0.027402760460972786, -0.0845552459359169], [-0.054421860724687576, -0.1003776341676712, 0.03138907626271248, 0.056052256375551224, -0.055082812905311584, -0.13142257928848267, -0.017455436289310455, 0.041538942605257034, 0.053192950785160065, -0.05088009312748909, -0.05939174070954323, -0.010127895511686802, -0.05317387357354164, -0.08605799823999405, -0.022429121658205986, -0.004882842302322388, -0.24844706058502197, 0.11166372150182724, 0.05847636237740517, -0.026227550581097603, -0.08956114202737808], [0.08445596694946289, -0.05740342289209366, 0.013979100622236729, 0.11283799260854721, -0.15472321212291718, -0.14856792986392975, -0.08345862478017807, 0.06718873232603073, 0.08388141542673111, 0.14435623586177826, 0.04979615658521652, -0.03190505504608154, -0.15738289058208466, -0.062127236276865005, -0.13431677222251892, 0.04146615415811539, 0.00901645515114069, 0.0148245207965374, 0.09592162817716599, -0.09510837495326996, 0.15202085673809052], [0.13110694289207458, -0.11845865100622177, 0.1846305876970291, 0.22752855718135834, -0.1888751983642578, 0.03904928267002106, 0.14776039123535156, 0.10284601151943207, 0.07729072123765945, 0.08534346520900726, 0.1673794686794281, -0.13004562258720398, -0.21010582149028778, 0.06331206858158112, -0.018903816118836403, -0.047757845371961594, -0.14881063997745514, -0.05645425245165825, 0.09373287111520767, 0.15171703696250916, 0.04655199870467186], [-0.007625995669513941, -0.1593995839357376, -0.08829133212566376, -0.19159053266048431, 0.018359500914812088, -0.05570010840892792, 0.12488788366317749, 0.11834556609392166, -0.12863439321517944, 0.10027483105659485, 0.11432318389415741, -0.06079898402094841, -0.1522519439458847, 0.14333577454090118, -0.14732301235198975, 0.07383759319782257, 0.0905202254652977, -0.003188915317878127, -0.18857641518115997, 0.06844025105237961, -0.15236131846904755], [-0.1750054806470871, 0.07073570042848587, -0.0691995769739151, -0.1826331466436386, -0.15003715455532074, -0.01864912360906601, -0.0883018746972084, -0.056647319346666336, -0.022218430414795876, 0.15601424872875214, -0.15157732367515564, -0.017575815320014954, 0.04371701553463936, -0.002155730966478586, -0.013239228166639805, 0.02800840325653553, 0.015846943482756615, -0.05101870000362396, -0.1596454381942749, -0.07526880502700806, -0.12265540659427643], [-0.0945429876446724, 0.17383433878421783, 0.14762046933174133, 0.0606415718793869, 0.07039855420589447, -0.13559287786483765, 0.08276503533124924, -0.1028338074684143, 0.010922250337898731, -0.041383691132068634, -0.06545203179121017, -0.04292582347989082, 0.055763524025678635, 0.10616163164377213, 0.18649587035179138, -0.05531615391373634, 0.11585498601198196, 0.07298817485570908, 0.03688053786754608, -0.09914465993642807, -0.004102006088942289], [-0.16191419959068298, -0.14196895062923431, -0.04450641945004463, 0.0696449801325798, -0.03426608443260193, -0.10397002100944519, 0.06923579424619675, -0.06332679092884064, -0.15725360810756683, 0.1697102189064026, -0.15796209871768951, 0.021928833797574043, -0.13655757904052734, -0.058028098195791245, 0.0017993372166529298, -0.041521213948726654, 0.0007916495087556541, -0.09651124477386475, -0.16094325482845306, -0.13552804291248322, 0.019996341317892075], [-0.1228780448436737, -0.13259218633174896, -0.01597488857805729, 0.04917376488447189, 0.1540246158838272, -0.10993390530347824, 0.13073177635669708, 0.05936262384057045, -0.03333055600523949, 0.05548454076051712, -0.0371113084256649, -0.14055562019348145, -0.04261075332760811, 0.11215915530920029, 0.05437730252742767, -0.037942856550216675, -0.16343678534030914, 0.15373249351978302, 0.14656345546245575, -0.010833850130438805, -0.16833585500717163], [0.039250366389751434, -0.07988211512565613, -0.01800614781677723, -0.09053224325180054, 0.08124707639217377, 0.0163279976695776, 0.1571846753358841, 0.05657053738832474, -0.07858552783727646, -0.09653913229703903, -0.0025689187459647655, 0.09921789169311523, 0.05926988273859024, 0.16553643345832825, 0.07436537742614746, 0.01782335713505745, 0.04524661600589752, 0.13507677614688873, 0.07961855828762054, -0.08978702872991562, -0.0574783980846405], [0.0616864450275898, -0.0734812542796135, -0.048254720866680145, 0.11803047358989716, -0.10260534286499023, 0.05735734477639198, 0.04932974651455879, 0.10314816236495972, -0.05195881798863411, -0.08988918364048004, -0.08581997454166412, 0.09066476672887802, -0.0014602200826629996, -0.058586813509464264, 0.0862554982304573, 0.07433424890041351, -0.15097220242023468, 0.04132490232586861, -0.0358806774020195, -0.011749167926609516, 0.050488922744989395], [0.22292965650558472, 0.15460015833377838, -0.15168754756450653, -0.03205539658665657, 0.06989111751317978, 0.07270104438066483, -0.06938325613737106, -0.04335809126496315, 0.15569017827510834, 0.16458532214164734, 0.044890277087688446, 0.11910151690244675, 0.00534102413803339, -0.02987613156437874, 0.02968771755695343, -0.007471281103789806, 0.07630262523889542, -0.033304665237665176, 0.05206749960780144, -0.013071823865175247, 0.18167218565940857], [-0.01639224961400032, 0.03813818097114563, 0.11280067265033722, 0.08362097293138504, -0.0652860775589943, 0.1215401291847229, 0.11462417989969254, 0.22824303805828094, -0.0038835315499454737, 0.0360182523727417, -0.03207496926188469, 0.05281964689493179, -0.042474087327718735, 0.008738989941775799, 0.10022766888141632, -0.1820707619190216, -0.10554563999176025, -0.12500403821468353, 0.07838859409093857, 0.015076924115419388, 0.10590632259845734], [0.03317717835307121, -0.07164927572011948, 0.11021167039871216, -0.1034637913107872, 0.11986442655324936, -0.06130708009004593, -0.0061708856374025345, -0.012461619451642036, 0.0675010234117508, -0.12444192916154861, -0.0010442532366141677, 0.06560056656599045, 0.07634066045284271, 0.025663724169135094, 0.08807119727134705, 0.08164499700069427, 0.03468058258295059, 0.12626969814300537, 0.16720516979694366, 0.05480650067329407, 0.1298820525407791], [-0.000947600812651217, 0.12556611001491547, 0.2319353073835373, -0.09290848672389984, -0.06919180601835251, 0.07887732982635498, -0.11818476021289825, -0.14640288054943085, -0.07110168784856796, 0.20839416980743408, -0.1638721227645874, -0.029285747557878494, -0.05469954013824463, -0.05759496986865997, 0.06750091165304184, -0.164218470454216, 0.01041514240205288, 0.19429270923137665, 0.14928443729877472, -0.05680221691727638, -0.12884193658828735], [0.0043327417224645615, -0.11524177342653275, 0.08564568310976028, 0.0018219569465145469, 0.036204081028699875, -0.02638457529246807, 0.0623103603720665, -0.0816318690776825, -0.02934412658214569, -0.17430651187896729, -0.043135106563568115, -0.07444071024656296, 0.016913583502173424, 0.059456776827573776, 0.06331571936607361, 0.12555377185344696, -0.0031418893486261368, -0.03693920373916626, -0.13677208125591278, -0.14034457504749298, -0.11576909571886063], [-0.05513850972056389, 0.009308326058089733, -0.21517515182495117, -0.06446626782417297, 0.1144743338227272, 0.04428926110267639, -0.06338073313236237, 0.11959365010261536, -0.08246594667434692, -0.1038757711648941, -0.15709321200847626, 0.13095569610595703, -0.1508883833885193, -0.10669541358947754, -0.04534217715263367, 0.1487756222486496, 0.030979899689555168, -0.05346380174160004, 0.1055421382188797, -0.14143124222755432, -0.13117969036102295], [-0.04860888794064522, 0.10986550897359848, -0.049967098981142044, -0.06301014870405197, 0.06395495682954788, -0.08940529078245163, -0.05009133368730545, -0.0028125608805567026, 0.09021280705928802, 0.1406054049730301, 0.001488418085500598, -0.1228288933634758, 0.19493301212787628, 0.01515153143554926, 0.09889910370111465, -0.02269941195845604, -0.12346644699573517, -0.007416747510433197, 0.10034768283367157, 0.10208675265312195, 0.008262679912149906], [-0.11032561957836151, -0.10957033932209015, 0.10960344970226288, -0.029964642599225044, 0.16896073520183563, 0.15094523131847382, 0.1573253720998764, -0.048915717750787735, -0.12753082811832428, 0.06114373728632927, 0.003565969644114375, -0.04682294279336929, -0.14967156946659088, -0.1287538707256317, -0.15522633492946625, 0.10460840165615082, -0.25343629717826843, 0.06688927114009857, 0.15682154893875122, 0.03822167590260506, 0.05273272842168808], [-0.023693222552537918, -0.0041410550475120544, -0.09270484000444412, 0.0420830138027668, -0.08685663342475891, -0.07287432253360748, -0.044326454401016235, 0.04187465086579323, -0.13204210996627808, 0.1723787933588028, -0.1503000408411026, 0.0511435940861702, -0.05975746735930443, 0.09643204510211945, -0.12544988095760345, -0.13662174344062805, -0.05411187931895256, -0.06398005783557892, -0.12686218321323395, -0.1730269491672516, 0.14128975570201874], [0.06378001719713211, 0.008301874622702599, 0.11628897488117218, -0.05743507668375969, -0.03945118561387062, 0.13030238449573517, -0.08302104473114014, -0.06662876904010773, 0.1331649273633957, -0.09143470972776413, 0.07531709969043732, 0.02469598315656185, 0.07152930647134781, 0.03986268863081932, -0.11130494624376297, 0.15998601913452148, 0.08247759193181992, -0.07366826385259628, -0.07819577306509018, -0.10437733680009842, 0.18744395673274994], [-0.04439696669578552, 0.03739507496356964, 0.017487727105617523, 0.03444761037826538, -0.12798833847045898, 0.12510135769844055, 0.09671612828969955, -0.00958545133471489, -0.0983579084277153, 0.11636591702699661, 0.08674420416355133, -0.08018983155488968, 0.13927359879016876, 0.06021964177489281, 0.042723532766103745, -0.09813195466995239, 0.16838319599628448, 0.09559156000614166, -0.11713294684886932, -0.01036764681339264, -0.047470953315496445], [0.11045753210783005, -0.01764010265469551, -0.0958198681473732, 0.12285076081752777, 0.031900133937597275, -0.05202213302254677, 0.0010757723357528448, -0.04699253663420677, 0.09300872683525085, -0.007103422190994024, -0.054894521832466125, 0.037700407207012177, 0.05765004828572273, 0.027221592143177986, -0.030245903879404068, -0.020512601360678673, -0.10645122826099396, -0.08075323700904846, 0.06267387419939041, 0.10004755854606628, -0.015753353014588356], [0.020889513194561005, 0.04456984996795654, -0.09425750374794006, 0.094263955950737, 0.031563203781843185, -0.09763142466545105, -0.05045601725578308, 0.07859203219413757, 0.027531607076525688, -0.0781412124633789, -0.10207036882638931, 0.023655083030462265, 0.10732918977737427, -0.093450628221035, -0.058161377906799316, 0.05084500089287758, -0.16826990246772766, 0.10125812888145447, -0.04342443868517876, 0.1638055145740509, 0.002138623036444187], [-0.05685945600271225, 0.0901956558227539, -0.06063346937298775, -0.1430417150259018, -0.10470881313085556, -0.18203109502792358, -0.08770174533128738, -0.02730189636349678, -0.05621642246842384, 0.04251010715961456, 0.09887434542179108, 0.08846599608659744, -0.08866427093744278, 0.0922950729727745, -0.10612456500530243, 0.042888276278972626, -0.16781409084796906, -0.06924772262573242, -0.11552426964044571, 0.007065869867801666, 0.14568550884723663], [-0.14514286816120148, 0.07149690389633179, -0.07838001102209091, -0.030620720237493515, -0.10096260160207748, 0.12086652219295502, 0.17592649161815643, 0.0006353778881020844, 0.16607868671417236, 0.028872434049844742, 0.08548416942358017, -0.07620611786842346, -0.034607645124197006, -0.030525345355272293, 0.18311339616775513, -0.06025531888008118, -0.10946749895811081, 0.13280761241912842, -0.09735662490129471, 0.014643801376223564, 0.07231580466032028], [-0.028727024793624878, 0.04256296530365944, 0.11322592198848724, 0.13708314299583435, -0.14306692779064178, -0.06406909972429276, -0.08495370298624039, -0.10644284635782242, -0.0770796537399292, 0.09725354611873627, 0.11460460722446442, -0.05008048564195633, -0.14741972088813782, -0.020083913579583168, 0.026208309456706047, -0.014239253476262093, -0.16922643780708313, 0.1401810646057129, -0.09413298964500427, -0.01773160696029663, -0.030485263094305992], [0.017466723918914795, 0.09694219380617142, -0.0833766981959343, 0.04410466179251671, -0.007768196053802967, 0.1342785656452179, 0.09186402708292007, 0.04113803431391716, 0.026959355920553207, -0.008575533516705036, -0.016400717198848724, 0.10505983233451843, -0.01548097189515829, 0.112532839179039, -0.11434637755155563, -0.1220220997929573, -0.12809866666793823, -0.07860112190246582, -0.049352455884218216, -0.013126288540661335, 0.11493861675262451], [-0.12010669708251953, 0.07833445072174072, -0.016846220940351486, -0.0527031347155571, -0.021739041432738304, 0.12480124831199646, -0.12657570838928223, -0.0968356505036354, 0.04849977418780327, 0.07091355323791504, 0.08660706877708435, 0.005233058240264654, 0.21064502000808716, 0.21533915400505066, -0.15964418649673462, -0.022581085562705994, -0.09879365563392639, -0.08097796142101288, 0.10640980303287506, -0.02551562897861004, -0.17823946475982666], [-0.01821167953312397, -0.007716688793152571, 0.06642631441354752, -0.05922241881489754, -0.04848537594079971, 0.18982361257076263, 0.025129443034529686, -0.15762105584144592, -0.10777364671230316, 0.05149189382791519, -0.17216548323631287, 0.08183528482913971, -0.1283487230539322, 0.02923589199781418, 0.12171029299497604, 0.09518519043922424, 0.0014531664783135056, -0.010608851909637451, 0.03261538967490196, -0.0027129300870001316, -0.14562617242336273], [-0.10957436263561249, 0.06086176261305809, -0.09075096994638443, -0.12666666507720947, 0.07979784905910492, -0.03866807371377945, -0.05970076844096184, 0.04223787784576416, -0.061702899634838104, 0.16751030087471008, -0.04202441871166229, -0.10843515396118164, 0.19116459786891937, 0.04044419899582863, -0.016196196898818016, -0.024886054918169975, 0.040892068296670914, 0.08715822547674179, -0.039239365607500076, -0.06979150325059891, -0.039961282163858414], [0.12127435207366943, 0.02821185812354088, 0.013803980313241482, -0.1663106381893158, 0.009757821448147297, -0.12178126722574234, -0.17400763928890228, 0.039540763944387436, 0.05742640793323517, -0.0456705279648304, -0.06955869495868683, 0.0704653188586235, 0.050474461168050766, -0.1375950425863266, 0.16302937269210815, -0.10634682327508926, 0.2587719261646271, -0.06444858759641647, 0.011021853424608707, 0.09497877955436707, 0.007199455518275499], [0.20764727890491486, -0.07598447054624557, -0.1510142982006073, 0.14183132350444794, 0.20300550758838654, 0.05784160643815994, -0.1555093228816986, 0.11469372361898422, -0.04185054823756218, -0.1398060917854309, 0.11629389971494675, 0.10197513550519943, 0.1836940497159958, -0.012996437028050423, 0.11715593934059143, -0.10759531706571579, 0.029305577278137207, 0.13266196846961975, -0.11949017643928528, -0.06733676046133041, 0.045662786811590195], [0.021190717816352844, 0.06754181534051895, -0.1360197812318802, 0.0877331793308258, 0.16952942311763763, 0.06607117503881454, -0.042173098772764206, 0.138027623295784, -0.0029772850684821606, 0.04465067386627197, -0.020139148458838463, -0.14855043590068817, 0.060375865548849106, 0.08096875250339508, -0.03130340576171875, -0.15500201284885406, 0.2538955509662628, -0.09953338652849197, 0.15539661049842834, -0.11816297471523285, -0.01743192784488201], [0.004375838208943605, -0.12580443918704987, 0.011357737705111504, -0.017159229144454002, 0.12837561964988708, -0.09540354460477829, -0.0695028007030487, 0.04001809284090996, -0.06134505942463875, 0.14144961535930634, -0.042760685086250305, 0.06573508679866791, 0.028794944286346436, 0.05037527531385422, 0.07169284671545029, 0.04722825810313225, -0.0203901007771492, -0.05127299949526787, 0.047793399542570114, 0.09397585690021515, -0.05702954903244972], [0.09216015785932541, 0.13639914989471436, -0.33301928639411926, -0.05112177133560181, 0.1079477071762085, 0.15801768004894257, 0.09911234676837921, -0.02647772803902626, 0.10525979846715927, 0.06424742192029953, -0.06932681798934937, 0.16062749922275543, -0.1402280628681183, -0.02677925117313862, 0.06207792088389397, -0.1003187745809555, -0.021443547680974007, 0.14326462149620056, 0.011138327419757843, 0.024726679548621178, -0.16317495703697205], [-0.05696383863687515, 0.08901017904281616, 0.0740639865398407, -0.05082889273762703, -0.10292383283376694, 0.02770877443253994, -0.06519465148448944, -0.09532157331705093, -0.13446523249149323, -0.0838208943605423, -0.06539475917816162, -0.03544875234365463, 0.1747608482837677, 0.14812137186527252, 0.02347254753112793, 0.022370900958776474, 0.09955726563930511, 0.05941290780901909, -0.1409967541694641, -0.018035627901554108, -0.022031188011169434], [-0.018927928060293198, -0.07988438755273819, 0.060417186468839645, -0.0907495766878128, -0.1040523424744606, 0.039970435202121735, 0.08544273674488068, -0.06990274786949158, -0.1264214962720871, -0.009201266802847385, -0.05500524118542671, -0.19337506592273712, 0.01139290165156126, 0.13711123168468475, 0.0894131138920784, 0.012418909929692745, 0.17819245159626007, 0.12069714814424515, -0.08279534429311752, -0.0714988261461258, 0.03677235171198845], [0.04670334979891777, 0.11831466108560562, 0.045263372361660004, 0.05359330400824547, -0.00469830259680748, -0.0761183425784111, -0.10239599645137787, 0.06149086728692055, -0.13317279517650604, -0.0534316822886467, -0.21171097457408905, 0.03816254436969757, -0.01665535196661949, -0.10324496775865555, -0.0493084080517292, 0.04030958563089371, 0.35003572702407837, 0.04003148153424263, 0.15192200243473053, -0.08943910896778107, -0.026646869257092476], [-0.11795466393232346, -0.01898586004972458, 0.061409514397382736, 0.16190174221992493, -0.05490617826581001, -0.0117628313601017, 0.12959235906600952, -0.08476154506206512, 0.0076331994496285915, -0.1038849875330925, -0.003133996855467558, 0.016520904377102852, -0.12938188016414642, 0.1419786512851715, -0.013860730454325676, 0.016284655779600143, 0.08497954159975052, 0.02906411699950695, 0.005384623538702726, -0.00828001368790865, -0.025903897359967232], [0.08957206457853317, 0.10756109654903412, -0.025401249527931213, 0.16102632880210876, 0.08531790971755981, -0.048666104674339294, 0.07428259402513504, 0.11806308478116989, 0.00899545568972826, 0.04650326445698738, 0.04126057028770447, -0.12937882542610168, -0.04549235478043556, -0.03517265245318413, 0.1677612066268921, -0.10995529592037201, -0.12243920564651489, 0.09191066771745682, 0.06669067591428757, 0.0674392357468605, 0.05969870835542679], [0.017099348828196526, 0.04125196114182472, -0.020006628707051277, 0.004459972959011793, 0.060073502361774445, -0.06881267577409744, 0.15079984068870544, 0.1184382513165474, 0.1077960878610611, 0.023055050522089005, 0.033685941249132156, 0.035638898611068726, 0.10091323405504227, -0.0564127080142498, 0.13708384335041046, -0.09881500899791718, -0.20257699489593506, 0.06073081120848656, 0.1113092452287674, 0.000686504237819463, -0.1519920825958252], [-0.059909921139478683, -0.04730831831693649, 0.10077294707298279, -0.11524251103401184, 0.09141257405281067, -0.02451835758984089, 0.1283687800168991, -0.06140023097395897, -0.10037402808666229, 0.001374645042233169, -0.056911006569862366, 0.041477661579847336, 0.20597313344478607, -0.04709068313241005, -0.015448328107595444, 0.07133449614048004, 0.15483340620994568, -0.06016042083501816, 0.07801809161901474, 0.034265074878931046, -0.12828080356121063]], "b1": [-0.146578848361969, 0.21397699415683746, 0.16361021995544434, -0.19162839651107788, 0.2340751737356186, 0.16726036369800568, -0.04428057000041008, -0.03888396918773651, 0.14492829144001007, 0.08143456280231476, -0.15660588443279266, -0.005667887162417173, 0.037917692214250565, -0.11429169774055481, -0.1662244200706482, 0.05895979329943657, -0.032265614718198776, 0.17500580847263336, 0.047068335115909576, 0.1597239375114441, -0.08311615884304047, 0.12552949786186218, 0.2075202912092209, 0.07673962414264679, 0.04872314631938934, -0.12538404762744904, 0.15316101908683777, 0.02833971194922924, -0.09308526664972305, -0.01022593304514885, -0.09499937295913696, -0.016259603202342987, 0.10038835555315018, 0.11760743707418442, 0.2180202454328537, 0.1160225048661232, 0.23565293848514557, 0.03796853870153427, -0.11243031173944473, 0.15870706737041473, -0.14556199312210083, -0.08339823782444, -0.059304576367139816, 0.0423872172832489, -0.07488853484392166, 0.06527110934257507, -0.04469341039657593, -0.11430367082357407, -0.02487708069384098, 0.0893171951174736, 0.014940040186047554, 0.038217693567276, -0.05226394906640053, 0.09403806179761887, 0.023847654461860657, -0.02840859815478325, 0.01674179546535015, -0.06984967738389969, 0.08062440156936646, 0.022152723744511604, -0.15719659626483917, 0.10236255824565887, 0.12471894919872284, 0.17905035614967346, -0.043867696076631546, -0.14817099273204803, -0.1874830722808838, 0.1556762456893921, -0.10345928370952606, -0.11205988377332687, -0.004404962994158268, -0.04854882135987282, -0.06850976496934891, -0.20178969204425812, -0.11507720500230789, 0.10505816340446472, 0.22384102642536163, 0.12437138706445694, 0.04394173622131348, -0.06059694290161133, -0.017397470772266388, 0.18692617118358612, 0.060116495937108994, 0.028077594935894012, 0.16289614140987396, 0.08278650790452957, 0.1843789666891098, -0.09045372158288956, 0.06323818862438202, 0.10101137310266495, 0.13342082500457764, -0.0452108308672905, -0.01824764907360077, -0.14273110032081604, 0.17747095227241516, 0.19509099423885345], "W2": [[0.06881165504455566, 0.010705954395234585, 0.004551966208964586, -0.061917662620544434, -0.01910930685698986, -0.033077422529459, -0.06393522024154663, -0.010874769650399685, 0.06697691977024078, 0.06979762762784958, -0.09793511033058167, 0.07459443807601929, -0.05467668175697327, 0.004352831281721592, 0.0629485547542572, 0.018641086295247078, -0.02273857221007347, -0.09324776381254196, 0.10191997140645981, 0.005199209321290255, 0.10655704885721207, -0.06377013027667999, 0.07309646159410477, 0.07027405500411987, -0.02007703110575676, -0.004194617737084627, 0.060206539928913116, 0.0760401040315628, -0.04080121964216232, 0.05319185554981232, -0.0864877924323082, 0.02034199982881546, -0.047678109258413315, 0.007495669182389975, 0.07270921021699905, 0.05260345712304115, 0.02337154932320118, 0.0938517153263092, -0.010232574306428432, -0.06463907659053802, 0.026595603674650192, 0.07295680791139603, -0.07440698891878128, 0.02205912582576275, -0.012324737384915352, 0.06664987653493881, -0.10043662041425705, -0.04520759359002113, 0.03496376797556877, 0.07161960005760193, 0.059021297842264175, 0.05114210024476051, 0.07149999588727951, -0.04514021426439285, -0.04192538931965828, 0.0017639275174587965, -0.04318566620349884, 0.016094261780381203, 0.10083258152008057, -0.03712522238492966, -0.03435062617063522, -0.053608961403369904, -0.01952705718576908, -0.06178103759884834, 0.0028405110351741314, -0.11354441940784454, -0.06122603639960289, 0.01880907267332077, -0.006943755783140659, -0.027069631963968277, 0.052979618310928345, 0.04315993934869766, -0.05817863345146179, -0.07619786262512207, -0.019684014841914177, 0.03958991914987564, -0.007652208674699068, 0.042831286787986755, -0.0607542060315609, 0.0015299351653084159, -0.027496447786688805, 0.08853445947170258, -0.016357168555259705, 0.032290589064359665, 0.10132654756307602, 0.012594158761203289, -0.020133808255195618, -0.018513336777687073, -0.037512388080358505, -0.00849435105919838, 0.05143994465470314, 0.1430780291557312, -0.03869739919900894, -0.08084432035684586, 0.058523744344711304, 0.02595374919474125], [0.05771928280591965, -0.010203546844422817, 0.070782870054245, 0.032173626124858856, -0.018963687121868134, 0.04506341740489006, 0.014813639223575592, -0.010291686281561852, 0.0032006707042455673, 0.014016594737768173, 0.06586277484893799, -0.01556993555277586, 0.001497726421803236, 0.0814567282795906, -0.005259164609014988, -0.006830464117228985, 0.08641423285007477, 0.05193137004971504, -0.033926546573638916, 0.027893580496311188, 0.06453397125005722, 0.06115753948688507, 0.02259724959731102, -0.04962434619665146, 0.014375577680766582, 0.02108503319323063, -0.05038182809948921, 0.04856511950492859, 0.07086572051048279, -0.03086993284523487, -0.0009285209234803915, 0.07196034491062164, 0.057140644639730453, 0.05282672867178917, 0.0019036250887438655, -0.05762285739183426, -0.06673245131969452, -0.028496531769633293, -0.002455491106957197, 0.059909358620643616, 0.047407638281583786, 0.08641382306814194, -0.008163398131728172, -0.024495208635926247, 0.01773238740861416, 0.032320551574230194, -0.004613323602825403, 0.07377326488494873, -0.002817960688844323, -0.003961541689932346, -0.07500454038381577, 0.03902439773082733, 0.019137166440486908, -0.009411645121872425, 0.01057365257292986, 0.011743412353098392, 0.01660681515932083, 0.010473629459738731, -0.009615525603294373, 0.005017314106225967, 0.06998639553785324, -0.007299231365323067, -0.006523875519633293, -0.001436458551324904, 0.06162106618285179, 0.09657485783100128, -0.002825257834047079, -0.058410413563251495, 0.07902386039495468, 0.08996611833572388, 0.027627019211649895, 0.019288474693894386, 0.07993386685848236, 0.03160923346877098, 0.00418138550594449, -0.004196080844849348, -0.00045306090032681823, 0.058155357837677, 0.03547277674078941, -0.025306882336735725, 0.02480728179216385, -0.004666964057832956, 0.07685723900794983, 0.009027746506035328, 0.07627328485250473, 0.022334903478622437, 0.061485521495342255, -0.030757039785385132, 0.13050450384616852, 0.011050330474972725, 0.04615632817149162, -0.0352734811604023, 0.05210414528846741, -0.06341225653886795, 0.044070031493902206, 0.030449535697698593], [0.06476012617349625, 0.045141298323869705, -0.014862417243421078, -0.06111614406108856, 0.03553260862827301, -0.03557358682155609, -0.08280456066131592, 0.017928479239344597, 0.0410904623568058, -0.010394422337412834, -0.0247365552932024, 0.1130625307559967, 0.08170698583126068, -0.1270148754119873, 0.05421052500605583, 0.028350507840514183, 0.04660384729504585, -0.04997355863451958, -0.035609565675258636, -0.0022204627748578787, 0.006881402339786291, -0.0780382752418518, 0.018429050222039223, 0.03451652452349663, 0.09834709018468857, 0.006457178387790918, 0.08258036524057388, 0.04556756094098091, -0.12231940776109695, -0.06924740225076675, -0.1156296655535698, -0.10922332108020782, -0.05823085084557533, 0.06438589096069336, -0.02785651944577694, 0.11225777864456177, -0.01934460736811161, 0.11189315468072891, 0.029233790934085846, -0.06941784173250198, -0.025262923911213875, -0.06197478249669075, -0.08142822235822678, -0.024894021451473236, -0.05334597826004028, 0.013104945421218872, 0.04792702943086624, -0.051646068692207336, 0.1257973313331604, -0.05713285133242607, -0.05589655414223671, 0.10787981748580933, 0.020811613649129868, 0.06028096005320549, -0.05118855461478233, 0.10990917682647705, -0.015451772138476372, 0.048725228756666183, 0.09606257826089859, -0.045086342841386795, -0.004831146914511919, 0.017715327441692352, -0.03286978602409363, 0.002807037904858589, 0.039505332708358765, -0.10239695757627487, -0.056424181908369064, 0.01206694170832634, -0.007881822995841503, -0.0655159130692482, 0.10451971739530563, -0.04222278669476509, 0.015462719835340977, -0.09863755851984024, 0.030165839940309525, 0.01556410826742649, 0.08310180902481079, 0.0049561383202672005, 0.004246438387781382, -0.05226212367415428, 0.0181130263954401, -0.007620867807418108, 0.028200149536132812, 0.12702949345111847, 0.05303396284580231, 0.06542718410491943, 0.05761270225048065, -0.03942573815584183, -0.0810384452342987, 0.06776431202888489, 0.07182706892490387, 0.19097605347633362, 0.04841858148574829, 0.03058871068060398, 0.05253647640347481, 0.10849086195230484], [0.04055663198232651, 0.0629974827170372, 0.01829143799841404, -0.019347036257386208, 0.01707991398870945, 0.04025169461965561, 0.010267378762364388, 0.00566261587664485, 0.03671180084347725, 0.05593554303050041, -0.006668721325695515, -0.011879153549671173, -0.028372230008244514, -0.026053395122289658, -0.03757495805621147, 0.0674922913312912, -0.03015798144042492, 0.0037097828462719917, 0.06098121777176857, 0.015808407217264175, 0.009464401751756668, 0.07040068507194519, -0.021051157265901566, 0.015052135102450848, -0.020312761887907982, 0.01637859083712101, -0.0022296570241451263, 0.06887209415435791, -0.015833158046007156, -0.0406513437628746, -0.059271469712257385, -0.06794427335262299, 0.0705534815788269, -0.056768205016851425, 0.031031599268317223, 0.003574214642867446, -0.04663136973977089, 0.04993351176381111, -0.04816879704594612, -0.041350606828927994, -0.05008677393198013, 0.0006897010607644916, 0.06149410083889961, 0.04168516770005226, -0.05621711537241936, -0.038817569613456726, -0.02605227753520012, -0.06691081821918488, 0.0066257198341190815, 0.015616846270859241, -0.05126643180847168, -0.02386888861656189, 0.03458593785762787, 0.019645925611257553, 0.02925720438361168, 0.08270668238401413, 0.0909871906042099, -0.0137026971206069, 0.08477683365345001, 0.008929471485316753, -0.05082431063055992, 0.019599031656980515, 0.03683865815401077, -0.02251087687909603, -0.04714159294962883, 0.012967760674655437, 0.030357154086232185, 0.035135071724653244, -0.07091689109802246, 0.020255539566278458, -0.03140850365161896, -0.0354556106030941, -0.003391654696315527, -0.059915877878665924, -0.0589328296482563, 0.048063013702631, 0.006690407171845436, 0.04966041073203087, -0.027506615966558456, 0.012638513930141926, -0.03045884519815445, -0.03825928270816803, -0.012554366141557693, 0.007206120993942022, -0.04949210211634636, -0.015125690959393978, 0.04526485130190849, -0.03763069584965706, -0.01675538904964924, -0.0011682410258799791, 0.060648076236248016, 0.005296229384839535, -0.012666262686252594, 0.0691196471452713, -0.014242429286241531, 0.02271292731165886], [-0.01631135120987892, -0.012395995669066906, 0.07271989434957504, -0.10321702063083649, 0.07508780062198639, -0.07912745326757431, 0.014020777307450771, 0.010568312369287014, -0.06567338109016418, -0.02450445666909218, -0.10705725103616714, 0.06100626289844513, -0.042687010020017624, -0.0809507817029953, 0.035064950585365295, 0.07873113453388214, 0.05706631392240524, 0.015893271192908287, 0.05829476937651634, 0.048222124576568604, -0.02085687592625618, -0.013511520810425282, 0.1026841253042221, 0.05907323583960533, 0.045723382383584976, -0.1100797951221466, 0.09121281653642654, -0.027852512896060944, -0.06462100893259048, 0.07540477067232132, -0.057444870471954346, -0.017387695610523224, -0.06226110830903053, 0.0597294382750988, -0.075522281229496, 0.09384237974882126, 0.048665136098861694, 0.09142474085092545, -0.045667123049497604, 0.05627448111772537, -0.016264863312244415, 0.0727696493268013, -0.023330695927143097, -0.003054653760045767, -0.08711306750774384, 0.014267572201788425, 0.08049104362726212, -0.053265560418367386, -0.051531754434108734, -0.0016153111355379224, -0.06568315625190735, 0.07937531173229218, -0.049655504524707794, 0.07981274276971817, -0.061141375452280045, 0.006711033638566732, 0.06698626279830933, 0.029233206063508987, 0.0326208621263504, 0.05339846760034561, 0.049868084490299225, -0.052711475640535355, -0.030242448672652245, -0.019149992614984512, -0.10306455194950104, -0.09767337143421173, -0.04251950606703758, -0.009501347318291664, 0.019767647609114647, -0.011531551368534565, -0.07681675255298615, 0.008619766682386398, -0.05288168415427208, -0.04853738471865654, 0.00903394166380167, -0.06441294401884079, -0.0759861171245575, 0.09543827176094055, 0.07801124453544617, 0.041380152106285095, 0.022715846076607704, 0.07596223801374435, 0.04709485173225403, -0.019315434619784355, 0.016542915254831314, -0.08824853599071503, 0.07760853320360184, -0.018189070746302605, 0.05502612516283989, -0.05430321767926216, -0.02168188989162445, 0.14459362626075745, -0.00792058277875185, -0.02683514729142189, 0.012413837015628815, 0.0026638961862772703], [0.07070223987102509, -0.013172660954296589, -0.01861436478793621, 0.03868091106414795, -0.10961510241031647, -0.003911864478141069, 0.029392370954155922, 0.01497591845691204, 0.028109341859817505, -0.07538550347089767, 0.03180772066116333, -0.025761619210243225, -0.06234392151236534, 0.14794714748859406, -0.028023239225149155, 0.01750604622066021, 0.12049371004104614, -0.05477273464202881, -0.0009081587777473032, -0.03338829055428505, -0.01516031939536333, -0.110185906291008, -0.001991734141483903, -0.018320873379707336, 0.03897682577371597, 0.1309923529624939, 0.03378993272781372, -0.05645520240068436, 0.009394570253789425, -0.055914007127285004, 0.046027135103940964, -0.03729557245969772, -0.07979682832956314, -0.054425839334726334, -0.08290400356054306, -0.07788775116205215, -0.08661361038684845, 0.0008258973248302937, 0.04494287818670273, 0.06487347930669785, -0.029525356367230415, 0.049557678401470184, 0.003554251277819276, -0.004945777356624603, -0.0022180676460266113, -0.050907477736473083, 0.06159178167581558, 0.035224027931690216, -0.04098426178097725, 0.031038418412208557, -0.04466605186462402, 0.014554652385413647, -0.10724307596683502, -0.06822127103805542, -0.024617284536361694, -0.05239453539252281, -0.06205467879772186, 0.09728085994720459, 0.08228792250156403, -0.07544019818305969, 0.1407155692577362, -0.037761081010103226, 0.0525033064186573, -0.05716017633676529, 0.012127391062676907, 0.03275289386510849, -0.021636946126818657, -0.04186692461371422, 0.0012618704931810498, 0.044311169534921646, -0.01969270408153534, -0.05794190987944603, 0.08248036354780197, -0.0407884381711483, -0.00835173949599266, -0.0266815647482872, -0.06833026558160782, 0.10126858949661255, -0.006652729585766792, -0.001430738135240972, 0.023323271423578262, -0.001552662462927401, 0.0328797772526741, 0.029752010479569435, 0.00907520018517971, -0.039780374616384506, 0.046863723546266556, 0.02345343865454197, 0.09773948788642883, -0.09639766812324524, 0.052950795739889145, -0.05670057237148285, 0.022306839004158974, 0.05434556305408478, -0.008476768620312214, -0.0009088318911381066], [-0.0037252905312925577, -0.04162079468369484, -0.08802621066570282, 0.04177786782383919, 0.013398483395576477, -0.059905510395765305, 0.001266251434572041, 0.04180155321955681, -0.03471733629703522, -0.1107826977968216, 0.0731998085975647, -0.028540747240185738, -0.0778636783361435, 0.04697195440530777, 0.07459303736686707, 0.0009309860179200768, -0.01100825984030962, 0.03852090984582901, -0.003268030472099781, 0.05553010106086731, -0.010401885956525803, -0.009789965115487576, -0.014478640630841255, -0.042011696845293045, -0.05943837761878967, -0.012311751022934914, 0.04493638873100281, -0.023356394842267036, -0.00929140206426382, -0.10928240418434143, -0.012971555814146996, -0.015466678887605667, 0.008223148994147778, -0.05544538423418999, -0.11425796896219254, -0.0016506295651197433, 0.02309102565050125, -0.09307469427585602, -0.09618797153234482, -0.06429845839738846, -0.03603467717766762, -0.04872080311179161, -0.044430773705244064, 0.05833012983202934, 0.020889887586236, -0.0024312406312674284, -0.09776455163955688, 0.05854352191090584, -0.0303875170648098, 0.009538745507597923, -0.08397714793682098, -0.04019305482506752, -0.03610304743051529, -0.09397077560424805, 0.03045376017689705, -0.10496586561203003, 0.04150884225964546, -0.014989444054663181, -0.008165537379682064, 0.08047270029783249, 0.033223509788513184, -0.0931301936507225, 0.07030611485242844, -0.00680884812027216, -0.047271016985177994, 0.002124123740941286, 0.09668588638305664, 0.006955650169402361, 0.016810428351163864, 0.03286962956190109, -0.04312727600336075, 0.034964654594659805, -0.07485346496105194, 0.03861701861023903, -0.07694510370492935, -0.08597753942012787, -0.0641137957572937, 0.00915624387562275, -0.08506142348051071, 0.053170014172792435, -0.010228811763226986, -0.05633733049035072, -0.09343761205673218, -0.07929631322622299, -0.08596551418304443, -0.08339077979326248, -0.04828040301799774, 0.008947495371103287, -0.06455908715724945, 0.041670363396406174, 0.05718359723687172, -0.004657900892198086, 0.04382092133164406, 0.0005260639591142535, 0.05494741350412369, -0.00167280703317374], [0.0082522789016366, -0.0017912151524797082, -0.015161375515162945, 0.011710692197084427, -0.013922091573476791, -0.010575512424111366, -0.003686025273054838, -0.008268564008176327, -0.011651520617306232, 0.01764116808772087, 0.002744928002357483, 0.0021140417084097862, 0.009468916803598404, -0.001631698221899569, 0.003950111567974091, 0.00908727291971445, 0.006697485223412514, 0.0291669350117445, -0.007765123620629311, 0.02483486384153366, -0.0027057984843850136, 0.009346033446490765, -0.018600625917315483, -0.007295823190361261, -0.009542842395603657, 0.00236449739895761, -0.005157872568815947, 0.002915009157732129, -0.007187496405094862, 0.014347617514431477, -0.0005085291340947151, 0.01066630333662033, 0.022021085023880005, -0.003112901234999299, 0.00407275278121233, 0.013807258568704128, -0.012836987152695656, -0.0019871785771101713, -0.0028455061838030815, 0.0032013901509344578, 0.0006194370798766613, 0.013781016692519188, 0.0008563614683225751, 0.0014442205429077148, -0.001425370224751532, -0.011445647105574608, -0.004471191670745611, 0.007524001877754927, 0.010469848290085793, -0.010038322769105434, 0.0047569372691214085, -0.002758486894890666, 0.000923953193705529, 0.003932276740670204, 0.004985371138900518, -0.015290760435163975, 0.03227125108242035, 0.011655474081635475, -0.005441173445433378, -0.0013497121399268508, 0.00757504953071475, -0.004422980826348066, 0.010878152213990688, 0.0049534947611391544, 0.015538022853434086, 0.0022476566955447197, 0.0026317329611629248, -0.03207607567310333, 0.005935964174568653, 0.0040422240272164345, -0.009899684228003025, 0.008586455136537552, 0.002115560695528984, 0.006352718453854322, -0.003639895236119628, 0.02155368961393833, -0.008643615990877151, 0.0027853588107973337, -0.009248004294931889, -0.002605242421850562, 0.006593618541955948, -0.007879280485212803, -0.011482780799269676, -0.0009295112686231732, 0.0008419842342846096, -0.009636534377932549, -0.011736348271369934, -0.0019521941430866718, 0.020508553832769394, 0.0054850270971655846, -0.018548108637332916, -0.013460099697113037, -0.00197070580907166, -0.002592063043266535, -0.005319521296769381, 0.03732360899448395], [0.0882331132888794, 0.06383109837770462, -0.01810031197965145, 0.06303981691598892, 0.008756386116147041, -0.05399462580680847, 0.0872943326830864, 0.010025765746831894, -0.014516397379338741, -0.07463008910417557, -0.07846727222204208, 0.0669817104935646, 0.0887041836977005, -0.10786295682191849, -0.06910604238510132, 0.04861560836434364, 0.07816486060619354, -0.11775123327970505, -0.07538089901208878, 0.10003690421581268, 0.06880857050418854, 0.04537622630596161, 0.05950755253434181, -0.07480800896883011, 0.06654853373765945, -0.008391289040446281, 0.04684354364871979, 0.12724989652633667, 0.01923721842467785, 0.09175831079483032, -0.011728409677743912, 0.006690180394798517, -0.07072467356920242, 0.08124013245105743, 0.07957854866981506, -0.0029597869142889977, 0.08766601234674454, -0.011424100026488304, 0.01880279928445816, -0.002563329180702567, 0.05125515162944794, -0.060279764235019684, 0.006124295759946108, -0.017844779416918755, -0.0554877333343029, -0.09000245481729507, -0.10426020622253418, -0.04678589850664139, 0.02369227446615696, 0.04581410065293312, 0.02716754376888275, 0.04267087206244469, 0.05975012853741646, 0.0352938212454319, 0.0994160994887352, 0.04753914475440979, 0.04378975182771683, -0.0787460207939148, 0.025829071179032326, 0.013695132918655872, 0.04684370383620262, -0.018954774364829063, 0.09671241044998169, 0.01727329194545746, -0.000560793443582952, -0.02255123481154442, -0.09724064916372299, 0.07578828930854797, -0.004003999289125204, -0.013092869892716408, -0.015728864818811417, 0.06717608124017715, 0.0030830204486846924, -0.12504538893699646, 0.015862468630075455, 0.05366220325231552, 0.08729693293571472, 0.02517743781208992, 0.032637178897857666, 0.022327041253447533, -0.07563627511262894, -0.010264260694384575, 0.055074919015169144, -0.08151315152645111, -0.08498506247997284, -0.11023145914077759, -0.02762737311422825, 0.03044646605849266, -0.09289772063493729, 0.05724488943815231, -0.06176207959651947, -0.0948038250207901, 0.018030768260359764, 0.05697285011410713, 0.04641851782798767, -0.060192134231328964], [0.10371964424848557, 0.0641711950302124, -0.08354824781417847, 0.0023702499456703663, 0.04874385520815849, 0.005668895319104195, -0.07667075097560883, -0.009623668156564236, 0.024701081216335297, 0.03197667747735977, -0.029716677963733673, -0.005024148151278496, 0.0563613697886467, 0.016858374699950218, -0.04229583218693733, -0.021521052345633507, 0.03853413835167885, 0.0633516013622284, 0.05882521718740463, 0.010959874838590622, 0.03293574973940849, -0.024520335718989372, 0.01859264448285103, 0.08237932622432709, -0.018475808203220367, -0.13601234555244446, 0.0034859280567616224, 0.02725650928914547, 0.05058768764138222, -0.07528056204319, -0.13377469778060913, -0.08351172506809235, -0.05982803925871849, -0.04404443874955177, 0.0840916857123375, 0.08320334553718567, 0.08069248497486115, 0.10599317401647568, -0.0049000084400177, -0.03070901893079281, 0.05449625849723816, -0.09402680397033691, -0.02470230497419834, -0.023989995941519737, -0.1446835696697235, -0.0629887804389, -0.03505565598607063, 0.00010655940423021093, 0.10095192492008209, 0.003604630474001169, 0.042104657739400864, -0.04017500579357147, 0.043928857892751694, -0.038825713098049164, 0.03011326864361763, -0.06546777486801147, -0.0576651394367218, 0.07607464492321014, 0.06344573944807053, 0.005964509677141905, -0.044759321957826614, 0.07740603387355804, 0.08534932136535645, -0.07693537324666977, -0.08668181300163269, -0.1247541680932045, -0.03086523339152336, 0.011486921459436417, -0.06769687682390213, 0.020206259563565254, 0.03837570548057556, 0.08302639424800873, -0.04852568730711937, -0.08734766393899918, 0.07598621398210526, -0.021056337282061577, -0.06804532557725906, 0.08557943254709244, 0.07278671115636826, -0.033572837710380554, 0.08913765847682953, 0.053172290325164795, 0.06327123939990997, 0.05696485564112663, 0.04491041973233223, -0.08037589490413666, -0.018622633069753647, -0.051520269364118576, -0.029661010950803757, 0.06542015820741653, -0.023979544639587402, 0.0016327560879290104, -0.06670624762773514, -0.1303340345621109, -0.05593476817011833, -0.0364668145775795], [-0.011429610662162304, -0.03355589136481285, 0.055933866649866104, -0.03829806670546532, -0.05486090108752251, 0.0008608255302533507, -0.04785000532865524, 0.024064430966973305, -0.036742426455020905, 0.05815044045448303, -0.004920670296996832, 0.007252570707350969, -0.04239249229431152, -0.004138731397688389, -0.024771055206656456, -0.0535803884267807, -0.03718084096908569, 0.055844515562057495, 0.029530854895710945, -0.02608366496860981, -0.04782677814364433, -0.056631315499544144, 0.027156751602888107, 0.07294456660747528, 0.06544710695743561, -0.046485986560583115, 0.07571127265691757, 0.06378305703401566, 0.043434884399175644, -0.031083354726433754, -0.03934726119041443, -0.007594936992973089, 0.03964156657457352, 0.029847512021660805, 0.03024933859705925, 0.09791702777147293, 0.044908758252859116, 0.07431836426258087, -0.01612694188952446, -0.00937032513320446, 0.05471983179450035, 0.012998412363231182, -0.036126524209976196, -0.01900091953575611, -0.023009488359093666, 0.05099702253937721, -0.00967421568930149, -0.02401701733469963, -0.004903117660433054, 0.07650838047266006, -0.012421159073710442, -0.0004357901634648442, -0.054024890065193176, -0.05475548654794693, 0.004438565578311682, -0.05097169056534767, 0.057738155126571655, -0.008006825111806393, -0.0004731363442260772, -0.06939386576414108, 0.005574570968747139, 0.060480087995529175, -0.06321476399898529, -0.016395946964621544, 0.029358934611082077, -0.09278786182403564, -0.038898296654224396, -0.05107303336262703, -0.056708019226789474, 0.05405169725418091, -0.008422309532761574, 0.03310748189687729, -0.0027963141910731792, -0.0025107988622039557, 0.02360641583800316, 0.017156438902020454, 0.05619240179657936, 0.06680048257112503, -0.01992925815284252, 0.03633040934801102, -0.0553358718752861, -0.05821581184864044, 0.012287357822060585, 0.030460428446531296, 0.02433287724852562, -0.08771537244319916, 0.014636192470788956, -0.06679309159517288, 0.0749557837843895, 0.02479172870516777, 0.09165365993976593, 0.017679784446954727, -0.043508294969797134, -0.04898417741060257, 0.05839938297867775, 0.02943301759660244], [0.10984063148498535, -0.050321802496910095, -0.022460663691163063, 0.06964439898729324, 0.021674219518899918, -0.015536341816186905, -0.055515628308057785, 0.05409327894449234, -0.035764314234256744, -0.04741930216550827, 0.10728701204061508, 0.014815140515565872, -0.09430695325136185, 0.13582688570022583, 0.05068979412317276, 0.02066277526319027, 0.08564677834510803, -0.10031821578741074, -0.07803737372159958, 0.012874063104391098, -0.098138727247715, -0.08531373739242554, -0.0864410251379013, -0.05850384384393692, 0.02374686487019062, 0.05588161572813988, 0.0244036503136158, -0.05376500263810158, 0.018138209357857704, 0.09249149262905121, 0.06174037232995033, -0.10501772165298462, -0.06927823275327682, -0.11803526431322098, 0.02057669870555401, 0.014309868216514587, -0.045244000852108, -0.04223176836967468, 0.04582236707210541, -0.015068425796926022, 0.01935434713959694, -0.020629389211535454, -0.04428994283080101, 0.004949210677295923, 0.010191747918725014, 0.051795460283756256, -0.05205057933926582, -0.0034302501007914543, -0.0608077235519886, -0.12248153239488602, -0.060136135667562485, 0.006855282932519913, -0.0304279662668705, -0.00747364666312933, -0.05946679413318634, 0.007609798572957516, -0.0751744881272316, 0.11459358036518097, 0.02539558708667755, -0.08472789824008942, 0.06726913154125214, -0.10431624203920364, -0.08364981412887573, 0.038970720022916794, 0.024300144985318184, 0.07382027804851532, -0.020909547805786133, -0.00834509078413248, 0.05932706966996193, 0.12146908044815063, -0.053472187370061874, -0.0990254208445549, 0.1272846907377243, 0.0662200003862381, 0.0272898580878973, 0.07418453693389893, -0.06289587169885635, -0.0355331227183342, 0.02252770960330963, 0.016491692513227463, -0.06011543795466423, 0.010445013642311096, 0.08388189226388931, 0.06916940957307816, 0.04112203046679497, 0.08608732372522354, -0.01062583364546299, 0.10289789736270905, 0.0036772259045392275, -0.04529128968715668, -0.09074726700782776, -0.08003748208284378, -0.09456566721200943, -0.00644292589277029, 0.06354199349880219, -0.05106038227677345], [0.010255299508571625, -0.0416441336274147, -0.020254548639059067, 0.06315356492996216, 0.006222855299711227, 0.019989648833870888, 0.02023998834192753, 0.04152077063918114, -0.046069350093603134, 0.08918944001197815, 0.013792445883154869, 0.05291636288166046, 0.02943483740091324, 0.008434798568487167, -0.013776913285255432, -0.030361805111169815, 0.025061244145035744, -0.01761099137365818, -0.038865718990564346, -0.0240250825881958, 0.005896247923374176, 0.0721391811966896, 0.029908185824751854, 0.03886554762721062, -0.005917081609368324, -0.01737877167761326, 0.012114659883081913, 0.02661791816353798, 0.0473310723900795, -0.06345846503973007, 0.005040419287979603, 0.009158166125416756, -0.005125174764543772, 0.00424688495695591, -0.01913178339600563, 0.03578578680753708, -0.06243401765823364, -0.001900147064588964, -0.01990891993045807, -0.006743691395968199, 0.018641522154211998, -0.011516996659338474, 0.04842418059706688, 0.023480134084820747, 0.03979502245783806, 0.025951363146305084, -0.02027762122452259, 0.06495318561792374, 0.002001534216105938, -0.02204860933125019, -0.01841774210333824, 0.03148868307471275, -0.05509738624095917, 0.0025786980986595154, 0.009453866630792618, 0.05822119116783142, -0.0010135774500668049, -0.039087068289518356, -0.019178612157702446, -0.07331207394599915, -0.016924725845456123, 0.018163036555051804, -0.003316278802230954, 0.020675785839557648, 0.08855059742927551, 0.04543912038207054, 0.05622659996151924, 0.0055442615412175655, 0.03142320364713669, -0.00040306925075128675, 0.020484209060668945, 0.02738482691347599, 0.018251892179250717, 0.04635252803564072, -0.03164559602737427, 0.016669103875756264, -0.005259240046143532, -0.027791980654001236, -0.04589597508311272, -0.028699319809675217, -0.0013903541257604957, -0.062073156237602234, 0.009463811293244362, 0.006865214090794325, 0.0537119060754776, 0.05149364843964577, 0.02818351611495018, 0.06387081742286682, 0.01524970680475235, -0.005692449864000082, 0.021704817190766335, -0.04498250409960747, -0.023769669234752655, -0.006892039440572262, 0.007922972552478313, -0.016290588304400444], [-0.029415138065814972, 0.04237646237015724, 0.08398142457008362, 0.05044638738036156, 0.0036065641324967146, 0.08066436648368835, -0.10056803375482559, 0.07842490077018738, 0.09266065061092377, 0.031933676451444626, -0.1071106344461441, 0.09522310644388199, 0.03763415664434433, -0.012538750655949116, 0.04264827072620392, 0.011086946353316307, 0.04625046253204346, -0.12027877569198608, 0.029026055708527565, 0.012779820710420609, 0.06296110153198242, -0.03956184908747673, -0.06086568906903267, -0.016356635838747025, 0.006539289839565754, 0.009262439794838428, -0.061150386929512024, 0.10485538095235825, -0.007221453823149204, 0.030156319960951805, -0.10906434804201126, -0.06163797900080681, -0.012622488662600517, 0.08921099454164505, 0.11467147618532181, 0.05126542970538139, -0.06787560135126114, -0.031297408044338226, -0.0800834372639656, 0.07752147316932678, 0.004077582620084286, 0.01489097811281681, 0.023641284555196762, 0.0903051421046257, 0.04347580298781395, -0.019813671708106995, -0.08376862853765488, -0.09649685770273209, -0.06823574751615524, 0.025146277621388435, -0.04609222337603569, -0.10780823230743408, -0.060701701790094376, -0.04751474782824516, 0.05917137861251831, -0.07708761841058731, 0.08461069315671921, -0.03683139383792877, 0.05582582578063011, 0.08657604455947876, -0.04739895462989807, -0.03864529728889465, 0.023425661027431488, 0.030476387590169907, -0.07817865908145905, 0.07243160903453827, 0.065558061003685, 0.01843894273042679, 0.06540486216545105, -0.044763412326574326, 0.019157493487000465, -0.04866261035203934, -0.045437540858983994, 0.011833746917545795, -0.009738219901919365, -0.047903381288051605, 0.07055909186601639, -0.044074609875679016, 0.00026980487746186554, -0.014619601890444756, -0.0656878650188446, 0.010789352469146252, -0.03360811620950699, 0.09299346059560776, -0.11045921593904495, 0.025955980643630028, 0.024064362049102783, 0.08081144094467163, 0.06108738109469414, -0.05144685134291649, 0.011291048489511013, -0.04514523595571518, -0.052204679697752, -0.04386017844080925, 0.0514666773378849, -0.025285976007580757], [-0.0073756445199251175, 0.04033197462558746, -0.07717031985521317, -0.0347135029733181, 0.03917534649372101, -0.06113245338201523, -0.01068017166107893, 0.03639736771583557, -0.04688383266329765, -0.050016749650239944, 0.01089838519692421, -0.04817625507712364, 0.0108425822108984, 0.044626060873270035, 0.027592038735747337, 0.06597703695297241, 0.013379891403019428, -0.033478740602731705, -0.04662991315126419, 0.08992323279380798, 0.008309771306812763, 0.013389816507697105, 0.08835180103778839, 0.05806276947259903, 0.07103578746318817, -0.07718344777822495, 0.061779625713825226, 0.07249268889427185, 0.02816782332956791, 0.09231933206319809, 0.00831423420459032, 0.024418503046035767, -0.028980275616049767, 0.1073646992444992, 0.08742934465408325, 0.06319610029459, 0.048974040895700455, -0.03957417607307434, -0.07437169551849365, 0.03695743903517723, 0.03295646607875824, -0.07319219410419464, 0.08677900582551956, 0.06103650853037834, -0.06508894264698029, 0.03921936824917793, 0.06307966262102127, -0.07040522992610931, 0.028435228392481804, 0.09705323725938797, -0.013953188434243202, 0.027720002457499504, 0.06777002662420273, 0.0897970050573349, -0.021704144775867462, 0.04362306743860245, 0.024438759312033653, -0.02107269875705242, 0.10392341017723083, -0.018306320533156395, -0.05913439393043518, 0.06223496049642563, -0.03654726222157478, -0.04517547786235809, -0.0232872124761343, -0.020538071170449257, -0.057862285524606705, 0.08143185079097748, -0.04187430068850517, -0.021367833018302917, -0.05570947006344795, -0.0014199777506291866, 0.022445177659392357, -0.028577381744980812, 0.018752198666334152, 0.013227014802396297, -0.07330124825239182, 0.011152874678373337, 0.02136324718594551, 0.07661687582731247, 0.029721323400735855, 0.04350423440337181, 0.05217621475458145, -0.016494588926434517, 0.07227901369333267, -0.005957840010523796, 0.019000563770532608, 0.04630846902728081, -0.065420001745224, -0.007358881179243326, 0.07720260322093964, 0.003988618031144142, 0.037641189992427826, 0.08355195075273514, 0.053306303918361664, -0.04595200717449188], [0.022959306836128235, -0.01004728302359581, -0.0018320097588002682, 0.024303343147039413, -0.0348040796816349, -0.010709185153245926, 0.0065369755029678345, -0.024274533614516258, 0.09948534518480301, -0.018269088119268417, 0.03584738075733185, -0.019249387085437775, -0.013098289258778095, 0.05213836953043938, 0.01805965229868889, -0.03640175238251686, 0.10586857795715332, -0.018637564033269882, -0.004060071427375078, -0.03243919461965561, 0.014252958819270134, -0.04457036778330803, -0.035289715975522995, -0.01060442440211773, 0.019786037504673004, 0.04217774048447609, 0.02212332747876644, 4.377330333227292e-05, 0.007760744076222181, -0.008364827372133732, 0.02638842537999153, 0.014264668338000774, 0.004031845834106207, -0.053266823291778564, -0.028977714478969574, -0.050765395164489746, -0.06874728947877884, -0.0016519797500222921, 0.046718522906303406, -0.006454812828451395, 0.01809842139482498, 0.04286698251962662, -0.017870495095849037, 0.045264486223459244, 0.004565773531794548, -0.019581498578190804, 0.05431746318936348, 0.005194912198930979, -0.028733491897583008, -0.05389287322759628, -0.025566527619957924, -0.009630902670323849, -0.04433772712945938, -0.030357183888554573, -0.030323132872581482, -0.04370081052184105, -0.03869282081723213, 0.1194472461938858, 0.03880142793059349, -0.03500009700655937, 0.09543834626674652, -0.04206792637705803, 0.001520024728961289, -0.046245988458395004, -0.027877546846866608, -0.003094941144809127, 0.016495205461978912, -0.06441760063171387, 0.029797902330756187, 0.0315777063369751, -0.02856065146625042, -0.03300345316529274, 0.05794348567724228, 0.009586029686033726, -0.009497033432126045, -0.01710757426917553, 0.010750767774879932, 0.03851013258099556, -0.030032966285943985, -0.02783389575779438, -0.020731033757328987, -0.022443780675530434, 0.06252999603748322, -0.04230044409632683, -0.015596254728734493, 0.026014788076281548, 0.021111687645316124, 0.022688517346978188, 0.0700632631778717, -0.056842245161533356, -0.02381049282848835, -0.043636150658130646, -0.015067650005221367, 0.005296570248901844, -0.03618999570608139, 0.03919827565550804], [0.0338149294257164, -0.02924991212785244, 0.06821519136428833, 0.014151164330542088, -0.08287274837493896, 0.00670971255749464, -0.044959720224142075, -0.035475462675094604, -0.06807370483875275, -0.018259644508361816, 0.028526969254016876, 0.02078845724463463, 0.03574787452816963, -0.02627139538526535, 0.04172133654356003, 0.0007573707844130695, -0.03553509712219238, -0.013417919166386127, -0.03924509882926941, 0.003053204622119665, -0.0029448929708451033, 0.009712751023471355, 0.013552162796258926, -0.010433390736579895, 0.053254567086696625, 0.007208087015897036, -0.021519850939512253, 0.010679089464247227, 0.05017286539077759, 0.036795489490032196, -0.034693919122219086, 0.024884922429919243, -0.00794347282499075, 0.01464946661144495, -0.0014555997913703322, 0.01268603652715683, 0.005913023371249437, -0.039992786943912506, -0.025496309623122215, 0.044650133699178696, -0.03566417843103409, 0.01922430656850338, -0.04645418003201485, 0.010344483889639378, -0.024883940815925598, 0.0578276701271534, 0.07051973789930344, -0.017255596816539764, -0.029659122228622437, -0.06153205782175064, 0.05686049908399582, -0.05621517449617386, 0.045723430812358856, -0.04978532716631889, -0.0171009860932827, 0.024705855175852776, -0.07747570425271988, -0.002687192754819989, -0.042860884219408035, 0.04089401662349701, -0.02917959727346897, -0.05385218933224678, -0.049928899854421616, 0.05248904228210449, 0.03354691341519356, 0.05504148080945015, 0.02081652544438839, -0.05151696875691414, 0.06529200822114944, -0.019474312663078308, 0.010192285291850567, -0.041786078363657, -0.017628131434321404, -0.018050307407975197, 0.0019041941268369555, 0.028006846085190773, -0.07306535542011261, 0.03492734953761101, -0.06486072391271591, -0.006574616301804781, 0.04327687621116638, 0.02681019715964794, 0.0314875990152359, -0.025032423436641693, 0.056608669459819794, 0.06868919730186462, -0.061673205345869064, -0.02899445779621601, 0.042917124927043915, -0.0001361525064567104, 0.05109279975295067, -0.0074090901762247086, -0.05943843349814415, 0.005665682256221771, -0.04529520124197006, -0.03342962637543678], [0.03595508262515068, -0.03237275034189224, -0.026384450495243073, -0.04414476454257965, 0.06212494149804115, 0.024176862090826035, -0.03465767949819565, -0.06124182417988777, 0.03814743831753731, 0.02884873002767563, -0.050231628119945526, 0.0031560203060507774, 0.008727717213332653, -0.06550318747758865, 0.011121850460767746, -0.02409856766462326, 0.006893771700561047, -0.07882846891880035, -0.0032198287080973387, -0.01099557988345623, 0.05019717290997505, -0.052337903529405594, 0.015391461551189423, -0.018860038369894028, -0.023838862776756287, -0.04715766757726669, 0.03461618348956108, 0.0215168297290802, -0.068992018699646, 0.0017907096771523356, -0.06554360687732697, -0.01489304844290018, -0.01696552149951458, -0.007646093610674143, 0.02960125543177128, -0.008473595604300499, -0.044713232666254044, 0.04669264331459999, -0.01129226852208376, 0.01543139573186636, -0.030768021941184998, -0.034419946372509, 0.030856193974614143, -0.012604884803295135, -0.0009171593701466918, -0.020001592114567757, -0.020429514348506927, -0.03556344285607338, 0.02124664932489395, 0.04132780060172081, -0.02257625013589859, -0.01651613786816597, 0.017839616164565086, 0.06533259898424149, -0.024691220372915268, 0.010792508721351624, -0.011733180843293667, 0.031079789623618126, 0.0405653677880764, -0.012083912268280983, -0.009161734953522682, 0.03143564984202385, -0.014236941002309322, -0.0037232316099107265, -0.01886124350130558, -0.005472245160490274, -0.05376940220594406, 0.05379210039973259, -0.011215630918741226, -0.009814433753490448, -0.005549646448343992, -0.028109725564718246, -0.001136747538112104, -0.03673989325761795, 0.005030810367316008, 0.0034784895833581686, 0.016436969861388206, 0.030696792528033257, -0.008863444440066814, 0.030602512881159782, -0.00578945642337203, 0.0432131290435791, 0.025961026549339294, 0.07209280878305435, 0.02724124677479267, 0.02207098715007305, -0.011754225939512253, 0.021680019795894623, -0.06524056196212769, 0.0474604107439518, -0.01256265863776207, 0.046584222465753555, -0.03194207325577736, -0.042472053319215775, -0.023303980007767677, 0.021395409479737282], [0.03405867516994476, 0.013654774986207485, 0.0018394122598692775, 0.009071781300008297, 0.04766260087490082, 0.02311399206519127, -0.017008215188980103, 0.023887863382697105, 0.013552135787904263, -0.04763351380825043, 0.0012840289855375886, 0.025491192936897278, 0.026077035814523697, -0.0017803231021389365, 0.01957656256854534, 0.002287753392010927, 0.009542496874928474, -0.026596734300255775, 0.00422410573810339, 0.05792919173836708, -0.013843778520822525, 0.023895541206002235, 0.0076915244571864605, 0.04595721513032913, -0.04634230211377144, -0.006662707310169935, 0.006102107465267181, 0.040196530520915985, -0.05802769586443901, 0.0020413645543158054, -0.02319878339767456, 0.0017281521577388048, -0.01022528950124979, 0.05785899609327316, -0.00797992292791605, 0.008122226223349571, 0.02462405152618885, 0.016334302723407745, -0.003915239591151476, -0.014075144194066525, 0.03193352371454239, -0.0064888205379247665, 0.010350423865020275, 0.039476241916418076, 0.013110746629536152, 0.01550797838717699, -0.04947496950626373, -0.013198117725551128, 0.02930953912436962, 0.0665055364370346, 0.01740288734436035, -0.03384486585855484, 0.03693775832653046, 0.04985782131552696, 0.0052539994940161705, -0.05267289653420448, -0.029490111395716667, -0.022162308916449547, 0.03774695098400116, -0.0015204742085188627, -0.0012650333810597658, 0.0490366630256176, 0.01431279443204403, -0.04221608489751816, 0.015289845876395702, -0.06345119327306747, -0.01739877462387085, 0.03009854070842266, 0.022471437230706215, -0.030042482540011406, -0.0002820587542373687, 0.05219659209251404, -0.0018998521845787764, 0.024529125541448593, 0.007567569613456726, -0.0029731758404523134, -0.019464990124106407, 0.009819505736231804, 0.032296206802129745, 0.02716459333896637, 0.026713736355304718, 0.038030967116355896, -0.016565261408686638, 0.019264401867985725, -0.022374030202627182, -0.018784690648317337, -0.06147660315036774, -0.006785400211811066, -0.05569276213645935, -0.016792302951216698, -0.004081714432686567, -0.009778033941984177, 0.041219308972358704, 0.04497133940458298, 0.04880028963088989, -0.00931498408317566], [0.09216558188199997, 0.09052331745624542, -0.05571582168340683, -0.030826404690742493, -0.020495614036917686, 0.09753800928592682, 0.040257833898067474, -0.05143095180392265, 0.06788706034421921, -0.004725500009953976, 0.00913701206445694, -0.015517128631472588, -0.009129922837018967, -0.045571181923151016, 0.012375961989164352, 0.07103896886110306, -0.012935198843479156, 0.044648248702287674, -0.055758364498615265, 0.01145909819751978, 0.08757352828979492, -0.051885008811950684, -0.07425101846456528, -0.05304894968867302, 0.0292813740670681, 0.0207128394395113, -0.03046531416475773, 0.06007525697350502, -0.0910932794213295, -0.030883872881531715, -0.025071261450648308, -0.06955046951770782, 0.03674433380365372, -0.05249522998929024, 0.01728009060025215, 0.03143162652850151, 0.09671265631914139, -0.045382946729660034, -0.0687393769621849, 0.06776569783687592, 0.017759336158633232, -0.0772601068019867, 0.07225430756807327, 0.07036571204662323, 0.0005295888986438513, -0.013862543739378452, 0.04728059470653534, 0.06501222401857376, -0.027707461267709732, 0.06363742053508759, 0.012520913034677505, -0.0841052308678627, 0.08328617364168167, 0.016284186393022537, -0.04191412776708603, 0.03507937490940094, -0.07679520547389984, 0.03377356752753258, 0.021625099703669548, -0.030402276664972305, 0.0027802148833870888, 0.06523387134075165, -0.027177177369594574, 0.07791034877300262, -0.1014556959271431, -0.05997711420059204, -0.10146480798721313, 0.10972945392131805, 0.04025353118777275, -0.09198754280805588, -0.006962968967854977, 0.07343743741512299, -0.06381379812955856, -0.01883922517299652, -0.027667749673128128, -0.07414986938238144, 0.05953928828239441, 0.033426858484745026, -0.0535060353577137, -0.056737471371889114, -0.02759251929819584, -0.02185007371008396, 0.04455959051847458, 0.014054200612008572, 0.02156762219965458, -0.07710584998130798, -0.0797450914978981, -0.07652190327644348, -0.10788014531135559, -0.027579039335250854, 0.003571604611352086, -0.012729865498840809, -0.08044275641441345, -0.00624054903164506, 0.07640448212623596, -0.03353281319141388], [-0.06056540459394455, 0.03649340197443962, -0.014653575606644154, 0.018794992938637733, -0.1100822389125824, 0.0660322904586792, -0.07654711604118347, -0.04393605515360832, -0.040563128888607025, 0.04359891265630722, -0.051806941628456116, -0.04038960114121437, 0.05318080633878708, -0.03225640580058098, 0.04114316403865814, 0.035759493708610535, 0.0844741240143776, 0.07086025178432465, -0.055278874933719635, 0.04774823039770126, 0.06756629049777985, -0.028352854773402214, 0.006273830775171518, 0.0016070068813860416, -0.03702796995639801, -0.011064233258366585, -0.038155581802129745, -0.024271918460726738, -0.06531915813684464, -0.07176751643419266, 0.025407128036022186, -0.05436747521162033, 0.020669423043727875, 0.016905203461647034, -0.0015507359057664871, -0.05308259278535843, 0.07600467652082443, -0.06680683046579361, 0.00332109397277236, 0.07207754999399185, 0.08815161883831024, -0.014209859073162079, -0.11387434601783752, -0.022489380091428757, 0.05996975675225258, -0.09859280288219452, -0.05943354591727257, 0.02320416457951069, -0.04997707158327103, -0.022750461474061012, -0.026003072038292885, 0.062068913131952286, -0.03109986148774624, -0.09782986342906952, -0.09413059800863266, -0.0909491702914238, -0.045344382524490356, 0.025051115080714226, 0.013535862788558006, -0.10092665255069733, -0.02991228736937046, -0.04695454612374306, -0.007048399653285742, -0.02155202440917492, -0.03481915593147278, 0.05945143476128578, 0.008261333219707012, -0.07088791579008102, 0.0597115084528923, 0.07508392632007599, 0.08865737169981003, -0.07141008228063583, -0.07658836245536804, 0.10396704822778702, -0.056685905903577805, -0.056693267077207565, -0.019620073959231377, -0.08691298961639404, 0.004499149043112993, -0.03703383356332779, 0.06053543835878372, -0.00798280443996191, -0.1011263057589531, 0.04989657923579216, -0.026584619656205177, 0.004425691440701485, -0.094328872859478, 0.013509349897503853, 0.08147383481264114, -0.08035081624984741, -0.06014810875058174, -0.04623698443174362, -0.10810866951942444, -0.0155533067882061, -0.08814455568790436, -0.005115262698382139], [0.00901854783296585, -0.12433667480945587, -0.05545912683010101, -0.017130715772509575, -0.02778620645403862, -0.07753876596689224, -0.014500056393444538, -0.02069251425564289, -0.0444643571972847, -0.10199170559644699, -0.031403105705976486, -0.014078274369239807, 0.017632313072681427, -0.10768667608499527, 0.030138308182358742, -0.046826738864183426, -0.06310757249593735, 0.0855865553021431, -0.05152614042162895, 0.01323368214070797, -0.024207133799791336, -0.013908755034208298, -0.06250809133052826, -0.005604881327599287, -0.008430429734289646, -0.10367952287197113, 0.017782315611839294, 0.033746715635061264, -0.034854065626859665, -0.11906511336565018, -0.08841419965028763, 0.06294357776641846, 0.02414306066930294, -0.009361917153000832, 0.046354129910469055, -0.03605727478861809, -0.004419985692948103, -0.0063187796622514725, -0.04940151795744896, -0.08736735582351685, 0.0020216747652739286, -0.04866471886634827, -0.0838370993733406, -0.01912429742515087, -0.007584094535559416, -0.046202652156353, 0.020536096766591072, -0.018654778599739075, -0.03466114401817322, -0.05002810060977936, -0.013005135580897331, -0.01458286214619875, 0.05358511209487915, -0.028600409626960754, 0.016723427921533585, -0.03027457371354103, 0.00514996936544776, 0.0019169618608430028, -0.08865434676408768, 0.005831345450133085, -0.05527079850435257, -0.007588808424770832, -0.021995345130562782, -0.07363299280405045, -0.026385677978396416, -0.013022043742239475, 0.009830350987613201, -0.06343270093202591, 0.052333198487758636, -0.09278138726949692, -0.04544200748205185, -0.014514544978737831, -0.08216919749975204, 0.03739113360643387, -0.005073802545666695, -0.08285871148109436, -0.010747824795544147, -0.0753249004483223, -0.05369541794061661, 0.002710300264880061, -0.07094427198171616, 0.008166249841451645, -0.07720540463924408, -0.06759648025035858, -0.037466537207365036, -0.016854645684361458, -0.014305721037089825, -0.007316367235034704, -0.10910377651453018, 0.01483350433409214, -0.05202990025281906, -0.06252133101224899, -0.02988603338599205, -0.024887025356292725, -0.028106773272156715, -0.042028188705444336], [0.1188916265964508, -0.11465398967266083, -0.041746482253074646, 0.0018284434918314219, -0.036739662289619446, 0.052966464310884476, -0.08437014371156693, -0.0031840845476835966, -0.02591724880039692, -0.10634062439203262, 0.06698260456323624, -0.04263855516910553, 0.04150015488266945, 0.08791345357894897, 0.08642503619194031, -0.052300695329904556, 0.051416944712400436, -0.08073505759239197, -0.0177445150911808, -0.029618071392178535, 0.0018416596576571465, -0.02823004499077797, 0.012979378923773766, 0.037698131054639816, 0.06021580100059509, 0.036461856216192245, 0.008032872341573238, -0.026466410607099533, -0.061345767229795456, -0.06573080271482468, 0.005430840887129307, -0.04569588601589203, 0.07615223526954651, -0.02325385995209217, -0.03570026531815529, 0.025856563821434975, -0.03428775444626808, -0.0597124882042408, 0.07501374185085297, 0.08679386973381042, 0.08346571028232574, -0.030279597267508507, -0.015629762783646584, -0.013373191468417645, -0.032866738736629486, 0.04711480811238289, -0.019173091277480125, -0.039399050176143646, -0.12269370257854462, -0.06119680032134056, 0.013041331432759762, -0.05470578745007515, -0.07139544934034348, -0.044265516102313995, -0.04505286365747452, -0.0033955953549593687, -0.09355636686086655, 0.09479622542858124, 0.07187187671661377, -0.013236799277365208, 0.149785116314888, -0.11911571770906448, 0.03629368543624878, 0.03314030542969704, -0.03033384121954441, -0.006378932856023312, 0.09522631019353867, -0.09643930941820145, -0.05044260621070862, 0.1075742170214653, -0.02283012680709362, -0.06278690695762634, 0.11682872474193573, 0.013626927509903908, 0.06261438131332397, 0.026851166039705276, -0.09155590087175369, 0.07572094351053238, 0.036777764558792114, 0.024380410090088844, -0.04826853796839714, -0.09553202986717224, -0.08157119154930115, -0.011078611016273499, 0.013297646306455135, 0.0015538455918431282, -0.052395861595869064, 0.007076547481119633, -0.039458271116018295, -0.023453230038285255, -0.07717511802911758, -0.08414427191019058, 0.06184026598930359, -0.06493762135505676, 0.06420319527387619, 0.033189963549375534], [0.07269901782274246, 0.03710978105664253, 0.041608307510614395, 0.09009500592947006, -0.0802001878619194, 0.04618058726191521, 0.04036737233400345, 0.043660763651132584, 0.0694861188530922, -0.04382157325744629, 0.05451392009854317, -0.013881581835448742, -0.00833023339509964, 0.079165019094944, 0.05154377594590187, 0.015289421193301678, 0.15024173259735107, -0.02688460797071457, -0.016765296459197998, -0.051083773374557495, 0.049441028386354446, -0.015218707732856274, 0.05084525793790817, -0.03208266943693161, -0.07862988114356995, 0.15193773806095123, -0.057804401963949203, 0.02405613847076893, -0.024483393877744675, 0.06932783871889114, 0.0339897982776165, -0.07730244845151901, 0.051468729972839355, -0.08565515279769897, -0.013681981712579727, 0.003966381307691336, -0.09001179039478302, -0.02309093251824379, 0.0957920178771019, -0.001383200054988265, -0.026776926591992378, -0.06088661029934883, 0.04254322126507759, -0.05779392272233963, 0.011264724656939507, 0.04584168270230293, -0.05936791002750397, 0.042615242302417755, -0.047100894153118134, -0.11544725298881531, 0.0005977896507829428, 0.014430203475058079, -0.058817990124225616, -0.067972831428051, -0.0608501099050045, -0.004571732133626938, -0.12270881980657578, 0.07939151674509048, 0.0014692061813548207, 0.010318208485841751, 0.1307123452425003, -0.09901683032512665, 0.01940101757645607, 0.012031479738652706, 0.012943043373525143, 0.015175753273069859, -0.012876956723630428, -0.017711499705910683, 0.0275937058031559, -0.0019421076867729425, -0.04109269753098488, -0.042445119470357895, 0.028885571286082268, 0.05151500925421715, -0.04380561411380768, -0.055288251489400864, -0.08022846281528473, 0.03364601358771324, -0.039588816463947296, -0.04472879320383072, -0.020453795790672302, 0.0038787859957665205, -0.0026799875777214766, -0.019700530916452408, 0.06138227507472038, 0.00902770645916462, 0.03779846802353859, 0.06934531778097153, -0.04211682453751564, -0.10939417034387589, -0.04738413915038109, -0.022643961012363434, -0.07879345864057541, 0.0011397682828828692, 0.013543420471251011, -0.0515967421233654], [-0.01534613873809576, 0.010276411660015583, -0.06027115136384964, -0.03635319322347641, 0.055930037051439285, 0.011759571731090546, -0.0462806336581707, -0.03489087522029877, -0.04524732008576393, -0.026032723486423492, -0.032557275146245956, -0.015590674243867397, -0.01494805607944727, -0.048739511519670486, -0.01052634697407484, -0.0160862747579813, -0.0018416724633425474, 0.006693130824714899, 0.031078150495886803, 0.0635022446513176, -0.00831571314483881, 0.015356221236288548, -0.015054463408887386, 0.04087206721305847, 0.024204816669225693, -0.02750074677169323, 0.006381587591022253, 0.008362687192857265, -0.03644988313317299, -0.00881221704185009, -0.03720555081963539, -0.02938278764486313, -0.04584136977791786, -0.004791360814124346, 0.020847328007221222, 0.07899696379899979, -0.004795495420694351, 0.010831063613295555, 0.0017931892070919275, 0.037321899086236954, -0.026188727468252182, -0.026641955599188805, -0.021152757108211517, -0.007042477838695049, -0.03116399236023426, -0.008112356998026371, 0.0001721395383356139, -0.019015109166502953, 0.012403752654790878, 0.01689746417105198, 0.05456157028675079, -0.009535419754683971, -0.008258631452918053, 0.034708306193351746, -0.010576418600976467, -0.018371567130088806, -0.010285071097314358, 0.023419227451086044, 0.013907031156122684, -0.025626642629504204, -0.005524773616343737, 0.0056826272048056126, 0.02981257438659668, -0.04331711307168007, -0.025790080428123474, -0.022023003548383713, -0.049309540539979935, 0.04971042275428772, -0.0074563235975801945, 0.01792902685701847, -0.03564586117863655, 0.0019802204333245754, 0.007858398370444775, -0.03219108283519745, -0.009044189937412739, 0.01815374754369259, -0.04188244417309761, -0.004709706176072359, -0.016439585015177727, 0.012871082872152328, 0.026003902778029442, 0.04360732063651085, 0.06787697970867157, 0.01418341789394617, 0.02092771604657173, -0.04094449803233147, 0.04226996749639511, -0.01136869378387928, -0.0011203739559277892, -0.049388587474823, -0.03573136404156685, 0.05652954801917076, -0.03869634494185448, -0.04730435088276863, -0.004143555648624897, 0.05015576630830765], [-0.0707668885588646, 0.06339004635810852, 0.05132881551980972, 0.05562940984964371, 0.04181552678346634, -0.0387321412563324, 0.07464825361967087, 0.0753849595785141, 0.02230282872915268, 0.07571138441562653, -0.016697237268090248, 0.049739498645067215, 0.049201298505067825, 0.0837806761264801, 0.032815564423799515, 0.08465658128261566, -0.06587684154510498, 0.052987296134233475, -0.02916351519525051, -0.08685965836048126, 0.0811362937092781, -0.023054854944348335, -0.04298965260386467, 0.013781375251710415, -0.04147425293922424, -0.022873299196362495, -0.04838314652442932, -0.010659448802471161, -0.011320043355226517, -0.08459682017564774, -0.05698328837752342, 0.05712775141000748, 0.009447271004319191, -0.08312961459159851, 0.05388093739748001, -0.0006786783924326301, -0.09516873955726624, 0.017507024109363556, 0.02921551652252674, -0.06883834302425385, 0.025544309988617897, 0.06140086427330971, 0.0852099359035492, 0.02338438481092453, 0.005796812009066343, 0.09322725981473923, 0.02917022444307804, 0.054756034165620804, -0.028241008520126343, -0.07249023020267487, 0.08133787661790848, 0.004122103564441204, 0.059320904314517975, 0.03425144776701927, 0.019060539081692696, -0.07769694924354553, 0.04512496665120125, 0.06670232862234116, 0.008008253760635853, 0.07699369639158249, -0.06927847117185593, -0.05613911896944046, -0.028731880709528923, 0.044493380934000015, -0.00544008007273078, -0.024669885635375977, 0.07160204648971558, -0.056105658411979675, 0.08211938291788101, 0.053823184221982956, 0.013480161316692829, 0.05218852311372757, 0.0327017717063427, -0.01735568791627884, 0.027841979637742043, -0.07388730347156525, -0.03251304104924202, -0.023901352658867836, -0.05868365615606308, -0.039969008415937424, 0.02905276231467724, -0.09132381528615952, -0.05454553663730621, -0.049231480807065964, -0.042240262031555176, 0.055254433304071426, 0.01612432859838009, 0.034147970378398895, 0.003560679266229272, 0.0735207125544548, 0.03356146439909935, 0.008252034895122051, -0.020767847076058388, -0.009736762382090092, 0.0626847967505455, -0.010434945113956928], [0.04568903148174286, 0.07527905702590942, -0.08302092552185059, -0.09107255935668945, -0.01275184378027916, -0.004052038304507732, -0.021038001403212547, 0.06916802376508713, 0.062472984194755554, 0.026367248967289925, -0.013802715577185154, -0.058901071548461914, 0.1013059988617897, -0.05758918821811676, -0.014947258867323399, 0.08353523910045624, -0.018053624778985977, -0.08542860299348831, 0.10526878386735916, 0.06265728920698166, -0.0175502747297287, 0.020397748798131943, -0.009668398648500443, -0.010172703303396702, 0.011332082562148571, -0.0843254029750824, -0.013414939865469933, 0.078731007874012, -0.08117436617612839, 0.09997926652431488, -0.10765446722507477, 0.03616265580058098, -0.08526420593261719, 0.0221723485738039, -0.02688673324882984, -0.003189093666151166, 0.08869668841362, 0.09349877387285233, -0.031390152871608734, 0.09636618942022324, 0.07214779406785965, 0.059821631759405136, 0.01781213842332363, -0.03714435547590256, -0.009623023681342602, -0.051561955362558365, -0.028380202129483223, 0.046237438917160034, -0.07412385940551758, -0.010651717893779278, -0.0267327893525362, -0.03515397384762764, 0.053596340119838715, -0.004005730617791414, 0.10143748670816422, -0.0005951159400865436, 0.04233236983418465, -0.047393471002578735, -0.0012144279899075627, 0.08856014907360077, -0.06223836913704872, -0.005755399353802204, 0.0642981231212616, 0.10194677859544754, -0.02602982334792614, 0.04111349210143089, -0.01244849432259798, -0.013730672188103199, -0.08652255684137344, 0.04507320374250412, 0.06892377883195877, 0.05739987641572952, 0.02749968320131302, -0.05252029001712799, -0.002045714296400547, -0.08193039894104004, -0.007047959137707949, 0.0468604676425457, 0.09444306790828705, 0.0020059954840689898, 0.0002172483946196735, -0.014253262430429459, -0.07141660153865814, 0.038804568350315094, 0.08007092773914337, -0.07126527279615402, -0.04468797892332077, 0.02814936265349388, -0.006976321805268526, -0.012912971898913383, -0.05348583310842514, 0.023002926260232925, 0.07543227076530457, -0.08930721879005432, -0.0587783120572567, 0.09782518446445465], [-0.04523913934826851, -0.04377937689423561, -0.06861724704504013, 0.0038135359063744545, -0.023130124434828758, -0.026764487847685814, -0.01905432902276516, 0.06704798340797424, 0.09846425801515579, 0.040350403636693954, 0.004783195909112692, -0.030321530997753143, -0.038544442504644394, 0.03148876130580902, -0.06084275618195534, -0.030424965545535088, -0.03779464215040207, -0.01595417596399784, -0.049449726939201355, -0.0476396344602108, 0.029867704957723618, -0.06555143743753433, -0.014928090386092663, -0.009775341488420963, -0.04440687969326973, 0.00600102823227644, 0.1069602221250534, 0.05890211835503578, 0.04807440936565399, -0.04652995616197586, -0.06583661586046219, 0.04584450647234917, -0.03437000513076782, 0.10160965472459793, 0.0391402505338192, -0.004409345332533121, 0.10596900433301926, 0.013494950719177723, -0.024474773555994034, 0.023454394191503525, 0.009729385375976562, 0.013983253389596939, 0.045518502593040466, -0.01352164801210165, -0.09891611337661743, 0.05551321059465408, 0.04816246032714844, 0.0030720513314008713, -0.06862085312604904, -0.026501011103391647, 0.006904473528265953, -0.08450504392385483, 0.07984878867864609, -0.027948565781116486, -0.046106331050395966, 0.038989655673503876, 0.05643343925476074, -0.04378388077020645, 0.013706616126000881, 0.028780722990632057, -0.05745648220181465, 0.0391804501414299, -0.060933489352464676, 0.08445869386196136, -0.09635937958955765, -0.09531685709953308, -0.1273476928472519, 0.0021782172843813896, -0.057714663445949554, -0.010013262741267681, -0.03222702443599701, 0.0710243359208107, 0.059072837233543396, 0.025914311408996582, 0.0728512555360794, -0.05733416602015495, -0.016642652451992035, -0.0521366149187088, 0.018840432167053223, 0.07234344631433487, 0.0055129267275333405, 0.039886120706796646, 0.10289125889539719, 0.061393655836582184, -0.01877198927104473, -0.09192360937595367, 0.05553346872329712, 0.05716106668114662, -0.0054479846730828285, 0.01792534999549389, -0.003392576240003109, 0.13313670456409454, 0.01736258901655674, -0.049536388367414474, 0.015403256751596928, 0.06176876649260521], [0.06250924617052078, -0.1126246228814125, 0.04506201297044754, 0.018712198361754417, -0.030431462451815605, 0.029059387743473053, 0.0195454228669405, -0.02044774405658245, 0.020213350653648376, -0.08569540083408356, 0.06422267854213715, -0.06813471764326096, -0.05201319232583046, 0.039096321910619736, 0.04247257113456726, -0.0381629541516304, 0.12356679141521454, -0.09030825644731522, -0.018965741619467735, -0.021370533853769302, 0.0718497559428215, -0.0036221796181052923, 0.0067883944138884544, -0.05597924441099167, -0.0425211638212204, 0.02351953461766243, -0.02824660763144493, 0.04250267893075943, -0.029686089605093002, 0.005441753659397364, 0.081935815513134, -0.03744828328490257, -0.08089867979288101, -0.074806347489357, 0.022124579176306725, 0.018607987090945244, -0.03375781700015068, -0.026626678183674812, 0.061288654804229736, -0.07443056255578995, 0.023354746401309967, 0.03658689185976982, -0.0778733342885971, 0.03079736791551113, -0.028625652194023132, 0.0033744163811206818, -0.018836287781596184, -0.02048121578991413, -0.057195331901311874, -0.07904189079999924, -0.10114799439907074, -0.0395958237349987, -0.09747914969921112, -0.05074768140912056, -0.05481397733092308, -0.030023565515875816, 0.017870163545012474, 0.12263181805610657, 0.0601574070751667, -0.0012573752319440246, 0.12445079535245895, -0.053132202476263046, 0.021521512418985367, -0.058273252099752426, -0.019868584349751472, -0.01790436916053295, 0.025304105132818222, 0.03852924704551697, -0.0015369352186098695, 0.052596114575862885, -0.03854687139391899, -0.007068464066833258, -0.0017360252095386386, 0.08661465346813202, -0.12345945835113525, 0.030976785346865654, -0.039232660084962845, -0.013306405395269394, -0.09254617989063263, -0.001703132875263691, -0.08657055348157883, -0.0616900660097599, -0.046881143003702164, 0.0445941723883152, -0.09767116606235504, 0.0775802731513977, 0.009032893925905228, 0.07117719203233719, 0.016142675653100014, 0.05521513149142265, -0.10353584587574005, -0.06093139946460724, -0.03493848443031311, 0.0015393244102597237, -0.07459263503551483, -0.012519573792815208], [0.07523443549871445, 0.008639691397547722, -0.05822088196873665, 0.06133769452571869, 0.02425645850598812, -0.09206795692443848, 0.017812281847000122, 0.052260760217905045, -0.041730862110853195, -0.08862598985433578, 0.026319367811083794, 0.04579029604792595, -0.020368652418255806, 0.028232838958501816, -0.0025166955310851336, 0.04134557396173477, 0.047058895230293274, 0.036347080022096634, -0.01768425852060318, -0.0068288929760456085, -0.024584414437413216, 0.05645640939474106, -0.07400976121425629, -0.0674041286110878, -0.006218019872903824, 0.12031196802854538, 0.02726750448346138, -0.048511769622564316, 0.02616450935602188, -0.06441100686788559, 0.02338063344359398, -0.0210260022431612, -0.06509502977132797, -0.08776868879795074, -0.040790874511003494, -0.06577172875404358, 0.03145423159003258, 0.00987090915441513, 0.026922155171632767, -0.020741265267133713, 0.04898635298013687, 0.005123739130795002, 0.03052518144249916, -0.05971107259392738, 0.03502880036830902, -0.07205375283956528, -0.02797173522412777, -0.034358736127614975, -0.07025295495986938, 0.025641510263085365, 0.06261871755123138, 0.04204274341464043, -0.086558997631073, -0.0760703831911087, -0.04057775437831879, 0.022630730643868446, -0.06666798144578934, 0.059731993824243546, 0.043526541441679, -0.05047321692109108, 0.06994324922561646, -0.05863703414797783, -0.018813030794262886, -0.04345022141933441, -0.04496606066823006, -0.007649100851267576, 0.039686139672994614, 0.047163043171167374, -0.04004257172346115, -0.006382963620126247, -0.022933917120099068, -0.0434115007519722, 0.05932241678237915, 0.008540511131286621, 0.033538516610860825, 0.06095552071928978, 0.027555590495467186, 0.0163800697773695, -0.025279223918914795, -0.05414709076285362, -0.05824814364314079, -0.023363197222352028, 0.040843017399311066, 0.05000443384051323, 0.057816341519355774, 0.03592871502041817, -0.0924263596534729, -0.0026034365873783827, 0.09165950119495392, 0.0413515605032444, 0.008718244731426239, -0.047091882675886154, 0.016172004863619804, 0.014167028479278088, -0.027227085083723068, -0.04873786121606827], [-0.037001464515924454, -0.05932387337088585, -0.0790943130850792, 0.07211059331893921, -0.10909603536128998, 0.026061709970235825, -0.07443349808454514, -0.0007784001645632088, -0.06498416513204575, -0.046879447996616364, -0.0039052320644259453, 0.03375858813524246, 0.021109407767653465, -0.06054436415433884, 0.06553758680820465, -0.00833737850189209, 0.02525031752884388, -0.03247341513633728, 0.020490476861596107, -0.00204316433519125, 0.032483652234077454, -0.012632476165890694, 0.06692089140415192, 0.061093125492334366, -0.04454658180475235, -0.047402605414390564, 0.007039001677185297, -0.10044083744287491, -0.060652174055576324, -0.08312644809484482, -0.010806037113070488, -0.03357905149459839, -0.04670524224638939, 0.011307812295854092, -0.02511356770992279, 0.0326041616499424, -0.033577870577573776, 0.003664759686216712, -0.06802757829427719, -0.009882264770567417, -0.07341639697551727, -0.0016246746527031064, -0.07290960103273392, 0.018262138590216637, 0.004871412180364132, 0.00129812047816813, -0.011858511716127396, 0.024067416787147522, -0.06797356903553009, 0.04105613753199577, 0.00043305414146743715, -0.006898312829434872, -0.004612085875123739, 0.003951150458306074, -0.08941461890935898, -0.009690023958683014, -0.08896767348051071, 0.07657372206449509, -0.04790567234158516, -0.03859078139066696, 0.007909313775599003, 0.04933934658765793, -0.006094043608754873, -0.04251769930124283, -0.0309539046138525, -0.019627751782536507, -0.03533773124217987, -0.09448891878128052, 0.08604422211647034, -0.03499791771173477, -0.05649855360388756, -0.011437906883656979, -0.00163321103900671, -0.01328491885215044, 0.04946766421198845, 0.0554770827293396, -0.014000286348164082, -0.024049177765846252, 0.03978343307971954, -0.01320161484181881, -0.01862587407231331, -0.04651890695095062, -0.029807286337018013, 0.01958453468978405, -0.06891341507434845, 0.014119642786681652, 0.03695881739258766, 0.019688284024596214, 0.08538328111171722, -0.08471232652664185, 0.03352254256606102, -0.01423481572419405, 0.030404256656765938, 0.02832137793302536, -0.01388025376945734, 0.002857352141290903], [-0.09262587875127792, 0.036598090082407, 0.023038124665617943, -0.03736505284905434, -0.10109312832355499, 0.02766086719930172, -0.03540869802236557, -0.014565329998731613, -0.056671418249607086, 0.039530348032712936, -0.026852454990148544, -0.021444138139486313, 0.026491302996873856, -0.03436405584216118, 0.02121465839445591, 0.030236368998885155, -0.010041157715022564, -0.0695425420999527, 0.005141088273376226, 0.014708233065903187, -0.012759734876453876, 0.07273447513580322, 0.015365312807261944, -0.05727250874042511, -0.05181329697370529, -0.014196465723216534, 0.06465481966733932, 0.052862074226140976, -0.013111957348883152, 0.07630917429924011, 0.008305723778903484, -0.08071374893188477, -0.029205065220594406, 0.046086110174655914, 0.030382612720131874, 0.05605246126651764, -0.02173013985157013, 0.04297524690628052, 0.04917934909462929, -0.030696868896484375, -0.026170020923018456, -0.015853220596909523, -0.11335503309965134, -0.057666849344968796, 0.08263607323169708, 0.04680553451180458, -0.07257571816444397, -0.0700654685497284, 0.07153798639774323, -0.014969400130212307, -0.014161106199026108, -0.05424010008573532, 0.018718760460615158, 0.004078751429915428, -0.09411381930112839, 0.0317126102745533, -0.07586454600095749, -0.07342292368412018, -0.012828591279685497, 0.005670171696692705, -0.032547835260629654, -0.038289207965135574, -0.05427543446421623, 0.07871056348085403, -0.07903653383255005, -0.02664870023727417, -0.04529840499162674, -0.04758896306157112, 0.03176382556557655, 0.012217450886964798, -0.08373533934354782, -0.0160063486546278, -0.031494952738285065, 0.07919077575206757, -0.05999012291431427, -0.09543879330158234, 0.06938912719488144, -0.06525696069002151, 0.010113302618265152, -0.08107174187898636, -0.06549844890832901, 0.0011486569419503212, 0.0562518946826458, -0.016561513766646385, 0.08493682742118835, 0.10242129117250443, 0.01587381400167942, -0.01709936372935772, 0.09033061563968658, -0.06241694465279579, -0.07917583733797073, -0.056127533316612244, -0.11968465894460678, 0.051139216870069504, 0.020451828837394714, -0.06888049840927124]], "b2": [0.02001373842358589, -0.07121194899082184, 0.027713891118764877, 0.057082343846559525, 0.04298469424247742, -0.12763437628746033, -0.05942315608263016, -0.027373919263482094, 0.036680594086647034, 0.02960415557026863, 0.011042521335184574, -0.04001324623823166, -0.07232227921485901, -0.02330191060900688, 0.11402785032987595, -0.09460920840501785, 0.04163837805390358, 0.1069703996181488, -0.02493263967335224, 0.10844825208187103, 0.07084468007087708, 0.03186650574207306, -0.01296864915639162, -0.03055269457399845, 0.07398232817649841, -0.011727157980203629, -0.00577453663572669, 0.12830941379070282, -0.0242008063942194, -0.09569002687931061, -0.06842612475156784, -0.041887152940034866], "W3": [[-0.1500881463289261, 0.06233307346701622, -0.18341055512428284, -0.040416836738586426, -0.2295537292957306, 0.09405209869146347, 0.1501133143901825, 0.011883760802447796, -0.16466550529003143, -0.141086146235466, -0.05109785869717598, 0.15982240438461304, 0.04503323882818222, -0.16078446805477142, -0.065036341547966, 0.061170849949121475, 0.027033092454075813, -0.05031342804431915, -0.020610282197594643, -0.08199993520975113, 0.16455084085464478, 0.1052025556564331, 0.19048333168029785, 0.11819032579660416, -0.05190537869930267, 0.17964313924312592, -0.15819105505943298, -0.1068258211016655, 0.14426228404045105, 0.09126129001379013, 0.12906399369239807, 0.10578160732984543]], "b3": [0.14572584629058838]}, "2022": {"W1": [[-0.174261212348938, -0.09931448101997375, 0.055085115134716034, -0.24886342883110046, 0.14340126514434814, -0.09694782644510269, 0.04327962547540665, -0.01927802339196205, -0.009109198115766048, 0.281678169965744, -0.004358748439699411, 0.05603468418121338, -0.027139244601130486, 0.07186395674943924, 0.021712476387619972, -0.08786073327064514, -0.17764946818351746, -0.07071516662836075, -0.1101517528295517, -0.0020328150130808353, -0.05189613997936249], [0.11226573586463928, -0.07040838897228241, 0.06898801028728485, 0.10302096605300903, -0.024025481194257736, -0.12004206329584122, 0.11740205436944962, -0.13863413035869598, 0.0662279799580574, 0.11212852597236633, -0.0561368353664875, -0.06194186210632324, -0.23236115276813507, -0.00879848375916481, -0.14678612351417542, -0.07033604383468628, -0.2609899640083313, -0.05773979425430298, 0.08044197410345078, -0.012754018418490887, -0.0628429651260376], [0.10991855710744858, -0.050266679376363754, 0.05096517875790596, 0.07773487269878387, 0.09213978052139282, -0.07314399629831314, -0.04194723814725876, -0.022059733048081398, 0.055519625544548035, 0.06252289563417435, 0.06449957937002182, 0.007672268897294998, -0.026043497025966644, 0.02558516152203083, -0.013763644732534885, -0.047047097235918045, -0.02533765695989132, -0.034182820469141006, -0.0036979392170906067, -0.03447689861059189, 0.02294917218387127], [-0.08014820516109467, -0.03313969820737839, -0.1715080887079239, 0.07999661564826965, 0.09356917440891266, -0.02394334226846695, -0.0440802276134491, -0.09147471934556961, 0.020847273990511894, 0.15530335903167725, -0.15417011082172394, -0.05123809352517128, -0.14108788967132568, -0.07822085916996002, 0.08931373804807663, 0.13652265071868896, -0.09878786653280258, 0.09009013324975967, 0.05574619770050049, 0.017572708427906036, -0.08416731655597687], [0.11861315369606018, -0.15795907378196716, 0.06932452321052551, 0.14693023264408112, 0.11471319943666458, 0.04825415462255478, -0.0749223604798317, 0.060061149299144745, -0.08613111823797226, -0.04950308799743652, -0.03490924462676048, 0.19223900139331818, -0.025947025045752525, 0.23422162234783173, -0.10626604408025742, 0.024494007229804993, 0.013571839779615402, 0.08025463670492172, 0.013918978162109852, 0.009782012552022934, 0.06408066302537918], [0.13150325417518616, 0.11337672173976898, -0.1358460783958435, -0.06411381810903549, 0.11892927438020706, 0.14329949021339417, 0.14139479398727417, 0.09461463242769241, -0.07741308212280273, 0.04772346466779709, 0.12784230709075928, 0.1283126175403595, 0.01550083328038454, 0.0800500437617302, -0.036185890436172485, -0.06138006970286369, -0.05740409344434738, 0.026918256655335426, -0.037077538669109344, -0.07142855226993561, -0.034557584673166275], [-0.09573650360107422, 0.13867636024951935, -0.10811252146959305, 0.08430919051170349, -0.11863305419683456, -0.10342597961425781, 0.10043083876371384, -0.1468902975320816, -0.07141000777482986, -0.0073506832122802734, -0.09511829167604446, -0.11300403624773026, 0.041878387331962585, 0.07595235854387283, -0.0014299222966656089, -0.027645722031593323, 0.008242695592343807, -0.010896882973611355, 0.15622512996196747, -0.09205877780914307, 0.08783700317144394], [0.04949130862951279, -0.05162874236702919, 0.10268862545490265, 0.022942353039979935, -0.005010360851883888, -0.05554644390940666, -0.05928162485361099, 0.008719121105968952, -0.1079624593257904, 0.15199850499629974, -0.06496522575616837, -0.005192952696233988, 0.027303002774715424, -0.051897626370191574, 0.05261065810918808, -0.034577563405036926, -0.17498192191123962, 0.0626143366098404, 0.09405698627233505, -0.04062191769480705, 0.06611839681863785], [0.025312308222055435, 0.034750740975141525, -0.12488830089569092, -0.18665193021297455, -0.17431288957595825, -0.15826047956943512, 0.019912775605916977, 0.01250114943832159, 0.10395429283380508, -0.05374039709568024, -0.13183076679706573, 0.08619760721921921, 0.03302953019738197, -0.057725515216588974, -0.00730894785374403, 0.030119528993964195, -0.09075265377759933, -0.14307112991809845, -0.10620656609535217, -0.14434461295604706, -0.1583632528781891], [0.08345504850149155, 0.026261523365974426, -0.010332697071135044, 0.015080067329108715, 0.11909354478120804, -0.08783731609582901, -0.03711465373635292, -0.01760464534163475, 0.02543705329298973, -0.025631604716181755, -0.01705782115459442, 0.08619868010282516, -0.11783621460199356, 0.07341239601373672, -0.07073839008808136, 0.053709838539361954, -0.13742533326148987, 0.09150086343288422, 0.08409141004085541, -0.02028905786573887, 0.09189251065254211], [-0.1698680818080902, 0.10039648413658142, 0.01346737239509821, -0.15157873928546906, -0.1266682744026184, -0.11644851416349411, -0.04721449688076973, 0.08313588798046112, 0.0077554602175951, -0.17334015667438507, -0.010012377053499222, 0.15132012963294983, -0.09391443431377411, 0.06159740686416626, 0.06459198892116547, 0.005460871383547783, 0.014335696585476398, -0.06058619171380997, -0.09034160524606705, -0.08846385776996613, 0.10057106614112854], [-0.014358116313815117, 0.11055444180965424, 0.025324445217847824, -0.10648120194673538, 0.02625233307480812, -0.06054528430104256, -0.08077959716320038, -0.08296466618776321, 0.07296375930309296, 0.08209186047315598, -0.10687986016273499, 0.020839815959334373, 0.09560487419366837, 0.06170403212308884, 0.11283858120441437, -0.04412779584527016, -0.14941789209842682, -0.08226870745420456, -0.02963479608297348, 0.09010981768369675, 0.12582051753997803], [-0.010816960595548153, 0.06597068905830383, 0.023472491651773453, -0.09972307085990906, -0.04111800715327263, -0.03898769989609718, -0.040408723056316376, -0.015897074714303017, 0.09364720433950424, 0.012154566124081612, -0.026120351627469063, -0.08778295665979385, -0.07368636876344681, 0.001897901645861566, -0.05402861163020134, 0.04025747627019882, -0.13475129008293152, -0.04259384050965309, -0.014892936684191227, 0.07371658086776733, 0.03518423065543175], [-0.07687334716320038, -0.10399539023637772, -0.20466499030590057, -0.1569756418466568, -0.049394991248846054, -0.02032981812953949, -0.11200191080570221, 0.11886674165725708, 0.07222282886505127, 0.1747090369462967, -0.031649403274059296, -0.09640788286924362, -0.16898442804813385, -0.05941012129187584, -0.12866687774658203, -0.11586660146713257, -0.059807244688272476, 0.1350371092557907, 0.07235856354236603, -0.18722450733184814, -0.14047357439994812], [-0.06127524375915527, -0.03662215173244476, 0.10880851000547409, 0.03241785243153572, 0.06905359029769897, 0.023396287113428116, -0.0677911564707756, -0.029264312237501144, 0.023009968921542168, 0.037258002907037735, 0.004695078358054161, -0.08907762169837952, 0.05742912366986275, 0.10935036838054657, 0.011337376199662685, -0.009081054478883743, 0.04224476218223572, -0.06804899126291275, 0.013299847953021526, -0.09563376009464264, -0.09700575470924377], [-0.012172753922641277, -0.008450251072645187, -0.01819991134107113, 0.09060162305831909, -0.07802741974592209, -0.023515872657299042, -0.06564600765705109, 0.00824884045869112, 0.03674877807497978, -0.03723973408341408, -0.013338304124772549, 0.05850660800933838, 0.04388526454567909, -0.018148135393857956, -0.07111435383558273, -0.024424711242318153, -0.11506405472755432, -0.06608940660953522, 0.041581589728593826, -0.04132840409874916, 0.11316152662038803], [-0.10865403711795807, -0.12960250675678253, -0.08758025616407394, -0.23295770585536957, 0.005858733784407377, -0.012570023536682129, -0.015752727165818214, -0.029818035662174225, -0.1539033204317093, 0.07412226498126984, -0.08694488555192947, 0.07490888237953186, -0.10023102909326553, -0.09194961190223694, -0.026226330548524857, -0.011947308667004108, -0.034759752452373505, -0.059405576437711716, -0.12977392971515656, 0.01773471012711525, 0.09899043291807175], [-0.03856430947780609, 0.05151287093758583, 0.22855709493160248, -0.1792660802602768, -0.016541797667741776, 0.14075520634651184, -0.1696586012840271, 0.07937337458133698, 0.05899803340435028, -0.09955441951751709, -0.018625855445861816, -0.19004729390144348, 0.031968869268894196, 0.0012875421671196818, -0.06983130425214767, 0.0045729875564575195, 0.20566678047180176, -0.08341147005558014, -0.0039564454928040504, -0.029462361708283424, -0.08230119943618774], [-0.04712928831577301, -0.04537488892674446, 0.014110219664871693, 0.031432345509529114, 0.05795414373278618, -0.10155593603849411, 0.011181127279996872, -0.10282368957996368, 0.040408048778772354, -0.004194879904389381, -0.06242819130420685, -0.00773816229775548, -0.1377442628145218, -0.1024186760187149, 0.027347242459654808, 0.05705692619085312, 0.13351838290691376, 0.11126676946878433, 0.14600975811481476, -0.038431666791439056, -0.0710873007774353], [-0.00824722833931446, -0.0422806516289711, 0.09957177191972733, -0.00530705600976944, -0.0762566477060318, -0.026466690003871918, -0.031616464257240295, 0.04765501990914345, -0.029579883441329002, -0.028291700407862663, 0.03163861855864525, 0.09002150595188141, -0.11960789561271667, 0.01933964528143406, -0.036525823175907135, -0.006030568853020668, -0.19959408044815063, 0.05061989650130272, -0.0508602112531662, -0.03851308301091194, -0.06842977553606033], [-0.08166506886482239, -0.016790026798844337, 0.0512634702026844, -0.15555836260318756, 0.0999818965792656, 0.09044922143220901, 0.04095746576786041, -0.03848813474178314, -0.053011879324913025, -0.00598534569144249, 0.014757247641682625, -0.11392561346292496, -0.08141423761844635, 0.01691659726202488, 0.17734187841415405, 0.022019701078534126, -0.22710351645946503, 0.05043007433414459, -0.08531753718852997, 0.04557628184556961, -0.055395178496837616], [0.020212436094880104, 0.07133647799491882, -0.17328791320323944, 0.01729651913046837, -0.0037905368953943253, 0.11000659316778183, -0.05306793749332428, 0.036014243960380554, -0.04690570384263992, -0.12553630769252777, -0.09374850988388062, -0.02503451704978943, -0.021520651876926422, -0.0765853077173233, 0.14748819172382355, 0.01558643952012062, -0.009569589979946613, 0.03965393081307411, 0.009871854446828365, 0.05081907659769058, -0.10393474251031876], [0.0007210971089079976, -0.058805614709854126, -0.062224388122558594, 0.047456227242946625, 0.08782996237277985, 0.11986225843429565, 0.040224675089120865, -0.05144476518034935, 0.013704977929592133, -0.05882905051112175, 0.12612298130989075, 0.005911208223551512, 0.12100447714328766, -0.1145525649189949, 0.0997033417224884, -0.09325747191905975, -0.098872110247612, 0.09424614161252975, -0.1271810382604599, -0.10512952506542206, 0.13338878750801086], [0.12660850584506989, 0.05495724081993103, 0.025676395744085312, -0.06071225181221962, 0.051337677985429764, -0.008332700468599796, -0.10096726566553116, -0.15006503462791443, 0.0991477221250534, -0.04409034177660942, 0.06710277497768402, -0.09494292736053467, -0.1602165699005127, -0.03453429415822029, 0.12686093151569366, 0.019158460199832916, 0.11030685901641846, -0.08895908296108246, -0.034075941890478134, -0.04186997935175896, -0.005927758291363716], [-0.08835198730230331, 0.07160089164972305, 0.07898105680942535, -0.04636344686150551, 0.057334236800670624, -0.12520326673984528, 0.08222239464521408, 0.13813455402851105, -0.1231415793299675, -0.0516246035695076, -0.05012130364775658, 0.020866340026259422, -0.21609452366828918, -0.07683033496141434, 0.1590866893529892, 0.06168355792760849, 0.05061975121498108, -0.08579593151807785, 0.09445399045944214, -0.015819193795323372, -0.017111264169216156], [-0.1248432993888855, 0.1389961987733841, -0.030424397438764572, 0.07441270351409912, -0.027592606842517853, -0.08381275832653046, 0.10293883085250854, -0.03647478669881821, 0.022858910262584686, 0.14264284074306488, -0.07844731956720352, 0.08471812307834625, -0.20929187536239624, 0.03307970240712166, -0.05167761072516441, -0.02800232730805874, 0.020308170467615128, 0.08233532309532166, -0.199650838971138, -0.11367624998092651, -0.08696284890174866], [-0.04750216007232666, -0.06801188737154007, -0.05429477244615555, -0.12577390670776367, -0.11671007424592972, -0.11693043261766434, -0.047973569482564926, -0.06043006852269173, 0.05307215824723244, 0.06492073088884354, 0.08854098618030548, -0.02920542284846306, 0.03168758377432823, 0.012515087611973286, 0.11225662380456924, 0.06093992292881012, -0.050163522362709045, 0.054990287870168686, -0.016466641798615456, -0.05957893654704094, 0.13627731800079346], [-0.13165010511875153, -0.07203789055347443, 0.014191149733960629, 0.008955237455666065, 0.10541891306638718, -0.08315274119377136, 0.008736814372241497, -0.010960062965750694, 0.18500840663909912, 0.06151766702532768, 0.096409872174263, 0.051192138344049454, 0.19714246690273285, 0.14822356402873993, 0.15760287642478943, 0.042604152113199234, -0.15453028678894043, -0.09634671360254288, -0.08601164072751999, -0.06305475533008575, 0.0350036546587944], [-0.10692328214645386, 0.08926787227392197, 0.130121648311615, 0.07610764354467392, 0.016758892685174942, 0.12146994471549988, 0.10564140975475311, -0.03435609117150307, 0.1067001223564148, -0.009543867781758308, 0.13220058381557465, 0.062110401690006256, -0.06551453471183777, 0.03248165547847748, 0.0719003900885582, 0.05613739416003227, -0.006477447226643562, 0.12796704471111298, 0.04050574079155922, -0.057913731783628464, 0.14435125887393951], [0.16120553016662598, 0.08776049315929413, 0.002816444728523493, -0.00725562172010541, 0.05356939882040024, 0.13889074325561523, 0.1342928111553192, -0.02649536356329918, 0.1535356640815735, 0.17248409986495972, -0.14009812474250793, 0.0894017145037651, -0.2265806496143341, 0.038939788937568665, 0.07757069170475006, -0.008566784672439098, 0.0507499985396862, 0.13172784447669983, 0.03259851038455963, -0.11553248763084412, 0.02233107015490532], [0.1379096955060959, 0.1251538246870041, -0.20676037669181824, -0.01961677148938179, 0.004282827489078045, -0.10107521712779999, 0.15273721516132355, 0.015169448219239712, -0.06128828600049019, 0.12092819064855576, -0.03439001739025116, -0.05175444856286049, -0.12423999607563019, -0.04898497089743614, -0.05077934265136719, 0.05212802812457085, -0.02191135101020336, 0.09989631921052933, 0.014333512634038925, -0.19762685894966125, 0.01900476962327957], [-0.011084312573075294, 0.06806878745555878, 0.1701621264219284, -0.06726867705583572, 0.05537385493516922, -0.011931337416172028, 0.14253634214401245, -0.009415642358362675, 0.030101779848337173, -0.12531666457653046, 0.10307842493057251, -0.09663134813308716, -0.10646454989910126, -0.01936463825404644, -0.05666342005133629, 0.05158122628927231, -0.03340965509414673, -0.06737107783555984, -0.060939520597457886, -0.06010846793651581, 0.013914489187300205], [-0.00268986145965755, -0.043921541422605515, 0.17830626666545868, -0.067979596555233, -0.10184384882450104, -0.07548597455024719, -0.04421675205230713, 0.0675906166434288, 0.0757061019539833, -0.09612631052732468, 0.1567937135696411, -0.1701563149690628, 0.07439925521612167, 0.10706926882266998, -0.05177544057369232, 0.04638709872961044, 0.13937155902385712, 0.02794385328888893, -0.08896990865468979, 0.13700681924819946, -0.0423080250620842], [-0.13674631714820862, 0.08208052814006805, 0.19996479153633118, 0.13334959745407104, 0.1095842644572258, 0.017896603792905807, 0.005505571141839027, 0.02179509401321411, -0.0818554162979126, -0.01455300860106945, -0.14400573074817657, -0.07317003607749939, -0.1437249630689621, -0.10935615003108978, -0.05593346804380417, -0.013180457055568695, -0.19181619584560394, -0.05827357620000839, 0.10762298852205276, -0.0095230583101511, 0.1079241931438446], [0.026498453691601753, 0.1349228471517563, 0.16687318682670593, -0.10981187224388123, -0.027067381888628006, 0.1033308207988739, -0.039239268749952316, 0.05645471438765526, -0.018417280167341232, 0.08319549262523651, -0.10994549095630646, 0.07419514656066895, -0.03892117366194725, -0.05845525860786438, 0.1594506800174713, -0.13454550504684448, -0.2096281200647354, -0.05100950971245766, 0.10605630278587341, -0.029449816793203354, 0.021914971992373466], [0.168430358171463, 0.1308106780052185, 0.04814821481704712, 0.14227840304374695, 0.11160585284233093, -0.0626867339015007, -0.1536094844341278, -0.06225084140896797, 0.06123284623026848, 0.11712998151779175, -0.13396978378295898, 0.002006846247240901, -0.16165944933891296, -0.07703538239002228, -0.09599387645721436, 0.03893576189875603, 0.27009403705596924, -0.07354187965393066, -0.11868402361869812, 0.08728541433811188, 0.09169845283031464], [-0.01630658656358719, 0.07828250527381897, 0.04327799752354622, 0.013666732236742973, 0.08585689961910248, 0.14706291258335114, -0.0908883735537529, -0.13301339745521545, 0.10528430342674255, -0.07608810812234879, 0.06531266868114471, -0.11394963413476944, -0.24818386137485504, -0.06249987334012985, -0.07718345522880554, 0.08582354336977005, 0.07328677177429199, -0.012965538538992405, 0.07453155517578125, 0.004210391081869602, 0.04796410724520683], [-0.06950081139802933, -0.01016837079077959, -0.1197659894824028, 0.05087123066186905, 0.050541967153549194, 0.0053529865108430386, 0.12119463831186295, 0.07635770738124847, -0.0856114849448204, 0.13850751519203186, -0.015955524519085884, -0.02590635046362877, 0.18046949803829193, 0.039263930171728134, -0.02355637028813362, 0.013938467018306255, 0.11673830449581146, 0.10491973906755447, -0.16837023198604584, -0.13605378568172455, 0.0054845428094267845], [-0.09362322092056274, -0.10323178023099899, -0.10535114258527756, 0.07013048976659775, -0.061703749001026154, 0.10790067166090012, 0.016124894842505455, 0.024487394839525223, -0.0316181555390358, 0.18660122156143188, 0.06293801963329315, 0.038096655160188675, -0.1251966655254364, 0.07160355895757675, -0.058848511427640915, 0.06620670855045319, 0.02499704249203205, 0.09815183281898499, -0.13321806490421295, -0.09657865017652512, -0.0466809943318367], [-0.07096993178129196, 0.09479974210262299, -0.056965939700603485, -0.11622354388237, 0.11038599908351898, 0.010110979899764061, 0.026430709287524223, 0.0175665020942688, -0.010848169215023518, 0.06942836195230484, -0.05945848673582077, 0.1291128247976303, -0.046750109642744064, 0.024810094386339188, -0.08251190185546875, -0.08350022882223129, 0.005485447123646736, -0.045675698667764664, -0.020268511027097702, -0.018558640033006668, -0.08219042420387268], [-0.11933247745037079, -0.07166662067174911, -0.06876985728740692, -0.11843173205852509, 0.08748407661914825, -0.03116888925433159, 0.13732852041721344, -0.08268389105796814, 0.06442401558160782, 0.023092038929462433, 0.1259213089942932, -0.04515750706195831, 0.05487442389130592, -0.03666165471076965, 0.0012864512391388416, 0.028179293498396873, -0.07875777781009674, -0.0683874785900116, -0.08794517070055008, 0.0388459786772728, -0.052396804094314575], [-0.049975767731666565, 0.04515775665640831, -0.007536675781011581, 0.035971615463495255, -0.0404614694416523, 0.11773858964443207, 0.07845094054937363, -0.1348261684179306, -0.05677700415253639, -0.06637616455554962, -0.01956528052687645, 0.026874929666519165, -0.05963755026459694, 0.08157002925872803, 0.07226170599460602, -0.024715032428503036, 0.07362040877342224, -0.041608743369579315, 0.07748154550790787, -0.008819131180644035, 0.07639381289482117], [0.020858382806181908, -0.03942883759737015, 0.08525490015745163, 0.0008383256499655545, 0.07432432472705841, -0.025799760594964027, 0.03734796494245529, -0.04719534143805504, 0.013928464613854885, 0.15648703277111053, -0.06024728715419769, -0.005275815259665251, -0.18103677034378052, 0.11751512438058853, -0.07581956684589386, -0.0073350272141397, -0.14344634115695953, -0.022389337420463562, 0.10423500090837479, -0.01826542802155018, 0.0058523728512227535], [-0.009106792509555817, -0.019034378230571747, -0.012957928702235222, 0.017776712775230408, 0.01317621860653162, 0.05603021755814552, -0.022436421364545822, 0.008434525690972805, -0.02288169600069523, -0.054908763617277145, 0.019742514938116074, 0.053541235625743866, 0.07512491941452026, -0.00035036823828704655, -0.05807345360517502, -0.053772974759340286, -0.04250238090753555, -0.05014268308877945, -0.020499521866440773, -0.011908960528671741, -0.0075631896033883095], [0.14487579464912415, -0.10383876413106918, -0.08906108886003494, 0.15499496459960938, -0.08204308897256851, -0.07626593112945557, -0.024891996756196022, 0.08254486322402954, -0.005148892756551504, 0.17882035672664642, 0.06971650570631027, -0.021510090678930283, 0.06064193695783615, 0.012335039675235748, 0.13090358674526215, 0.033100735396146774, -0.09664681553840637, -0.10603926330804825, 0.1684405505657196, -0.012232714332640171, 0.14203716814517975], [0.006764667574316263, -0.025784242898225784, -0.05778149515390396, 0.06596644222736359, -0.07182645797729492, -0.03990995138883591, -0.06889893114566803, -0.004811576567590237, 0.01705017499625683, -0.008769161067903042, -0.08425384759902954, 0.10312409698963165, 0.07351285219192505, 0.059323687106370926, 0.14513391256332397, 0.044078025966882706, 0.01817149482667446, 0.040274836122989655, -0.017220325767993927, 0.10995984822511673, 0.06001700088381767], [-0.10543116927146912, -0.036072876304388046, 0.11697706580162048, 0.09417206048965454, -0.013011607341468334, 0.08923999220132828, -0.11378967761993408, -0.030363278463482857, -0.09953996539115906, -0.07750797271728516, -0.10055384039878845, 0.05428102985024452, -0.006420868914574385, 0.023005709052085876, 0.06370195746421814, 0.060179516673088074, 0.1586087942123413, 0.015396170318126678, -0.09942461550235748, 0.0456809438765049, -0.03740520030260086], [0.07054152339696884, -0.03443038836121559, -0.13779158890247345, 0.1205107569694519, -0.04572395235300064, 0.08890333026647568, -0.12030144035816193, -0.08852416276931763, -0.030502909794449806, 0.04568395018577576, -0.07442129403352737, -0.1504659354686737, -0.0855564996600151, 0.05639813095331192, 0.017682503908872604, -0.06568732112646103, -0.12296296656131744, 0.0006965007050894201, 0.009035227820277214, 0.02707146853208542, -0.055538054555654526], [0.02697557583451271, 0.03930268436670303, 0.13575585186481476, -0.1649877429008484, -0.07766557484865189, 0.10401208698749542, -0.06209013611078262, -0.07820142805576324, 0.08268820494413376, 0.1615390181541443, -0.11155399680137634, -0.14796721935272217, 0.11432997137308121, 0.049225836992263794, 0.15278494358062744, -0.01874939538538456, 0.056428954005241394, 0.0419841893017292, 0.03294974938035011, 0.017957111820578575, -0.10419679433107376], [-0.1972820907831192, 0.03573831170797348, 0.14421531558036804, 0.042050376534461975, -0.12004265189170837, 0.1223561018705368, 0.12143365293741226, -0.15717360377311707, 0.12919831275939941, -0.07504734396934509, 0.030482232570648193, 0.014905287884175777, -0.03512057289481163, 0.007690608035773039, -0.046602167189121246, 0.08551293611526489, -0.2601432204246521, 0.033995985984802246, 0.09715206921100616, 0.04125184938311577, -0.06055663898587227], [0.12127580493688583, -0.017628995701670647, 0.14140835404396057, -0.016354423016309738, 0.09104780107736588, 0.14077799022197723, 0.08214887976646423, -0.1372634768486023, -0.07685287296772003, 0.11936058104038239, 0.009883086197078228, 0.05646850913763046, 0.017919057980179787, -0.038730595260858536, 0.00917728990316391, -0.06107676401734352, -0.08043626695871353, 0.03802356868982315, -0.08457255363464355, 0.1088433563709259, 0.15376350283622742], [0.060836732387542725, 0.05715833604335785, -0.04857468605041504, -0.06387583166360855, 0.048493996262550354, 0.08038879185914993, -0.025119954720139503, 0.043025825172662735, 0.055641885846853256, 0.07076811045408249, -0.039911460131406784, -0.05722721293568611, 0.060673221945762634, -0.09148497879505157, -0.03056539222598076, 0.02640356495976448, 0.16034035384655, 0.043947406113147736, 0.14020833373069763, -0.01159973256289959, -0.13276846706867218], [-0.0213127750903368, 0.12606877088546753, 0.13570091128349304, -0.04239843785762787, 0.0738525465130806, -0.08909109234809875, -0.002914020325988531, 0.10359334945678711, 0.07195470482110977, 0.051104675978422165, 0.010214079171419144, -0.22002597153186798, -0.0836060643196106, -0.01860116980969906, 0.1725662648677826, 0.08262366056442261, -0.09687132388353348, 0.06460211426019669, -0.14250658452510834, -0.028999416157603264, -0.08557982742786407], [-0.041136182844638824, -0.04197040945291519, -0.08815059065818787, 0.06666140258312225, -0.04063396155834198, 0.07261683791875839, -0.04888215288519859, -0.09091614931821823, 5.44663671462331e-05, 0.03286946564912796, 0.16427871584892273, -0.04966972768306732, 0.00925504881888628, -0.010068044066429138, -0.11669111251831055, -0.006243565119802952, -0.1524975746870041, -0.04054996743798256, -0.031121239066123962, 0.03148917481303215, -0.06546872109174728], [-0.04731035232543945, -0.08486568182706833, 0.03955456614494324, 0.03080025687813759, -0.04817933216691017, -0.10898822546005249, -0.009190653450787067, 0.03983252868056297, 0.04401795566082001, -0.028817057609558105, -0.03678157553076744, -0.0031382953748106956, -0.04550866410136223, -0.07617437094449997, -0.011655978858470917, 0.0061882915906608105, -0.25115400552749634, 0.09425891190767288, 0.03858074173331261, -0.02443300373852253, -0.06468628346920013], [0.08296888321638107, -0.05859033018350601, 0.008738161996006966, 0.0684618353843689, -0.14418186247348785, -0.11416099220514297, -0.0701904445886612, 0.06468947976827621, 0.06750614196062088, 0.13333261013031006, 0.022959496825933456, -0.014702669344842434, -0.13414937257766724, -0.05959579348564148, -0.10947726666927338, 0.028958246111869812, 0.04817808419466019, -0.0026344142388552427, 0.0902991071343422, -0.06011754646897316, 0.12219035625457764], [0.1223703995347023, -0.11101914942264557, 0.16132792830467224, 0.20950660109519958, -0.1546931117773056, 0.050088390707969666, 0.1461074948310852, 0.07941871136426926, 0.06285466253757477, 0.06307035684585571, 0.13041409850120544, -0.11639955639839172, -0.18972261250019073, 0.06043513864278793, -0.012364465743303299, -0.026282750070095062, -0.1346072107553482, -0.05919817462563515, 0.08324407786130905, 0.14302675426006317, 0.03895080089569092], [-0.015060922130942345, -0.1182294711470604, -0.0727950930595398, -0.19057953357696533, 0.02647392265498638, -0.047843508422374725, 0.09386434406042099, 0.0880512148141861, -0.10525750368833542, 0.10361040383577347, 0.08973796665668488, -0.022035736590623856, -0.13618889451026917, 0.11681538820266724, -0.0908895954489708, 0.03690594434738159, 0.08332143723964691, 0.005774017423391342, -0.17194589972496033, 0.0715281069278717, -0.12256179749965668], [-0.2003934681415558, 0.04762972891330719, -0.08605635166168213, -0.20309051871299744, -0.10455726832151413, -0.04334238916635513, -0.07796530425548553, -0.06623274832963943, -0.057018935680389404, 0.17856919765472412, -0.11009057611227036, 0.02359042316675186, 0.012952773831784725, 0.004775101784616709, -0.017897240817546844, 0.022683370858430862, 0.03906256705522537, -0.04967059567570686, -0.19039861857891083, -0.07159556448459625, -0.13646595180034637], [-0.0872923955321312, 0.15105389058589935, 0.12488856166601181, 0.052928730845451355, 0.07097577303647995, -0.11411256343126297, 0.06742789596319199, -0.08649514615535736, 0.004187409300357103, -0.04037347808480263, -0.03476634994149208, -0.04138040915131569, 0.03387801721692085, 0.08469287306070328, 0.16788257658481598, -0.048998016864061356, 0.10639028251171112, 0.07243461161851883, 0.0060368929989635944, -0.07396452873945236, -0.005077436566352844], [-0.16306205093860626, -0.11924678087234497, -0.014875199645757675, 0.048485640436410904, -0.018252205103635788, -0.09733027964830399, 0.04347444698214531, -0.042847491800785065, -0.1423732489347458, 0.1674281805753708, -0.14341959357261658, 0.06547103077173233, -0.14269697666168213, -0.04927649721503258, -0.03340837359428406, -0.04567583277821541, 0.0072184475138783455, -0.0853334590792656, -0.16596665978431702, -0.111305370926857, 0.01703469827771187], [-0.11104190349578857, -0.11827731132507324, -0.012218658812344074, 0.04090823978185654, 0.15280357003211975, -0.0981157198548317, 0.10949322581291199, 0.03744114190340042, -0.02635089121758938, 0.05940369889140129, -0.027996931225061417, -0.1292066127061844, -0.04679776355624199, 0.09959319233894348, 0.04819202050566673, -0.023531362414360046, -0.16326753795146942, 0.13750506937503815, 0.12386293709278107, -0.004077085759490728, -0.14310994744300842], [0.017097648233175278, -0.058245331048965454, 0.00357983261346817, -0.05434902384877205, 0.04895850270986557, 0.010453895665705204, 0.10599711537361145, 0.03434928506612778, -0.04825679957866669, -0.06630107760429382, 0.018199555575847626, 0.06137809157371521, 0.02197236567735672, 0.10371273756027222, 0.06108354404568672, 0.01227644458413124, 0.017805764451622963, 0.07827512919902802, 0.04417024552822113, -0.03592772036790848, -0.033697936683893204], [0.042851269245147705, -0.03608294576406479, -0.01796758733689785, 0.06266821175813675, -0.06977749615907669, 0.03830213472247124, 0.031416602432727814, 0.06867750734090805, -0.020456967875361443, -0.03105120360851288, -0.048642560839653015, 0.05515056475996971, 0.007058554794639349, -0.03670666366815567, 0.08163747936487198, 0.03050646185874939, -0.12214642018079758, 0.03061271458864212, -0.01608429104089737, -0.015215955674648285, 0.04065180942416191], [0.22081203758716583, 0.1394030898809433, -0.145626500248909, -0.035152241587638855, 0.0672348365187645, 0.06318037956953049, -0.08287681639194489, -0.030687589198350906, 0.14105334877967834, 0.18983416259288788, 0.012410856783390045, 0.09810085594654083, 0.0233704075217247, -0.03367014601826668, -0.007301453035324812, 0.00378644117154181, 0.035546764731407166, -0.023264514282345772, 0.08115029335021973, -0.0016648523742333055, 0.15682712197303772], [-0.025079095736145973, 0.032635901123285294, 0.10709625482559204, 0.06553807854652405, -0.057437747716903687, 0.08035220950841904, 0.10926902294158936, 0.1923372447490692, 0.003455352270975709, 0.020979637280106544, -0.0018733539618551731, 0.027466420084238052, -0.04919128865003586, 0.028478678315877914, 0.08060716092586517, -0.17343659698963165, -0.09964075684547424, -0.10301487147808075, 0.06712740659713745, -0.01830270141363144, 0.09173611551523209], [0.03062431886792183, -0.06782415509223938, 0.0849495679140091, -0.1056627631187439, 0.11132736504077911, -0.04654742404818535, -0.015358420088887215, -0.001085219206288457, 0.057801708579063416, -0.12653908133506775, -0.003334532957524061, 0.035145923495292664, 0.07052848488092422, 0.01542075164616108, 0.09217682480812073, 0.05065510421991348, 0.03463517501950264, 0.10272957384586334, 0.15707065165042877, 0.06088077649474144, 0.11031901091337204], [0.005051237531006336, 0.12017857283353806, 0.21159161627292633, -0.0738811269402504, -0.04405037686228752, 0.05499425157904625, -0.09909041225910187, -0.13840939104557037, -0.06286218017339706, 0.21313048899173737, -0.13281579315662384, -0.010474375449120998, -0.05714251846075058, -0.0528934970498085, 0.058324214071035385, -0.15315979719161987, -0.004088497254997492, 0.1832926869392395, 0.1317557841539383, -0.033523984253406525, -0.1075453832745552], [0.0014014393091201782, -0.09256166964769363, 0.09977668523788452, -0.001351259765215218, 0.0329936183989048, -0.017350785434246063, 0.052697379142045975, -0.0571260042488575, -0.017031900584697723, -0.1869434267282486, -0.029594935476779938, -0.08422252535820007, 0.0027444767765700817, 0.04941626265645027, 0.048971936106681824, 0.07505092024803162, -0.010218461975455284, -0.03392580896615982, -0.11531335115432739, -0.13857270777225494, -0.08394703269004822], [-0.05122394487261772, 0.015449201688170433, -0.176608607172966, -0.06217377632856369, 0.10179929435253143, 0.05265139043331146, -0.06857533752918243, 0.10340064018964767, -0.06706495583057404, -0.0778035819530487, -0.13567425310611725, 0.12666156888008118, -0.1263495534658432, -0.10098028182983398, -0.020767731592059135, 0.13894039392471313, 0.0011668873485177755, -0.04234125465154648, 0.08638612180948257, -0.11716891825199127, -0.10584705322980881], [-0.006272559054195881, 0.06183633953332901, -0.03871428593993187, -0.049863629043102264, 0.06067484989762306, -0.049946606159210205, -0.026492664590477943, -0.0020429175347089767, 0.049729254096746445, 0.10058717429637909, -0.028978684917092323, -0.07270047813653946, 0.08053071796894073, -0.004533686209470034, 0.05396820977330208, -0.014052050188183784, -0.06813542544841766, -0.007446968927979469, 0.08665697276592255, 0.05870062857866287, -0.005303171928972006], [-0.08071193099021912, -0.08567656576633453, 0.11188054084777832, -0.03680936247110367, 0.1453481912612915, 0.12275704741477966, 0.14950022101402283, -0.05558032914996147, -0.09302028268575668, 0.06808581203222275, 0.017021281644701958, -0.04148579388856888, -0.12671761214733124, -0.11316310614347458, -0.10096247494220734, 0.06707437336444855, -0.2416030913591385, 0.06578735262155533, 0.13783478736877441, 0.0407940074801445, 0.05984307453036308], [-0.035049159079790115, -0.02086556889116764, -0.08570101857185364, 0.02224932238459587, -0.07087257504463196, -0.07031495869159698, -0.056766387075185776, 0.04484182596206665, -0.12094050645828247, 0.1787567287683487, -0.14006894826889038, 0.06453599035739899, -0.058592312037944794, 0.07958940416574478, -0.10842853784561157, -0.12511369585990906, -0.03567728027701378, -0.0784827172756195, -0.12893983721733093, -0.16416913270950317, 0.12633003294467926], [0.05427424982190132, 0.022751016542315483, 0.10776383429765701, -0.06718048453330994, -0.022725015878677368, 0.13153350353240967, -0.08054705709218979, -0.0516226626932621, 0.11983335763216019, -0.07495737075805664, 0.06443491578102112, 0.020838694646954536, 0.051483627408742905, 0.03064638003706932, -0.09445460140705109, 0.12177994102239609, 0.07059777528047562, -0.051946382969617844, -0.07058640569448471, -0.12145539373159409, 0.16771897673606873], [-0.017394552007317543, 0.00844376627355814, 0.018824290484189987, 0.007989206351339817, -0.059787120670080185, 0.08544156700372696, 0.05994933471083641, -0.01770392805337906, -0.03514554351568222, 0.042232878506183624, 0.05146883428096771, -0.030513279139995575, 0.07621031254529953, 0.017466913908720016, 0.03658784180879593, -0.04408161714673042, 0.09280306100845337, 0.03915727883577347, -0.04349784925580025, 0.0024857609532773495, -0.014608128927648067], [0.1031564474105835, 0.004055477678775787, -0.059326596558094025, 0.10662566125392914, 0.03206629678606987, -0.039265941828489304, 0.011359937489032745, -0.03169061243534088, 0.07252071052789688, 0.006649899296462536, -0.041763920336961746, 0.009952359832823277, 0.07098182290792465, 0.01796315237879753, -0.005968586076050997, -0.002994930138811469, -0.07468877732753754, -0.036018576472997665, 0.08170409500598907, 0.09242991358041763, -0.003816665383055806], [0.012707158923149109, 0.03385641798377037, -0.06059875339269638, 0.06779646873474121, 0.03260401636362076, -0.08479296416044235, -0.04718683660030365, 0.06841785460710526, 0.022525835782289505, -0.068672314286232, -0.06875015050172806, 0.027772460132837296, 0.05636194720864296, -0.08771470189094543, -0.05391126498579979, 0.03622923418879509, -0.16291247308254242, 0.08344019204378128, -0.0434652641415596, 0.12477192282676697, 0.001769944909028709], [-0.06718690693378448, 0.07374642789363861, -0.056838877499103546, -0.14877326786518097, -0.0793987438082695, -0.15320855379104614, -0.061094507575035095, -0.030882248654961586, -0.058634206652641296, 0.06168067082762718, 0.07229317724704742, 0.09940320253372192, -0.0796491876244545, 0.07485317438840866, -0.06366311758756638, 0.03651817888021469, -0.1432047039270401, -0.05487540736794472, -0.11331240832805634, -0.004201506730169058, 0.11162721365690231], [-0.11535970866680145, 0.0635620579123497, -0.0679846853017807, -0.002198445377871394, -0.07578699290752411, 0.11086519062519073, 0.1465831696987152, -0.006335299927741289, 0.13989748060703278, 0.028964126482605934, 0.08112777769565582, -0.06504841148853302, -0.014307950623333454, -0.034249745309352875, 0.1871868520975113, -0.036127734929323196, -0.09053580462932587, 0.11505970358848572, -0.08517973124980927, 0.025708002969622612, 0.06849746406078339], [-0.02837042510509491, 0.02911144681274891, 0.08864136040210724, 0.0854734405875206, -0.08315926045179367, -0.05134467035531998, -0.050607793033123016, -0.05624585598707199, -0.0601721927523613, 0.042148325592279434, 0.07143368571996689, -0.024301327764987946, -0.09988968819379807, -0.0067903390154242516, 0.004982129670679569, -0.010027103126049042, -0.11773429065942764, 0.09879673272371292, -0.06391460448503494, -0.004467328544706106, -0.034169480204582214], [0.006279673893004656, 0.042543135583400726, -0.06602433323860168, 0.048075348138809204, 0.0008695600554347038, 0.10314355790615082, 0.06006361544132233, 0.022608181461691856, 0.013110858388245106, -0.024186216294765472, 0.010403377935290337, 0.06611040979623795, -0.047947824001312256, 0.07589054852724075, -0.07561933249235153, -0.08471544086933136, -0.10474611073732376, -0.052045948803424835, -0.037469394505023956, -0.00798130128532648, 0.07449499517679214], [-0.1044808104634285, 0.06655647605657578, 0.006733757443726063, -0.037554845213890076, -0.018794143572449684, 0.10717650502920151, -0.11838363856077194, -0.09605076164007187, 0.035240814089775085, 0.057283855974674225, 0.09437757730484009, 0.0016495775198563933, 0.1974872350692749, 0.18293540179729462, -0.11768805235624313, -0.015345074236392975, -0.08267007023096085, -0.064812071621418, 0.07554378360509872, -0.030760398134589195, -0.14370663464069366], [-0.012249760329723358, -0.01872374303638935, 0.06320762634277344, -0.06346099078655243, -0.04924987629055977, 0.13684521615505219, 0.015165789052844048, -0.13952374458312988, -0.1061158999800682, 0.07496359199285507, -0.1634509563446045, 0.08404666185379028, -0.1334398239850998, 0.02516348846256733, 0.1004076600074768, 0.06144704669713974, 0.021613527089357376, -0.0187061820179224, 0.03078734315931797, -0.006219764705747366, -0.14031611382961273], [-0.0785502940416336, 0.03907155990600586, -0.07914828509092331, -0.08524596691131592, 0.07764869928359985, -0.02132178284227848, -0.04554188996553421, 0.021937934681773186, -0.05369119346141815, 0.12517935037612915, -0.024995999410748482, -0.07067450135946274, 0.18251293897628784, 0.016944237053394318, -0.013870391063392162, -0.001553099020384252, 0.014415299519896507, 0.058467164635658264, -0.04818442091345787, -0.028623118996620178, -0.035053953528404236], [0.10143807530403137, 0.0006165364175103605, 0.006189195439219475, -0.13495618104934692, -0.01048981212079525, -0.07718147337436676, -0.16260233521461487, 0.05326289311051369, 0.030404509976506233, 0.004646484274417162, -0.06494435667991638, 0.06143408268690109, 0.031688012182712555, -0.1287883073091507, 0.10880975425243378, -0.0890873596072197, 0.24016410112380981, -0.06485249847173691, 0.017916666343808174, 0.0695754811167717, -0.0069182259030640125], [0.18810072541236877, -0.05878779664635658, -0.14509432017803192, 0.12560966610908508, 0.1876479536294937, 0.05163249745965004, -0.14125199615955353, 0.10526026040315628, -0.04357873275876045, -0.12987591326236725, 0.099649578332901, 0.09618516266345978, 0.1636376827955246, -0.00630324287340045, 0.10868324339389801, -0.1008516252040863, 0.017439354211091995, 0.13044703006744385, -0.10314196348190308, -0.05508667603135109, 0.03411741554737091], [0.025157103314995766, 0.05265088006854057, -0.12261150032281876, 0.07657785713672638, 0.13190089166164398, 0.0641854926943779, -0.05703128129243851, 0.12945249676704407, -0.013267988339066505, 0.04306535795331001, -0.026511458680033684, -0.12495279312133789, 0.0227050743997097, 0.05480412393808365, -0.03496444970369339, -0.13348141312599182, 0.27176910638809204, -0.08155002444982529, 0.1323545277118683, -0.09054619818925858, -0.022109540179371834], [-0.005460008978843689, -0.08727850764989853, -0.026304468512535095, -0.017720544710755348, 0.09414699673652649, -0.06612415611743927, -0.05097773298621178, 0.02623259648680687, -0.043534088879823685, 0.12602540850639343, -0.0324924997985363, 0.08299233019351959, 0.018019674345850945, 0.02615588717162609, 0.04710404574871063, 0.046511974185705185, -0.018950214609503746, -0.0424320288002491, 0.028397945687174797, 0.09262577444314957, -0.039173346012830734], [0.0761590227484703, 0.1358717679977417, -0.3527945578098297, -0.03219423070549965, 0.08820642530918121, 0.16821765899658203, 0.07483549416065216, -0.02271150052547455, 0.09811234474182129, 0.06347228586673737, -0.05269719660282135, 0.14958834648132324, -0.11351921409368515, -0.024569958448410034, 0.021294236183166504, -0.0658273920416832, -0.04509427025914192, 0.14152464270591736, 0.017529107630252838, 0.059801384806632996, -0.136648491024971], [-0.04552064090967178, 0.04978913068771362, 0.06639263033866882, -0.02380692958831787, -0.06498800963163376, 0.001480160397477448, -0.03245421126484871, -0.06111240014433861, -0.0762939378619194, -0.05831490829586983, -0.019248083233833313, -0.03341352567076683, 0.10617125034332275, 0.10269768536090851, 0.009919671341776848, 0.009631337597966194, 0.10142026841640472, 0.034735724329948425, -0.0944741815328598, -0.02255420945584774, -0.01014709658920765], [-0.012823178432881832, -0.0641632229089737, 0.05148736760020256, -0.057421181350946426, -0.09410347789525986, 0.025719132274389267, 0.059960734099149704, -0.05011855810880661, -0.093374103307724, -0.003987269941717386, -0.03242561221122742, -0.17604844272136688, 0.010687434114515781, 0.1073153018951416, 0.06738653033971786, 0.010372345335781574, 0.18254567682743073, 0.09149356931447983, -0.06038205698132515, -0.059854112565517426, 0.028432795777916908], [0.03890295699238777, 0.0782550796866417, 0.04751850664615631, -0.022196736186742783, -0.04402491822838783, -0.04845469444990158, -0.10418859869241714, 0.09669625759124756, -0.131408229470253, -0.016988828778266907, -0.16517555713653564, -0.01022761408239603, 0.059827957302331924, -0.10001006722450256, -0.015767846256494522, 0.013482245616614819, 0.2764277160167694, 0.02734832838177681, 0.11580858379602432, -0.09517550468444824, -0.059602972120046616], [-0.06572286784648895, -0.031016690656542778, 0.03532695397734642, 0.12859641015529633, -0.04431382194161415, -0.009451670572161674, 0.09616922587156296, -0.05549393966794014, 0.007920799776911736, -0.08038956671953201, -0.0029227547347545624, 0.027790367603302002, -0.14953964948654175, 0.1074027493596077, -0.022102907299995422, 0.0022486846428364515, 0.061111100018024445, -0.0011472930200397968, 0.01069012563675642, -0.009001999162137508, -0.015012111514806747], [0.08183746784925461, 0.05988525226712227, -0.026188138872385025, 0.12152942270040512, 0.07014067471027374, -0.04045699164271355, 0.07367817312479019, 0.07196314632892609, 0.00706872995942831, 0.0375700369477272, 0.01904047653079033, -0.08993221074342728, -0.041125278919935226, -0.007726763375103474, 0.12929943203926086, -0.09634685516357422, -0.09445147216320038, 0.048089686781167984, 0.07590679079294205, 0.047203756868839264, 0.038874927908182144], [0.0126497782766819, 0.032462045550346375, 0.002762874122709036, -0.006392904557287693, 0.04892289265990257, -0.06307363510131836, 0.10555948317050934, 0.08937554061412811, 0.08224610239267349, 0.03628218173980713, 0.018410976976156235, 0.03572222590446472, 0.05236915126442909, -0.04955402389168739, 0.12433774769306183, -0.06899472326040268, -0.1851322203874588, 0.049616534262895584, 0.07911069691181183, -0.0032351466361433268, -0.11167012155056], [-0.05538157746195793, -0.044067807495594025, 0.08819632977247238, -0.09470928460359573, 0.05890778824687004, -0.03133566677570343, 0.1058136373758316, -0.061802830547094345, -0.09134665876626968, -0.0011468057055026293, -0.03851744160056114, 0.03275332227349281, 0.18420618772506714, -0.04763360694050789, -0.01615161821246147, 0.05660410225391388, 0.15952196717262268, -0.05106552690267563, 0.055862486362457275, 0.027619393542408943, -0.1149514839053154]], "b1": [-0.12625323235988617, 0.21286773681640625, 0.11896148324012756, -0.17307886481285095, 0.2175169736146927, 0.15204885601997375, -0.0371813103556633, -0.02518823742866516, 0.13305334746837616, 0.08420911431312561, -0.14831119775772095, -0.002631236333400011, 0.03563961014151573, -0.11569076031446457, -0.14836227893829346, 0.046281784772872925, -0.035464756190776825, 0.16302689909934998, 0.04975677654147148, 0.14502933621406555, -0.05902644246816635, 0.09415848553180695, 0.1731300801038742, 0.06721623986959457, 0.04225907474756241, -0.12559716403484344, 0.13162527978420258, 0.030851686373353004, -0.05421729385852814, -0.0012338875094428658, -0.09192980825901031, -0.012399842031300068, 0.08923356235027313, 0.11808716505765915, 0.19293169677257538, 0.11533327400684357, 0.23044590651988983, 0.0367802195250988, -0.09971165657043457, 0.11297638714313507, -0.11798808723688126, -0.05749323591589928, -0.038673948496580124, 0.0314977802336216, -0.06916336715221405, 0.05413689464330673, -0.03570503368973732, -0.09576928615570068, -0.024083945900201797, 0.08937540650367737, 0.014349395409226418, 0.03926241770386696, -0.03360532596707344, 0.08739354461431503, 0.030805954709649086, -0.024962574243545532, 0.023709123954176903, -0.07731035351753235, 0.059103917330503464, 0.025980057194828987, -0.17313963174819946, 0.1011197417974472, 0.07300785183906555, 0.13208885490894318, -0.04675712808966637, -0.1331871896982193, -0.17053359746932983, 0.14282232522964478, -0.08694121986627579, -0.10879038274288177, -0.005992434918880463, -0.0371355265378952, -0.08902693539857864, -0.17857865989208221, -0.07470078766345978, 0.06973680108785629, 0.21126459538936615, 0.10599875450134277, 0.039924316108226776, -0.05322876200079918, -0.012391415424644947, 0.16787725687026978, 0.05548158288002014, 0.016531722620129585, 0.1403713971376419, 0.07746456563472748, 0.17490340769290924, -0.07647603750228882, 0.05218987539410591, 0.07688315957784653, 0.12248877435922623, -0.024764161556959152, -0.0021694391034543514, -0.1007443442940712, 0.13882991671562195, 0.16796693205833435], "W2": [[0.06879922747612, 0.01482413336634636, 0.00446871854364872, -0.045303381979465485, -0.02233405038714409, -0.03234190121293068, -0.05824355036020279, -0.0037652156315743923, 0.0572078637778759, 0.06724313646554947, -0.09457898139953613, 0.06286755949258804, -0.04756970703601837, 0.008922656066715717, 0.034547846764326096, 0.009606696665287018, -0.010153467766940594, -0.08878627419471741, 0.09148558229207993, 0.006776086986064911, 0.10294346511363983, -0.07171017676591873, 0.06924214214086533, 0.061333801597356796, -0.017376752570271492, 0.012134364806115627, 0.056156035512685776, 0.06955864280462265, -0.03617825731635094, 0.056748759001493454, -0.06154672056436539, 0.005789507180452347, -0.03598910570144653, 0.00907738134264946, 0.06730920076370239, 0.06007632985711098, 0.022661244496703148, 0.07480745762586594, -0.007759926840662956, -0.060369640588760376, 0.031508252024650574, 0.07349535077810287, -0.0670010969042778, 0.020695993676781654, -0.014638389460742474, 0.05978911370038986, -0.08454359322786331, -0.03731849044561386, 0.03643662855029106, 0.07247209548950195, 0.0640062466263771, 0.0350138358771801, 0.06864800304174423, -0.04261317476630211, -0.021672293543815613, 0.009271332062780857, -0.032493747770786285, 0.017500950023531914, 0.1109733060002327, -0.03054703027009964, -0.021359721198678017, -0.04802510142326355, -0.026230495423078537, -0.05715245380997658, -0.004158569499850273, -0.10125628858804703, -0.05943173170089722, 0.027151890099048615, -0.02199486456811428, -0.03790127858519554, 0.03820078447461128, 0.04405677691102028, -0.05700117349624634, -0.06733625382184982, -0.018113672733306885, 0.029783396050333977, -0.00817467924207449, 0.03617660328745842, -0.05268315598368645, 0.01160146202892065, -0.03087223879992962, 0.08163069933652878, -0.003951400984078646, 0.028075601905584335, 0.09531991928815842, 0.003712200094014406, -0.011533839628100395, -0.01978614367544651, -0.04692444950342178, -0.006527476478368044, 0.05292302742600441, 0.09655352681875229, -0.029994867742061615, -0.07047402113676071, 0.0512966550886631, 0.03254057466983795], [0.04694443196058273, -0.00275800540111959, 0.044713739305734634, 0.026555007323622704, -0.022783217951655388, 0.02814588136970997, 0.0044610933400690556, -0.018808040767908096, 0.0029701553285121918, 0.008726361207664013, 0.061249300837516785, -0.005950897932052612, 0.015203757211565971, 0.06545159220695496, 0.0011536331148818135, 0.0022563531529158354, 0.061793308705091476, 0.037784427404403687, -0.01878085359930992, 0.030248677358031273, 0.04874676093459129, 0.04709038883447647, 0.013130664825439453, -0.026983607560396194, 0.02087991312146187, 0.031249629333615303, -0.03374682739377022, 0.04154186323285103, 0.05615107715129852, -0.030483044683933258, -0.0016170671442523599, 0.056219033896923065, 0.026871757581830025, 0.043614841997623444, -0.0013449571561068296, -0.056493129581213, -0.04597904905676842, -0.026593318209052086, -0.006094443611800671, 0.05214747413992882, 0.016127057373523712, 0.04598705843091011, -0.003399668959900737, -0.025973618030548096, 0.016394926235079765, 0.014959522522985935, -0.01794767752289772, 0.044647492468357086, -0.007744056638330221, -0.0074491617269814014, -0.07912792265415192, 0.02265317551791668, 0.020220812410116196, -0.006188362371176481, 0.005080683622509241, 0.00501451687887311, 0.018736237660050392, 0.015613878145813942, -0.01563478633761406, 0.0036632316187024117, 0.060084834694862366, 0.0004534459440037608, -0.001390499877743423, -0.002094003837555647, 0.056168872863054276, 0.06375884264707565, -7.805282075423747e-05, -0.049320150166749954, 0.054837390780448914, 0.05742952600121498, 0.029363300651311874, 0.0015924426261335611, 0.06183275207877159, 0.011583353392779827, -0.008637054823338985, -0.005370903294533491, 0.0018862917786464095, 0.06504731625318527, 0.02951519563794136, -0.0014622763264924288, 0.019691774621605873, -0.012324406765401363, 0.043801408261060715, 0.002556830644607544, 0.04993567615747452, 0.013355382718145847, 0.03221037611365318, -0.0206363033503294, 0.11457952111959457, 0.0033874157816171646, 0.02422480098903179, -0.019586432725191116, 0.03173819184303284, -0.03804263100028038, 0.0386744923889637, 0.012595889158546925], [0.06654885411262512, 0.05228321626782417, -0.01062860805541277, -0.049890290945768356, 0.030792225152254105, -0.03491750732064247, -0.08418438583612442, 0.01829918660223484, 0.035753022879362106, -0.011358681134879589, -0.034356024116277695, 0.09339004755020142, 0.0637054368853569, -0.11335143446922302, 0.03715945780277252, 0.013180430978536606, 0.0471477247774601, -0.037437062710523605, -0.021229248493909836, -0.0019164510304108262, 0.012605003081262112, -0.07410719990730286, 0.012185058556497097, 0.028838807716965675, 0.09286807477474213, 0.013149011880159378, 0.07377711683511734, 0.044860128313302994, -0.13254955410957336, -0.05603452026844025, -0.08511392027139664, -0.1127241775393486, -0.05003269389271736, 0.0655408501625061, -0.026955517008900642, 0.11699911206960678, -0.01177488174289465, 0.10608070343732834, 0.02615281194448471, -0.0641983300447464, -0.008423462510108948, -0.05108144134283066, -0.0622166246175766, -0.018485229462385178, -0.06585250794887543, 0.005658215843141079, 0.04221953824162483, -0.05049958452582359, 0.11865164339542389, -0.05897962301969528, -0.047818347811698914, 0.09337227046489716, 0.01635524444282055, 0.05034860596060753, -0.04317592829465866, 0.11305959522724152, -0.009141964837908745, 0.0494062714278698, 0.10592024028301239, -0.05045633390545845, 0.0004801168688572943, 0.014570376835763454, -0.03733342885971069, -0.006367824971675873, 0.020233258605003357, -0.0937352105975151, -0.05899371579289436, 0.016579674556851387, -0.02143010124564171, -0.0660628229379654, 0.07825718075037003, -0.021711578592658043, -0.0024051046930253506, -0.09041117131710052, 0.013062965124845505, 0.003535730764269829, 0.0675271525979042, -0.0007115997723303735, -0.007726922165602446, -0.05076144263148308, 0.002864514011889696, -0.0019400623859837651, 0.03692939877510071, 0.12453590333461761, 0.06002195551991463, 0.061283960938453674, 0.06502770632505417, -0.04170577600598335, -0.08940579742193222, 0.060788486152887344, 0.0706409364938736, 0.1748763471841812, 0.04723111540079117, 0.025810549035668373, 0.03757373243570328, 0.11233218759298325], [0.0276795681566, 0.05225767940282822, 0.013720511458814144, -0.02066112868487835, 0.01689947210252285, 0.028404859825968742, -0.00872581172734499, 0.008077174425125122, 0.02695789933204651, 0.04686088487505913, -0.011995350010693073, -0.007889469154179096, -0.021221043542027473, -0.015559191815555096, -0.019907468929886818, 0.04438675567507744, -0.014680404216051102, 0.0037343150470405817, 0.03888236731290817, 0.008168798871338367, 0.005902289878576994, 0.04948636144399643, -0.01522540207952261, 0.0034816674888134003, -0.012026609852910042, 0.005344670731574297, 0.002922717947512865, 0.04505513235926628, -0.022451085969805717, -0.027876101434230804, -0.026896363124251366, -0.05854448303580284, 0.05516178905963898, -0.048064086586236954, 0.03084736317396164, 0.020344870164990425, -0.03882478550076485, 0.029824214056134224, -0.027701660990715027, -0.025424504652619362, -0.023142917081713676, -0.008979620411992073, 0.048577893525362015, 0.029901541769504547, -0.04470761492848396, -0.021354639902710915, -0.008769212290644646, -0.047320105135440826, 0.0030601471662521362, -0.00010132154420716688, -0.038125887513160706, -0.01295002643018961, 0.025899209082126617, 0.007414679974317551, 0.015611856244504452, 0.07638619840145111, 0.073189876973629, -0.0038813105784356594, 0.07450822740793228, 0.0019161339150741696, -0.03840566799044609, 0.0020719014573842287, 0.01664181426167488, -0.014009232632815838, -0.04064131900668144, 0.008647074922919273, 0.014597664587199688, 0.03354944288730621, -0.05197126418352127, 0.010951684787869453, -0.029548630118370056, -0.026167768985033035, -0.010238482616841793, -0.0394158773124218, -0.04790017008781433, 0.030498649924993515, -0.0002420013042865321, 0.028692645952105522, -0.028842581436038017, -0.0018621392082422972, -0.02406233549118042, -0.01788756437599659, -0.004254422150552273, 0.007150465622544289, -0.026003524661064148, -0.0036168748047202826, 0.04285692051053047, -0.02535712905228138, -0.026609208434820175, -0.0017009462462738156, 0.040722768753767014, 0.011057100258767605, -0.019046533852815628, 0.04562264308333397, -0.014965365640819073, 0.020707938820123672], [-0.033273980021476746, -0.008921808563172817, 0.07160990685224533, -0.10807911306619644, 0.07403592020273209, -0.07665333896875381, 0.0011922414414584637, 0.012523707933723927, -0.06476614624261856, -0.022821910679340363, -0.10545050352811813, 0.044902585446834564, -0.05291881039738655, -0.07153621315956116, 0.02384219318628311, 0.06894099712371826, 0.054062072187662125, 0.0239874254912138, 0.06852829456329346, 0.036501698195934296, -0.03353782743215561, -0.013364002108573914, 0.10575580596923828, 0.06198178604245186, 0.04651276394724846, -0.09705647826194763, 0.08200269937515259, -0.022800061851739883, -0.061889681965112686, 0.07411270588636398, -0.05377576872706413, -0.033531006425619125, -0.05216725543141365, 0.05374535545706749, -0.07170452922582626, 0.09786850214004517, 0.047363605350255966, 0.08844035118818283, -0.03970005363225937, 0.051273614168167114, -0.012907428666949272, 0.0771038606762886, -0.028133241459727287, 9.370758198201656e-05, -0.08504766970872879, 0.017087643966078758, 0.07950948178768158, -0.044096510857343674, -0.044977787882089615, -0.00014789143460802734, -0.05078766867518425, 0.0741376206278801, -0.05074566602706909, 0.07282303273677826, -0.05736851692199707, 0.007573672570288181, 0.07008517533540726, 0.029912903904914856, 0.04155841842293739, 0.05280831828713417, 0.04437357559800148, -0.06108477711677551, -0.030565299093723297, -0.016785576939582825, -0.10784224420785904, -0.07809778302907944, -0.03589940071105957, -0.0009346415172331035, 0.004208457190543413, -0.01328401267528534, -0.08880413323640823, 0.0003441854496486485, -0.05376271530985832, -0.039419904351234436, 0.012837819755077362, -0.060342904180288315, -0.0766478106379509, 0.07754421979188919, 0.07896941900253296, 0.03117774985730648, 0.01789751462638378, 0.07852292060852051, 0.05309053137898445, -0.014895354397594929, 0.034760113805532455, -0.07867930084466934, 0.09246336668729782, -0.025443825870752335, 0.03906544670462608, -0.04905750975012779, -0.015622841194272041, 0.12595747411251068, -0.008025355637073517, -0.022839367389678955, 0.0047666048631072044, 0.012036807835102081], [0.07979537546634674, -0.018539665266871452, -0.01970830000936985, 0.048352066427469254, -0.08714357763528824, -0.006368371192365885, 0.03189953416585922, -0.0024229579139500856, 0.01685073785483837, -0.047013889998197556, 0.03645548224449158, -0.02897745743393898, -0.04098290205001831, 0.12414973229169846, -0.007952808402478695, 0.009510448202490807, 0.109298475086689, -0.03695633262395859, -0.011223376728594303, -0.02255835570394993, -0.01939849741756916, -0.08742081373929977, -0.0026249403599649668, -0.015823855996131897, 0.03684651479125023, 0.11694613099098206, 0.008696842007339, -0.038798581808805466, 0.012794490903615952, -0.05485087260603905, 0.030241433531045914, -0.008815247565507889, -0.05060058459639549, -0.022641994059085846, -0.08093573153018951, -0.06774955987930298, -0.0669260025024414, 0.012308681383728981, 0.029780041426420212, 0.0633615106344223, -0.00602788245305419, 0.03247605636715889, 0.0017054203199222684, -0.008629733696579933, 0.0019014272838830948, -0.02534693479537964, 0.037259746342897415, 0.02249107137322426, -0.04499693959951401, 0.026986919343471527, -0.04133692756295204, 0.004496625158935785, -0.09450960159301758, -0.04486604034900665, -0.01559385471045971, -0.049642398953437805, -0.025358572602272034, 0.08571777492761612, 0.07217733561992645, -0.061174411326646805, 0.1365930438041687, -0.031660046428442, 0.03330779820680618, -0.03779413178563118, 0.006282013840973377, 0.030706051737070084, 0.0014078360982239246, -0.05619817599654198, -0.0064051044173538685, 0.036136530339717865, -0.007976249791681767, -0.044390372931957245, 0.07920757681131363, -0.015073898248374462, -0.0026626738253980875, 0.00037214544136077166, -0.03746975213289261, 0.0918409526348114, -0.016998743638396263, 0.004188800696283579, 0.015769489109516144, -0.000933916075155139, 0.027431391179561615, 0.02577098086476326, 0.011858897283673286, -0.015906330198049545, 0.017254991456866264, 0.024712296202778816, 0.08231856673955917, -0.06939826905727386, 0.03191988915205002, -0.03352218121290207, 0.01018619816750288, 0.04650317132472992, -0.015255517326295376, 0.0047739907167851925], [-0.01639171876013279, -0.050069380551576614, -0.07989587634801865, 0.0316922664642334, 0.006884833797812462, -0.06473476439714432, 0.011432959698140621, 0.026268985122442245, -0.035579863935709, -0.1040177270770073, 0.050229258835315704, -0.034807562828063965, -0.05207210034132004, 0.022085389122366905, 0.07406409084796906, -0.00524114491418004, -0.027979854494333267, 0.07017350941896439, -0.010584869422018528, 0.05642060190439224, 0.0006767704617232084, -0.016780104488134384, -0.02217944525182247, -0.022609079256653786, -0.03926262632012367, -0.022894585505127907, 0.03432510793209076, -0.03090096265077591, -0.00950501300394535, -0.12196578085422516, -0.026412738487124443, 0.021927854046225548, 0.022032685577869415, -0.029567191377282143, -0.10247526317834854, -0.008805446326732635, 0.022505832836031914, -0.08540881425142288, -0.08619167655706406, -0.06650377810001373, -0.03697703033685684, -0.02762318216264248, -0.059884559363126755, 0.04312271252274513, 0.0034796122927218676, -0.007281300611793995, -0.05866173654794693, 0.040731120854616165, -0.008563054725527763, 0.0017792598810046911, -0.08568842709064484, -0.04890420287847519, 0.012767397798597813, -0.08308472484350204, 0.038169149309396744, -0.09906618297100067, 0.02976391837000847, -0.03115931898355484, -0.023959144949913025, 0.08674782514572144, 0.007689699064940214, -0.07171861082315445, 0.05800781399011612, -0.005662068724632263, -0.05868622660636902, 0.0019199661910533905, 0.058581478893756866, 0.0027501455042511225, 0.028557684272527695, 0.014667326584458351, -0.030813241377472878, 0.024163536727428436, -0.06712382286787033, 0.04414292052388191, -0.037192609161138535, -0.07990614324808121, -0.05341208726167679, -0.004900856874883175, -0.07883737981319427, 0.05501148849725723, -0.012992290779948235, -0.0550348237156868, -0.08333498239517212, -0.056969888508319855, -0.07065629959106445, -0.0733879804611206, -0.043183766305446625, -0.008085469715297222, -0.07158143073320389, 0.048391859978437424, 0.06870637834072113, 0.009079928509891033, 0.0468958355486393, -0.003546804189682007, 0.0442899726331234, 0.0061268797144293785], [0.0014904632698744535, -0.0012487303465604782, -0.0025276103988289833, 0.004825879354029894, -0.004336371552199125, -0.0031856363639235497, 0.0006375289522111416, -0.0018768365262076259, -0.0005114107625558972, 0.0018530171364545822, 0.002186748431995511, 0.0014306060038506985, 0.002531890058889985, 0.0012041968293488026, 0.0002012476179515943, 0.0009547623922117054, -0.0009326621657237411, 0.01021075714379549, -0.002209651516750455, 0.004811484832316637, 0.002140782540664077, 0.003204090753570199, -0.006480828858911991, -0.000893395917955786, -0.0020251437090337276, 0.0005149806966073811, -0.0010241462150588632, 0.0011613761307671666, -0.0018495646072551608, 6.03044536546804e-05, 0.0002642472682055086, 0.003804609179496765, 0.0029865747783333063, 0.0007046997779980302, 0.00032928501605056226, 0.0005690910038538277, -0.0029013650491833687, -0.0012905693147331476, -0.0008070127805694938, -3.961742186220363e-05, -0.0009088487131521106, 0.0019147783750668168, -0.0003270729212090373, -0.0011198286665603518, 0.000495738466270268, -0.0010703826555982232, -0.002698007505387068, 0.002624396001920104, 0.002684414852410555, -0.002097140997648239, -0.0022374382242560387, 5.801864244858734e-05, -0.0004206994781270623, -0.00019812195387203246, 0.0011318634497001767, -0.00415710499510169, 0.008449752815067768, 0.0003002120356541127, -0.00443139486014843, -0.0008045673603191972, 0.0007465627859346569, 0.003605328034609556, 0.001546895713545382, 0.00110305764246732, 0.0046432241797447205, 0.0007088010897859931, -0.0003354724030941725, -0.011007547378540039, 0.0029782101046293974, 0.0011780341155827045, 0.0005102901486679912, 0.0018949245568364859, 0.0012692237505689263, 0.0009073816472664475, -0.0005767949623987079, 0.00275748735293746, -0.0002453020424582064, 0.0015515444101765752, -0.002911192364990711, -0.00023659721773583442, 0.0014326294185593724, -0.004259829875081778, -0.0034203149843961, 6.056488928152248e-05, 2.0590887288562953e-05, -0.0031376087572425604, -0.0036465879529714584, 0.0008472049958072603, 0.00810360349714756, 0.00046504908823408186, -0.0030009772162884474, -0.0024390665348619223, 0.0009540803148411214, -0.000774610263761133, 0.0009887117194011807, 0.008664127439260483], [0.09701919555664062, 0.06024313345551491, -0.017815280705690384, 0.05269210785627365, 0.0075158788822591305, -0.05212952941656113, 0.054138921201229095, 0.017974859103560448, -0.010145619511604309, -0.0626126304268837, -0.08853533118963242, 0.0705268532037735, 0.08226756006479263, -0.10197117179632187, -0.07869859784841537, 0.04212344065308571, 0.07953286170959473, -0.14729958772659302, -0.06795065850019455, 0.0922430157661438, 0.07227155566215515, 0.026626691222190857, 0.054167333990335464, -0.07644644379615784, 0.056538477540016174, -0.019197164103388786, 0.05313359573483467, 0.12742628157138824, 0.018488330766558647, 0.08986721187829971, -0.009742150083184242, -0.006250576116144657, -0.07927504181861877, 0.0739893689751625, 0.0797818973660469, 0.003396653803065419, 0.07551596313714981, -0.03257031738758087, 0.0012817581882700324, -0.004159888718277216, 0.05862201750278473, -0.05629908666014671, 0.0018393023638054729, -0.02209477126598358, -0.05637679994106293, -0.08646290004253387, -0.09785974770784378, -0.05864150822162628, 0.012099780142307281, 0.04522718861699104, 0.03181558474898338, 0.01260199025273323, 0.05612429603934288, 0.018028555437922478, 0.10371358692646027, 0.04744676873087883, 0.034638017416000366, -0.07266506552696228, 0.043285638093948364, 0.008698217570781708, 0.03392496705055237, -0.018421852961182594, 0.08092277497053146, 0.019245870411396027, -0.009500697255134583, -0.023309767246246338, -0.09580288082361221, 0.07153438776731491, -0.017815833911299706, -0.028063368052244186, -0.025534944608807564, 0.06818195432424545, -0.02103607729077339, -0.11647889018058777, -0.001252025831490755, 0.04245512932538986, 0.07668038457632065, 0.026516379788517952, 0.03059832751750946, 0.013564448803663254, -0.07418369501829147, -0.01611308753490448, 0.05477744713425636, -0.08802549540996552, -0.09785960614681244, -0.12254656851291656, -0.038084715604782104, 0.02501368708908558, -0.10377416759729385, 0.04121840000152588, -0.06484343856573105, -0.16230140626430511, 0.0021690225694328547, 0.04747798293828964, 0.048770662397146225, -0.06618461012840271], [0.1015852689743042, 0.061450134962797165, -0.07291451096534729, 0.00310668651945889, 0.04546595737338066, 0.005319791380316019, -0.07801145315170288, 0.0007448006654158235, 0.023623084649443626, 0.03598185256123543, -0.04254068806767464, -0.0008449219749309123, 0.04916084557771683, 0.017745690420269966, -0.04187415912747383, -0.011632975190877914, 0.03857087716460228, 0.05353879928588867, 0.051607269793748856, 0.011290142312645912, 0.037700578570365906, -0.027333635836839676, 0.03480098024010658, 0.06410110741853714, -0.01958216167986393, -0.11458760499954224, 0.012207020074129105, 0.03991956263780594, 0.059518709778785706, -0.07509394735097885, -0.10539058595895767, -0.08329354226589203, -0.052280738949775696, -0.041799675673246384, 0.0802701786160469, 0.07720977813005447, 0.07633422315120697, 0.08452995121479034, -0.00858497153967619, -0.03037850931286812, 0.05351068079471588, -0.07832098007202148, -0.029114045202732086, -0.015190224163234234, -0.14456219971179962, -0.056711453944444656, -0.03194161504507065, -0.00815875269472599, 0.08967692404985428, 0.019250413402915, 0.050435882061719894, -0.047794073820114136, 0.04473426565527916, -0.029127223417162895, 0.0414416678249836, -0.05463067442178726, -0.06054109334945679, 0.06472356617450714, 0.06926492601633072, 0.012852788902819157, -0.03855098411440849, 0.07090970873832703, 0.06433496624231339, -0.05922110378742218, -0.09378913789987564, -0.10130772739648819, -0.03579825162887573, 0.01509349700063467, -0.07191174477338791, 0.004126854706555605, 0.0137897077947855, 0.07630857080221176, -0.05131547898054123, -0.05519980937242508, 0.0638352632522583, -0.03883105143904686, -0.05750390887260437, 0.08108548820018768, 0.08536703139543533, -0.02107222191989422, 0.06639603525400162, 0.06004316732287407, 0.06283184885978699, 0.04651137441396713, 0.03223768249154091, -0.06467002630233765, -0.01041295938193798, -0.046837013214826584, -0.04797210544347763, 0.05579371750354767, -0.020255815237760544, -0.02001652680337429, -0.05653270334005356, -0.11773179471492767, -0.04161803051829338, -0.029756492003798485], [-0.015534084290266037, -0.030755456537008286, 0.04782852157950401, -0.03665916621685028, -0.04367101192474365, -0.0005367127596400678, -0.03897314518690109, 0.026741115376353264, -0.030964476987719536, 0.05236101895570755, -0.01240502018481493, 5.522481387743028e-07, -0.03822583332657814, -0.001016655471175909, -0.019974837079644203, -0.04178733006119728, -0.02113223262131214, 0.04658276215195656, 0.026914719492197037, -0.026486191898584366, -0.04209958761930466, -0.04893757775425911, 0.03158897906541824, 0.058864593505859375, 0.05099974572658539, -0.035841431468725204, 0.05814671516418457, 0.042378123849630356, 0.04288113862276077, -0.02119479700922966, -0.020071670413017273, -0.015091262757778168, 0.03519275039434433, 0.024904659017920494, 0.030703864991664886, 0.09305377304553986, 0.03978373855352402, 0.05060585215687752, -0.010531429201364517, -0.010448211804032326, 0.04573599249124527, 0.016185615211725235, -0.028803156688809395, -0.010291696526110172, -0.017214953899383545, 0.0425146110355854, -0.0032490799203515053, -0.022276345640420914, -0.0014725562650710344, 0.06408551335334778, 0.00048786099068820477, 4.539554356597364e-05, -0.0440865196287632, -0.04366384074091911, 0.0016339017311111093, -0.032079026103019714, 0.052999287843704224, -0.002188781276345253, 0.011888878419995308, -0.055429451167583466, 0.0034582968801259995, 0.03434034809470177, -0.05021699145436287, -0.01124534197151661, 0.026019873097538948, -0.060905855149030685, -0.02687281370162964, -0.036395519971847534, -0.0504014678299427, 0.034299086779356, -0.014227787964046001, 0.02708365023136139, -0.006513890344649553, 0.005121916066855192, 0.01927863247692585, 0.012038679793477058, 0.04204091429710388, 0.041762575507164, -0.01392234954982996, 0.023576881736516953, -0.044557344168424606, -0.037285592406988144, 0.019098220393061638, 0.01810361072421074, 0.03037862479686737, -0.0621655248105526, 0.02048281580209732, -0.048027750104665756, 0.04909935221076012, 0.018995700404047966, 0.07824387401342392, 0.010624808259308338, -0.038352612406015396, -0.034957412630319595, 0.04212093725800514, 0.03110678866505623], [0.12044462561607361, -0.06863002479076385, -0.020291779190301895, 0.0663934126496315, 0.006475788541138172, -0.023142699152231216, -0.03890519589185715, 0.056430574506521225, -0.027618063613772392, -0.05174224078655243, 0.09907383471727371, 0.03063412755727768, -0.06973351538181305, 0.13390430808067322, 0.03116730786859989, 0.022389980033040047, 0.08530329167842865, -0.09968525171279907, -0.07629796117544174, 0.0007004029466770589, -0.09705645591020584, -0.08900700509548187, -0.07764717936515808, -0.05938262864947319, 0.016865957528352737, 0.05104886740446091, 0.02613680064678192, -0.03820178657770157, -0.0031113203149288893, 0.07612599432468414, 0.04132313281297684, -0.0926189124584198, -0.0762457549571991, -0.11098819226026535, 0.022754784673452377, 0.0023963507264852524, -0.061098888516426086, -0.023711109533905983, 0.03491003438830376, -0.0007138038636185229, 0.014296909794211388, -0.0388677716255188, -0.048090074211359024, -0.00015737437934149057, 0.02408108115196228, 0.04950699582695961, -0.05030805245041847, -0.006781114265322685, -0.045260075479745865, -0.11520859599113464, -0.050084080547094345, 0.00564880995079875, -0.058500852435827255, -0.02011151984333992, -0.06392818689346313, 0.011007042601704597, -0.08215194195508957, 0.11019392311573029, 0.03254030644893646, -0.08420267701148987, 0.07962706685066223, -0.09657170623540878, -0.07092717289924622, 0.030092043802142143, 0.03586891293525696, 0.06696488708257675, -0.004965490195900202, -0.005969211459159851, 0.0335504524409771, 0.10395932197570801, -0.030476368963718414, -0.10773801803588867, 0.13692715764045715, 0.07356730848550797, 0.018105946481227875, 0.06804690510034561, -0.05764048546552658, -0.02104405127465725, 0.006032753270119429, 0.0006532628904096782, -0.05148378759622574, 0.014640231616795063, 0.08255001157522202, 0.06631200760602951, 0.04945368319749832, 0.07383270561695099, -0.02079494297504425, 0.09929563105106354, -0.0033119346480816603, -0.044248804450035095, -0.09312257915735245, -0.05019824579358101, -0.10059043020009995, -0.011881937272846699, 0.04836985468864441, -0.03984233736991882], [-0.01691180281341076, -0.04788922145962715, -0.01728132739663124, 0.03043399751186371, -0.0014540760312229395, 0.015124285593628883, 0.01803058572113514, 0.03973676636815071, -0.02295049838721752, 0.06336147338151932, -0.007183431647717953, 0.05244358256459236, 0.022425055503845215, -0.0032802585046738386, -0.0028398267459124327, -0.015432775020599365, -0.03019827790558338, -0.01569424383342266, -0.0244700089097023, -0.033972349017858505, 0.0012834701919928193, 0.06160644814372063, 0.017797715961933136, 0.021757377311587334, -0.017988916486501694, -0.04446393623948097, 0.015810977667570114, 0.022253328934311867, 0.0351155549287796, -0.05389507859945297, -0.00148996920324862, -0.003253981238231063, -0.021571220830082893, -0.008237162604928017, -0.006568923592567444, 0.013057122007012367, -0.053658779710531235, -0.011822547763586044, -0.02285332977771759, -0.017670763656497, -0.004281318746507168, -0.011635715141892433, 0.02688186801970005, 0.004378045443445444, 0.035722099244594574, 0.02008698508143425, -0.022306188941001892, 0.03866778686642647, 0.02098271995782852, -0.012436316348612309, -0.010896139778196812, 0.028815852478146553, -0.029111359268426895, -0.002702901139855385, -0.0016834246926009655, 0.04263349995017052, -0.017388010397553444, -0.05174517631530762, -0.04202762991189957, -0.0442587174475193, -0.05036069080233574, 0.01887134090065956, 0.00537991663441062, 0.014220839366316795, 0.08604912459850311, 0.026518523693084717, 0.036722101271152496, 0.015842093154788017, 0.021527457982301712, -0.008390267379581928, 0.030694207176566124, 0.010929438285529613, -0.013876611366868019, 0.027528034523129463, -0.018823858350515366, 0.014108400791883469, 0.0005641725729219615, -0.022462692111730576, -0.021295402199029922, -0.01984669640660286, 0.0009445830364711583, -0.03408302366733551, -0.00768530648201704, 0.0026838474441319704, 0.03185515105724335, 0.025414379313588142, 0.01731596700847149, 0.038789648562669754, 0.024114428088068962, -0.006862840615212917, 0.014234496280550957, -0.016411874443292618, -0.028647765517234802, -0.009339505806565285, 0.017484718933701515, -0.01671389676630497], [-0.00996832363307476, 0.04524962604045868, 0.07960817962884903, 0.04438847675919533, 0.0061459592543542385, 0.07578085362911224, -0.0958678126335144, 0.08081939071416855, 0.08593465387821198, 0.04091426730155945, -0.10525861382484436, 0.09057603776454926, 0.04042968899011612, -0.0124555304646492, 0.022452736273407936, 0.011290735565125942, 0.057473570108413696, -0.13240279257297516, 0.02439999021589756, 0.022347427904605865, 0.0706547275185585, -0.04394698515534401, -0.05515707656741142, -0.023935681208968163, 0.010195177048444748, 0.0015358907403424382, -0.05027792975306511, 0.10538709908723831, -0.0036186801735311747, 0.03271220996975899, -0.08307646960020065, -0.0531005896627903, -0.015723954886198044, 0.09347580373287201, 0.11757954210042953, 0.05584319680929184, -0.06245555356144905, -0.03791624307632446, -0.07647068053483963, 0.07206333428621292, 0.010206134058535099, 0.005173963028937578, 0.025183511897921562, 0.07822822779417038, 0.03633766621351242, -0.021590769290924072, -0.07302224636077881, -0.09445729851722717, -0.07672243565320969, 0.031836677342653275, -0.040278080850839615, -0.11565959453582764, -0.03907746076583862, -0.05105823650956154, 0.07194188982248306, -0.0675206333398819, 0.08189351111650467, -0.03136095404624939, 0.06743277609348297, 0.07973726093769073, -0.04411235824227333, -0.035759005695581436, 0.015430928207933903, 0.032763831317424774, -0.08270934224128723, 0.0671008825302124, 0.04992515593767166, 0.019100409001111984, 0.04640963673591614, -0.044479064643383026, 0.010363428853452206, -0.031075578182935715, -0.05279368534684181, -0.0028784212190657854, -0.027923479676246643, -0.048365890979766846, 0.06759648025035858, -0.03496873006224632, -0.0009562235791236162, -0.010740543715655804, -0.060416288673877716, 0.007164607755839825, -0.02426060661673546, 0.06971341371536255, -0.11722691357135773, 0.018056008964776993, 0.01523172203451395, 0.07020010054111481, 0.04637867957353592, -0.05335186421871185, 0.0010980138322338462, -0.09107328951358795, -0.05402378737926483, -0.038180943578481674, 0.05480920150876045, -0.029645459726452827], [-0.00199296185746789, 0.03390052914619446, -0.06658951938152313, -0.02879687398672104, 0.03387335687875748, -0.05166487768292427, -0.017475564032793045, 0.036789555102586746, -0.03835441544651985, -0.03859446197748184, -0.00547410361468792, -0.03653039410710335, 0.002570014214143157, 0.038695961236953735, 0.008872948586940765, 0.04932933300733566, 0.018498891964554787, -0.034837767481803894, -0.026636231690645218, 0.07195141166448593, 0.009578343480825424, 0.005074626300483942, 0.08259522914886475, 0.046969059854745865, 0.05843766778707504, -0.06539268046617508, 0.0560736358165741, 0.059995777904987335, 0.02623371221125126, 0.08780481666326523, 0.01330318208783865, 0.002299250103533268, -0.022016583010554314, 0.09257955104112625, 0.08201145380735397, 0.0704265758395195, 0.04186874255537987, -0.03136833384633064, -0.0541565828025341, 0.026297811418771744, 0.03657801076769829, -0.058810681104660034, 0.07176440209150314, 0.05221349373459816, -0.05963660776615143, 0.03509620204567909, 0.05891935154795647, -0.059742044657468796, 0.026790494099259377, 0.08614558726549149, -0.0010374094126746058, 0.01653193309903145, 0.05384553596377373, 0.06848142296075821, -0.01154700480401516, 0.044220950454473495, 0.020001765340566635, -0.014009779319167137, 0.10713167488574982, -0.02060742862522602, -0.04898902028799057, 0.04488619789481163, -0.03537943214178085, -0.0350821278989315, -0.025889312848448753, -0.01761784590780735, -0.0494670569896698, 0.079161137342453, -0.04807395488023758, -0.021032769232988358, -0.05504907667636871, -8.258660818682984e-05, 0.0026187102776020765, -0.02142290212213993, 0.015148588456213474, 0.0082927905023098, -0.06295385211706161, 0.0032124482095241547, 0.02147657610476017, 0.05742131918668747, 0.0154593326151371, 0.04065420478582382, 0.054419126361608505, -0.014751089736819267, 0.0702071487903595, -0.006665647495537996, 0.021195843815803528, 0.03389018028974533, -0.07006554305553436, -0.009935341775417328, 0.06850603967905045, -0.015272211283445358, 0.021838389337062836, 0.07071999460458755, 0.0448940135538578, -0.03732873126864433], [0.029232239350676537, -0.027681395411491394, -0.013917668722569942, 0.033118270337581635, -0.03784218803048134, -0.0024626401718705893, 0.011254346929490566, -0.024486515671014786, 0.07719164341688156, -0.01865474134683609, 0.03252267837524414, -0.015435955487191677, -0.003963090013712645, 0.045303117483854294, 0.002320188097655773, -0.01831572875380516, 0.09076589345932007, -0.0136775067076087, -0.008920364081859589, -0.022598644718527794, 0.005176708102226257, -0.030233781784772873, -0.020010804757475853, -0.005971620324999094, 0.018904225900769234, 0.04109257832169533, 0.007708139251917601, -0.00741635262966156, -0.0006944226333871484, -0.013184902258217335, 0.018991701304912567, 0.012577175162732601, -0.006771606858819723, -0.024260982871055603, -0.0312194861471653, -0.033623106777668, -0.048612259328365326, 0.00440216576680541, 0.02809865027666092, 0.011404376477003098, 0.01816416159272194, 0.012452338822185993, -0.022376371547579765, 0.018547553569078445, 0.0013420484028756618, -0.0016759919235482812, 0.029493674635887146, 0.0015920484438538551, -0.02294832654297352, -0.03125491365790367, -0.02143585868179798, -0.011456352658569813, -0.04140140116214752, -0.019495556131005287, -0.01990080066025257, -0.02800365909934044, -0.02299734763801098, 0.09682808071374893, 0.040344636887311935, -0.03316137194633484, 0.08621487021446228, -0.034592319279909134, 0.0022103700321167707, -0.02324933558702469, -0.01319397147744894, -0.004101727623492479, 0.005912595894187689, -0.05546414852142334, 0.0095438864082098, 0.027185125276446342, -0.010871280916035175, -0.024127013981342316, 0.05291347578167915, 0.0015292686875909567, -0.002103823935613036, -0.0006616001483052969, 0.00661192974075675, 0.03978380933403969, -0.024467194452881813, -0.016658512875437737, -0.0078040845692157745, -0.013204752467572689, 0.05203689634799957, -0.011069866828620434, -0.0007837449666112661, 0.013257683254778385, -0.013232148252427578, 0.015280205756425858, 0.05729707330465317, -0.03442491218447685, -0.029069695621728897, -0.021325526759028435, -0.02336142212152481, 0.004195443354547024, -0.028728999197483063, 0.02724628522992134], [0.011497572995722294, -0.03338008001446724, 0.04657094553112984, 0.0038046611007303, -0.06167733669281006, 0.005158285144716501, -0.020420055836439133, -0.019110580906271935, -0.03925614058971405, -0.019748849794268608, 0.017186177894473076, 0.017318012192845345, 0.01883327215909958, -0.010196994990110397, 0.018649626523256302, -0.0009078218135982752, -0.021922947838902473, -0.0015952254179865122, -0.02564280293881893, -0.0026884984690696, 0.003113816026598215, 0.013597628101706505, 0.01110667921602726, -0.004476963542401791, 0.03238653764128685, -0.010101576335728168, -0.008190754801034927, 0.012629056349396706, 0.03852084279060364, 0.023264527320861816, -0.014060434885323048, 0.019575441256165504, -0.0045101940631866455, 0.003542209044098854, 0.0029636137187480927, 0.004664299078285694, 0.0022508224938064814, -0.020493758842349052, -0.01726098544895649, 0.024637505412101746, -0.017241498455405235, 0.00798044167459011, -0.03684045001864433, 0.0022166280541568995, -0.01055394671857357, 0.037653107196092606, 0.04378047585487366, -0.006373839918524027, -0.013395596295595169, -0.051327917724847794, 0.03473661094903946, -0.02594801038503647, 0.038473621010780334, -0.03039725124835968, -0.014355233870446682, 0.017002545297145844, -0.06433011591434479, -0.005899408366531134, -0.03360343351960182, 0.032559800893068314, -0.02249210700392723, -0.03298971801996231, -0.028355777263641357, 0.034023284912109375, 0.034373484551906586, 0.027000272646546364, 0.014818611554801464, -0.038445841521024704, 0.04427371174097061, -0.009600835852324963, 0.012306774966418743, -0.030832581222057343, -0.008100791834294796, -0.004468108061701059, -0.0013281403807923198, 0.013116459362208843, -0.05230678617954254, 0.01966194063425064, -0.044246215373277664, -0.006516758818179369, 0.017926650121808052, 0.011004985310137272, 0.016451148316264153, -0.00946801993995905, 0.038547661155462265, 0.04729967936873436, -0.04365215450525284, -0.01129359845072031, 0.03748061880469322, 0.001259448123164475, 0.03871499374508858, -0.004462665878236294, -0.0391412116587162, 0.0028107997495681047, -0.029547804966568947, -0.02328616939485073], [0.028875425457954407, -0.023581266403198242, -0.016060413792729378, -0.02555082179605961, 0.03981594368815422, 0.012369285337626934, -0.02542758546769619, -0.03975750878453255, 0.0203697569668293, 0.022754831239581108, -0.03077809140086174, 0.0006576754967682064, 0.002688289387151599, -0.0379098579287529, 0.002240175614133477, -0.014389494433999062, 0.009625745937228203, -0.059516433626413345, 0.003547623986378312, -0.003452734788879752, 0.03892681002616882, -0.035183824598789215, 0.01886090822517872, -0.016732238233089447, -0.011982746422290802, -0.019567390903830528, 0.019820863381028175, 0.022045237943530083, -0.034978460520505905, 0.0012058224529027939, -0.03143386170268059, -0.012522149831056595, -0.006740532349795103, -0.004276412073522806, 0.0173457283526659, -0.0029309550300240517, -0.02509784884750843, 0.01988605596125126, -0.0109802121296525, 0.009746238589286804, -0.013429970480501652, -0.019434088841080666, 0.017662663012742996, -0.0053301649168133736, -0.01826057955622673, -0.014238200150430202, -0.011633871123194695, -0.020374951884150505, -0.004638202488422394, 0.03564850613474846, -0.011687325313687325, -0.02242402918636799, 0.023255757987499237, 0.029989948496222496, -0.005042799282819033, 0.0021801460534334183, -0.016458580270409584, 0.020354432985186577, 0.03309331461787224, -7.124629337340593e-05, -0.007509010378271341, 0.01645623706281185, -0.01436782255768776, -0.0032526582945138216, -0.031487297266721725, -0.008366037160158157, -0.024313874542713165, 0.0333561971783638, -0.01715228147804737, -0.012524783611297607, -0.017523376271128654, -0.014372576028108597, -0.009118505753576756, -0.015392281115055084, -0.007299238350242376, -0.010258480906486511, 0.005566003732383251, 0.01976897194981575, 0.0028068930841982365, 0.0205089021474123, -0.007537476252764463, 0.030704142525792122, 0.014581257477402687, 0.025779375806450844, 0.0041214171797037125, 0.017992697656154633, -0.003279771190136671, 0.0025982894003391266, -0.0553663931787014, 0.023466574028134346, -0.014511937275528908, 0.009547192603349686, -0.017139526084065437, -0.023316528648138046, -0.013030107133090496, 0.010916931554675102], [0.013690865598618984, 0.002499100985005498, -0.002637108089402318, -0.0017449232982471585, 0.016784729436039925, 0.0092686228454113, -0.004949827678501606, 0.006319578271359205, 0.004631712567061186, -0.02262914925813675, -0.0041679078713059425, 0.0056884754449129105, 0.00018664009985513985, -0.0038551671896129847, 0.0005925064324401319, -0.0028815476689487696, 0.006484962999820709, -0.01697620004415512, 0.0020698460284620523, 0.020154424011707306, -0.0054908678866922855, 0.006773222703486681, 0.008614984340965748, 0.015378479845821857, -0.01851673610508442, -0.004059546161442995, 0.0035735617857426405, 0.015416139736771584, -0.030868632718920708, -2.8844704502262175e-05, -0.007994739338755608, -0.00419664429500699, -0.0022326656617224216, 0.02409880980849266, -0.00993838720023632, 0.0024301621597260237, 0.010017827153205872, 0.004241640213876963, -0.001446424750611186, -0.003773817792534828, 0.012000791728496552, 0.001778415055014193, -0.0026458725333213806, 0.010658147744834423, -0.002414194168522954, 0.004841291345655918, -0.013249523937702179, -0.008303681388497353, 0.013238455168902874, 0.03480149060487747, 0.012318390421569347, -0.011846481822431087, 0.014126245863735676, 0.011690440587699413, 4.50374573119916e-05, -0.02312808856368065, -0.024273216724395752, -0.004693388473242521, 0.020451268181204796, -0.000876949867233634, -0.0009952360996976495, 0.013227101415395737, 0.002521305810660124, -0.01517010759562254, 0.000922623963560909, -0.025200972333550453, -0.006906329654157162, 0.013218535110354424, 0.001150778611190617, -0.009159361943602562, -0.005628529004752636, 0.020558426156640053, -0.0038386178202927113, 0.009206430055201054, 0.003735768375918269, -0.004227862693369389, -0.011065957136452198, 0.0004926256369799376, 0.019547322764992714, 0.0058600883930921555, 0.0025853377301245928, 0.015567858703434467, -0.002185686258599162, 0.0032852545846253633, -0.010258665308356285, -0.00566262099891901, -0.019557537510991096, -0.0034918412566184998, -0.03087444417178631, -0.005703075788915157, -0.0006844362360425293, -0.003170166164636612, 0.011801478452980518, 0.012979469262063503, 0.01512680109590292, -0.0017383433878421783], [0.09731286019086838, 0.08125032484531403, -0.04814645275473595, -0.02548392489552498, -0.01428334228694439, 0.07959877699613571, 0.005576657131314278, -0.03864824026823044, 0.05727546662092209, 0.0014023343101143837, -0.006590195931494236, -0.0069300769828259945, -0.008116137236356735, -0.038324691355228424, 0.0026653341483324766, 0.054581400007009506, -0.008885636925697327, 0.025721890851855278, -0.03638690337538719, 0.008985117077827454, 0.08667485415935516, -0.04996204748749733, -0.06060531735420227, -0.05780009180307388, 0.020693371072411537, 0.01697617396712303, -0.02019858919084072, 0.06597984582185745, -0.07918794453144073, -0.02925187721848488, -0.01595708355307579, -0.07401479780673981, 0.027455518022179604, -0.05408347398042679, 0.018574852496385574, 0.037539564073085785, 0.07853143662214279, -0.04330315440893173, -0.05593439191579819, 0.05665271356701851, 0.0183804240077734, -0.06292357295751572, 0.05875050649046898, 0.05438569560647011, -0.01807270757853985, -0.02009766362607479, 0.038334280252456665, 0.03576148301362991, -0.03146770969033241, 0.05579342320561409, 0.010785860940814018, -0.08098408579826355, 0.07277636229991913, 0.009992362931370735, -0.02715316414833069, 0.03438037633895874, -0.08165203034877777, 0.030606437474489212, 0.032357603311538696, -0.029865063726902008, -0.004125317558646202, 0.05531581863760948, -0.028885165229439735, 0.06440991908311844, -0.10140663385391235, -0.0552133247256279, -0.09383887052536011, 0.10171207785606384, 0.01966213248670101, -0.08718352019786835, -0.020154418423771858, 0.06090056896209717, -0.06492974609136581, -0.023755108937621117, -0.04039832949638367, -0.0776354968547821, 0.04619067162275314, 0.03573715314269066, -0.042810503393411636, -0.049526479095220566, -0.031962405890226364, -0.013234113343060017, 0.04212990775704384, 0.004076826386153698, 0.009154155850410461, -0.06690490245819092, -0.06580167263746262, -0.06174290180206299, -0.10904458910226822, -0.030024563893675804, -0.0018324179109185934, -0.037099551409482956, -0.0676460713148117, -0.01685081608593464, 0.06842398643493652, -0.03216423839330673], [-0.07265285402536392, 0.017330871894955635, -0.021531442180275917, 0.02945018745958805, -0.11263244599103928, 0.06685023009777069, -0.04343122988939285, -0.03979824110865593, -0.03588847443461418, 0.029985830187797546, -0.045279163867235184, -0.03520500659942627, 0.04636171832680702, -0.04264712333679199, 0.022081291303038597, 0.02059009112417698, 0.06232590600848198, 0.0715223029255867, -0.05855629965662956, 0.040536750108003616, 0.0763278603553772, -0.023583775386214256, 0.001511617680080235, 0.00024446009774692357, -0.0319523960351944, -0.0391596183180809, -0.030888212844729424, -0.03408931568264961, -0.06722404807806015, -0.07583902031183243, 0.0195255596190691, -0.02749052457511425, 0.012304932810366154, 0.02493816241621971, -0.00046390300849452615, -0.057295843958854675, 0.06884795427322388, -0.06627090275287628, -0.017809268087148666, 0.056917376816272736, 0.08248709887266159, -0.014888504520058632, -0.12465003877878189, -0.023446323350071907, 0.044309716671705246, -0.07791607826948166, -0.05095641314983368, 0.018020616844296455, -0.017525749281048775, -0.02439836971461773, -0.03009919822216034, 0.04922923818230629, 0.0021763755939900875, -0.08987098932266235, -0.07465066015720367, -0.08310968428850174, -0.051241643726825714, 0.011600334197282791, 0.004440752789378166, -0.0921756699681282, -0.05177877098321915, -0.04049311950802803, -0.001836689654737711, -0.01898615062236786, -0.029464056715369225, 0.028517259284853935, -0.004668896552175283, -0.06386867165565491, 0.05582202225923538, 0.06361151486635208, 0.08011723309755325, -0.05044751986861229, -0.08177594840526581, 0.06267685443162918, -0.04362982138991356, -0.05165679007768631, -0.01633540727198124, -0.08491640537977219, 0.0037679625675082207, -0.02946641482412815, 0.04464951902627945, -0.013936666771769524, -0.0846448540687561, 0.04274577274918556, -0.023130595684051514, -0.007918897084891796, -0.10184752941131592, 0.009589004330337048, 0.08539997786283493, -0.07286687195301056, -0.05356289818882942, -0.024341141805052757, -0.10663311928510666, -0.01808016188442707, -0.08045599609613419, -0.006811134517192841], [-0.043837908655405045, -0.1426132470369339, -0.04633639380335808, -0.03085922636091709, -0.037613771855831146, -0.07030048221349716, 0.0020307113882154226, -0.004305451177060604, -0.018836362287402153, -0.09134313464164734, -0.02429196424782276, 0.006464493926614523, 0.02143741585314274, -0.07711756229400635, 0.026398006826639175, -0.03623679652810097, -0.05684225633740425, 0.0954432487487793, -0.04308072105050087, 0.012566453777253628, 0.013597636483609676, -0.004510181490331888, -0.06050755828619003, 0.005364555865526199, 0.009888784028589725, -0.10886882990598679, 0.02663383260369301, 0.01469564251601696, -0.05396130681037903, -0.13455097377300262, -0.06260082125663757, 0.06384927034378052, 0.026949476450681686, 0.018190156668424606, 0.056169543415308, -0.04751915484666824, -0.02542119473218918, -0.046962909400463104, -0.07880168408155441, -0.06795378029346466, -0.01694846712052822, -0.032824017107486725, -0.10860645025968552, -0.013018042780458927, -0.023014146834611893, -0.0129788126796484, 0.03311595693230629, -0.028097927570343018, 0.004925010725855827, -0.06411867588758469, -0.04121188074350357, -0.04561002179980278, 0.09887664020061493, -0.04366966709494591, 0.029141077771782875, -0.04226658120751381, -0.05183107778429985, -0.04016874358057976, -0.07998812943696976, 0.03375769779086113, -0.08529000729322433, -0.008632166311144829, -0.0037718366365879774, -0.029134593904018402, -0.04480360075831413, -0.010487064719200134, 0.00026899436488747597, -0.04044220224022865, 0.05557269603013992, -0.058615054935216904, -0.0162898488342762, -0.018496954813599586, -0.06511574238538742, 0.006892710458487272, -0.00019572601013351232, -0.06585320830345154, -0.005946920718997717, -0.07091926038265228, -0.04705772176384926, 0.0031689510215073824, -0.04884292185306549, -0.016176676377654076, -0.060469161719083786, -0.05462781712412834, -0.013978295028209686, -0.04103167727589607, -0.035327281802892685, -0.01598934642970562, -0.08685208857059479, 0.023020189255475998, -0.01750965043902397, -0.03635942563414574, -0.041945844888687134, -0.0260724276304245, 0.00613700645044446, -0.03239750117063522], [0.12586131691932678, -0.12471843510866165, -0.04132961854338646, 0.01903396099805832, -0.04514569789171219, 0.04563048481941223, -0.07421955466270447, -0.017762526869773865, -0.027554070577025414, -0.10326622426509857, 0.07115281373262405, -0.03301539644598961, 0.0327826663851738, 0.0850004106760025, 0.06886431574821472, -0.042530301958322525, 0.05934102088212967, -0.0699181854724884, -0.027141286060214043, -0.029837964102625847, -0.0023329562973231077, -0.045485980808734894, 0.00899132713675499, 0.031105631962418556, 0.06113205850124359, 0.03933623433113098, 0.0018340334063395858, -0.016118787229061127, -0.06696683168411255, -0.0762956291437149, -0.011303738690912724, -0.03220096230506897, 0.07126595079898834, -0.01747889444231987, -0.04278264939785004, 0.02041698433458805, -0.04462531581521034, -0.04563693702220917, 0.06719192862510681, 0.09205072373151779, 0.08682677894830704, -0.035727597773075104, -0.02608625404536724, -0.013715523295104504, -0.01145257893949747, 0.04596267268061638, -0.007429501507431269, -0.04983242601156235, -0.11827166378498077, -0.05997868627309799, 0.003964101430028677, -0.07115879654884338, -0.08540406078100204, -0.0458173006772995, -0.05095575004816055, -0.00037224721745587885, -0.06784523278474808, 0.10165061056613922, 0.07603224366903305, -0.026510195806622505, 0.1603935807943344, -0.12472894787788391, 0.0337446853518486, 0.023129833862185478, -0.027813421562314034, 0.003244793973863125, 0.09130389243364334, -0.10556983947753906, -0.04389536753296852, 0.09574759751558304, -0.023675469681620598, -0.05122610926628113, 0.12612593173980713, 0.01881321892142296, 0.05656898021697998, 0.029260452836751938, -0.08966749161481857, 0.08077255636453629, 0.015362313017249107, 0.01609690673649311, -0.035973526537418365, -0.0831863284111023, -0.06594505161046982, -0.013593518175184727, 0.024391429498791695, 0.002544892020523548, -0.06991986930370331, 0.02039777673780918, -0.041444215923547745, -0.021092982962727547, -0.08064159005880356, -0.06393503397703171, 0.040413543581962585, -0.04509994015097618, 0.041386302560567856, 0.03679847717285156], [0.0782579630613327, 0.010693775489926338, 0.026692993938922882, 0.07188953459262848, -0.0864718109369278, 0.027453588321805, 0.03981953114271164, 0.0392158143222332, 0.06572733074426651, -0.05026455223560333, 0.05916490778326988, -0.0034961295314133167, -0.01185114961117506, 0.07106856256723404, 0.02660433202981949, 0.008737222291529179, 0.13672234117984772, -0.041166629642248154, -0.024880902841687202, -0.05045070871710777, 0.02304254285991192, -0.03290441259741783, 0.02931000292301178, -0.03016633912920952, -0.06641145795583725, 0.13063643872737885, -0.041242629289627075, 0.011147532612085342, -0.035263463854789734, 0.054981354624032974, 0.02482161484658718, -0.05130956321954727, 0.0195277351886034, -0.07868511229753494, -0.018554365262389183, -0.011068490333855152, -0.09682191908359528, -0.007505415007472038, 0.07000461220741272, 0.007684327196329832, -0.01068537775427103, -0.051539335399866104, 0.02388778328895569, -0.04890817031264305, 0.022638317197561264, 0.0325324647128582, -0.048704832792282104, 0.01677272841334343, -0.03938247635960579, -0.09982774406671524, -0.0038894491735845804, 0.003649994730949402, -0.0727144256234169, -0.06259538978338242, -0.056321579962968826, -0.002921139355748892, -0.11071460694074631, 0.07348547130823135, 0.008799029514193535, -0.01333802379667759, 0.1272069364786148, -0.0865146741271019, 0.012461202219128609, -0.000668574939481914, 0.019095439463853836, 0.015529566444456577, 0.0034321548882871866, -0.02405109815299511, 0.010679412633180618, 0.0010022723581641912, -0.020804980769753456, -0.06080884486436844, 0.046757981181144714, 0.044247228652238846, -0.02366916462779045, -0.03698471933603287, -0.0702100619673729, 0.03917456790804863, -0.04785392805933952, -0.04559829458594322, -0.022135358303785324, -0.0006787116290070117, 0.008906089700758457, -0.005073703825473785, 0.05556285008788109, -0.006859529297798872, 0.013767939060926437, 0.05791904777288437, -0.04306821897625923, -0.08926999568939209, -0.04789680987596512, -0.019563153386116028, -0.08018601685762405, -0.006013384088873863, -0.0024801234249025583, -0.03916717320680618], [-0.021759992465376854, 0.017348099499940872, -0.03500555828213692, -0.03785421699285507, 0.04825432971119881, 0.003983174450695515, -0.051908690482378006, -0.016841182485222816, -0.038419075310230255, -0.013892862014472485, -0.029990864917635918, -0.022041521966457367, -0.011824917048215866, -0.020848723128437996, -0.0026625741738826036, -0.011279696598649025, -0.0027296922635287046, 0.015569081529974937, 0.03105253167450428, 0.04420368745923042, -0.02066020667552948, 0.00980538222938776, -0.025202831253409386, 0.004756709095090628, 0.01972814090549946, -0.005550905596464872, -0.012982894666492939, 0.0019992615561932325, -0.05269487574696541, -0.014701967127621174, -0.014761457219719887, -0.040080104023218155, -0.023527145385742188, -0.0009523715707473457, 0.02337339147925377, 0.07173001021146774, -0.010881052352488041, -0.0006633538869209588, 0.001038057147525251, 0.02748485654592514, -0.02072390541434288, -0.034350182861089706, -0.00868692435324192, -0.0026116189546883106, -0.03798918053507805, -0.009484476409852505, 0.008084352128207684, -0.012926907278597355, -0.008162783458828926, 0.0028234636411070824, 0.019411752000451088, -0.008753019385039806, -0.007629896514117718, 0.011963976547122002, -0.006385334767401218, -0.0029463511891663074, 0.004394941031932831, 0.018876155838370323, 0.010160881094634533, -0.019067855551838875, -0.012114867568016052, -0.013237277045845985, 0.00801120325922966, -0.018514273688197136, -0.059325698763132095, 0.0037566388491541147, -0.030291108414530754, 0.049007922410964966, -0.020541435107588768, 0.014348442666232586, -0.043702516704797745, -0.005219371523708105, -0.0035962825641036034, -0.0392860472202301, -0.025375721976161003, -0.0036149516236037016, -0.035573508590459824, 0.0011908634332939982, -0.035593096166849136, 0.009616291150450706, -0.0009521975298412144, 0.03075293079018593, 0.04252390190958977, -0.0037287927698343992, 0.02056967094540596, -0.017778808251023293, 0.04691411927342415, -0.015297405421733856, -0.029355937615036964, -0.03170757740736008, -0.039788052439689636, 0.0467030294239521, -0.025131575763225555, -0.027386657893657684, -0.010157056152820587, 0.031843751668930054], [-0.09491755068302155, 0.05709338188171387, 0.04678760841488838, 0.049387093633413315, 0.03845136612653732, -0.040951747447252274, 0.07442495971918106, 0.07359598577022552, 0.017223874107003212, 0.06654655933380127, -0.027229592204093933, 0.05469794571399689, 0.05041131004691124, 0.059190113097429276, 0.03374207764863968, 0.07644569128751755, -0.10327765345573425, 0.05497763305902481, -0.026537131518125534, -0.08925340324640274, 0.07748590409755707, -0.015299618244171143, -0.04555191472172737, 0.020786982029676437, -0.04047999531030655, -0.05679306387901306, -0.043091535568237305, -0.013834932819008827, -0.013411314226686954, -0.08661355823278427, -0.05961599573493004, 0.06151178479194641, 0.003197522135451436, -0.07680612057447433, 0.05501868575811386, -0.008283548057079315, -0.08881380409002304, -0.0043525733053684235, 0.001883895369246602, -0.07725752890110016, 0.004074787721037865, 0.0517108291387558, 0.07405932992696762, 0.012927399948239326, 0.007652770262211561, 0.08731600642204285, 0.021041056141257286, 0.05745166540145874, -0.014888170175254345, -0.06865426898002625, 0.07658044248819351, 0.008365637622773647, 0.07101504504680634, 0.028210995718836784, 0.01654898375272751, -0.08198187500238419, 0.03582523763179779, 0.02815573289990425, -0.029181499034166336, 0.08336416631937027, -0.10467619448900223, -0.04310309886932373, -0.02006632089614868, 0.04502806439995766, -0.002298644743859768, -0.022298824042081833, 0.0663677528500557, -0.049581363797187805, 0.08432423323392868, 0.030980724841356277, 0.023351063951849937, 0.05192921683192253, 0.00490583386272192, -0.02175159752368927, 0.030470987781882286, -0.0690590962767601, -0.027738066390156746, -0.035169824957847595, -0.047715552151203156, -0.03408614918589592, 0.026601634919643402, -0.09354572743177414, -0.06559815257787704, -0.051404982805252075, -0.0435323566198349, 0.04167487099766731, 0.013421246781945229, 0.03172209486365318, 0.007758046500384808, 0.07113080471754074, 0.038155991584062576, 0.022078972309827805, -0.021104799583554268, -0.007135441992431879, 0.06554118543863297, -0.012147939763963223], [0.04774875566363335, 0.06762544065713882, -0.0778336375951767, -0.08848850429058075, -0.012159986421465874, -0.003521198872476816, -0.03207985311746597, 0.06713636219501495, 0.05997154861688614, 0.02527237869799137, -0.021197712048888206, -0.04459028318524361, 0.08637479692697525, -0.052360936999320984, -0.02317124977707863, 0.07396751642227173, -0.007052725646644831, -0.09816819429397583, 0.08713816106319427, 0.05605461448431015, -0.012433193624019623, 0.013174625113606453, -0.0006328478921204805, -0.012503438629209995, 0.008478555828332901, -0.08373698592185974, -0.0037777291145175695, 0.08468052744865417, -0.07318662852048874, 0.09429628401994705, -0.10235662758350372, 0.020375818014144897, -0.08131910860538483, 0.018403563648462296, -0.0185854472219944, 0.0001496937475167215, 0.07877644896507263, 0.0775320902466774, -0.03427792713046074, 0.08531410992145538, 0.07181453704833984, 0.06562024354934692, 0.004400615114718676, -0.03455480560660362, -0.013995984569191933, -0.04353627189993858, -0.025906160473823547, 0.03128194808959961, -0.06963185966014862, -0.007467770017683506, -0.015411663800477982, -0.054712358862161636, 0.05657316744327545, -0.011532559059560299, 0.09747529029846191, -0.0035102006513625383, 0.035072874277830124, -0.040860615670681, 0.018019119277596474, 0.07997994124889374, -0.059078000485897064, -0.012644185684621334, 0.05736643821001053, 0.09781178832054138, -0.0333254300057888, 0.03693564608693123, -0.019879592582583427, -0.01101880706846714, -0.08499167859554291, 0.034441255033016205, 0.04711095616221428, 0.0538940504193306, 0.01019748579710722, -0.04189816862344742, -0.002043036976829171, -0.0793004184961319, -0.008889120072126389, 0.040047675371170044, 0.09443338960409164, -0.006095814052969217, -0.0012520961463451385, -0.009269632399082184, -0.057100508362054825, 0.02529495768249035, 0.06240297853946686, -0.07153932005167007, -0.04362126439809799, 0.014612719416618347, -0.019047170877456665, -0.014140119776129723, -0.05010693892836571, -0.026416491717100143, 0.06313959509134293, -0.08784928917884827, -0.05085773393511772, 0.0916450023651123], [-0.02431677095592022, -0.04643460735678673, -0.06285909563302994, 0.0037836816627532244, -0.020639706403017044, -0.024050502106547356, -0.025752566754817963, 0.05817847326397896, 0.08939849585294724, 0.038333114236593246, -0.007645495235919952, -0.01899874582886696, -0.040845535695552826, 0.025490859523415565, -0.053743425756692886, -0.03070896677672863, -0.02394372597336769, -0.025768272578716278, -0.030239082872867584, -0.04698996990919113, 0.0324246920645237, -0.06124217063188553, 0.00018609259859658778, -0.013128112070262432, -0.040212828665971756, 0.006158721167594194, 0.09982770681381226, 0.06955448538064957, 0.050924669951200485, -0.04042883589863777, -0.058827854692935944, 0.02511870674788952, -0.034088876098394394, 0.08883824944496155, 0.034762267023324966, 0.004264429211616516, 0.09321313351392746, 0.014262627810239792, -0.022377189248800278, 0.02095659077167511, 0.021708671003580093, 0.025039587169885635, 0.0323348231613636, -0.008586824871599674, -0.10557806491851807, 0.0511513389647007, 0.04283013194799423, -0.007409644313156605, -0.06010996550321579, -0.018167510628700256, 0.020029183477163315, -0.08006679266691208, 0.07083310186862946, -0.024515246972441673, -0.033133577555418015, 0.028969842940568924, 0.04283905774354935, -0.0332614965736866, 0.03365710750222206, 0.02782939001917839, -0.04574461653828621, 0.03258730098605156, -0.05655479431152344, 0.07333789765834808, -0.09942615777254105, -0.08252882957458496, -0.10505909472703934, 0.004158377181738615, -0.06275390088558197, -0.014118542894721031, -0.04084530845284462, 0.06565555185079575, 0.03372108191251755, 0.028955800458788872, 0.06755764782428741, -0.057918351143598557, -0.018120819702744484, -0.04750309884548187, 0.029286928474903107, 0.059759803116321564, 0.00021674636809621006, 0.04294036328792572, 0.1039503887295723, 0.0492849238216877, -0.011688120663166046, -0.07578820735216141, 0.053560011088848114, 0.04261217266321182, -0.018626336008310318, 0.013592958450317383, 0.0001342222822131589, 0.08588921278715134, 0.01217334158718586, -0.04864068329334259, 0.016163811087608337, 0.061921145766973495], [0.06939774751663208, -0.1192915290594101, 0.027832550927996635, 0.027688680216670036, -0.03839956969022751, 0.02851269021630287, 0.017211424186825752, -0.018599696457386017, 0.02341199479997158, -0.07182189077138901, 0.05914405360817909, -0.050077978521585464, -0.03345724195241928, 0.039138276129961014, 0.015574993565678596, -0.02619408816099167, 0.11889424920082092, -0.08051417768001556, -0.026356486603617668, -0.023583341389894485, 0.05021120235323906, -0.01638195477426052, 0.0018890023929998279, -0.05259884148836136, -0.039459072053432465, 0.021860143169760704, -0.024566954001784325, 0.02052485942840576, -0.032702889293432236, -0.00688110152259469, 0.06396226584911346, -0.030059320852160454, -0.0762862041592598, -0.06034340336918831, 0.013406259939074516, 0.010996504686772823, -0.04241609573364258, -0.014936152845621109, 0.04695133864879608, -0.04683806374669075, 0.025344636291265488, 0.010437996126711369, -0.07692499458789825, 0.02465738356113434, -0.020364683121442795, 0.004618862643837929, -0.014339941553771496, -0.02469608001410961, -0.05628957226872444, -0.06829753518104553, -0.08150003105401993, -0.033032212406396866, -0.09503494203090668, -0.04584678262472153, -0.04405536130070686, -0.018544793128967285, -0.0005129303317517042, 0.11505340039730072, 0.06761738657951355, -0.02888634242117405, 0.12359503656625748, -0.05750131234526634, 0.015708070248365402, -0.04979514330625534, -0.006771545857191086, -0.01634371094405651, 0.01708519086241722, 0.030162043869495392, -0.009966542012989521, 0.046686410903930664, -0.027244562283158302, -0.016536304727196693, 0.019418248906731606, 0.05970752611756325, -0.0832064300775528, 0.03101818449795246, -0.03131880238652229, -0.00116033386439085, -0.086471788585186, -0.005715259816497564, -0.055700771510601044, -0.04509603604674339, -0.03282395005226135, 0.036401890218257904, -0.07566247880458832, 0.05744652450084686, -0.012384786270558834, 0.057038139551877975, 0.01820947974920273, 0.043474260717630386, -0.10240603983402252, -0.039443932473659515, -0.052256423979997635, -0.007848606444895267, -0.06975279003381729, -0.00771449925377965], [0.08146311342716217, -0.0021565898787230253, -0.0510718934237957, 0.058779239654541016, 0.0027596293948590755, -0.07145878672599792, 0.022891849279403687, 0.036906056106090546, -0.026151273399591446, -0.07276714593172073, 0.04203160107135773, 0.02647394873201847, -0.014292710460722446, 0.03375345095992088, -0.008803697302937508, 0.019345605745911598, 0.0557275153696537, 0.02440851740539074, -0.018290072679519653, -0.0015499471919611096, -0.024316370487213135, 0.04596927762031555, -0.06497617065906525, -0.052605196833610535, -2.119925056831562e-06, 0.11466459184885025, 0.013735290616750717, -0.049896713346242905, 0.018535053357481956, -0.052534569054841995, 0.021204812452197075, -0.0068528857082128525, -0.05960739776492119, -0.06085629016160965, -0.03213499113917351, -0.06079327315092087, 0.01929459720849991, 0.010612398386001587, 0.023942558094859123, 0.0024560652673244476, 0.022968687117099762, 0.005798936355859041, 0.018553106114268303, -0.04570790380239487, 0.024090170860290527, -0.048479657620191574, -0.023264601826667786, -0.014555207453668118, -0.05187651142477989, 0.023634478449821472, 0.04080374166369438, 0.030527763068675995, -0.07939541339874268, -0.05713753029704094, -0.028630148619413376, 0.006729340646415949, -0.04908173531293869, 0.059776537120342255, 0.05072920024394989, -0.04911596700549126, 0.07920058816671371, -0.047124795615673065, -0.011187483556568623, -0.03076021373271942, -0.03276941180229187, 0.003286920487880707, 0.0180329792201519, 0.04478965699672699, -0.031141864135861397, 0.00707591837272048, -0.0069764782674610615, -0.04157247766852379, 0.06933067739009857, -0.001180302700959146, 0.03331519663333893, 0.04694457724690437, 0.026943892240524292, 0.023829501122236252, -0.019614743068814278, -0.035504892468452454, -0.026593798771500587, -0.022697141394019127, 0.04558839648962021, 0.040362339466810226, 0.049166180193424225, 0.018262621015310287, -0.08551812171936035, 0.008928087539970875, 0.09537890553474426, 0.03147147595882416, 0.0040894076228141785, -0.026873096823692322, 0.009259616024792194, 0.01267270091921091, -0.02124137245118618, -0.04045393317937851], [-0.06556820124387741, -0.07067522406578064, -0.07590422034263611, 0.06202623248100281, -0.0970233827829361, 0.02864764630794525, -0.03991466388106346, 0.0037739924155175686, -0.055074967443943024, -0.04839242249727249, -0.019658813253045082, 0.031216658651828766, 0.018100878223776817, -0.06647288799285889, 0.041953153908252716, -0.011568701826035976, -0.010490219108760357, -0.018417397513985634, 0.0027730504516512156, -0.01249675452709198, 0.04348938539624214, 0.0065897731110453606, 0.05855245888233185, 0.05288676545023918, -0.03635893017053604, -0.09259457886219025, 0.01166000496596098, -0.0734991505742073, -0.05753860995173454, -0.07691190391778946, -0.006780383177101612, -0.026918256655335426, -0.0465664267539978, 0.013412849977612495, -0.007393415551632643, 0.019944021478295326, -0.0346805676817894, -0.011624298058450222, -0.07677896320819855, -0.022369755432009697, -0.05612703412771225, -0.0008467782172374427, -0.0825057178735733, 0.012594813480973244, 0.0016899191541597247, 0.01881617121398449, 3.8654558011330664e-05, 0.01395748183131218, -0.018167896196246147, 0.02828701213002205, -0.0036083466839045286, -0.00243942067027092, 0.015111844055354595, -0.006611091550439596, -0.053399015218019485, -0.01451314426958561, -0.0958188995718956, 0.031392503529787064, -0.06084071844816208, -0.0236184224486351, -0.04896887391805649, 0.0460379421710968, 0.003789410227909684, -0.023593800142407417, -0.020051496103405952, -0.027302805334329605, -0.029546668753027916, -0.07112690806388855, 0.075471892952919, -0.031218532472848892, -0.026060158386826515, 0.00027298115310259163, -0.04420449584722519, -0.029584497213363647, 0.042258720844984055, 0.04468720778822899, -0.004779168404638767, -0.03868843615055084, 0.04424552619457245, -0.016149096190929413, -0.018823567777872086, -0.04714302718639374, -0.01977381855249405, 0.010155498050153255, -0.04422401264309883, 0.010001945309340954, 0.01845991425216198, 0.01751410774886608, 0.08750535547733307, -0.07079164683818817, 0.03657848760485649, 0.0024675221648067236, 0.007577515207231045, 0.023889629170298576, -0.001001889817416668, 0.005629215855151415], [-0.10061673074960709, 0.012678110972046852, 0.01559400837868452, -0.030419932678341866, -0.09752567112445831, 0.03225712105631828, -0.015506687574088573, -0.00852139201015234, -0.04828566312789917, 0.02794317901134491, -0.035550039261579514, -0.008502205833792686, 0.02145903743803501, -0.035809267312288284, 0.018452128395438194, 0.024190684780478477, -0.04977146536111832, -0.06026160344481468, -0.02142193354666233, -0.0058204480446875095, 0.00024496045080013573, 0.07995261251926422, 0.022579165175557137, -0.04607522115111351, -0.0550432987511158, -0.06915474683046341, 0.06677376478910446, 0.054107386618852615, -0.005820111371576786, 0.06630884110927582, 0.009641400538384914, -0.054042525589466095, -0.03701396659016609, 0.025182269513607025, 0.03433958441019058, 0.0368424691259861, -0.02451442740857601, 0.012650573626160622, 0.007615338545292616, -0.03872227668762207, -0.016669979318976402, -0.006847390905022621, -0.11968790739774704, -0.050937000662088394, 0.0741773247718811, 0.04910261556506157, -0.05582347884774208, -0.042499057948589325, 0.08241815865039825, -0.023630987852811813, -0.014372275210916996, -0.03692184388637543, 0.030564192682504654, -0.00179070676676929, -0.0835416242480278, 0.029226379469037056, -0.09301112592220306, -0.0901348888874054, -0.04168148338794708, 0.010437236167490482, -0.08531937003135681, -0.029047708958387375, -0.033848196268081665, 0.0725063756108284, -0.05523047596216202, -0.031321845948696136, -0.031003344804048538, -0.03710072860121727, 0.04208173230290413, 0.0019044072832912207, -0.059628281742334366, -0.026743274182081223, -0.04885031282901764, 0.06868444383144379, -0.037835001945495605, -0.08684363961219788, 0.06058530509471893, -0.05801495164632797, 0.0316542349755764, -0.06967764347791672, -0.04958179220557213, -0.0046294499188661575, 0.043753013014793396, -0.006189832929521799, 0.07167438417673111, 0.09316762536764145, 0.0036574648693203926, -0.011911863461136818, 0.09514632821083069, -0.048794329166412354, -0.06211435794830322, -0.02081671729683876, -0.10792775452136993, 0.040890466421842575, 0.02238399349153042, -0.05754171684384346]], "b2": [0.021727826446294785, -0.06581674516201019, 0.02798455022275448, 0.05589974671602249, 0.046010572463274, -0.1347099095582962, -0.05965246260166168, -0.019385822117328644, 0.03673212602734566, 0.03201679885387421, 0.012005343101918697, -0.04585399478673935, -0.0608188770711422, -0.020334439352154732, 0.1112699955701828, -0.0974912941455841, 0.037322573363780975, 0.09427617490291595, -0.01965491846203804, 0.10678041726350784, 0.06813012063503265, 0.028957989066839218, -0.021355552598834038, -0.04115462675690651, 0.06917672604322433, -0.011998394504189491, -0.003584829857572913, 0.12638460099697113, -0.029435113072395325, -0.0952397957444191, -0.06553181260824203, -0.04152020439505577], "W3": [[-0.14812301099300385, 0.06615634262561798, -0.1910509467124939, -0.04653064161539078, -0.23700417578220367, 0.10627096891403198, 0.16044911742210388, 0.008259503170847893, -0.177328959107399, -0.1518845111131668, -0.05362774431705475, 0.17554809153079987, 0.052064597606658936, -0.17073728144168854, -0.06804455816745758, 0.06834742426872253, 0.03313416242599487, -0.04900691658258438, -0.015645738691091537, -0.0917234793305397, 0.1677989363670349, 0.13899682462215424, 0.2067747414112091, 0.12503787875175476, -0.06590916216373444, 0.18901987373828888, -0.15677890181541443, -0.11285357922315598, 0.14757226407527924, 0.1025172770023346, 0.14264002442359924, 0.12127445638179779]], "b3": [0.14379507303237915]}, "2023": {"W1": [[-0.18103112280368805, -0.07128512859344482, 0.029086457565426826, -0.27776652574539185, 0.13259756565093994, -0.08326708525419235, 0.03549261391162872, -0.010900242254137993, -0.019164366647601128, 0.28180068731307983, 0.006574819330126047, 0.06761014461517334, -0.03197126463055611, 0.05429457500576973, 0.017380455508828163, -0.04585738480091095, -0.18726730346679688, -0.0430908203125, -0.12786325812339783, -0.017423242330551147, -0.051348064094781876], [0.09800666570663452, -0.06175849959254265, 0.06011296808719635, 0.09531714767217636, -0.01845376193523407, -0.09880156815052032, 0.12026173621416092, -0.14703986048698425, 0.06589680165052414, 0.10287468135356903, -0.039241958409547806, -0.05395769327878952, -0.22916556894779205, 0.0030394718050956726, -0.1292009800672531, -0.07127411663532257, -0.24695450067520142, -0.04624359309673309, 0.06933046877384186, -0.021278444677591324, -0.05244116857647896], [0.08088353276252747, -0.01748196966946125, 0.033687982708215714, 0.06874535977840424, 0.06983108073472977, -0.03945719823241234, -0.005028975661844015, -0.02772265113890171, 0.02744671143591404, 0.037809304893016815, 0.011008859612047672, 0.012032740749418736, -0.044691141694784164, 0.024210400879383087, 0.0035985405556857586, -0.03625988960266113, -0.03671227768063545, -0.007822273299098015, 0.016482651233673096, -0.0011167449411004782, 0.011263119988143444], [-0.08725541830062866, -0.027006845921278, -0.17765803635120392, 0.06411880254745483, 0.10538281500339508, -0.02813321352005005, -0.024732429534196854, -0.07399744540452957, 0.0037924866192042828, 0.16289056837558746, -0.1250421106815338, -0.028290877118706703, -0.13772529363632202, -0.059443555772304535, 0.05222895368933678, 0.12854310870170593, -0.08675151318311691, 0.07843397557735443, 0.0294803474098444, 0.0250500850379467, -0.08556428551673889], [0.10893028974533081, -0.1333850473165512, 0.04199184849858284, 0.13702192902565002, 0.10121336579322815, 0.04134510084986687, -0.06551174074411392, 0.0475543774664402, -0.0799952894449234, -0.042553555220365524, -0.024667493999004364, 0.18418994545936584, -0.016063014045357704, 0.2045622169971466, -0.1270269751548767, 0.004893573932349682, 0.0018545256461948156, 0.06511382758617401, 0.019668933004140854, 0.01717313379049301, 0.04302896931767464], [0.08208314329385757, 0.0924406424164772, -0.09708360582590103, -0.06403703987598419, 0.09000450372695923, 0.12024964392185211, 0.12208756059408188, 0.08050575852394104, -0.06619840860366821, 0.0407593809068203, 0.10378447920084, 0.09163515269756317, 0.009484443813562393, 0.06587772816419601, -0.014575181528925896, -0.03361818194389343, -0.048233263194561005, 0.022774145007133484, -0.04272003844380379, -0.04489782452583313, -0.030951429158449173], [-0.08123544603586197, 0.1448451280593872, -0.1293415129184723, 0.05985108017921448, -0.12765616178512573, -0.09753728657960892, 0.09327273815870285, -0.13228215277194977, -0.05944076180458069, 0.00969178881496191, -0.09573692828416824, -0.11302955448627472, 0.0632300078868866, 0.07003435492515564, -0.0020770819392055273, -0.0382075160741806, 0.018549121916294098, 0.00047666626051068306, 0.1719111055135727, -0.0949731171131134, 0.09332817792892456], [0.03571444749832153, -0.0052889673970639706, 0.0952397808432579, 0.00986957736313343, -0.00836948398500681, -0.04285028204321861, -0.04653026908636093, 0.022778019309043884, -0.07270171493291855, 0.12069032341241837, -0.03984343260526657, -0.024955440312623978, -0.00018099785665981472, -0.03972052410244942, 0.052793797105550766, -0.012830980122089386, -0.17201732099056244, 0.06503145396709442, 0.06627313792705536, -0.032475702464580536, 0.04410724714398384], [0.02156042493879795, 0.031750597059726715, -0.10113310813903809, -0.15404832363128662, -0.13174454867839813, -0.11425404250621796, 0.009873087517917156, 0.011470706202089787, 0.09073232114315033, -0.03532182425260544, -0.0998181626200676, 0.05759621039032936, 0.01618894189596176, -0.05024036020040512, -0.0008915463113225996, 0.026349522173404694, -0.09076523780822754, -0.10666609555482864, -0.08102841675281525, -0.09265219420194626, -0.11770586669445038], [0.07973568886518478, 0.04248763248324394, -0.004880380816757679, 0.010376980528235435, 0.08647936582565308, -0.06519956886768341, -0.02403276041150093, -0.020927567034959793, 0.024215664714574814, -0.0008654353441670537, -0.015214856714010239, 0.08402976393699646, -0.10889506340026855, 0.06036292761564255, -0.0766855999827385, 0.021199679002165794, -0.10941444337368011, 0.09754316508769989, 0.08159514516592026, -0.0324060320854187, 0.07183674722909927], [-0.15792357921600342, 0.10124185681343079, 0.013527455739676952, -0.14191871881484985, -0.09016304463148117, -0.08905255049467087, -0.03118090145289898, 0.06547423452138901, 0.009820232167840004, -0.15766705572605133, -0.002909756964072585, 0.15952830016613007, -0.09623681753873825, 0.056395236402750015, 0.05996118485927582, 0.004771443549543619, 0.003476909128949046, -0.04445637762546539, -0.0888134315609932, -0.07690207660198212, 0.09577082842588425], [-0.009280262514948845, 0.0806727483868599, 0.015053416602313519, -0.0890982449054718, 0.04306318610906601, -0.04844377562403679, -0.07157914340496063, -0.05339638143777847, 0.056645795702934265, 0.07948067784309387, -0.08689520508050919, -0.003008698346093297, 0.06887781620025635, 0.05104099214076996, 0.10999976098537445, -0.023163601756095886, -0.16949373483657837, -0.06011468544602394, -0.02470717951655388, 0.08249696344137192, 0.09971732646226883], [-0.009000611491501331, 0.04107053950428963, 0.019723178818821907, -0.10075599700212479, -0.029221832752227783, -0.032021962106227875, -0.024675551801919937, -0.0075613330118358135, 0.06289840489625931, 0.021006939932703972, -0.031429141759872437, -0.0739082470536232, -0.04775698482990265, -0.009131207130849361, -0.006327187642455101, 0.02569776028394699, -0.1352342814207077, -0.03215927258133888, -0.013285278342664242, 0.04527415707707405, 0.018518269062042236], [-0.07610395550727844, -0.0864836797118187, -0.19630420207977295, -0.14337529242038727, -0.06144154816865921, -0.02039426937699318, -0.10522989928722382, 0.10368821024894714, 0.052020855247974396, 0.16694624722003937, -0.038895439356565475, -0.08336630463600159, -0.15294459462165833, -0.04955374449491501, -0.11481184512376785, -0.10236342996358871, -0.04284529760479927, 0.12362080067396164, 0.059737421572208405, -0.15844054520130157, -0.13609442114830017], [-0.03823896124958992, -0.029277877882122993, 0.096192367374897, 0.006186519283801317, 0.04464416205883026, 0.020094186067581177, -0.061281971633434296, -0.005309470929205418, 0.012766526080667973, 0.031649887561798096, -0.004889253526926041, -0.05665084719657898, 0.056946903467178345, 0.07407514750957489, 0.0074903336353600025, -0.014773541130125523, 0.05232718214392662, -0.051361992955207825, 0.0123573187738657, -0.06754308938980103, -0.066167913377285], [-9.360550757264718e-05, -0.014627345837652683, -0.00793720968067646, 0.06496341526508331, -0.070885568857193, -0.02220943197607994, -0.05682274326682091, 0.008690706454217434, 0.03582129254937172, -0.01865018717944622, -0.006575234234333038, 0.04974763095378876, 0.010647079907357693, -0.017324933782219887, -0.05251345783472061, -0.01818894036114216, -0.10616319626569748, -0.05332405865192413, 0.04070145636796951, -0.031197765842080116, 0.09653422981500626], [-0.09644068032503128, -0.08467529714107513, -0.09565293043851852, -0.22835035622119904, 0.015939828008413315, -0.03213336691260338, -0.006191491149365902, -0.012828510254621506, -0.11820787191390991, 0.10537096112966537, -0.0546155720949173, 0.09160321205854416, -0.09128737449645996, -0.06233993172645569, -0.016620740294456482, 0.001991319702938199, -0.024161322042346, -0.03635789081454277, -0.11347551643848419, 0.001956055872142315, 0.06764139235019684], [-0.023733701556921005, 0.04334065318107605, 0.2395557016134262, -0.20310010015964508, -0.013587580993771553, 0.11903370171785355, -0.1654718816280365, 0.0706578865647316, 0.037954170256853104, -0.09277619421482086, -0.04330822825431824, -0.1565137356519699, 0.02813038043677807, -0.012015048414468765, -0.07578663527965546, -0.010588852688670158, 0.2083325982093811, -0.07406248152256012, 0.008881602436304092, -0.061667200177907944, -0.07902443408966064], [-0.04292437061667442, -0.03568015620112419, 0.017830343917012215, 0.03217098489403725, 0.05599148944020271, -0.09905053675174713, 0.009614895097911358, -0.10431422293186188, 0.046104975044727325, -0.004458996467292309, -0.04101946949958801, -0.012177872471511364, -0.1522907316684723, -0.10048948973417282, 0.02150099165737629, 0.04374320060014725, 0.13149580359458923, 0.10462809354066849, 0.13254833221435547, -0.041191503405570984, -0.06061585992574692], [-0.004601782653480768, -0.035492055118083954, 0.08820659667253494, -0.010130830109119415, -0.07093467563390732, -0.01147131621837616, -0.02944348193705082, 0.049375277012586594, -0.030532611533999443, -0.006466575898230076, 0.016749659553170204, 0.0704687237739563, -0.11533567309379578, 0.018374158069491386, -0.038763344287872314, -0.019128797575831413, -0.18027590215206146, 0.04235752671957016, -0.04226462170481682, -0.046691909432411194, -0.0639771893620491], [-0.059171125292778015, -0.003373968880623579, 0.05989275872707367, -0.14165350794792175, 0.09098442643880844, 0.06281566619873047, 0.031370799988508224, -0.020779667422175407, -0.059855952858924866, 0.002497568493708968, -0.021005181595683098, -0.1322597712278366, -0.05935375392436981, -0.003399524837732315, 0.16906879842281342, 0.02355620078742504, -0.22716248035430908, 0.04646174609661102, -0.062431734055280685, 0.020074227824807167, -0.060946524143218994], [0.019689735025167465, 0.06731916218996048, -0.18001596629619598, 0.010055228136479855, 0.0191582590341568, 0.10264790803194046, -0.04934970661997795, 0.02119487151503563, -0.03479430451989174, -0.10997048020362854, -0.07196219265460968, 0.0005510984919965267, -0.01485391054302454, -0.06819571554660797, 0.13225147128105164, 0.006806678604334593, 0.009370299987494946, 0.03454170748591423, 0.01599200628697872, 0.05267062783241272, -0.08135802298784256], [-0.008187877014279366, -0.05398103594779968, -0.05956282094120979, 0.0362248420715332, 0.08717048168182373, 0.09841280430555344, 0.04530628025531769, -0.05963052064180374, 0.008670818991959095, -0.06143936142325401, 0.12784461677074432, -0.0019714070949703455, 0.1135803833603859, -0.09794378280639648, 0.07346728444099426, -0.073335200548172, -0.0850231871008873, 0.08024817705154419, -0.11521156877279282, -0.08554401993751526, 0.11065345257520676], [0.10709674656391144, 0.0515219047665596, 0.021804489195346832, -0.04732763394713402, 0.05787088721990585, -0.021963799372315407, -0.0713927373290062, -0.14010566473007202, 0.08364564180374146, -0.026134798303246498, 0.06302297860383987, -0.07777809351682663, -0.1490524560213089, -0.04295621067285538, 0.09882519394159317, 0.02168184332549572, 0.10187681019306183, -0.0635993480682373, -0.023758385330438614, -0.031546931713819504, -0.005986989010125399], [-0.0716986283659935, 0.05425513908267021, 0.0747373104095459, -0.03356024995446205, 0.03265068680047989, -0.09254492074251175, 0.059615608304739, 0.14378835260868073, -0.10912581533193588, -0.029106486588716507, -0.049888964742422104, 0.015994861721992493, -0.20747080445289612, -0.07716313004493713, 0.14544875919818878, 0.05842144414782524, 0.06143520399928093, -0.07335934042930603, 0.08218597620725632, -0.01935831643640995, -0.025163104757666588], [-0.14421910047531128, 0.10927968472242355, -0.03254101425409317, 0.05411701649427414, -0.006904608570039272, -0.08223661035299301, 0.09307415783405304, -0.02998409979045391, 0.01762728951871395, 0.13302144408226013, -0.05112956836819649, 0.09410926699638367, -0.21288788318634033, 0.028917016461491585, -0.03177967295050621, -0.019825203344225883, 0.019184719771146774, 0.05807672068476677, -0.21183259785175323, -0.08319107443094254, -0.07763120532035828], [-0.03180556744337082, -0.06676468253135681, -0.04695821553468704, -0.1114184781908989, -0.09823641926050186, -0.09767741709947586, -0.0466439351439476, -0.0465090312063694, 0.045720603317022324, 0.06272348016500473, 0.06948735564947128, -0.02767249196767807, 0.01116571668535471, -0.0003224531246814877, 0.10455586016178131, 0.04045712947845459, -0.041526731103658676, 0.0365382544696331, -0.004975373391062021, -0.04100620001554489, 0.114771768450737], [-0.1153702437877655, -0.0764961987733841, 0.0036042199935764074, -0.005928725004196167, 0.10566128045320511, -0.06846056133508682, -0.0015673526795580983, -0.015516038052737713, 0.17544208467006683, 0.046085964888334274, 0.07905449718236923, 0.02341299317777157, 0.18608401715755463, 0.13144968450069427, 0.1552547961473465, 0.041772421449422836, -0.14334198832511902, -0.0956016257405281, -0.07555314898490906, -0.0480702705681324, 0.036345694214105606], [-0.09380750358104706, 0.0848950445652008, 0.11926530301570892, 0.07236799597740173, 0.014967652037739754, 0.09797544777393341, 0.09191001206636429, -0.03530820459127426, 0.09157729148864746, -0.014436122961342335, 0.11753082275390625, 0.07983073592185974, -0.045171108096838, 0.041404709219932556, 0.05935712158679962, 0.030754437670111656, -0.004414885304868221, 0.11387569457292557, 0.02820049598813057, -0.046303268522024155, 0.12157177180051804], [0.14929069578647614, 0.07961539924144745, 0.023891212418675423, -0.010843230411410332, 0.04061877354979515, 0.12235020101070404, 0.12226226180791855, -0.025371341034770012, 0.15232038497924805, 0.16092804074287415, -0.10397529602050781, 0.07826179265975952, -0.21290160715579987, 0.02970019355416298, 0.06475481390953064, -0.013398244045674801, 0.02173885516822338, 0.12113486975431442, 0.027469275519251823, -0.09936762601137161, 0.031002026051282883], [0.1095181480050087, 0.11665630340576172, -0.21067547798156738, -0.023511730134487152, 0.005104603245854378, -0.08115944266319275, 0.13750889897346497, 0.007337232120335102, -0.060505304485559464, 0.10558310151100159, -0.02462410181760788, -0.040589772164821625, -0.10435494035482407, -0.03114638477563858, -0.044744573533535004, 0.08133438974618912, -0.00380675564520061, 0.09570435434579849, -0.003926068544387817, -0.17398995161056519, 0.015386409126222134], [-0.013494991697371006, 0.06771668791770935, 0.16113689541816711, -0.05549071356654167, 0.048445794731378555, -0.018315082415938377, 0.11038290709257126, -0.00440182164311409, 0.017798449844121933, -0.11592597514390945, 0.08992219716310501, -0.07235848158597946, -0.09644840657711029, -0.004640023689717054, -0.053313661366701126, 0.037447553128004074, -0.03903675079345703, -0.045039787888526917, -0.0536629892885685, -0.02670753188431263, 0.009451635181903839], [-0.0029777076561003923, -0.025910882279276848, 0.1490003913640976, -0.0581696480512619, -0.07386218011379242, -0.06431537121534348, -0.037857118993997574, 0.05014216527342796, 0.05439827963709831, -0.07561928778886795, 0.13306736946105957, -0.13060666620731354, 0.04089454934000969, 0.09064802527427673, -0.046351250261068344, 0.021921705454587936, 0.12000694870948792, 0.030521521344780922, -0.06718454509973526, 0.10749772191047668, -0.030048491433262825], [-0.126106858253479, 0.0811421349644661, 0.18540829420089722, 0.11120781302452087, 0.10173909366130829, 0.01418042741715908, 0.003659667447209358, 0.0333620049059391, -0.0748920738697052, 0.011080772615969181, -0.12303541600704193, -0.07148140668869019, -0.13980741798877716, -0.10784036666154861, -0.04711400717496872, -0.015423348173499107, -0.1778748631477356, -0.042882632464170456, 0.09666677564382553, -0.008761540055274963, 0.09803292900323868], [0.02047640085220337, 0.1297662854194641, 0.15324807167053223, -0.11206994205713272, -0.011375006288290024, 0.08848600834608078, -0.0317206047475338, 0.056387390941381454, -0.01660161465406418, 0.08730737864971161, -0.09985238313674927, 0.04240676015615463, -0.03682544454932213, -0.05261004716157913, 0.15541096031665802, -0.11051838099956512, -0.22474750876426697, -0.029120784252882004, 0.08677645027637482, -0.03238099440932274, 0.017969993874430656], [0.15690529346466064, 0.1208483949303627, 0.04950529336929321, 0.1363459825515747, 0.10638131201267242, -0.07353027164936066, -0.13642723858356476, -0.05689316615462303, 0.046593986451625824, 0.09941016137599945, -0.11624546349048615, 0.006597031839191914, -0.1449947953224182, -0.07703632861375809, -0.10710541903972626, 0.027538105845451355, 0.26721733808517456, -0.06617448478937149, -0.11259753257036209, 0.0665893480181694, 0.07240492850542068], [-0.013202433474361897, 0.06898528337478638, 0.04218306019902229, 0.010596461594104767, 0.06311070919036865, 0.13065293431282043, -0.07379871606826782, -0.1373574435710907, 0.10162457823753357, -0.05355850234627724, 0.07212179154157639, -0.10237845033407211, -0.2316553145647049, -0.06693615019321442, -0.07385209202766418, 0.07468634843826294, 0.07534846663475037, -0.018370337784290314, 0.070552758872509, -0.014473920688033104, 0.04593902826309204], [-0.056029535830020905, -0.0046014864929020405, -0.09238167852163315, 0.030633896589279175, 0.04142861068248749, 0.009150784462690353, 0.10366050899028778, 0.06768040359020233, -0.0766594335436821, 0.11194794625043869, -0.014579414390027523, -0.02633323147892952, 0.1716507226228714, 0.03600753843784332, -0.054290734231472015, -0.00015310829621739686, 0.09723686426877975, 0.0934780016541481, -0.1393279731273651, -0.11670884490013123, -0.0008538420079275966], [-0.08861356228590012, -0.09110206365585327, -0.11983920633792877, 0.06525794416666031, -0.0358315110206604, 0.117937371134758, 0.006771046202629805, 0.026255080476403236, -0.006705550476908684, 0.190071702003479, 0.07055265456438065, 0.04675274342298508, -0.11970633268356323, 0.060835208743810654, -0.051895853132009506, 0.0829446092247963, 0.016276439651846886, 0.09888803958892822, -0.12606140971183777, -0.08719361573457718, -0.018901849165558815], [-0.06800654530525208, 0.08802858740091324, -0.06809883564710617, -0.12212112545967102, 0.10241082310676575, -0.004606288392096758, 0.04483994096517563, -0.006425172556191683, -0.025944218039512634, 0.08925771713256836, -0.04858586564660072, 0.14674948155879974, -0.05438614636659622, 0.037490829825401306, -0.09276607632637024, -0.07579845190048218, 0.021240144968032837, -0.037685927003622055, -0.03387448936700821, -0.030948787927627563, -0.07964426279067993], [-0.07432281225919724, -0.0515621043741703, -0.043347034603357315, -0.0858309343457222, 0.053722530603408813, -0.022448819130659103, 0.09803947806358337, -0.06663309782743454, 0.03821881115436554, 0.02893386036157608, 0.08911176770925522, -0.023995699360966682, 0.04989653080701828, -0.03436382859945297, -0.0025037287268787622, 0.016831830143928528, -0.055742133408784866, -0.049045391380786896, -0.06005842983722687, 0.021098511293530464, -0.03458527848124504], [-0.02623545564711094, 0.03376232087612152, -0.006432637572288513, 0.011667640879750252, -0.012620355933904648, 0.08682432025671005, 0.054420482367277145, -0.10605243593454361, -0.03674868866801262, -0.035313308238983154, 0.0026746741496026516, 0.02237527072429657, -0.03886113688349724, 0.04390909895300865, 0.0589318573474884, -0.032137613743543625, 0.055904317647218704, -0.017173005267977715, 0.04336056858301163, -0.012772983871400356, 0.049349378794431686], [0.013474411331117153, -0.015984710305929184, 0.06797023117542267, 0.007983595132827759, 0.04679732024669647, -0.025915764272212982, 0.048385605216026306, -0.05685262754559517, 0.00614507682621479, 0.10953052341938019, -0.034390635788440704, 0.019511763006448746, -0.14262926578521729, 0.08911113440990448, -0.0548664815723896, -0.0079482551664114, -0.1448594480752945, -0.004868886433541775, 0.07172589004039764, -0.015134770423173904, -0.0014762282371520996], [-0.005556275602430105, -0.02395654283463955, -0.012132384814321995, 0.021557655185461044, 0.017928889021277428, 0.057080335915088654, -0.024759164080023766, 0.00918728206306696, -0.02575894258916378, -0.05804446339607239, 0.017310114577412605, 0.04682936146855354, 0.07890342175960541, -0.0034756397362798452, -0.0616600476205349, -0.05267144367098808, -0.04323963820934296, -0.04346281290054321, -0.01933765597641468, -0.0032820950727909803, -0.012342899106442928], [0.13603706657886505, -0.06302476674318314, -0.08355294913053513, 0.14142802357673645, -0.07006808370351791, -0.060916755348443985, -0.0015796968946233392, 0.0560004748404026, 0.00572539446875453, 0.18573807179927826, 0.06972706317901611, -0.011521023698151112, 0.07409924268722534, 0.026172524318099022, 0.11155591160058975, 0.02359672449529171, -0.0753263384103775, -0.06520231813192368, 0.16135238111019135, -0.02377130649983883, 0.1409008502960205], [0.0004384450730867684, -0.0035149354953318834, -0.009126688353717327, 0.033620089292526245, -0.0632699579000473, -0.04288078472018242, -0.05290193855762482, 0.015367359854280949, -0.005980317946523428, 0.0023042913526296616, -0.04312912002205849, 0.07076207548379898, 0.057740021497011185, 0.02094990573823452, 0.12975291907787323, 0.0031518929172307253, 0.034819986671209335, 0.03191469609737396, -0.011784971691668034, 0.05356642231345177, 0.0190627109259367], [-0.060266513377428055, -0.028690099716186523, 0.11037879437208176, 0.02186112105846405, -0.025787582620978355, 0.046012382954359055, -0.11631669104099274, 0.020453965291380882, -0.08655466884374619, -0.02716112695634365, -0.08180825412273407, 0.038661208003759384, 0.009160201996564865, -0.009202573448419571, 0.04595602676272392, 0.028859924525022507, 0.14525119960308075, 0.001670082681812346, -0.0572969950735569, 0.018437324091792107, -0.046958230435848236], [0.05296868085861206, -0.02067338302731514, -0.1640310287475586, 0.1094743087887764, -0.03374399244785309, 0.07787121087312698, -0.10458658635616302, -0.08213771879673004, -0.04102553054690361, 0.020531175658106804, -0.06041060388088226, -0.11325402557849884, -0.09439718723297119, 0.05209864303469658, -0.023808233439922333, -0.06443388015031815, -0.11958715319633484, 0.0027133391704410315, 0.006165506783872843, 0.02411303110420704, -0.0619526170194149], [0.025862134993076324, 0.03965670242905617, 0.1102243959903717, -0.15278588235378265, -0.05028930678963661, 0.07249797880649567, -0.05926113203167915, -0.046536680310964584, 0.05208886042237282, 0.1473490595817566, -0.10458168387413025, -0.15855208039283752, 0.1347655951976776, 0.035389821976423264, 0.14741133153438568, -0.018827958032488823, 0.06246963515877724, 0.04487822204828262, 0.030090762302279472, 0.011107434518635273, -0.08994726091623306], [-0.1754947006702423, 0.03460308536887169, 0.12332764267921448, 0.03239284083247185, -0.11927773058414459, 0.11054381728172302, 0.11146432161331177, -0.14633795619010925, 0.11536979675292969, -0.05576927959918976, 0.03292839974164963, 0.00870569609105587, -0.023704344406723976, 0.010220624506473541, -0.03647433966398239, 0.06506439298391342, -0.23303139209747314, 0.03171444684267044, 0.07920733094215393, 0.036880992352962494, -0.05352529510855675], [0.0948723778128624, -0.0034078634344041348, 0.12132776528596878, -0.016336889937520027, 0.0675613060593605, 0.10631367564201355, 0.0658850148320198, -0.11510146409273148, -0.06204770877957344, 0.10557737201452255, 0.010046501643955708, 0.03903048485517502, 0.016537800431251526, -0.01972450315952301, 0.01662842370569706, -0.04905228689312935, -0.07077597826719284, 0.033201269805431366, -0.06751543283462524, 0.0761379525065422, 0.11406753957271576], [0.07257549464702606, 0.048844482749700546, -0.0007783969049341977, -0.08209992200136185, 0.026549339294433594, 0.05927104130387306, -0.053683992475271225, 0.06503187119960785, 0.04061298072338104, 0.09089238941669464, -0.04327196255326271, -0.05730627104640007, 0.09516820311546326, -0.07747510075569153, -0.009580548852682114, -0.00016990370932035148, 0.15029212832450867, 0.04051749408245087, 0.127767875790596, -0.021251197904348373, -0.08374230563640594], [-0.016306867823004723, 0.12695592641830444, 0.10720499604940414, -0.04083499684929848, 0.06259410828351974, -0.06224391236901283, -0.004320777021348476, 0.08374902606010437, 0.04507788270711899, 0.05203749239444733, -0.013799682259559631, -0.2239258736371994, -0.06786958128213882, -0.03087618015706539, 0.19516372680664062, 0.05950084328651428, -0.08095807582139969, 0.06891371309757233, -0.11569542437791824, -0.04720687493681908, -0.08980298787355423], [-0.036470916122198105, -0.0561680942773819, -0.08574002236127853, 0.06760110706090927, -0.021726829931139946, 0.07281190901994705, -0.031272102147340775, -0.10040590167045593, -0.0013735840329900384, -0.009370432235300541, 0.15259626507759094, -0.036602627485990524, 0.01667959988117218, -0.009149828925728798, -0.12345454841852188, -0.009352440014481544, -0.14252689480781555, -0.05285133793950081, -0.02941550314426422, 0.021261397749185562, -0.05649959668517113], [-0.034317802637815475, -0.061447400599718094, 0.039926812052726746, 0.009633959271013737, -0.029028601944446564, -0.09028377383947372, -0.00047878691111691296, 0.025906503200531006, 0.03539257124066353, 0.0020660606678575277, -0.03698480874300003, -0.0137687548995018, -0.0408824123442173, -0.07007402926683426, 0.011698422022163868, 0.005471357610076666, -0.24648015201091766, 0.07906676083803177, 0.032530076801776886, -0.029899902641773224, -0.05074033513665199], [0.07267297804355621, -0.052744198590517044, 0.0018677948974072933, 0.05965639650821686, -0.13740015029907227, -0.09452936798334122, -0.0816817358136177, 0.08195378631353378, 0.06382091343402863, 0.10960739850997925, 0.03765035793185234, -0.01516434270888567, -0.14010776579380035, -0.07713225483894348, -0.09374973177909851, 0.019970402121543884, 0.041641056537628174, -0.00011034806811949238, 0.07613683491945267, -0.06515093147754669, 0.11317216604948044], [0.11559630185365677, -0.10534895211458206, 0.137461319565773, 0.19146887958049774, -0.13488638401031494, 0.046253498643636703, 0.15405400097370148, 0.05404379218816757, 0.05534030497074127, 0.0601578950881958, 0.11235994845628738, -0.09449836611747742, -0.16373103857040405, 0.06039779260754585, -0.00903200265020132, -0.031157372519373894, -0.11313923448324203, -0.05718217045068741, 0.07760639488697052, 0.13111244142055511, 0.03226519376039505], [-0.02135871909558773, -0.10063504427671432, -0.05740572512149811, -0.18598684668540955, 0.035919155925512314, -0.05086987838149071, 0.08521828800439835, 0.07050786912441254, -0.08388275653123856, 0.10245680809020996, 0.10056107491254807, -0.01777832768857479, -0.1296757012605667, 0.10243765264749527, -0.06815927475690842, 0.03478747233748436, 0.09795553237199783, 0.006577508524060249, -0.16134847700595856, 0.08326655626296997, -0.09864296764135361], [-0.19623690843582153, 0.05224977433681488, -0.06495153158903122, -0.21101820468902588, -0.09355022013187408, -0.04613899812102318, -0.06586034595966339, -0.03613124415278435, -0.07556226849555969, 0.19559665024280548, -0.11206477135419846, 0.04443849250674248, -0.010197467170655727, 0.006316567771136761, -0.014515949413180351, 0.04211510717868805, 0.06524407118558884, -0.030654381960630417, -0.1869882494211197, -0.051330503076314926, -0.14206193387508392], [-0.057621948421001434, 0.14031799137592316, 0.1028292179107666, 0.057688601315021515, 0.048854466527700424, -0.08401422202587128, 0.043778154999017715, -0.07405215501785278, 0.002076515695080161, -0.027645651251077652, -0.03507663309574127, -0.04266202822327614, 0.017715541645884514, 0.061424680054187775, 0.15318503975868225, -0.04609886556863785, 0.0779367983341217, 0.07376600056886673, 0.009002137929201126, -0.055493831634521484, -0.00609136838465929], [-0.15721763670444489, -0.09902802109718323, -0.005669541656970978, 0.03356800973415375, -0.010651769116520882, -0.08112409710884094, 0.039876267313957214, -0.030728016048669815, -0.11846668273210526, 0.16966624557971954, -0.1051449105143547, 0.09071521461009979, -0.13753294944763184, -0.045668866485357285, -0.031797852367162704, -0.033104173839092255, 0.037882667034864426, -0.07635533809661865, -0.16180779039859772, -0.08307318389415741, 0.018811773508787155], [-0.10011620819568634, -0.10304877161979675, -0.004778828471899033, 0.030118416994810104, 0.13253125548362732, -0.07272098958492279, 0.09812942147254944, 0.02408103086054325, -0.030009010806679726, 0.06174265593290329, -0.0351702943444252, -0.12584415078163147, -0.0431317500770092, 0.09393195807933807, 0.04706677421927452, -0.022499792277812958, -0.16011382639408112, 0.12395374476909637, 0.10673338919878006, -0.006241713184863329, -0.13260284066200256], [-0.0010473796864971519, -0.03640683740377426, 0.007955003529787064, -0.033771079033613205, 0.021288234740495682, 0.02223026007413864, 0.06358961015939713, 0.023348042741417885, -0.03131131827831268, -0.041106127202510834, 0.013422224670648575, 0.033891789615154266, -0.0025254199281334877, 0.0550943985581398, 0.04200611263513565, 0.00899818167090416, -0.006335781421512365, 0.0318128764629364, 0.014316760934889317, -0.013087424449622631, -0.022786470130085945], [0.03130773454904556, -0.018099579960107803, 0.004278486594557762, 0.04617847502231598, -0.06371255964040756, 0.02295314148068428, 0.02290496602654457, 0.04799395427107811, -0.006875530816614628, -0.01084697712212801, -0.026509765535593033, 0.039016276597976685, -0.0018768555019050837, -0.02918148599565029, 0.060812823474407196, 0.015707513317465782, -0.1135469451546669, 0.025211598724126816, -0.008730831556022167, -0.008425340056419373, 0.031492818146944046], [0.21466514468193054, 0.13021431863307953, -0.14600957930088043, -0.01902281865477562, 0.06197504326701164, 0.05491875857114792, -0.09744896739721298, -0.012797710485756397, 0.13716867566108704, 0.19126595556735992, -0.00435880571603775, 0.07699093967676163, 0.06330189108848572, -0.029666926711797714, 0.002436675364151597, 0.010345160961151123, 0.03547951951622963, -0.021298501640558243, 0.08976564556360245, 0.014268414117395878, 0.15169760584831238], [-0.029445871710777283, 0.032691143453121185, 0.11152683943510056, 0.06231819838285446, -0.04507498815655708, 0.07647788524627686, 0.09519274532794952, 0.15959687530994415, 0.01656830683350563, 0.007286646403372288, 0.01671365462243557, 0.029334427788853645, -0.02926751598715782, 0.024357357993721962, 0.07241111248731613, -0.17889373004436493, -0.09292146563529968, -0.08594921976327896, 0.05656561627984047, -0.025770487263798714, 0.09354298561811447], [0.04020790383219719, -0.056525591760873795, 0.0719308853149414, -0.09950181841850281, 0.10671648383140564, -0.03337022662162781, -0.00030313091701827943, -0.0072044194675982, 0.0500388965010643, -0.11226433515548706, -0.016312146559357643, 0.03070889599621296, 0.06117893382906914, 0.022904248908162117, 0.08894000202417374, 0.05846283584833145, 0.030413169413805008, 0.10117729008197784, 0.1559108942747116, 0.05216243490576744, 0.10105312615633011], [0.008551329374313354, 0.1185794323682785, 0.20646819472312927, -0.06178940087556839, -0.024828748777508736, 0.038790907710790634, -0.07989822328090668, -0.12486684322357178, -0.05261887237429619, 0.20569147169589996, -0.12703178822994232, -0.027418557554483414, -0.047019947320222855, -0.03738895803689957, 0.05361664295196533, -0.1464749425649643, -0.01929406449198723, 0.17820607125759125, 0.11918526887893677, -0.04011630639433861, -0.09257319569587708], [-0.01038544625043869, -0.07232220470905304, 0.08860372751951218, -0.01705300435423851, 0.016829926520586014, -0.022403115406632423, 0.04521641880273819, -0.03963027149438858, -0.025758586823940277, -0.1785409301519394, -0.024891942739486694, -0.0606534369289875, -0.004754389636218548, 0.047395505011081696, 0.04095454514026642, 0.06127409264445305, -0.023276593536138535, -0.02681872807443142, -0.10662171244621277, -0.11815018206834793, -0.07550680637359619], [-0.03077017329633236, 0.026798468083143234, -0.1654985249042511, -0.05862393230199814, 0.07900262624025345, 0.04713577404618263, -0.05475073307752609, 0.07874319702386856, -0.046071264892816544, -0.050670791417360306, -0.12432697415351868, 0.11790132522583008, -0.11676432937383652, -0.08265505731105804, -0.014792012982070446, 0.1348743438720703, 0.007637878879904747, -0.021092643961310387, 0.08548575639724731, -0.09785392135381699, -0.0762268528342247], [0.006355590187013149, 0.04349434748291969, -0.03720790520310402, -0.03916015475988388, 0.04636787995696068, -0.026449553668498993, -0.005497050937265158, -0.00573755893856287, 0.043160345405340195, 0.08215425908565521, -0.02644982933998108, -0.048364296555519104, 0.056016676127910614, 0.0011148500489071012, 0.023072289302945137, -0.0041348496451973915, -0.05801684036850929, -0.005085035227239132, 0.07286302000284195, 0.04741191864013672, 0.001001256750896573], [-0.05873670428991318, -0.08034081757068634, 0.10616390407085419, -0.03758036717772484, 0.12941907346248627, 0.11496787518262863, 0.14985714852809906, -0.07110361754894257, -0.07820174098014832, 0.07702004909515381, 0.015222991816699505, -0.03226979449391365, -0.11394164711236954, -0.11019621789455414, -0.07370904088020325, 0.04272348806262016, -0.23493751883506775, 0.05555601418018341, 0.13696739077568054, 0.036319904029369354, 0.05653945356607437], [-0.04312281683087349, -0.03207733482122421, -0.078981414437294, 0.015233040787279606, -0.0670052021741867, -0.057313308119773865, -0.05782235786318779, 0.04008730873465538, -0.10252001136541367, 0.17526903748512268, -0.10915432870388031, 0.07355407625436783, -0.048921357840299606, 0.05054615065455437, -0.06969834119081497, -0.10706944763660431, -0.012849364429712296, -0.08490192890167236, -0.12135181576013565, -0.137091264128685, 0.10536102205514908], [0.05281047150492668, 0.028902335092425346, 0.1063162237405777, -0.0667584016919136, -0.02329486981034279, 0.12854793667793274, -0.07746102660894394, -0.03971920907497406, 0.0975879654288292, -0.07478127628564835, 0.03899260237812996, 0.016821078956127167, 0.05182743817567825, 0.03579556196928024, -0.049976687878370285, 0.1365332156419754, 0.06402110308408737, -0.03710684925317764, -0.06184874102473259, -0.13340044021606445, 0.14765788614749908], [-0.015200946480035782, 0.010318527929484844, 0.015629049390554428, -0.015097202733159065, -0.053649887442588806, 0.06532781571149826, 0.050968050956726074, -0.014076225459575653, -0.03307487443089485, 0.038490619510412216, 0.03474850580096245, -0.012494072318077087, 0.06168220564723015, 0.010768335312604904, 0.02967221848666668, -0.017713118344545364, 0.0840839371085167, 0.033435966819524765, -0.03308292105793953, -0.001406787894666195, -0.01990252546966076], [0.07737954705953598, 0.01948825642466545, -0.053353503346443176, 0.11076901108026505, 0.04699089750647545, -0.01692485436797142, 0.01673164777457714, -0.01183557603508234, 0.057785507291555405, -0.018724720925092697, -0.01918143779039383, -0.012402018532156944, 0.09450376033782959, 0.013322750106453896, -0.003687093500047922, 0.014768209308385849, -0.07030553370714188, -0.00450329203158617, 0.06900667399168015, 0.10355176776647568, 0.012759379111230373], [0.006576950196176767, 0.02380029484629631, -0.04146452620625496, 0.05364826321601868, 0.029119040817022324, -0.06132728233933449, -0.0390695258975029, 0.05931225046515465, 0.016077574342489243, -0.050827980041503906, -0.05540189519524574, 0.014021983370184898, 0.06145179271697998, -0.07602173089981079, -0.018536940217018127, 0.03092585876584053, -0.14784160256385803, 0.06547672301530838, -0.04213425889611244, 0.11315561830997467, -0.002364443615078926], [-0.0474536158144474, 0.057706646621227264, -0.06455055624246597, -0.15236227214336395, -0.055792633444070816, -0.12022088468074799, -0.040472012013196945, -0.042738720774650574, -0.032589271664619446, 0.09310071915388107, 0.06647190451622009, 0.1108698770403862, -0.08344381302595139, 0.05542242154479027, -0.06117860972881317, 0.017436403781175613, -0.11057695746421814, -0.05298788100481033, -0.08481526374816895, -0.017872963100671768, 0.1103910282254219], [-0.09278886765241623, 0.05698491260409355, -0.052646055817604065, -0.0031850107479840517, -0.08057159185409546, 0.10039064288139343, 0.13154655694961548, -0.011344955302774906, 0.12649428844451904, 0.020302020013332367, 0.08000055700540543, -0.05921108275651932, -0.005200314801186323, -0.03207657113671303, 0.16276198625564575, -0.020676899701356888, -0.08685628324747086, 0.10107014328241348, -0.07093366235494614, 0.015166094526648521, 0.06499627232551575], [-0.011058700270950794, 0.032142892479896545, 0.07487684488296509, 0.057187724858522415, -0.050546955317258835, -0.038692034780979156, -0.030192503705620766, -0.02426856756210327, -0.04174788296222687, 0.015087420120835304, 0.04477410390973091, -0.0061857495456933975, -0.059797901660203934, -0.00696461321786046, -0.0028810647781938314, -0.006317489314824343, -0.07594602555036545, 0.08315594494342804, -0.031514640897512436, -0.0045583611354231834, -0.02608278952538967], [0.00785142369568348, 0.019997118040919304, -0.07945238798856735, 0.06012296304106712, 0.008405081927776337, 0.06663087755441666, 0.0695759654045105, -0.00919498410075903, 0.011157345958054066, -0.03728987276554108, 0.0196396317332983, 0.05497881770133972, -0.08348868042230606, 0.07317749410867691, -0.09252584725618362, -0.05850991606712341, -0.10201718658208847, -0.037424348294734955, -0.02249760739505291, -0.007392980623990297, 0.056157954037189484], [-0.10159877687692642, 0.056847233325242996, -0.005373841151595116, -0.03190967068076134, -0.0033829198218882084, 0.10034146904945374, -0.1059715673327446, -0.09739991277456284, 0.02290855161845684, 0.03328528627753258, 0.09200924634933472, 0.0036492033395916224, 0.20038747787475586, 0.18015044927597046, -0.13454824686050415, -0.012431912124156952, -0.08250593394041061, -0.0630161315202713, 0.05681891366839409, -0.015210801735520363, -0.13520008325576782], [-0.023557530716061592, -0.01902877725660801, 0.05744980648159981, -0.06606001406908035, -0.0549028143286705, 0.11495113372802734, 0.02483949437737465, -0.12925343215465546, -0.10535183548927307, 0.08831580728292465, -0.1492045819759369, 0.07515307515859604, -0.13688793778419495, 0.030969759449362755, 0.09134705364704132, 0.06460519134998322, 0.028339488431811333, -0.017622262239456177, 0.013445445336401463, -0.0018964149057865143, -0.1370418816804886], [-0.06423432379961014, 0.023754402995109558, -0.0651324912905693, -0.08352501690387726, 0.051829662173986435, 0.0027102013118565083, -0.04223349317908287, 0.03933293744921684, -0.056537047028541565, 0.09999121725559235, -0.03524363785982132, -0.05358421057462692, 0.18201547861099243, 0.007462598383426666, -0.014397747814655304, 0.01511139515787363, 0.03806743770837784, 0.039066363126039505, -0.0491446778178215, -0.015315263532102108, -0.04543311148881912], [0.06846299767494202, -0.005906612612307072, 0.029046405106782913, -0.1455681174993515, -0.01442728005349636, -0.059166714549064636, -0.1494406759738922, 0.0876241996884346, 0.0009622439974918962, 0.043641138821840286, -0.07324673980474472, 0.06984660774469376, 0.043787989765405655, -0.11033830046653748, 0.08289925754070282, -0.05993691086769104, 0.21837353706359863, -0.05221553146839142, 0.016126098111271858, 0.03331442177295685, -0.03125164285302162], [0.15711908042430878, -0.047669053077697754, -0.14398732781410217, 0.11734016239643097, 0.1821727603673935, 0.05551067739725113, -0.13981500267982483, 0.10502560436725616, -0.04237660393118858, -0.10184900462627411, 0.0955067127943039, 0.09351177513599396, 0.14686830341815948, 0.0015017802361398935, 0.10801304131746292, -0.10910329967737198, 0.010323379188776016, 0.12578599154949188, -0.0985441505908966, -0.04844801872968674, 0.026168977841734886], [0.023978175595402718, 0.04289795085787773, -0.08265510201454163, 0.04716169089078903, 0.1179402768611908, 0.049433790147304535, -0.04889558255672455, 0.11249858886003494, -0.02571643702685833, 0.042244765907526016, -0.02723475731909275, -0.10929776728153229, 0.02408137358725071, 0.036484211683273315, -0.03634192794561386, -0.10748713463544846, 0.23228760063648224, -0.05314387381076813, 0.10731442272663116, -0.08233033865690231, -0.033674027770757675], [-0.01878633350133896, -0.058372076600790024, -0.0421501062810421, -0.03828347846865654, 0.05678107589483261, -0.0489710196852684, -0.023686856031417847, 0.016290094703435898, -0.034674108028411865, 0.11406977474689484, -0.019994227215647697, 0.06556709110736847, 0.003695916384458542, 0.017915574833750725, 0.02281893417239189, 0.029029084369540215, -0.019954320043325424, -0.035517070442438126, 0.005230162292718887, 0.05530736222863197, -0.03177044540643692], [0.05552656576037407, 0.1302943229675293, -0.32458287477493286, -0.029147567227482796, 0.08177649974822998, 0.15174740552902222, 0.06948571652173996, -0.025807853788137436, 0.08810064941644669, 0.057568978518247604, -0.0313105471432209, 0.13450738787651062, -0.11416633427143097, -0.013629766181111336, 0.024578653275966644, -0.06796368956565857, -0.0343613475561142, 0.134348064661026, 0.010286380536854267, 0.05167962983250618, -0.1175660565495491], [-0.03398516774177551, 0.03450099751353264, 0.03022722713649273, -0.0158879142254591, -0.060406770557165146, 0.008594799786806107, -0.03597872704267502, -0.03570261597633362, -0.060563307255506516, -0.038369644433259964, -0.022736452519893646, -0.03675762191414833, 0.09561140835285187, 0.07851696014404297, 0.015421547926962376, 0.014138813130557537, 0.07479403913021088, 0.024989282712340355, -0.06964568793773651, -0.015144488774240017, -0.013181820511817932], [-0.015433630906045437, -0.03838491812348366, 0.0368637777864933, -0.05411067605018616, -0.08746422827243805, 0.012324800714850426, 0.05163527652621269, -0.04065253213047981, -0.0832342803478241, -0.008498343639075756, -0.036249466240406036, -0.17047016322612762, 0.010069412179291248, 0.08917365223169327, 0.06853024661540985, 0.007011472247540951, 0.14816376566886902, 0.08574818819761276, -0.05317065119743347, -0.047436799854040146, 0.014289701357483864], [0.043935377150774, 0.1001538336277008, 0.06251309812068939, -0.028874525800347328, -0.02016354352235794, -0.06538482755422592, -0.12199968099594116, 0.13363398611545563, -0.13482531905174255, 0.0036626604851335287, -0.17690829932689667, 0.001431398093700409, 0.04421447217464447, -0.10825632512569427, -0.01173408329486847, -0.003592930268496275, 0.2940421402454376, 0.05204920470714569, 0.11630813777446747, -0.07614969462156296, -0.07150496542453766], [-0.044924866408109665, -0.026227418333292007, 0.022839758545160294, 0.10916265100240707, -0.034276627004146576, -0.009609508328139782, 0.0789523497223854, -0.054998017847537994, 0.006318395026028156, -0.06466113775968552, 0.0014639052096754313, 0.02942838706076145, -0.1392286866903305, 0.08823212236166, -0.032117269933223724, 0.0023532421328127384, 0.062447477132081985, -0.002180923940613866, 0.012286516837775707, -0.01017643790692091, -0.011706594377756119], [0.04671576991677284, 0.03734225407242775, -0.03470703214406967, 0.09087127447128296, 0.06667139381170273, -0.040305886417627335, 0.04788750782608986, 0.04669414088129997, 0.007477843202650547, 0.0257661622017622, 0.028116988018155098, -0.04300965368747711, -0.023778188973665237, 0.005024563521146774, 0.09172061830759048, -0.08172117173671722, -0.07223203778266907, 0.025938112288713455, 0.047562871128320694, 0.07131358981132507, 0.02917507290840149], [0.006807062774896622, 0.03135539963841438, 0.003311195643618703, -0.014970965683460236, 0.05607844516634941, -0.048407189548015594, 0.07926410436630249, 0.08650244772434235, 0.05632028728723526, 0.04580916091799736, 0.005319340620189905, 0.015857277438044548, 0.03158959373831749, -0.04577404633164406, 0.13054285943508148, -0.04500387981534004, -0.20025408267974854, 0.048239562660455704, 0.05622846260666847, 0.0005304095684550703, -0.09319993108510971], [-0.04433956742286682, -0.02298407442867756, 0.07767793536186218, -0.08866564184427261, 0.05333072319626808, -0.023881560191512108, 0.09116183966398239, -0.05140083655714989, -0.07571344077587128, 0.00858569610863924, -0.02973640151321888, 0.01440490409731865, 0.1737983077764511, -0.04373014345765114, -0.019545679911971092, 0.050332676619291306, 0.1561025083065033, -0.02829282358288765, 0.04462992399930954, 0.03466328606009483, -0.09366390109062195]], "b1": [-0.11818134039640427, 0.20404280722141266, 0.07960540801286697, -0.15852844715118408, 0.20184046030044556, 0.12845905125141144, -0.03827623277902603, -0.009895935654640198, 0.11811517924070358, 0.07836314290761948, -0.13747891783714294, 0.00021274616301525384, 0.03182120621204376, -0.11193942278623581, -0.11848274618387222, 0.042132265865802765, -0.035431426018476486, 0.15243421494960785, 0.04825693368911743, 0.1305757313966751, -0.032822344452142715, 0.07280024141073227, 0.15405422449111938, 0.054861634969711304, 0.03389181196689606, -0.12535271048545837, 0.10929693281650543, 0.032701652497053146, -0.03176850825548172, 0.007177424151450396, -0.0882621556520462, -0.013621496967971325, 0.07549403607845306, 0.11433565616607666, 0.16929924488067627, 0.09957186132669449, 0.21763314306735992, 0.03511395677924156, -0.09487418085336685, 0.10113655775785446, -0.08717719465494156, -0.03231806308031082, -0.024844061583280563, 0.030703280121088028, -0.0663289725780487, 0.03960161656141281, -0.024254336953163147, -0.09136579185724258, -0.017772095277905464, 0.08636640012264252, 0.013455681502819061, 0.04060018062591553, -0.028660008683800697, 0.08777573704719543, 0.03487049788236618, -0.028278766199946404, 0.03197173401713371, -0.08119215816259384, 0.044198472052812576, 0.02614201046526432, -0.160310760140419, 0.0955575704574585, 0.032262034714221954, 0.09820004552602768, -0.04714352637529373, -0.11537350714206696, -0.15260420739650726, 0.13323576748371124, -0.0777379497885704, -0.0991276279091835, -0.0028569751884788275, -0.026847418397665024, -0.0845179334282875, -0.1587861180305481, -0.04810832440853119, 0.040111787617206573, 0.19249075651168823, 0.08940938860177994, 0.04044229909777641, -0.04350459575653076, -0.005323401652276516, 0.163389191031456, 0.05431893840432167, 0.009513752534985542, 0.1100248172879219, 0.07101482152938843, 0.16049422323703766, -0.058293189853429794, 0.03870375081896782, 0.06301732361316681, 0.10724203288555145, -0.023815080523490906, 0.008748559281229973, -0.07336589694023132, 0.11771523207426071, 0.14003443717956543], "W2": [[0.07280021160840988, 0.009420827962458134, 0.006169598083943129, -0.03528258576989174, -0.02432655356824398, -0.028004104271531105, -0.057581182569265366, -0.0041284505277872086, 0.042145855724811554, 0.05993473157286644, -0.08767189085483551, 0.05424409359693527, -0.0345367006957531, 0.018071137368679047, 0.031138023361563683, 0.005722980946302414, 0.0008730650879442692, -0.07964755594730377, 0.09388679265975952, 0.006336416583508253, 0.09963195770978928, -0.06850672513246536, 0.06704937666654587, 0.06561559438705444, -0.00459497282281518, 0.01621812954545021, 0.052303969860076904, 0.06488632410764694, -0.03332514315843582, 0.0536346398293972, -0.05262451991438866, 0.004356699995696545, -0.02720625139772892, 0.00960144866257906, 0.06808310747146606, 0.0646633505821228, 0.02797427773475647, 0.055451009422540665, -0.014900845475494862, -0.04872981458902359, 0.029122278094291687, 0.06644250452518463, -0.06171855702996254, 0.02039497159421444, -0.013860464096069336, 0.05437733232975006, -0.07006550580263138, -0.034483734518289566, 0.04429991543292999, 0.06684885919094086, 0.06173941120505333, 0.03544095531105995, 0.07441269606351852, -0.03280798718333244, -0.018939204514026642, 0.019580284133553505, -0.026831939816474915, 0.006872131023555994, 0.11280213296413422, -0.022088615223765373, -0.02114875428378582, -0.0364677794277668, -0.023007582873106003, -0.045867063105106354, -0.010107982903718948, -0.09239476919174194, -0.051211509853601456, 0.03425208479166031, -0.021395642310380936, -0.034371986985206604, 0.03335052356123924, 0.04554545134305954, -0.05294611677527428, -0.0594099722802639, -0.009960886090993881, 0.025480570271611214, -0.007228545378893614, 0.03407911956310272, -0.04404190555214882, 0.00989963486790657, -0.03608077019453049, 0.08542300760746002, 0.010361900553107262, 0.03140171989798546, 0.08124986290931702, -0.002736109308898449, -0.01400692481547594, -0.010012163780629635, -0.04683026298880577, -0.0008556779357604682, 0.05400581657886505, 0.10323471575975418, -0.025860542431473732, -0.06058412045240402, 0.04922332242131233, 0.030672727152705193], [0.03193569928407669, 0.012304149568080902, 0.02471226081252098, 0.027919352054595947, -0.016912374645471573, 0.01931946910917759, 0.00381159083917737, -0.012622905895113945, 0.004163940437138081, 0.013502880930900574, 0.04333166405558586, -0.00010858361929422244, 0.01219650823622942, 0.04776077717542648, 0.0003136441227979958, 0.004985094536095858, 0.03919064626097679, 0.029762782156467438, -0.01993226818740368, 0.025086183100938797, 0.044324424117803574, 0.031740568578243256, 0.004217793233692646, -0.025372127071022987, 0.009946518577635288, 0.0281120166182518, -0.023802366107702255, 0.029508046805858612, 0.04594310745596886, -0.022983942180871964, 0.00031680965912528336, 0.04055507853627205, 0.011355938389897346, 0.03320584073662758, -0.007892458699643612, -0.04101017490029335, -0.03255659341812134, -0.008061462081968784, 0.004939365666359663, 0.03569166362285614, 0.00707068108022213, 0.02778002992272377, 0.011485695838928223, -0.026932749897241592, 0.013376295566558838, 0.0045642792247235775, -0.016380183398723602, 0.03564839065074921, -0.012877663597464561, -0.0024212750140577555, -0.0628102719783783, 0.018269237130880356, 0.007928469218313694, -0.003503001295030117, 0.009770235978066921, -0.001360041555017233, 0.018337730318307877, 0.018592258915305138, -0.009547012858092785, -0.0070505510084331036, 0.04473154619336128, 0.009772398509085178, -0.00016630958998575807, -0.004930939991027117, 0.054938990622758865, 0.04020293429493904, -0.004781894851475954, -0.05202256143093109, 0.03109751082956791, 0.03810847923159599, 0.02045886404812336, 0.004107693675905466, 0.04156007617712021, 0.005226289853453636, -0.00790739618241787, -0.00230519101023674, 0.001953658415004611, 0.047706443816423416, 0.024956803768873215, -0.0007197536178864539, 0.023811211809515953, -0.019965844228863716, 0.023849952965974808, 0.007731254678219557, 0.04336395114660263, 0.0027465983293950558, 0.02192099578678608, -0.009820869192481041, 0.09242647886276245, -0.0027355325873941183, 0.011375191621482372, -0.016606638208031654, 0.01545958872884512, -0.02398327738046646, 0.03196313604712486, 0.014002705924212933], [0.06496862322092056, 0.05528063699603081, -0.004839243832975626, -0.040077093988657, 0.035946693271398544, -0.032745297998189926, -0.07744356989860535, 0.010735186748206615, 0.025797497481107712, -0.0170561745762825, -0.03532462567090988, 0.08719092607498169, 0.05057353898882866, -0.0952949970960617, 0.03866935521364212, 0.013215681537985802, 0.04954627901315689, -0.02730392850935459, -0.012031463906168938, -0.004199661780148745, 0.017930172383785248, -0.06298261880874634, 0.018748655915260315, 0.024435343220829964, 0.09760677069425583, 0.012646417133510113, 0.06621325761079788, 0.04709545150399208, -0.1583612561225891, -0.06351180374622345, -0.07997860014438629, -0.11564795672893524, -0.04223738610744476, 0.05426554009318352, -0.029115116223692894, 0.11643086373806, -0.012359962798655033, 0.09333670139312744, 0.010277804918587208, -0.05562206730246544, 0.0034566198009997606, -0.05922982096672058, -0.052539873868227005, -0.0032989862374961376, -0.059700753539800644, 0.01176767610013485, 0.050714634358882904, -0.0323469452559948, 0.12462761253118515, -0.06509604305028915, -0.04242360219359398, 0.10075812041759491, 0.015002790838479996, 0.061181336641311646, -0.04304569959640503, 0.11374222487211227, -0.0018191123381257057, 0.03857162222266197, 0.11394419521093369, -0.05677207186818123, 0.003991041798144579, 0.022890962660312653, -0.035387326031923294, -0.0025838243309408426, 0.015437586233019829, -0.09126794338226318, -0.04911259189248085, 0.014398104511201382, -0.013878360390663147, -0.06206229701638222, 0.07781015336513519, -0.005889635067433119, -0.004879124462604523, -0.08970420062541962, 0.008522032760083675, 0.014954669401049614, 0.07129881531000137, -0.010052353143692017, -0.013071530498564243, -0.05810411274433136, 0.010625869035720825, 0.013922219164669514, 0.03777541220188141, 0.13451245427131653, 0.05913126841187477, 0.06492786854505539, 0.05915704369544983, -0.027838297188282013, -0.10022307932376862, 0.06535688787698746, 0.07089338451623917, 0.18871456384658813, 0.03718557208776474, 0.027145272120833397, 0.037336286157369614, 0.11265470832586288], [0.024038100615143776, 0.04238550737500191, 0.008536274544894695, -0.01826319471001625, 0.010954126715660095, 0.019696900621056557, -0.012356980703771114, 0.006717650685459375, 0.014073367230594158, 0.03151392564177513, -0.010019779205322266, -0.006473989225924015, -0.01199799682945013, -0.009628303349018097, -0.008209269493818283, 0.02750355750322342, -0.005002980586141348, -0.0039826370775699615, 0.024031199514865875, 0.0073128207586705685, 0.001698464504443109, 0.03221456706523895, -0.006456557661294937, 0.004186748526990414, -0.0027037349063903093, 0.0022516942117363214, 0.0038404068909585476, 0.02864556759595871, -0.024006862193346024, -0.022305265069007874, -0.015743304044008255, -0.04124544933438301, 0.04025512933731079, -0.03723006322979927, 0.0267837755382061, 0.012254603207111359, -0.03011888824403286, 0.012002731673419476, -0.02142275497317314, -0.011864375323057175, -0.010657110251486301, -0.00945854652673006, 0.03478337824344635, 0.02292175032198429, -0.03634077310562134, -0.006556783802807331, -0.0028729133773595095, -0.0318441241979599, 0.005963858682662249, -0.0009333934867754579, -0.026740411296486855, -0.009539312683045864, 0.022895555943250656, 0.004109553527086973, 0.008762906305491924, 0.06421059370040894, 0.05937846750020981, -0.00517850648611784, 0.0602489598095417, 0.0006226038094609976, -0.02302015945315361, -0.001710062031634152, 0.0061785439029335976, -0.0064430539496243, -0.037218958139419556, 0.005760339088737965, 0.006858364213258028, 0.0307098850607872, -0.031227407976984978, 0.003207988105714321, -0.018522417172789574, -0.017307661473751068, -0.00547787407413125, -0.025487514212727547, -0.02768506109714508, 0.020709266886115074, -0.00014124540030024946, 0.019114676862955093, -0.01866546832025051, -0.0011969516053795815, -0.019764186814427376, -0.0031531904824078083, 0.0030895050149410963, 0.0038686504121869802, -0.023809127509593964, -0.0012377328239381313, 0.025687774643301964, -0.011622131802141666, -0.02366734854876995, 0.0012571957195177674, 0.028426023200154305, 0.01080626342445612, -0.014414798468351364, 0.025615209713578224, -0.011487895622849464, 0.008169162087142467], [-0.05176793411374092, -0.003842452308163047, 0.07707348465919495, -0.10665945708751678, 0.07138883322477341, -0.06623397767543793, -0.006069072987884283, 0.014458311721682549, -0.07262011617422104, -0.01167333871126175, -0.09424147754907608, 0.018455883488059044, -0.05433257669210434, -0.060644570738077164, 0.027107229456305504, 0.06448595970869064, 0.04519132152199745, 0.029296109452843666, 0.08700394630432129, 0.035855360329151154, -0.05637331306934357, -0.018362607806921005, 0.10428744554519653, 0.072080098092556, 0.05564943328499794, -0.0899958610534668, 0.06605733931064606, -0.034449052065610886, -0.06242085620760918, 0.0787191167473793, -0.03747059032320976, -0.032534729689359665, -0.04784245416522026, 0.0535922572016716, -0.07149611413478851, 0.10414194315671921, 0.04828977212309837, 0.0812593325972557, -0.04226849600672722, 0.05981152504682541, -0.01699841581285, 0.071315698325634, -0.02259381115436554, 0.010462213307619095, -0.0766262635588646, 0.0193246528506279, 0.07745186984539032, -0.040436822921037674, -0.03141993656754494, -0.002804888179525733, -0.045366961508989334, 0.08628055453300476, -0.045304328203201294, 0.07469339668750763, -0.05400020256638527, 0.0291912779211998, 0.07080630958080292, 0.017034996300935745, 0.04115859791636467, 0.059718333184719086, 0.036079976707696915, -0.06797467172145844, -0.028589284047484398, -0.00880152452737093, -0.10526606440544128, -0.06298848986625671, -0.028043320402503014, 0.010896079242229462, -0.00852569006383419, -0.012665707617998123, -0.08383254706859589, 0.005431806668639183, -0.0415755957365036, -0.04377103969454765, 0.01802060566842556, -0.05627765506505966, -0.07167859375476837, 0.07179974019527435, 0.07454563677310944, 0.030385099351406097, 0.017939982935786247, 0.08200190216302872, 0.05239338427782059, -0.014358104206621647, 0.045211128890514374, -0.07250434160232544, 0.09198325127363205, -0.021885858848690987, 0.04078054428100586, -0.03964918851852417, -0.013147993944585323, 0.15669965744018555, -0.008966635912656784, -0.009690096601843834, 0.0005377965280786157, 0.018975326791405678], [0.0811941996216774, -0.008628394454717636, -0.011521040461957455, 0.05454479902982712, -0.0798657163977623, -0.0023742658086121082, 0.02345593273639679, -0.0038065724074840546, 0.012153306975960732, -0.031548697501420975, 0.030933432281017303, -0.0175351332873106, -0.012189790606498718, 0.1079895943403244, -0.0015700478106737137, 0.009186563082039356, 0.09660227596759796, -0.02884036675095558, -0.008572869002819061, -0.01765633001923561, -0.018648549914360046, -0.054868195205926895, -0.013869850896298885, -0.01048057060688734, 0.033941008150577545, 0.11792068183422089, 0.010816521011292934, -0.02225826308131218, 0.01877591758966446, -0.03892724961042404, 0.034140028059482574, 0.005613876972347498, -0.03552338108420372, -0.008730989880859852, -0.07839571684598923, -0.04986731335520744, -0.04607747867703438, 0.02386442944407463, 0.039886463433504105, 0.060219906270504, -0.002493959851562977, 0.021832438185811043, 0.016121620312333107, -0.018045645207166672, 0.020365195348858833, -0.021335327997803688, 0.036054499447345734, 0.023995500057935715, -0.04097055643796921, 0.025373976677656174, -0.03445233032107353, 0.008754461072385311, -0.09126453846693039, -0.03773820027709007, -0.008600604720413685, -0.026604807004332542, -0.0006115589640103281, 0.08855462074279785, 0.07010352611541748, -0.054052479565143585, 0.12663042545318604, -0.02580939419567585, 0.022970888763666153, -0.02185765653848648, 0.026401344686746597, 0.027546364814043045, 0.00641687260940671, -0.05839988961815834, -0.006761123426258564, 0.0346134752035141, 0.0041465735994279385, -0.025996699929237366, 0.06943158060312271, -0.0031290201004594564, -0.003970067948102951, 0.016929863020777702, -0.018433908000588417, 0.0883408933877945, -0.01945878192782402, 0.010388732887804508, 0.008160632103681564, -0.011740396730601788, 0.030910499393939972, 0.031377580016851425, 0.021900566294789314, -0.007817890495061874, 0.010235981084406376, 0.027453819289803505, 0.07956444472074509, -0.05676036700606346, 0.015835823491215706, -0.031009236350655556, 0.003173351753503084, 0.038454100489616394, -0.022076956927776337, 0.010240241885185242], [-0.037211235612630844, -0.05444260686635971, -0.06892900168895721, 0.019082525745034218, 0.0003359222027938813, -0.06335539370775223, 0.014017924666404724, 0.025014974176883698, -0.026402419432997704, -0.09019235521554947, 0.025733422487974167, -0.0229166429489851, -0.02840639278292656, 0.018680162727832794, 0.056596796959638596, -0.01148232351988554, -0.040065061300992966, 0.0778653472661972, -0.00861680880188942, 0.04708867520093918, 0.024557985365390778, -0.010576004162430763, -0.026407260447740555, -0.019449887797236443, -0.03298114985227585, -0.046750497072935104, 0.028010869398713112, -0.023937758058309555, -0.0039966050535440445, -0.1056663766503334, -0.024811649695038795, 0.021893225610256195, 0.022480489686131477, -0.020371213555336, -0.07886338233947754, 0.0028631880413740873, 0.016362298280000687, -0.08057145029306412, -0.08304368704557419, -0.07616853713989258, -0.03481405973434448, -0.016735078766942024, -0.05532381311058998, 0.03142842277884483, -0.004786467179656029, -0.003963903523981571, -0.04494572430849075, 0.02898395247757435, 0.002806524047628045, -0.006209014914929867, -0.07567564398050308, -0.030908742919564247, 0.02500932849943638, -0.07734766602516174, 0.036230478435754776, -0.0811283141374588, 0.016174284741282463, -0.04256255179643631, -0.046488918364048004, 0.07871892303228378, -0.023360570892691612, -0.05283648520708084, 0.042983461171388626, -0.006722511723637581, -0.04889706149697304, 0.003682417795062065, 0.03616214543581009, 0.009017739444971085, 0.030655454844236374, 0.007975867949426174, -0.02515239082276821, 0.003128431737422943, -0.05430082976818085, 0.040498267859220505, -0.024920346215367317, -0.07610416412353516, -0.04203013703227043, -0.017464574426412582, -0.06943774223327637, 0.04544409364461899, -0.02803971618413925, -0.057215966284275055, -0.07339145243167877, -0.04897797480225563, -0.04977200925350189, -0.05629996955394745, -0.02870652638375759, -0.010190379805862904, -0.06409742683172226, 0.03885095566511154, 0.06419791281223297, 0.008200445212423801, 0.03532566875219345, -0.00398570392280817, 0.039902426302433014, 0.00772355729714036], [-0.00016433522978331894, -0.000696108618285507, -0.0002746467653196305, 0.0005127304466441274, -0.0009960425086319447, -0.0005207885405980051, 0.00019584005349315703, -0.00047894357703626156, 0.0008572187507525086, -8.331665594596416e-05, 0.000573660945519805, 0.00020382586808409542, 0.00017010098963510245, 0.0003303015255369246, 0.00014749154797755182, 8.154365787049755e-05, -0.0005917363450862467, 0.0024585374630987644, -0.0010333694517612457, -0.0002585737966001034, 0.000605070439632982, -0.00043735941289924085, -0.0016176799545064569, -0.0009246804984286427, -0.0010696783429011703, -0.0003999535401817411, -5.117544060340151e-05, 0.0003500449238345027, 0.0005096220411360264, -0.0006745335413143039, 0.00039805276901461184, 0.0004977314383722842, 0.00037247955333441496, -0.0005168460193090141, -0.00012635390157811344, -0.00018315069610252976, -0.0006720914389006793, 0.0006850084173493087, -0.0002518560504540801, -0.0004287923511583358, 8.678547601448372e-05, -9.428296471014619e-05, -0.00018652495054993778, -0.000506805197801441, 0.000850544311106205, -0.00020120818226132542, -0.000366375723388046, -7.84439907874912e-05, -0.00010228635801468045, -0.00035062097595073283, -0.000842008157633245, 0.0004496450419537723, -0.0008127413457259536, -0.0007970673614181578, -0.00037548778345808387, -0.0007734630489721894, 0.0006021270528435707, -1.4882378309266642e-05, -0.0011485477443784475, -0.0005335246096365154, -0.0003348184109199792, 0.0006849379860796034, -4.4680305109068286e-06, -0.00023845759278628975, 0.001892073079943657, 0.00021972770628053695, -0.00012503615289460868, -0.002158789662644267, 0.0006209799321368337, 0.00013117979688104242, 0.00016957111074589193, -0.0004739666765090078, 0.00038957037031650543, 0.000512600177899003, 0.00022376705601345748, -0.0004908886039629579, -0.0004317517450544983, 4.012763383798301e-05, -0.000845699047204107, -0.00019365816842764616, 7.047987764963182e-06, -0.0013928243424743414, -0.0011541839921846986, 0.0006981338374316692, 0.0009461722802370787, -0.0007182512781582773, -0.0001752699026837945, 0.00014285587531048805, 0.0013228649040684104, 8.985998283606023e-05, -0.00012822941062040627, -5.08814919157885e-05, -0.0001833322166930884, -0.00030405368306674063, 0.00014800977078266442, 0.0013525552349165082], [0.12009257078170776, 0.0596046969294548, -0.01187166664749384, 0.04698578268289566, 0.01036726962774992, -0.04582446441054344, 0.01955048181116581, 0.027453094720840454, -0.011894654482603073, -0.05100079998373985, -0.0905633345246315, 0.07817015051841736, 0.08528780937194824, -0.09315600246191025, -0.07269897311925888, 0.04383489117026329, 0.08368416875600815, -0.17018644511699677, -0.06009881570935249, 0.09019967168569565, 0.08097612112760544, 0.018878597766160965, 0.0505274198949337, -0.06799963116645813, 0.05018751323223114, -0.022417908534407616, 0.055352453142404556, 0.12435591965913773, 0.01413233857601881, 0.08505423367023468, -0.0175076462328434, -0.010584009811282158, -0.07154718041419983, 0.07613589614629745, 0.08505698293447495, -0.004750219639390707, 0.06690490245819092, -0.05452895909547806, -0.012437153607606888, 0.0077691832557320595, 0.05352584645152092, -0.05322926491498947, 0.008441387675702572, -0.016466228291392326, -0.052450962364673615, -0.07552804052829742, -0.10067689418792725, -0.0658239796757698, -0.002162416698411107, 0.04648536816239357, 0.03285090625286102, -0.01922079175710678, 0.06143316626548767, 0.016917254775762558, 0.11001619696617126, 0.0555291473865509, 0.037037599831819534, -0.07126600295305252, 0.045354392379522324, 0.0022004186175763607, 0.027651134878396988, -0.0003419783024583012, 0.0705779641866684, 0.02554190345108509, -0.025186460465192795, -0.0196163859218359, -0.09282179921865463, 0.06775254011154175, -0.020247183740139008, -0.02983885630965233, -0.02275485172867775, 0.07942214608192444, -0.025520578026771545, -0.11261086910963058, -0.007685136515647173, 0.040382616221904755, 0.07380951941013336, 0.03145208954811096, 0.02549288608133793, 0.014077682979404926, -0.07895490527153015, -0.006346846465021372, 0.058732494711875916, -0.09724637866020203, -0.12322323769330978, -0.12595973908901215, -0.05369625240564346, 0.034318957477808, -0.10491140186786652, 0.03074946254491806, -0.06570258736610413, -0.18302568793296814, 0.0026887657586485147, 0.04182761535048485, 0.0577840730547905, -0.08791913837194443], [0.11081915348768234, 0.061031367629766464, -0.060751110315322876, -0.0010955159086734056, 0.03831672668457031, 0.009833381511271, -0.08123253285884857, 0.006254914216697216, 0.014663493260741234, 0.03082205168902874, -0.04282157123088837, -0.0029236061964184046, 0.038169946521520615, 0.02029990591108799, -0.029503708705306053, -0.008013484068214893, 0.04316004365682602, 0.03670218214392662, 0.05103336274623871, 0.010995843447744846, 0.038596075028181076, -0.044924549758434296, 0.022418063133955002, 0.056505534797906876, -0.012122138403356075, -0.10467042773962021, 0.010000771842896938, 0.03569820895791054, 0.051892686635255814, -0.06169292330741882, -0.08015560358762741, -0.07437052577733994, -0.048037149012088776, -0.043675344437360764, 0.0786430612206459, 0.06553960591554642, 0.06700043380260468, 0.05519866198301315, -0.0117055494338274, -0.016217529773712158, 0.044669587165117264, -0.06631668657064438, -0.02124102972447872, -0.015378194861114025, -0.12961408495903015, -0.04283081367611885, -0.031479042023420334, -0.018244147300720215, 0.08518106490373611, 0.020303107798099518, 0.04571421816945076, -0.041666172444820404, 0.05204055458307266, -0.029311994090676308, 0.04111609607934952, -0.03357061743736267, -0.056553684175014496, 0.05068867653608322, 0.07145033776760101, 0.015575897879898548, -0.03739574924111366, 0.07042135298252106, 0.055135004222393036, -0.043848667293787, -0.10402021557092667, -0.09345566481351852, -0.04548199847340584, 0.024522878229618073, -0.06327562779188156, -0.0017580697312951088, -0.0018293479224666953, 0.07997865974903107, -0.045009247958660126, -0.04844542592763901, 0.05766766518354416, -0.04307237267494202, -0.060894113034009933, 0.07502909004688263, 0.07751331478357315, -0.019564997404813766, 0.047383952885866165, 0.06054165959358215, 0.06646755337715149, 0.03610172122716904, 0.011406327597796917, -0.08405788242816925, -0.020469319075345993, -0.0366438589990139, -0.04796219617128372, 0.05113426223397255, -0.01422260794788599, -0.007998323999345303, -0.04407993331551552, -0.10870114713907242, -0.027462124824523926, -0.027207164093852043], [-0.023005740717053413, -0.028549805283546448, 0.03981373831629753, -0.0344129242002964, -0.03794913738965988, 0.0018035756656900048, -0.031603191047906876, 0.01942155696451664, -0.03149920701980591, 0.044442348182201385, -0.008344555273652077, -0.009470591321587563, -0.029363024979829788, 0.002318629529327154, -0.011402041651308537, -0.03151974081993103, -0.013989593833684921, 0.041975561529397964, 0.035670552402734756, -0.021374782547354698, -0.04746146500110626, -0.039026301354169846, 0.0280794408172369, 0.05692245438694954, 0.04879599064588547, -0.023002909496426582, 0.039150699973106384, 0.022774571552872658, 0.03843875601887703, -0.015576242469251156, -0.008930644020438194, -0.011310734786093235, 0.031536128371953964, 0.021046863868832588, 0.023164497688412666, 0.08476929366588593, 0.0394681952893734, 0.033039409667253494, -0.010334049351513386, -0.0040542506612837315, 0.02953207492828369, 0.014476500451564789, -0.02637891098856926, -0.00205644266679883, -0.013248831033706665, 0.030722016468644142, 0.0004050224961247295, -0.01621697098016739, 0.005844817962497473, 0.05533280968666077, 0.004269368946552277, 0.006959935184568167, -0.03412356600165367, -0.02710459567606449, -0.0027327423449605703, -0.013606110587716103, 0.050557978451251984, -0.0067213852889835835, 0.015080142766237259, -0.036651611328125, 0.0026008659042418003, 0.01258804090321064, -0.03405413031578064, -0.007110385689884424, 0.020725874230265617, -0.04583923891186714, -0.01400713250041008, -0.022905822843313217, -0.04015948995947838, 0.0233701691031456, -0.012324816547334194, 0.021069122478365898, -0.005421444308012724, 0.004859996028244495, 0.020417023450136185, 0.012310123071074486, 0.032532691955566406, 0.031192200258374214, -0.011723258532583714, 0.01953231915831566, -0.03756497800350189, -0.01930159516632557, 0.021534346044063568, 0.010222898796200752, 0.02643158659338951, -0.050283532589673996, 0.016275152564048767, -0.030145255848765373, 0.03994037210941315, 0.01845664717257023, 0.06672472506761551, 0.0278544370085001, -0.02867598459124565, -0.02443300001323223, 0.025972994044423103, 0.027206433936953545], [0.11163625121116638, -0.0801306888461113, -0.02149556763470173, 0.05922267585992813, -0.007545615546405315, -0.01797952502965927, -0.029062874615192413, 0.05096549540758133, -0.017570922151207924, -0.049427248537540436, 0.08590799570083618, 0.02705685980618, -0.04627862945199013, 0.12931828200817108, 0.0250113345682621, 0.01991669274866581, 0.08074874430894852, -0.08544010668992996, -0.06996341049671173, -0.004948312416672707, -0.09475406259298325, -0.08363666385412216, -0.07860364764928818, -0.053138986229896545, 0.013016439974308014, 0.04588959366083145, 0.031958408653736115, -0.019800536334514618, 0.0023342410568147898, 0.06557030975818634, 0.03676867112517357, -0.07113721966743469, -0.06755892932415009, -0.09292597323656082, 0.022864382714033127, -0.0074552795849740505, -0.06098056584596634, -0.008185687474906445, 0.04378734529018402, 0.004461649805307388, 0.01551301870495081, -0.026399599388241768, -0.05557006224989891, -0.010763314552605152, 0.036662522703409195, 0.0399826318025589, -0.04319785535335541, -0.011656982824206352, -0.040699586272239685, -0.09338615834712982, -0.04209808260202408, 0.006439398508518934, -0.05802489444613457, -0.03053174912929535, -0.04929962009191513, 0.013675757683813572, -0.06947361677885056, 0.1069333627820015, 0.03984729200601578, -0.0675273984670639, 0.07403627038002014, -0.09139852970838547, -0.054366614669561386, 0.027563998475670815, 0.03874655067920685, 0.07138292491436005, -0.0013498696498572826, -0.011707568541169167, 0.013135484419763088, 0.09251280128955841, -0.02402605302631855, -0.09280884265899658, 0.12661078572273254, 0.0721302479505539, 0.017077913507819176, 0.06235531345009804, -0.04920120909810066, -0.012141498737037182, -0.0032725203782320023, 0.00211871275678277, -0.054258737713098526, 0.0008445467101410031, 0.07193402200937271, 0.057233892381191254, 0.060550395399332047, 0.05571915581822395, -0.022915324196219444, 0.08779233694076538, -0.004661638755351305, -0.04019344225525856, -0.08392661064863205, -0.03941288962960243, -0.09261520206928253, -0.0060345777310431, 0.0370369479060173, -0.025204334408044815], [-0.01619689166545868, -0.02313053607940674, -0.007982556708157063, 0.02732827700674534, 0.0061960965394973755, 0.010802505537867546, 0.020860930904746056, 0.04542655497789383, -0.007834947668015957, 0.06408921629190445, -0.005107235629111528, 0.049010977149009705, 0.017975235357880592, -0.005438115447759628, 9.158757166005671e-05, -0.0045393044129014015, -0.03682245686650276, -0.006274637300521135, -0.023418959230184555, -0.023836765438318253, 0.005529152695089579, 0.03688708692789078, -0.0015878052217885852, 0.008308293297886848, -0.021555813029408455, -0.03712831810116768, 0.008902427740395069, 0.015496037900447845, 0.03496202081441879, -0.028091860935091972, 0.008371023461222649, 0.00029625484603457153, -0.017659859731793404, -0.0008117019315250218, 0.004540966358035803, 0.008132824674248695, -0.04078692942857742, -0.0065240939147770405, -0.011644648388028145, -0.02289474941790104, -0.007270903792232275, -0.007954344153404236, 0.03827013447880745, -0.008773667737841606, 0.036003999412059784, 0.008650914765894413, -0.025047240778803825, 0.03255186229944229, 0.02817430905997753, -0.0015141326002776623, 0.0002525128365959972, 0.029221268370747566, -0.01672193594276905, -0.004234234802424908, 0.007553703151643276, 0.032227449119091034, -0.0059705632738769054, -0.04225785285234451, -0.042715124785900116, -0.031601663678884506, -0.04221510514616966, 0.029689770191907883, 0.0027642040513455868, 0.007225913926959038, 0.08454299718141556, 0.02804543823003769, 0.028286313638091087, 0.030534155666828156, 0.009737999178469181, -0.00473762908950448, 0.027929609641432762, 0.021290156990289688, -0.011049272492527962, 0.02186603657901287, -0.00809898879379034, 0.00942263938486576, -0.004054858349263668, -0.023727230727672577, -0.01969660446047783, -0.008477984927594662, 0.014110886491835117, -0.035213958472013474, -0.020345604047179222, 0.0012970201205462217, 0.029372191056609154, 0.005204984452575445, 0.017984991893172264, 0.0177443940192461, 0.026446789503097534, -0.0035242538433521986, 0.014614601619541645, -0.009088451974093914, -0.016714561730623245, -0.0020913565531373024, 0.024431509897112846, -0.008829177357256413], [0.020092766731977463, 0.049707043915987015, 0.0775398313999176, 0.03811762481927872, 0.008992358110845089, 0.07219432294368744, -0.09851178526878357, 0.08789987117052078, 0.0764947310090065, 0.04540089890360832, -0.10551446676254272, 0.0918819010257721, 0.05389680340886116, -0.0027227376122027636, 0.01110832765698433, 0.01548753958195448, 0.06506334990262985, -0.15154048800468445, 0.020231734961271286, 0.03006371296942234, 0.08283650875091553, -0.04532627388834953, -0.04909922555088997, -0.020195992663502693, 0.0075652203522622585, -0.001914328895509243, -0.037188079208135605, 0.09681200236082077, -0.0021837668027728796, 0.035151269286870956, -0.07503187656402588, -0.04818796366453171, -0.012946183793246746, 0.09986945241689682, 0.12433823943138123, 0.0470413938164711, -0.05682368576526642, -0.06013842299580574, -0.07683461159467697, 0.0769471749663353, 0.009121990762650967, 0.0020151257049292326, 0.03453157842159271, 0.07194176316261292, 0.034496475011110306, -0.01951056532561779, -0.07315537333488464, -0.09152762591838837, -0.08390041440725327, 0.03999035060405731, -0.03134879097342491, -0.12917250394821167, -0.027932608500123024, -0.04560016840696335, 0.08329065144062042, -0.0482209287583828, 0.08499245345592499, -0.029057011008262634, 0.06852785497903824, 0.06646059453487396, -0.0392703041434288, -0.017064234241843224, 0.012698588892817497, 0.03869447484612465, -0.0903494581580162, 0.06825844198465347, 0.03655407577753067, 0.023459890857338905, 0.0346425399184227, -0.04203023761510849, 0.00944367703050375, -0.009370959363877773, -0.04525768384337425, -0.009220215491950512, -0.03593328222632408, -0.04325062781572342, 0.06592348217964172, -0.018067017197608948, 0.005069396458566189, -0.0011644235346466303, -0.05858983099460602, 0.01028876006603241, -0.013147065415978432, 0.03405255824327469, -0.13866300880908966, 0.007367547135800123, 0.003988711629062891, 0.06746947765350342, 0.03662991151213646, -0.05679130554199219, -0.008798664435744286, -0.1193624809384346, -0.04471778869628906, -0.033556096255779266, 0.06095350906252861, -0.05314433202147484], [0.011702580377459526, 0.029817091301083565, -0.04985319823026657, -0.022503670305013657, 0.027671903371810913, -0.0387919545173645, -0.02655022032558918, 0.033777907490730286, -0.03578188270330429, -0.032020580023527145, -0.011432884261012077, -0.03066106326878071, 0.003424951108172536, 0.036319851875305176, 0.006212345790117979, 0.038105569779872894, 0.022229425609111786, -0.037771254777908325, -0.005451113451272249, 0.0615873709321022, 0.007253324147313833, -9.779239189811051e-05, 0.07458971440792084, 0.04420434683561325, 0.05766424164175987, -0.05305936187505722, 0.04392841085791588, 0.049219436943531036, 0.020227566361427307, 0.07999604195356369, 0.010166389867663383, -0.001645636628381908, -0.01758420467376709, 0.0848437249660492, 0.07587077468633652, 0.06357768177986145, 0.035979412496089935, -0.025130679830908775, -0.0490909107029438, 0.03169655799865723, 0.02626100741326809, -0.04864754155278206, 0.061692241579294205, 0.047225289046764374, -0.05860016494989395, 0.027447141706943512, 0.0471215583384037, -0.05106940120458603, 0.02432544343173504, 0.07577811181545258, 0.0019248425960540771, 0.010132729075849056, 0.05305344611406326, 0.054248396307229996, -0.004515510518103838, 0.04608139768242836, 0.017775556072592735, -0.016157524660229683, 0.10096973925828934, -0.012160386890172958, -0.03960913047194481, 0.04036756232380867, -0.023858552798628807, -0.02352181077003479, -0.036736562848091125, -0.013728324323892593, -0.041389964520931244, 0.07812125235795975, -0.04035884886980057, -0.016642071306705475, -0.046487294137477875, 0.008906267583370209, 0.0033787013962864876, -0.024049481377005577, 0.015623374842107296, 0.0052505917847156525, -0.05021714046597481, 0.004154978785663843, 0.018401166424155235, 0.047699835151433945, 0.002725322963669896, 0.04192913696169853, 0.05688789114356041, -0.010335827246308327, 0.05375774949789047, -0.0068696727976202965, 0.013737832196056843, 0.03041832149028778, -0.06245066598057747, -0.005077425390481949, 0.06063428521156311, -0.0001607683370821178, 0.01418482419103384, 0.05638564005494118, 0.041674092411994934, -0.033658962696790695], [0.023957306519150734, -0.02572835050523281, -0.014980010688304901, 0.028072506189346313, -0.03804768621921539, -0.0015767824370414019, 0.0062085301615297794, -0.0251422468572855, 0.06048306077718735, -0.018380889669060707, 0.024226246401667595, -0.018892941996455193, 0.004448134917765856, 0.034371111541986465, 0.00038584586582146585, -0.01023153867572546, 0.07095437496900558, -0.00011027893924620003, -0.008450484834611416, -0.01412731222808361, -0.00043853631359525025, -0.0197322778403759, -0.02334410324692726, -0.00675198994576931, 0.015400919131934643, 0.03896712884306908, 0.004157024901360273, -0.011359235271811485, 0.005469761788845062, -0.011146411299705505, 0.015559670515358448, 0.014849360100924969, -0.004986278247088194, -0.012077735736966133, -0.034839730709791183, -0.024708019569516182, -0.026194658130407333, 0.011222541332244873, 0.024379918351769447, 0.012118814513087273, 0.00947547797113657, 0.009035375900566578, -0.015905139967799187, 0.00014523496793117374, -0.003916028421372175, -0.009019214659929276, 0.020638713613152504, -0.0010783461621031165, -0.026892129331827164, -0.016060596331954002, -0.025063954293727875, -0.010410994291305542, -0.037201303988695145, -0.02136516198515892, -0.011052490212023258, -0.01819538325071335, -0.013258330523967743, 0.08256063610315323, 0.03581121936440468, -0.02503230795264244, 0.06638277322053909, -0.024663325399160385, 0.0029205940663814545, -0.01315590925514698, -0.00854091439396143, -0.004055854864418507, -0.00044119200902059674, -0.05544746667146683, 0.004389433655887842, 0.021250391378998756, -0.009269053116440773, -0.01877024956047535, 0.03662753105163574, 0.0038395309820771217, -0.0004504605894908309, -0.0022241082042455673, 0.002300589345395565, 0.035617437213659286, -0.022324852645397186, -0.006809951271861792, -0.011077356524765491, -0.01515729445964098, 0.04351821914315224, 0.0031678075902163982, 0.0070940288715064526, 0.002377535915002227, -0.011436101980507374, 0.00931558571755886, 0.04421066865324974, -0.021192362532019615, -0.02278101071715355, -0.019270211458206177, -0.015283772721886635, -0.0026222311425954103, -0.029736371710896492, 0.026157189160585403], [0.0027784749399870634, -0.029326803982257843, 0.023116691038012505, -0.000828695367090404, -0.04108678549528122, 0.0021973117254674435, -0.008692607283592224, -0.009473803453147411, -0.017278164625167847, -0.015428667888045311, 0.006650838535279036, 0.008133635856211185, 0.007003891281783581, -0.0023642864543944597, 0.0054214950650930405, -0.0021471790969371796, -0.010590626858174801, 0.0032446084078401327, -0.015935389325022697, -0.005339744500815868, 0.004066170193254948, 0.005213316064327955, 0.003590242937207222, -0.00683473190292716, 0.01405546348541975, -0.010798976756632328, -0.0031550179701298475, 0.006339567713439465, 0.02492017298936844, 0.014668387360870838, -0.002664433792233467, 0.0070081246085464954, -0.0037690226454287767, -0.0016468004323542118, 0.0032338693272322416, 0.0017760241171345115, -0.00025246888981200755, -0.005437969695776701, -0.0043545919470489025, 0.009975576773285866, -0.0030833107884973288, 0.002681220415979624, -0.02118007279932499, -0.002524352166801691, -0.004465124104171991, 0.015944916754961014, 0.017224259674549103, -0.004680323414504528, -0.006967634428292513, -0.0336310938000679, 0.01943708211183548, -0.00955180823802948, 0.022436926141381264, -0.01944391243159771, -0.009356522932648659, 0.00878931488841772, -0.04288257285952568, -0.001186650013551116, -0.020521065220236778, 0.016853563487529755, -0.010724718682467937, -0.01820276491343975, -0.01369865145534277, 0.013754202052950859, 0.026818398386240005, 0.012574245221912861, 0.00425372552126646, -0.026977600529789925, 0.01965293101966381, -0.0033736953046172857, 0.0056347595527768135, -0.02158968709409237, -0.0011238931911066175, 0.0022094082087278366, -0.0009422464645467699, 0.001292017986997962, -0.033748809248209, 0.005922758020460606, -0.030141472816467285, -0.00470191752538085, 0.0018291013548150659, 0.0006484412588179111, 0.0061082998290658, -0.0024605996441096067, 0.02512482926249504, 0.023895522579550743, -0.024321435019373894, -0.003432679222896695, 0.024679210036993027, -0.00105499557685107, 0.024549532681703568, -0.005743805319070816, -0.019538741558790207, -0.002549251075834036, -0.019903836771845818, -0.012858129106462002], [0.027208052575588226, -0.014598000794649124, -0.009109276346862316, -0.023913376033306122, 0.03084743395447731, 0.01287020742893219, -0.02215752750635147, -0.027097757905721664, 0.005910390522330999, 0.01220790110528469, -0.02239684946835041, 0.0010493004228919744, -6.067981667001732e-05, -0.029253650456666946, 0.0010983843822032213, -0.006887936033308506, 0.01048053428530693, -0.05278262123465538, 0.0017779230838641524, 0.0020526591688394547, 0.02496493235230446, -0.024051381275057793, 0.017481978982686996, -0.012136192992329597, -0.009364821016788483, -0.013250082731246948, 0.007728480268269777, 0.01216013915836811, -0.03132607415318489, -0.0023448197171092033, -0.025085091590881348, -0.013490019366145134, -0.0018882978474721313, -0.005628272425383329, 0.016321847215294838, -0.012173132039606571, -0.02116743102669716, 0.005961592774838209, -0.010407989844679832, 0.014342871494591236, -0.007409331388771534, -0.011450131423771381, 0.010090730153024197, 0.0036137732677161694, -0.02184157446026802, -0.005523816216737032, -0.007350885774940252, -0.012867464683949947, -0.004964327439665794, 0.029111724346876144, -0.0039018758106976748, -0.019155338406562805, 0.01853727176785469, 0.02483505941927433, 0.0006766150472685695, -0.001119751832447946, -0.012945093214511871, 0.009021037258207798, 0.025882383808493614, -0.00248277117498219, -0.006533487234264612, 0.011422179639339447, -0.008278567343950272, 0.002424769802019, -0.03689432889223099, -0.0056435721926391125, -0.01856413297355175, 0.027859287336468697, -0.014368549920618534, -0.009353475645184517, -0.012600420042872429, -0.002882904140278697, -0.007105348631739616, -0.012460301630198956, -0.006493082735687494, -0.003762808395549655, 0.006278324406594038, 0.014804346486926079, -4.3794847442768514e-05, 0.009163787588477135, -0.002527483506128192, 0.03334338963031769, 0.013987857848405838, 0.01181182824075222, -0.013119220733642578, 0.013590272516012192, -0.006374623626470566, -0.001130906050093472, -0.04367224499583244, 0.015388124622404575, -0.010870556347072124, 0.003692715195938945, -0.004294662736356258, -0.016928570345044136, -0.0051686204969882965, 0.0018176265293732285], [0.007495076861232519, -0.0008823216194286942, -0.0018572455737739801, -0.004023848567157984, 0.007503625936806202, 0.005202183034271002, -0.0029423239175230265, 0.00039645290235057473, -0.00034554366720840335, -0.011392481625080109, -0.002846971619874239, -9.649671710576513e-07, -0.002075143624097109, -0.0024715198669582605, -0.0004675623495131731, -0.0018529241206124425, 0.004394314717501402, -0.012915062718093395, 0.003116458188742399, 0.009013202972710133, -0.0037169167771935463, 0.0037330803461372852, 0.006729677319526672, 0.006535147782415152, -0.006400055252015591, -0.001082063652575016, 0.0006891650264151394, 0.005670010112226009, -0.01757129281759262, -0.003533314447849989, -0.0050025214441120625, -0.003416699357330799, -0.00040732070920057595, 0.01135160494595766, -0.007247636094689369, -0.0011220655869692564, 0.004880961962044239, -0.00048299640184268355, -0.0010574264451861382, 0.0006350446492433548, 0.0026445144321769476, 0.0019828504882752895, -0.003830535802990198, 0.004586406052112579, -0.00707349693402648, 0.0021379305981099606, -0.003738773986697197, -0.0037003967445343733, 0.0054141622968018055, 0.018650555983185768, 0.006043714005500078, -0.0069114649668335915, 0.00972437672317028, 0.006071241106837988, 0.000408850988605991, -0.009315842762589455, -0.014994148164987564, -0.002666008658707142, 0.0125540466979146, 0.0015238604974001646, -5.0296890549361706e-05, 0.003495581680908799, 0.0004630188341252506, -0.004476375877857208, -0.008783168159425259, -0.01078631542623043, -0.003348595928400755, 0.008101759478449821, -0.0019300025887787342, -0.003365438897162676, -0.004158127121627331, 0.010054097510874271, -0.002376577816903591, 0.0009300232050009072, 0.0013321810401976109, -0.0007231311756186187, -0.003843865590170026, 0.0008128329063765705, 0.011686848476529121, 0.0016550483414903283, -0.0007917959010228515, 0.01075882650911808, 0.002115000272169709, -0.00018827946041710675, -0.00928618386387825, -0.0012390804477036, -0.006817936431616545, -0.0011639982694759965, -0.016657844185829163, -0.0012332431506365538, 0.0005634607514366508, -0.0011520544067025185, 0.004781270399689674, 0.0026010579895228148, 0.006538895890116692, -0.0021081827580928802], [0.10682791471481323, 0.07452898472547531, -0.0426461398601532, -0.020902881398797035, -0.008385442197322845, 0.06664766371250153, -0.018561149016022682, -0.02903587743639946, 0.04511011764407158, -0.00044278684072196484, -0.011178987100720406, -0.0012165679363533854, -0.00475676404312253, -0.030359437689185143, 0.0008965997840277851, 0.04373549297451973, 0.0014814328169450164, 0.01230589672923088, -0.024428434669971466, 0.009839528240263462, 0.08396782726049423, -0.0477178730070591, -0.053071197122335434, -0.04908815398812294, 0.026433102786540985, 0.010835723020136356, -0.013775280676782131, 0.05779936909675598, -0.07545498013496399, -0.029188549146056175, -0.01294877752661705, -0.06992261111736298, 0.028022732585668564, -0.049527358263731, 0.02322803996503353, 0.036071352660655975, 0.06134796887636185, -0.04481776803731918, -0.0535704679787159, 0.05784289166331291, 0.012703110463917255, -0.05916549637913704, 0.05227292701601982, 0.04446785897016525, -0.02023714780807495, -0.007618999574333429, 0.03521965444087982, 0.01677255891263485, -0.02578374557197094, 0.046603862196207047, 0.003403151873499155, -0.0705987960100174, 0.07853594422340393, 0.004129142500460148, -0.01703747920691967, 0.043259717524051666, -0.07076075673103333, 0.022041313350200653, 0.039926908910274506, -0.026624418795108795, -0.0013177021173760295, 0.05288246273994446, -0.023796020075678825, 0.0578472875058651, -0.10447201132774353, -0.048086978495121, -0.08474864810705185, 0.10052251070737839, 0.010335461236536503, -0.07111824303865433, -0.0196567103266716, 0.05863058194518089, -0.056493405252695084, -0.023526862263679504, -0.032603099942207336, -0.07037353515625, 0.03646737337112427, 0.028551051393151283, -0.03793264925479889, -0.03975198417901993, -0.035599980503320694, -0.006183667108416557, 0.043397027999162674, -0.000390303524909541, 0.0015240886714309454, -0.06558036804199219, -0.0610864982008934, -0.039531901478767395, -0.10631155222654343, -0.021966680884361267, -0.0015177797758951783, -0.018097534775733948, -0.051686983555555344, -0.01957690343260765, 0.06760981678962708, -0.03541608527302742], [-0.07098088413476944, 0.008603115566074848, -0.021495405584573746, 0.033642854541540146, -0.11153865605592728, 0.05721209943294525, -0.033611733466386795, -0.02820567600429058, -0.024847161024808884, 0.02232217974960804, -0.03753980994224548, -0.030651018023490906, 0.049500953406095505, -0.023888321593403816, 0.017672594636678696, 0.014084084890782833, 0.0505024679005146, 0.08016887307167053, -0.050611644983291626, 0.033855367451906204, 0.07434448599815369, -0.02221028320491314, -0.003868852974846959, -0.0031735652592033148, -0.022703202441334724, -0.03671569377183914, -0.025958193466067314, -0.032955218106508255, -0.04969656094908714, -0.06273042410612106, 0.030531330034136772, -0.016849394887685776, 0.01505585853010416, 0.02765495516359806, 0.005965514108538628, -0.04982703551650047, 0.06520900130271912, -0.06036169081926346, -0.011976960115134716, 0.03841102868318558, 0.06148720905184746, -0.016852494329214096, -0.10106590390205383, -0.027065379545092583, 0.03696161136031151, -0.07202571630477905, -0.045724399387836456, 0.01744871586561203, -0.005653498228639364, -0.026724545285105705, -0.029939914122223854, 0.03809760883450508, 0.01632782071828842, -0.08489500731229782, -0.0618889220058918, -0.0660383328795433, -0.047621455043554306, 0.009471524506807327, -0.0008693857816979289, -0.06682763248682022, -0.052694808691740036, -0.030869217589497566, -0.0014974393416196108, -0.02282315120100975, -0.019233539700508118, 0.029380258172750473, 0.0037953914143145084, -0.05056785047054291, 0.05423988401889801, 0.04864226281642914, 0.06304222345352173, -0.048674289137125015, -0.05125448480248451, 0.0608995147049427, -0.03613898158073425, -0.05135561153292656, -0.018729139119386673, -0.07837305963039398, -0.0001920042122947052, -0.017124926671385765, 0.02181793376803398, -0.020687032490968704, -0.07333822548389435, 0.029179250821471214, -0.01707148551940918, -0.015601417981088161, -0.09331876039505005, 0.007186426315456629, 0.08073824644088745, -0.06335928291082382, -0.04690370336174965, -0.023634443059563637, -0.10063900053501129, -0.014088925905525684, -0.07565736025571823, -0.006081781815737486], [-0.06150560826063156, -0.13240297138690948, -0.03526536002755165, -0.02865028567612171, -0.038604944944381714, -0.06958147883415222, -0.0025440643075853586, 0.014665137976408005, 0.0026009269058704376, -0.06771000474691391, -0.031887881457805634, 0.0019193297484889627, 0.020803799852728844, -0.05517053231596947, 0.020234404131770134, -0.033485088497400284, -0.05805826932191849, 0.09182294458150864, -0.03716515004634857, -0.0029080864042043686, 0.02569170854985714, -0.013340136036276817, -0.06590217351913452, -0.004627021960914135, 0.0024662211071699858, -0.13330508768558502, 0.019081156700849533, -0.0007946940022520721, -0.0539499893784523, -0.10199687629938126, -0.04365399479866028, 0.03636534884572029, 0.020163370296359062, 0.020306196063756943, 0.0609714649617672, -0.03311982750892639, -0.03581327572464943, -0.05366745591163635, -0.09141985327005386, -0.07438529282808304, -0.023069672286510468, -0.031392257660627365, -0.0790618434548378, -0.008460323326289654, -0.027756469324231148, -0.011805366724729538, 0.016481656581163406, -0.023369964212179184, 0.01290972251445055, -0.07752615958452225, -0.032614897936582565, -0.0334354005753994, 0.08679257333278656, -0.05281687155365944, 0.018865810707211494, -0.033487461507320404, -0.06879629194736481, -0.054497916251420975, -0.09126702696084976, 0.02628268301486969, -0.09514923393726349, 0.003578695235773921, -0.004112196620553732, -0.029008937999606133, -0.03560013696551323, -0.005390586331486702, -0.002865320071578026, -0.018685968592762947, 0.048973727971315384, -0.03859923034906387, -0.015285821631550789, -0.03156084194779396, -0.040440633893013, 0.004193790256977081, -0.010649492964148521, -0.07310279458761215, -0.013815504498779774, -0.07426220178604126, -0.0636071041226387, -0.0057090711779892445, -0.05977316200733185, -0.02789130061864853, -0.05929672718048096, -0.05259273201227188, -0.005823457147926092, -0.044255081564188004, -0.020644085481762886, -0.008882373571395874, -0.07723002880811691, 0.0062337601557374, -0.01583895832300186, -0.03863344341516495, -0.05418439954519272, -0.022639557719230652, 0.0025355613324791193, -0.029205773025751114], [0.12055424600839615, -0.1300874501466751, -0.04280192032456398, 0.024162422865629196, -0.056035879999399185, 0.04208741337060928, -0.07330746948719025, -0.034363728016614914, -0.02440492995083332, -0.10185036063194275, 0.0678320899605751, -0.03930572792887688, 0.030292151495814323, 0.07965438812971115, 0.05512680485844612, -0.04033701494336128, 0.05882919207215309, -0.06112056225538254, -0.03006146475672722, -0.028708305209875107, -0.004288416355848312, -0.04468118026852608, 0.006784994620829821, 0.018981410190463066, 0.057979945093393326, 0.04794261232018471, -0.0025204250123351812, -0.011711309663951397, -0.044152453541755676, -0.07289906591176987, -0.013675937429070473, -0.02166646160185337, 0.06424630433320999, -0.01648731902241707, -0.053912702947854996, 0.010633734986186028, -0.051307398825883865, -0.01734364591538906, 0.06750956177711487, 0.08930782973766327, 0.07748779654502869, -0.023820910602808, -0.03020627237856388, -0.01868841052055359, -0.01893089897930622, 0.03557843342423439, 0.0039669242687523365, -0.056670114398002625, -0.12274323403835297, -0.04243406653404236, -0.00877821259200573, -0.06771258264780045, -0.09121908247470856, -0.050975505262613297, -0.04012688994407654, -0.00705722626298666, -0.051341913640499115, 0.11015107482671738, 0.07915150374174118, -0.028063379228115082, 0.15189102292060852, -0.11999616771936417, 0.032996851950883865, 0.019498303532600403, -0.04300977662205696, 0.005004905629903078, 0.07873435318470001, -0.11865892261266708, -0.04264760762453079, 0.08568058162927628, -0.02493930421769619, -0.049127452075481415, 0.11501085758209229, 0.011649702675640583, 0.05567195266485214, 0.030203990638256073, -0.0757984146475792, 0.0830233171582222, 0.014917433261871338, 0.021920980885624886, -0.05056207999587059, -0.07909087836742401, -0.05323069170117378, 0.0027418681420385838, 0.037907712161540985, 0.0079789524897933, -0.06962288171052933, 0.025080278515815735, -0.03441295400261879, -0.0197580736130476, -0.0821828618645668, -0.057547494769096375, 0.025997500866651535, -0.03116157092154026, 0.024009326472878456, 0.043872199952602386], [0.06913626939058304, -0.0016576714115217328, 0.011640254408121109, 0.059076469391584396, -0.08651459217071533, 0.02728806994855404, 0.036305539309978485, 0.03283723443746567, 0.06376023590564728, -0.048159461468458176, 0.047217752784490585, -0.008547579869627953, -0.0013832591939717531, 0.06400369107723236, 0.020630834624171257, 0.0008028147858567536, 0.1163409873843193, -0.021972419694066048, -0.023056183010339737, -0.043948616832494736, 0.0034311360213905573, -0.027998948469758034, 0.008591746911406517, -0.023077797144651413, -0.05260895565152168, 0.11584673821926117, -0.026560908183455467, 0.0006203189841471612, -0.018028004094958305, 0.04884760081768036, 0.024605678394436836, -0.02606496587395668, 0.014619981870055199, -0.05586830899119377, -0.015374117530882359, -0.022294938564300537, -0.07993666082620621, 0.005237567238509655, 0.06956543773412704, 0.009610121138393879, -0.00025974787422455847, -0.023519180715084076, 0.011437749490141869, -0.04427896440029144, 0.02530485764145851, 0.014902683906257153, -0.030257681384682655, 0.007088782265782356, -0.02758878655731678, -0.06398291885852814, 6.786042831663508e-06, 0.0054756044410169125, -0.06272673606872559, -0.05569285526871681, -0.03916819393634796, -0.0031178344506770372, -0.07983576506376266, 0.0719236508011818, 0.01451529935002327, -0.009794548153877258, 0.108816958963871, -0.07473839819431305, 0.01110911462455988, -0.000938398006837815, 0.016481317579746246, 0.02396862581372261, 0.0030296279583126307, -0.020009124651551247, 0.0006701149977743626, 0.002534963423386216, -0.01315472461283207, -0.04404405131936073, 0.04392995685338974, 0.03523673489689827, -0.007556158117949963, -0.02441415935754776, -0.05178138613700867, 0.03859037533402443, -0.04220817610621452, -0.026031361892819405, -0.03172215446829796, -0.009136402048170567, 0.01277200598269701, 0.005300990771502256, 0.05920103192329407, -0.01293895859271288, 0.008193541318178177, 0.04593975469470024, -0.029107864946126938, -0.06720162183046341, -0.0375395342707634, -0.014142218977212906, -0.060775939375162125, -0.003376051550731063, -0.010788673534989357, -0.021303284913301468], [-0.007008628454059362, 0.014721192419528961, -0.014104494825005531, -0.0237097330391407, 0.032763708382844925, 0.007767908275127411, -0.031390756368637085, -0.011797353625297546, -0.035229988396167755, -0.0071416315622627735, -0.018694816157221794, -0.013283791020512581, -0.010896359570324421, -0.017175892367959023, 0.0009780795080587268, -0.003563891863450408, 1.2408054317347705e-05, 0.006251994986087084, 0.019604912027716637, 0.030111996456980705, -0.01597062312066555, 0.0033467644825577736, 4.145113780396059e-05, 0.0026658393908292055, 0.015298488549888134, -0.003364480333402753, -0.0065908231772482395, 0.0004067476256750524, -0.037333011627197266, -0.008617222309112549, -0.009563068859279156, -0.025368278846144676, -0.009505365043878555, -0.004042299464344978, 0.01228045579046011, 0.04323089122772217, -0.010750539600849152, 0.0009866190375760198, -0.002601307351142168, 0.026529500260949135, -0.009910274296998978, -0.02511392906308174, -0.0054956222884356976, 0.009576923213899136, -0.027938446030020714, 0.0014388285344466567, 0.0052393036894500256, -0.008820410817861557, -0.0018975381972268224, 0.004985290113836527, 0.014933677390217781, -0.0020510032773017883, 0.00025981274666264653, 0.015561304055154324, -0.0032139201648533344, 0.004400379024446011, -0.0019314567325636744, 0.0060961563140153885, 0.01691974326968193, -0.011128153651952744, -0.004948212765157223, -0.0065116980113089085, 0.001507950248196721, -0.004657264333218336, -0.043962039053440094, 0.00038733810652047396, -0.01788676157593727, 0.03971462324261665, -0.01724730059504509, 0.006010980810970068, -0.019783886149525642, 0.003643567906692624, -0.0011762457434087992, -0.022847149521112442, -0.0121523542329669, 0.0018763989210128784, -0.014712058007717133, -0.00044231125502847135, -0.01155031193047762, 0.0040302458219230175, 0.0017054303316399455, 0.03299272805452347, 0.02709323912858963, 0.004223484545946121, 0.010673858225345612, -0.0009057277929969132, 0.028932813555002213, -0.0049354457296431065, -0.020669549703598022, -0.015506126917898655, -0.027713406831026077, 0.03965124860405922, -0.01702728495001793, -0.010626410134136677, -0.0015613171271979809, 0.021301958709955215], [-0.10844756662845612, 0.06067918986082077, 0.04304869472980499, 0.0561470091342926, 0.03932666406035423, -0.03861364722251892, 0.07114043086767197, 0.07828156650066376, 0.020690347999334335, 0.07140307128429413, -0.02564881183207035, 0.057359278202056885, 0.04483887180685997, 0.05203242227435112, 0.02167944796383381, 0.07078488916158676, -0.11739666759967804, 0.058399539440870285, -0.03021608106791973, -0.08157035708427429, 0.08314564824104309, -0.011788161471486092, -0.04841654375195503, 0.012337882071733475, -0.04617970064282417, -0.07248206436634064, -0.04133618250489235, -0.01448386162519455, -0.00822154525667429, -0.07436969876289368, -0.04240645468235016, 0.05327365919947624, -0.0006684712134301662, -0.06996116042137146, 0.06122748926281929, -0.005508040077984333, -0.08495063334703445, -0.01043179165571928, 1.9101682482869364e-05, -0.08658862859010696, -0.004016920458525419, 0.049264900386333466, 0.07422642409801483, -0.00031204443075694144, 0.005680854897946119, 0.07957395911216736, 0.011662141419947147, 0.0637311339378357, -0.004441727884113789, -0.05925138294696808, 0.07163440436124802, 0.014243521727621555, 0.07630664110183716, 0.0238431878387928, 0.02766065113246441, -0.08352308720350266, 0.03242116421461105, 0.003089564386755228, -0.04750015214085579, 0.08028892427682877, -0.11864219605922699, -0.02615489438176155, -0.017991481348872185, 0.042293936014175415, 0.0008588721975684166, -0.02014632150530815, 0.053210482001304626, -0.038483742624521255, 0.0699210911989212, 0.026789382100105286, 0.021253151819109917, 0.048593029379844666, -0.004854235332459211, -0.020871954038739204, 0.034048065543174744, -0.07124875485897064, -0.027263374999165535, -0.04209316521883011, -0.04194464161992073, -0.023627135902643204, 0.03788873925805092, -0.09953857958316803, -0.0708981454372406, -0.047811172902584076, -0.030541278421878815, 0.027576573193073273, 0.01735113561153412, 0.021108048036694527, 0.017803333699703217, 0.06365522742271423, 0.039663199335336685, 0.028885113075375557, -0.018042132258415222, -0.005385455675423145, 0.072013720870018, -0.009781046770513058], [0.044125717133283615, 0.06559969484806061, -0.06823916733264923, -0.10438690334558487, -0.017036693170666695, 6.55978947179392e-05, -0.04696464166045189, 0.06682107597589493, 0.045756854116916656, 0.024959063157439232, -0.025008829310536385, -0.04752814769744873, 0.07773397117853165, -0.042604394257068634, -0.020300306379795074, 0.06805643439292908, -0.0028146596159785986, -0.10924340784549713, 0.07850077748298645, 0.05411024019122124, -0.023113075643777847, -0.004067701753228903, 0.0026207633782178164, -0.0012836393434554338, 0.004209163598716259, -0.08414768427610397, -0.0033434960059821606, 0.05863383784890175, -0.06653862446546555, 0.09191805869340897, -0.08879609405994415, 0.011851648800075054, -0.0747150182723999, 0.01669464446604252, -0.014227570965886116, -0.0034909292589873075, 0.0747128278017044, 0.047947488725185394, -0.035590995103120804, 0.08898386359214783, 0.053592126816511154, 0.062481530010700226, 0.00472997035831213, -0.02645065449178219, -0.009749244898557663, -0.03817533701658249, -0.03353332728147507, 0.013551893644034863, -0.07772231847047806, -0.0037435342092067003, -0.012692725285887718, -0.06115410476922989, 0.04632110521197319, -0.009803147986531258, 0.09070529043674469, 0.012414943426847458, 0.036304548382759094, -0.040817052125930786, 0.009221312589943409, 0.06917057186365128, -0.06033514812588692, -0.02243049629032612, 0.04687274247407913, 0.09133685380220413, -0.039243925362825394, 0.03749975189566612, -0.025067850947380066, -0.006507805548608303, -0.08348069339990616, 0.02581934630870819, 0.029250454157590866, 0.0520387701690197, 0.008528202772140503, -0.04003123566508293, -0.002372779417783022, -0.07992390543222427, -0.016066420823335648, 0.045991845428943634, 0.09013427793979645, -0.0053848023526370525, -0.011935554444789886, 0.0024168584495782852, -0.04897397384047508, -0.011099847964942455, 0.03183840587735176, -0.07972223311662674, -0.04624755680561066, 0.002581423381343484, -0.02231167070567608, -0.01575530506670475, -0.05452260002493858, -0.03815551474690437, 0.058051783591508865, -0.08487776666879654, -0.047881875187158585, 0.07242291420698166], [-0.009012634865939617, -0.044417932629585266, -0.05449409782886505, 0.0030288826674222946, -0.0223532784730196, -0.013491855002939701, -0.032528188079595566, 0.05614062398672104, 0.07310362160205841, 0.03434343636035919, -0.006324462126940489, -0.02392069436609745, -0.033603865653276443, 0.027161795645952225, -0.037493154406547546, -0.027684347704052925, -0.01573006622493267, -0.0284042339771986, -0.007737837731838226, -0.04115298017859459, 0.027740364894270897, -0.06581006944179535, -0.0019440322648733854, -0.001015807269141078, -0.02224458009004593, 0.01514526829123497, 0.08609730005264282, 0.06012021377682686, 0.05122800171375275, -0.02909822389483452, -0.03537121042609215, 0.022193502634763718, -0.02630843035876751, 0.08023074269294739, 0.0379827544093132, 0.0041387733072042465, 0.08790356665849686, 0.004484099801629782, -0.022995473816990852, 0.029266715049743652, 0.018701864406466484, 0.02305392175912857, 0.031204475089907646, -0.004001709166914225, -0.10156753659248352, 0.04622800648212433, 0.03565031290054321, -0.014034205116331577, -0.04470730572938919, -0.011315003037452698, 0.020797692239284515, -0.07107897102832794, 0.08331740647554398, -0.023273447528481483, -0.027933143079280853, 0.03790469840168953, 0.041397757828235626, -0.03501711040735245, 0.04180668294429779, 0.039178989827632904, -0.04071730002760887, 0.030090423300862312, -0.04621802642941475, 0.06511649489402771, -0.10602487623691559, -0.06916403770446777, -0.09257514029741287, 0.020322272554039955, -0.05644461512565613, -0.014302419498562813, -0.04176053777337074, 0.06696255505084991, 0.027820274233818054, 0.022506505250930786, 0.06586700677871704, -0.05654788389801979, -0.02603817544877529, -0.03854779526591301, 0.030468152835965157, 0.06031416729092598, -0.009300166741013527, 0.04706322401762009, 0.10741046071052551, 0.041665200144052505, -0.01827298477292061, -0.07821158319711685, 0.03726072236895561, 0.03501911088824272, -0.01579534076154232, 0.015762565657496452, 0.005250005517154932, 0.09242433309555054, 0.011228661052882671, -0.04488717392086983, 0.01655115932226181, 0.05409977212548256], [0.06769576668739319, -0.11463328450918198, 0.013281980529427528, 0.02740613929927349, -0.04750266298651695, 0.02240113727748394, 0.008369190618395805, -0.02314886450767517, 0.027639703825116158, -0.06318826973438263, 0.049008503556251526, -0.051128409802913666, -0.013569981791079044, 0.03836672380566597, 0.00973978079855442, -0.019399015232920647, 0.11063527315855026, -0.05724547058343887, -0.023359378799796104, -0.01904219575226307, 0.029893476516008377, -0.02524731308221817, -0.010009242221713066, -0.04230105131864548, -0.02960142306983471, 0.024494409561157227, -0.021087851375341415, 0.004286590963602066, -0.016600552946329117, -0.015234028920531273, 0.05003200098872185, -0.013605684041976929, -0.061727579683065414, -0.04083792492747307, 0.001997248502448201, 0.0023667828645557165, -0.03919795900583267, 0.002100906800478697, 0.046906791627407074, -0.037266552448272705, 0.018975507467985153, 0.011028854176402092, -0.0665864646434784, 0.010421259328722954, -0.022826924920082092, -0.011053504422307014, -0.005289717111736536, -0.024544544517993927, -0.06037892773747444, -0.04718230664730072, -0.06822449713945389, -0.03073696792125702, -0.08882049471139908, -0.051011670380830765, -0.02700969949364662, -0.015270089730620384, 0.0043217940255999565, 0.1142164021730423, 0.06892536580562592, -0.02862139791250229, 0.110399529337883, -0.053378570824861526, 0.012439126148819923, -0.0406266525387764, -0.020169612020254135, -0.0034661791287362576, 0.012964565306901932, 0.01480916328728199, -0.013659931719303131, 0.0393897220492363, -0.026827488094568253, -0.019673265516757965, 0.02117755264043808, 0.04478210583329201, -0.0513731874525547, 0.02381756901741028, -0.02755700796842575, 0.004530405160039663, -0.08048734813928604, 0.000814195373095572, -0.05535675585269928, -0.04354917258024216, -0.023369483649730682, 0.03305468335747719, -0.0481392927467823, 0.03594740852713585, -0.014584868215024471, 0.04773915559053421, 0.012797504663467407, 0.03345774859189987, -0.08973631262779236, -0.029059140011668205, -0.04854077473282814, -0.011548157781362534, -0.06732692569494247, 0.0013337571872398257], [0.0836486667394638, 0.002651430433616042, -0.03286685049533844, 0.051531098783016205, -0.012207857333123684, -0.05799729377031326, 0.016098150983452797, 0.0311502143740654, -0.016039101406931877, -0.057527925819158554, 0.038883522152900696, 0.011789371259510517, 0.0008177791605703533, 0.035293273627758026, 0.001967754913493991, 0.0117116067558527, 0.05591963231563568, 0.02883923612535, -0.016596263274550438, 0.001575507572852075, -0.024707118049263954, 0.027129488065838814, -0.06131548061966896, -0.036387912929058075, 0.006209313403815031, 0.10885899513959885, 0.0159907229244709, -0.03792489320039749, 0.02069214917719364, -0.04069594293832779, 0.019797921180725098, 0.0113996472209692, -0.04697422310709953, -0.03523369878530502, -0.02779087796807289, -0.05134992673993111, 0.020915502682328224, 0.017378581687808037, 0.032970357686281204, 0.008318793028593063, 0.011513851583003998, 0.0032881705556064844, 0.024861887097358704, -0.040670327842235565, 0.032502103596925735, -0.038735922425985336, -0.015788545832037926, -0.010737622156739235, -0.04604320973157883, 0.01647154428064823, 0.027567531913518906, 0.023962058126926422, -0.06848561018705368, -0.055311817675828934, -0.023989342153072357, 0.010616819374263287, -0.02536345273256302, 0.06804357469081879, 0.05095683038234711, -0.04418089985847473, 0.0788397267460823, -0.04208448901772499, -0.0062699387781322, -0.02665773406624794, -0.014490549452602863, 0.01694156974554062, 0.01481013186275959, 0.034638117998838425, -0.015558945946395397, 0.011296777985990047, -0.005141865462064743, -0.028879208490252495, 0.06589122116565704, 0.009194956161081791, 0.02044595032930374, 0.03107691928744316, 0.010423325933516026, 0.0321391336619854, -0.026258505880832672, -0.021549994125962257, -0.021288936957716942, -0.029754219576716423, 0.04090852290391922, 0.035117845982313156, 0.0493665374815464, 0.009501872584223747, -0.06224147602915764, 0.013351688161492348, 0.0794268548488617, 0.01797577738761902, -0.004435430280864239, -0.03078826144337654, 0.004032142926007509, 0.006750697735697031, -0.028713706880807877, -0.027470098808407784], [-0.06171330809593201, -0.0720616951584816, -0.0635683685541153, 0.03732885792851448, -0.08783471584320068, 0.02303694188594818, -0.038346268236637115, -0.00119120255112648, -0.04130934178829193, -0.04626169055700302, -0.014279168099164963, 0.009429632686078548, 0.018025606870651245, -0.050751943141222, 0.029038412496447563, -0.015961939468979836, -0.019451024010777473, 0.0010835675057023764, -0.0037481116596609354, -0.012852183543145657, 0.052667874842882156, -0.0007477349718101323, 0.04457288607954979, 0.0379369892179966, -0.023956716060638428, -0.07966435700654984, 0.0035657002590596676, -0.05658445134758949, -0.03690914809703827, -0.07119516283273697, -0.00664243008941412, -0.008111819624900818, -0.02967889979481697, 0.01129125151783228, -0.00829368457198143, 0.02706761844456196, -0.030063895508646965, -0.008912655524909496, -0.0566670261323452, -0.02681955322623253, -0.037117697298526764, 0.001293049892410636, -0.06005106121301651, 0.0019686014857143164, -0.01050975825637579, 0.00718214176595211, 0.010629422031342983, -0.0015803194837644696, -0.01968185044825077, 0.026739001274108887, -0.014677039347589016, -0.009426116943359375, 0.027557019144296646, -0.019172221422195435, -0.03595521301031113, -0.007790790870785713, -0.07697218656539917, 0.027027780190110207, -0.0621015727519989, -0.012276840396225452, -0.05629339814186096, 0.03815324231982231, 0.006651018746197224, -0.023724541068077087, -0.028790779411792755, -0.02642739750444889, -0.022113367915153503, -0.05972207710146904, 0.06272351741790771, -0.02613878808915615, -0.027748918160796165, -0.006347617134451866, -0.03222532942891121, -0.013847540132701397, 0.04031164571642876, 0.014664427377283573, -0.013646827079355717, -0.034327730536460876, 0.038273122161626816, -0.0027974050026386976, -0.031161827966570854, -0.0451388843357563, -0.019446998834609985, -0.0005341311334632337, -0.03210872411727905, -0.0038824332877993584, 0.01753600686788559, 0.0026284598279744387, 0.07094398885965347, -0.0511585958302021, 0.03328315168619156, 0.002102500759065151, 0.0010921740904450417, 0.006283299531787634, -0.00496879406273365, 0.0077435532584786415], [-0.09143291413784027, -0.0030516432598233223, 0.007064696401357651, -0.03229596093297005, -0.09044566005468369, 0.02168319560587406, -0.014362000860273838, -0.0005539476405829191, -0.029418714344501495, 0.017123768106102943, -0.02854996547102928, -0.002635303884744644, 0.016074882820248604, -0.0257702823728323, 0.01591450534760952, 0.013995173387229443, -0.050547558814287186, -0.03938348591327667, -0.03389117494225502, -0.01532228197902441, 0.007415952626615763, 0.06336446106433868, 0.012246389873325825, -0.043260473757982254, -0.04435756057500839, -0.0900939330458641, 0.061117660254240036, 0.050397180020809174, -0.00486368453130126, 0.051345519721508026, 0.007255080621689558, -0.04205658659338951, -0.031125972047448158, 0.014829215593636036, 0.039596546441316605, 0.029758395627141, -0.028425339609384537, -0.002343436935916543, 0.0003476346901152283, -0.04259023442864418, -0.006957702338695526, -0.01184812281280756, -0.10646743327379227, -0.04737240821123123, 0.06589625030755997, 0.040871698409318924, -0.04624083265662193, -0.04144606739282608, 0.08132348209619522, -0.027666233479976654, -0.01673341542482376, -0.03141729533672333, 0.04809753969311714, -0.016377529129385948, -0.06817074120044708, 0.024655301123857498, -0.09450351446866989, -0.07804329693317413, -0.047215308994054794, 0.02072221040725708, -0.08501894772052765, -0.02531726472079754, -0.023171067237854004, 0.05281484127044678, -0.04834425449371338, -0.028117934241890907, -0.023447906598448753, -0.0220627598464489, 0.03716405853629112, -0.0029365522786974907, -0.0490960031747818, -0.03868916630744934, -0.03563365340232849, 0.0542878620326519, -0.02605462446808815, -0.08473492413759232, 0.043286897242069244, -0.05936247482895851, 0.025923756882548332, -0.05404770001769066, -0.05207519605755806, -0.01334367599338293, 0.023277603089809418, -0.013674842193722725, 0.06876253336668015, 0.0674695074558258, -0.0004952632007189095, -0.014565899036824703, 0.07587920874357224, -0.03205760195851326, -0.04684389755129814, -0.016189271584153175, -0.09116476029157639, 0.026890577748417854, 0.021285558119416237, -0.04309937730431557]], "b2": [0.025464089587330818, -0.0630481019616127, 0.03026348538696766, 0.054195109754800797, 0.049923159182071686, -0.13614779710769653, -0.060299742966890335, -0.010680217295885086, 0.038850802928209305, 0.03462139517068863, 0.01408082339912653, -0.05172562971711159, -0.05858762562274933, -0.015623745508491993, 0.10871750116348267, -0.09079278260469437, 0.030331283807754517, 0.09024401754140854, -0.010582409799098969, 0.10602420568466187, 0.06303215771913528, 0.021905576810240746, -0.030083784833550453, -0.04608394205570221, 0.06491526961326599, -0.013330415822565556, 0.0006072346004657447, 0.12527644634246826, -0.034716472029685974, -0.09651431441307068, -0.06795114278793335, -0.04244495928287506], "W3": [[-0.14964710175991058, 0.06388167291879654, -0.20895999670028687, -0.04604654759168625, -0.24674533307552338, 0.11750639230012894, 0.1676287204027176, 0.002925142180174589, -0.1944267749786377, -0.1573081761598587, -0.05782952904701233, 0.182954341173172, 0.05807621777057648, -0.1871320903301239, -0.07242071628570557, 0.07134173065423965, 0.027314167469739914, -0.05429954454302788, -0.015539159066975117, -0.10084526240825653, 0.16528940200805664, 0.1579592525959015, 0.21967986226081848, 0.12506116926670074, -0.05864126607775688, 0.20365428924560547, -0.16282853484153748, -0.12125112861394882, 0.14996552467346191, 0.11044245213270187, 0.13930350542068481, 0.1269633173942566]], "b3": [0.1397731751203537]}, "2024": {"W1": [[-0.15899908542633057, -0.08710678666830063, 0.023156331852078438, -0.2706564664840698, 0.11304058134555817, -0.07088945806026459, 0.039078615605831146, -0.027679134160280228, -0.035499949008226395, 0.285236656665802, -0.012363512068986893, 0.0845254436135292, -0.05403057858347893, 0.04469791054725647, 0.012393230572342873, -0.04774889349937439, -0.19075767695903778, -0.06514564156532288, -0.11544683575630188, -0.039516881108284, -0.06172143295407295], [0.09056740254163742, -0.0658329427242279, 0.05639670789241791, 0.08493078500032425, -0.008580301888287067, -0.09084118902683258, 0.10465319454669952, -0.15841397643089294, 0.07608240842819214, 0.08489533513784409, -0.01603764295578003, -0.03474607318639755, -0.2277943193912506, -0.005130436271429062, -0.1561262458562851, -0.07334742695093155, -0.24590431153774261, -0.04877510294318199, 0.059369008988142014, -0.028957992792129517, -0.03377087786793709], [0.05688923969864845, -0.0067869857884943485, 0.04705710709095001, 0.05303199589252472, 0.054225143045186996, -0.03209695965051651, 0.001338108442723751, -0.009800275787711143, 0.021982135251164436, 0.032281119376420975, -0.0008504445431753993, 0.016880877315998077, -0.040840163826942444, 0.01861829124391079, 0.0005213841213844717, -0.02197754755616188, -0.036638397723436356, -0.003210672875866294, 0.01572634093463421, 0.010667254216969013, 0.010719642043113708], [-0.08625039458274841, -0.01854258030653, -0.2087264060974121, 0.0636637806892395, 0.11263304948806763, -0.035651516169309616, -0.003723869565874338, -0.08745808899402618, 0.003224884858354926, 0.15850433707237244, -0.10854601860046387, -0.016094274818897247, -0.12788838148117065, -0.045952096581459045, 0.03208719566464424, 0.11032505333423615, -0.07565376162528992, 0.07414651662111282, 0.026186052709817886, 0.034086015075445175, -0.07671912759542465], [0.0974746122956276, -0.11069004982709885, 0.050040122121572495, 0.12474127858877182, 0.09864304959774017, 0.03439629077911377, -0.052957355976104736, 0.05209686607122421, -0.07045111805200577, -0.0406021885573864, -0.02958407811820507, 0.16017542779445648, -0.019845230504870415, 0.2058897614479065, -0.12675513327121735, 0.0037194720935076475, 0.028952544555068016, 0.06150578707456589, 0.015468591824173927, 0.03417114168405533, 0.03483160212635994], [0.061139464378356934, 0.06473159790039062, -0.07054809480905533, -0.05876244977116585, 0.0677378922700882, 0.09317616373300552, 0.09534063935279846, 0.06606347113847733, -0.05551239475607872, 0.02904324419796467, 0.07279681414365768, 0.07762516289949417, 0.005895300768315792, 0.04922955483198166, -0.006775769405066967, -0.021136580035090446, -0.05053134635090828, 0.0111959557980299, -0.02893655002117157, -0.03973465785384178, -0.026340030133724213], [-0.07041402161121368, 0.11732175946235657, -0.09903863817453384, 0.03253818303346634, -0.1140013039112091, -0.07835489511489868, 0.10767976939678192, -0.13419416546821594, -0.044394850730895996, 0.010800412856042385, -0.08093155175447464, -0.12030039727687836, 0.049774169921875, 0.06839870661497116, 0.0039032059721648693, -0.04438871517777443, 0.05314378812909126, -0.010189809836447239, 0.16120123863220215, -0.08690051734447479, 0.09282723814249039], [0.025211965665221214, -0.0031039684545248747, 0.08106134831905365, -0.00830982904881239, -0.009347124956548214, -0.028850769624114037, -0.024684781208634377, 0.00740192923694849, -0.048296164721250534, 0.1048872321844101, -0.026889491826295853, -0.02251325361430645, -0.003448614850640297, -0.027585286647081375, 0.04449821636080742, 0.0023085391148924828, -0.15718312561511993, 0.034279100596904755, 0.04361657053232193, -0.024416211992502213, 0.019388630986213684], [0.010994204320013523, 0.020637501031160355, -0.06059929355978966, -0.12789395451545715, -0.09751282632350922, -0.07835205644369125, 0.005976780783385038, -0.0011756087187677622, 0.07026142627000809, -0.010548185557126999, -0.06262405961751938, 0.037258829921483994, 0.007422198075801134, -0.04676900431513786, 0.0023140856064856052, 0.01559761818498373, -0.0839001014828682, -0.07736227661371231, -0.06542544066905975, -0.06778109818696976, -0.07845023274421692], [0.07516272366046906, 0.04315974935889244, 0.01657244563102722, 0.017306270077824593, 0.06821104884147644, -0.060427114367485046, -0.014899306930601597, -0.01659250259399414, 0.02758425660431385, 0.010876934975385666, -0.019419748336076736, 0.07833901047706604, -0.09941903501749039, 0.06322178989648819, -0.07940997928380966, 0.016899622976779938, -0.10380510985851288, 0.09166625142097473, 0.07546701282262802, -0.009676868095993996, 0.0644378513097763], [-0.15103238821029663, 0.0902828723192215, 0.0216768030077219, -0.13061746954917908, -0.07001957297325134, -0.09559503197669983, -0.029020870104432106, 0.0650915876030922, 0.0020508833695203066, -0.16215139627456665, 0.005837365053594112, 0.14741040766239166, -0.10198435932397842, 0.0658285990357399, 0.05751967802643776, 0.006354327313601971, -0.012979961931705475, -0.043375544250011444, -0.08970662951469421, -0.05379891395568848, 0.07897330075502396], [0.0014692884869873524, 0.04930496588349342, 0.030144231393933296, -0.06094656139612198, 0.04522746801376343, -0.033720411360263824, -0.042742665857076645, -0.042690690606832504, 0.04741938039660454, 0.07948126643896103, -0.06010866537690163, -0.015622809529304504, 0.05068882927298546, 0.03172052279114723, 0.1019711121916771, -0.014554696157574654, -0.14426852762699127, -0.04700183495879173, -0.012124214321374893, 0.06093693524599075, 0.07919124513864517], [-0.0033973311074078083, 0.016312792897224426, 0.006525923032313585, -0.09509266167879105, -0.04098575562238693, -0.03510116785764694, -0.020836718380451202, -0.01204261090606451, 0.04543052241206169, 0.018111135810613632, -0.027425162494182587, -0.05783902853727341, -0.04305708035826683, -0.012700201012194157, -0.0006054818513803184, 0.02495480328798294, -0.14200738072395325, -0.031895071268081665, -0.008476326242089272, 0.026723474264144897, 0.013202331960201263], [-0.07720276713371277, -0.06660943478345871, -0.1624096781015396, -0.12314935028553009, -0.03961091861128807, -0.025647081434726715, -0.08242969959974289, 0.08513987064361572, 0.04052460938692093, 0.16260622441768646, -0.026989763602614403, -0.06164751574397087, -0.1469782441854477, -0.04022920876741409, -0.08173906058073044, -0.10482962429523468, -0.03961234167218208, 0.10737470537424088, 0.05025096610188484, -0.13468527793884277, -0.12769919633865356], [-0.02469615265727043, -0.016440780833363533, 0.07657099515199661, -0.00045719847548753023, 0.020966392010450363, 0.005255457945168018, -0.042154375463724136, -0.00022036176233086735, 0.0046007586643099785, 0.026848291978240013, -0.011940272524952888, -0.03800887241959572, 0.02761257439851761, 0.048410337418317795, 0.0034634426701813936, -0.008933140896260738, 0.03160771355032921, -0.029179995879530907, 0.010738835670053959, -0.0419880747795105, -0.0413784384727478], [0.0037376584950834513, -0.02297755517065525, -0.001235193689353764, 0.034973617643117905, -0.06582457572221756, -0.031148435547947884, -0.04259703308343887, 0.004333832301199436, 0.024516550824046135, -0.009730186313390732, -0.004032098688185215, 0.03836408630013466, 0.008345335721969604, -0.019572623074054718, -0.06207234412431717, -0.011446371674537659, -0.09820510447025299, -0.04289478436112404, 0.031230386346578598, -0.02137908898293972, 0.07117187231779099], [-0.07841537147760391, -0.056231625378131866, -0.07124877721071243, -0.2192990630865097, 0.0019610458984971046, -0.012344684451818466, -0.023195957764983177, 0.001996492501348257, -0.08913341909646988, 0.13286346197128296, -0.045613911002874374, 0.08823196589946747, -0.08613231778144836, -0.05203483998775482, -0.0012099321465939283, 0.012005157768726349, -0.06751051545143127, -0.021372215822339058, -0.09266193956136703, 0.001554037444293499, 0.03172489255666733], [-0.023950176313519478, 0.03425349295139313, 0.22795264422893524, -0.19726677238941193, -0.03504680097103119, 0.10642887651920319, -0.16199541091918945, 0.08754344284534454, 0.016338177025318146, -0.06577403098344803, -0.0509980283677578, -0.11828774213790894, 0.02771628648042679, -0.016331221908330917, -0.0771150067448616, -0.02717021107673645, 0.2113061100244522, -0.06385130435228348, 0.011399511247873306, -0.0649057999253273, -0.07947669178247452], [-0.03706271946430206, -0.031848013401031494, 0.015477930195629597, 0.03082429990172386, 0.03158259391784668, -0.08425159007310867, 0.012366879731416702, -0.10028906911611557, 0.04465707018971443, -0.006451212335377932, -0.022731177508831024, -0.0051160408183932304, -0.1407923549413681, -0.08633239567279816, 0.010764073580503464, 0.028574442490935326, 0.10572133213281631, 0.0804455578327179, 0.10771210491657257, -0.04176482930779457, -0.043748605996370316], [0.010177144780755043, -0.03253541514277458, 0.08388408273458481, -0.0163132194429636, -0.06284793466329575, -0.02136426791548729, -0.021415414288640022, 0.025612248107790947, -0.023670559749007225, 0.00460717361420393, 0.009947176091372967, 0.06628964841365814, -0.11752855032682419, 0.01523023471236229, -0.04270952194929123, -0.007451796904206276, -0.17190775275230408, 0.03140922635793686, -0.02578219771385193, -0.04133877530694008, -0.04987461864948273], [-0.033025722950696945, -0.002996011171489954, 0.037995919585227966, -0.11893225461244583, 0.06121273338794708, 0.03705759719014168, 0.025263121351599693, -0.028154736384749413, -0.05537829175591469, 0.006684910971671343, -0.03755233436822891, -0.12362542748451233, -0.05284542962908745, -0.017255380749702454, 0.16573330760002136, 0.02058263309299946, -0.21081231534481049, 0.030607685446739197, -0.03689858689904213, 0.007116840686649084, -0.05802689865231514], [-0.0033018530812114477, 0.058521680533885956, -0.1633204221725464, 0.02041858620941639, 0.0002694269351195544, 0.0826738253235817, -0.041144344955682755, 0.009490016847848892, -0.03156960755586624, -0.13147348165512085, -0.04352886229753494, 0.026623118668794632, -0.022477231919765472, -0.04296732321381569, 0.12109652161598206, 0.0041994815692305565, 0.001347320037893951, 0.03233013674616814, -0.005280350334942341, 0.041977137327194214, -0.06013248488306999], [-0.012505017220973969, -0.054967306554317474, -0.04475485160946846, 0.024053284898400307, 0.08578450232744217, 0.087662473320961, 0.05087881535291672, -0.06570552289485931, 0.012610139325261116, -0.04134904593229294, 0.12026594579219818, -0.014471364207565784, 0.13023056089878082, -0.10890644788742065, 0.07705307006835938, -0.06996496766805649, -0.0828675702214241, 0.06347159296274185, -0.11156968027353287, -0.059781499207019806, 0.10462172329425812], [0.08754556626081467, 0.034932512789964676, 0.025829121470451355, -0.03173438087105751, 0.033845338970422745, -0.017978137359023094, -0.06549081206321716, -0.11043655872344971, 0.06894559413194656, -0.013658256269991398, 0.040988579392433167, -0.06641434878110886, -0.14411257207393646, -0.040690187364816666, 0.09053931385278702, 0.015031704679131508, 0.09179707616567612, -0.054813794791698456, -0.022273799404501915, -0.0241534486413002, -0.008150521665811539], [-0.05861010029911995, 0.03541979193687439, 0.07587996125221252, -0.015781892463564873, 0.014057760126888752, -0.06117784231901169, 0.03675362467765808, 0.11551887542009354, -0.08621692657470703, -0.015270677395164967, -0.04275527596473694, 0.016748784109950066, -0.19331350922584534, -0.06295546144247055, 0.13275545835494995, 0.040005117654800415, 0.060811854898929596, -0.06202547997236252, 0.05943659320473671, -0.019617103040218353, -0.030054191127419472], [-0.14558327198028564, 0.08360189944505692, -0.003704450558871031, 0.023076897487044334, -0.006170697510242462, -0.09116533398628235, 0.0864034965634346, -0.03226788714528084, 0.003001133445650339, 0.1347513496875763, -0.024571191519498825, 0.10214287042617798, -0.22074727714061737, 0.03993408381938934, -0.014902330935001373, -0.01787766069173813, 0.034273430705070496, 0.03869708999991417, -0.19933338463306427, -0.08465307950973511, -0.0698549821972847], [-0.028261074796319008, -0.05559925362467766, -0.016649991273880005, -0.09878898411989212, -0.09259533882141113, -0.08193225413560867, -0.03977111354470253, -0.04577614739537239, 0.033521730452775955, 0.06507641822099686, 0.05639348179101944, -0.024097779765725136, 0.027600953355431557, -0.014606686308979988, 0.09233397990465164, 0.030888468027114868, -0.028215477243065834, 0.030232323333621025, -0.008453712798655033, -0.030950045213103294, 0.09388177841901779], [-0.11199627816677094, -0.07872030138969421, 0.018433386459946632, -0.006422581151127815, 0.10369779914617538, -0.06562728434801102, 0.0009834023658186197, -0.024099133908748627, 0.16063876450061798, 0.03813585638999939, 0.08228173106908798, 0.010556526482105255, 0.18812325596809387, 0.11474519968032837, 0.14967231452465057, 0.039191097021102905, -0.12680980563163757, -0.09352139383554459, -0.0790877491235733, -0.04248110577464104, 0.033388011157512665], [-0.08182147145271301, 0.08725813776254654, 0.11027906835079193, 0.07517224550247192, 0.017824310809373856, 0.07789961248636246, 0.07445130497217178, -0.033146146684885025, 0.08830519765615463, -0.02167520858347416, 0.12017776817083359, 0.06880895793437958, -0.03374209254980087, 0.047747667878866196, 0.06015289947390556, 0.015139410272240639, -0.01474209688603878, 0.10949788987636566, 0.020070698112249374, -0.016523469239473343, 0.11111340671777725], [0.13703250885009766, 0.06591571122407913, 0.021063344553112984, 0.0008072122582234442, 0.021488312631845474, 0.10867921262979507, 0.12099113315343857, -0.03131718933582306, 0.13868039846420288, 0.14128118753433228, -0.09356317669153214, 0.06869155913591385, -0.1916429102420807, 0.028169341385364532, 0.06083724647760391, -0.0015800406690686941, 0.027960646897554398, 0.10408484935760498, 0.02200431190431118, -0.08956488221883774, 0.027554374188184738], [0.09710308164358139, 0.09904538840055466, -0.19933125376701355, -0.02694493718445301, 0.004338033962994814, -0.0817437618970871, 0.12433485686779022, 0.0038486183620989323, -0.0462261326611042, 0.10498005151748657, -0.016501111909747124, -0.031124530360102654, -0.09910854697227478, -0.020197879523038864, -0.028270870447158813, 0.05794382467865944, 0.0016317830886691809, 0.07889363169670105, -0.0019498404581099749, -0.15604090690612793, 0.021146554499864578], [-0.018479768186807632, 0.036602213978767395, 0.15760764479637146, -0.03940748795866966, 0.04187990352511406, -0.022085510194301605, 0.09869792312383652, 0.0035027791745960712, 0.010649594478309155, -0.10221640765666962, 0.07410364598035812, -0.06347757577896118, -0.09075204282999039, 0.010020711459219456, -0.04558391496539116, 0.022218145430088043, -0.04761562496423721, -0.049679894000291824, -0.05073898658156395, -0.016155041754245758, 0.008413427509367466], [-0.008201682940125465, -0.020921984687447548, 0.14085154235363007, -0.052139367908239365, -0.05821746587753296, -0.07071483880281448, -0.02317003905773163, 0.043618109077215195, 0.034260302782058716, -0.05848357081413269, 0.10515972971916199, -0.11186182498931885, 0.0287186149507761, 0.06966384500265121, -0.02582588978111744, 0.0013739675050601363, 0.10860836505889893, 0.013991475105285645, -0.057217273861169815, 0.09711505472660065, -0.025274742394685745], [-0.10894686728715897, 0.07786136120557785, 0.17573876678943634, 0.10095388442277908, 0.0953279510140419, 0.004815191496163607, -0.0059087397530674934, 0.032519083470106125, -0.060489360243082047, 0.01482705119997263, -0.10798237472772598, -0.05675850063562393, -0.15196453034877777, -0.09770268946886063, -0.03276428207755089, -0.0020117827225476503, -0.18105974793434143, -0.032170992344617844, 0.09260379523038864, -0.0030647434759885073, 0.09528299421072006], [0.017214609310030937, 0.10445553809404373, 0.12700209021568298, -0.10537362843751907, 0.0015830540796741843, 0.06046655401587486, -0.020297763869166374, 0.022680673748254776, -0.01785155013203621, 0.08602932095527649, -0.07956848293542862, 0.017353827133774757, -0.030867066234350204, -0.05118594318628311, 0.14648716151714325, -0.0820324495434761, -0.2435784488916397, -0.01663220487535, 0.06459641456604004, -0.022270428016781807, 0.011838290840387344], [0.14864231646060944, 0.10630763322114944, 0.03914191946387291, 0.12516270577907562, 0.0998157411813736, -0.07240873575210571, -0.11835470795631409, -0.055505458265542984, 0.0485023632645607, 0.07665818929672241, -0.11299480497837067, 0.010420802049338818, -0.1407299041748047, -0.06617780029773712, -0.10375339537858963, 0.02787224017083645, 0.2600964605808258, -0.06307461112737656, -0.10361464321613312, 0.05136706307530403, 0.06930577754974365], [-0.006491655018180609, 0.04724670201539993, 0.035854291170835495, 0.006769667845219374, 0.04667074978351593, 0.11249253153800964, -0.06792642176151276, -0.1338675320148468, 0.09370749443769455, -0.047898583114147186, 0.0601651668548584, -0.08545516431331635, -0.21843183040618896, -0.06434701383113861, -0.06763671338558197, 0.058958910405635834, 0.08225277811288834, -0.028647206723690033, 0.06779512017965317, -0.020661449059844017, 0.04010320082306862], [-0.052929919213056564, 0.00058561417972669, -0.08049315959215164, 0.006936168298125267, 0.024545911699533463, 0.007740319706499577, 0.10371273756027222, 0.0643598809838295, -0.07095412909984589, 0.10918612033128738, -0.011371222324669361, -0.019000481814146042, 0.17511843144893646, 0.031017107889056206, -0.04867105558514595, -0.0032550939358770847, 0.1080937534570694, 0.08782495558261871, -0.12808048725128174, -0.09846782684326172, -0.008005348034203053], [-0.0921301394701004, -0.07473401725292206, -0.10356229543685913, 0.07360303401947021, -0.026352522894740105, 0.09103807061910629, -0.00441740220412612, 0.027361052110791206, -0.008306900970637798, 0.17211955785751343, 0.07650550454854965, 0.0476030595600605, -0.11327649652957916, 0.05055782571434975, -0.04491118714213371, 0.07080184668302536, 0.010822810232639313, 0.0794479250907898, -0.1221972107887268, -0.09140481054782867, -0.015611506998538971], [-0.05887683108448982, 0.06555967032909393, -0.05388496443629265, -0.11244381964206696, 0.0716736763715744, -0.012833207845687866, 0.034499913454055786, -0.003067979123443365, -0.029325826093554497, 0.08619476109743118, -0.03242474049329758, 0.12767702341079712, -0.05159926414489746, 0.03148098662495613, -0.07412610203027725, -0.06701863557100296, 0.009156985208392143, -0.03273358196020126, -0.03856004402041435, -0.027256203815340996, -0.06550683826208115], [-0.04160275310277939, -0.03762802109122276, -0.018773823976516724, -0.05041348189115524, 0.03702681139111519, -0.014071013778448105, 0.06912904977798462, -0.05581125244498253, 0.025443045422434807, 0.017450600862503052, 0.060604196041822433, -0.010282115079462528, 0.05014526844024658, -0.03478006273508072, -0.0018266156548634171, 0.002850484801456332, -0.03036731295287609, -0.035586804151535034, -0.03797284513711929, 0.013751067221164703, -0.01726689003407955], [-0.0040281424298882484, 0.008683950640261173, 0.0018868118058890104, 0.002902212319895625, -0.00493536377325654, 0.0527205765247345, 0.03649295121431351, -0.065070241689682, -0.015629462897777557, -0.012944056652486324, 0.004229925572872162, 0.00831886101514101, -0.010561189614236355, 0.015829529613256454, 0.05189240723848343, -0.023171216249465942, 0.05810057371854782, -0.006697845179587603, 0.023242369294166565, -0.004286206793040037, 0.021388085559010506], [0.0008108143811114132, -0.0018130750395357609, 0.08283382654190063, -0.0008538763504475355, 0.025984633713960648, -0.025923442095518112, 0.03668798878788948, -0.04694841429591179, 0.0062950411811470985, 0.08495732396841049, -0.016941456124186516, 0.0234699584543705, -0.11646244674921036, 0.06507690995931625, -0.04488636925816536, -0.00835393462330103, -0.14087502658367157, 0.005975213833153248, 0.03836395591497421, -0.0036811514291912317, -0.0029180364217609167], [-0.0067973667755723, -0.018236791715025902, -0.003467442700639367, 0.019641153514385223, 0.00765431160107255, 0.03546364605426788, -0.015296637080609798, 0.004050623159855604, -0.016891881823539734, -0.04938973858952522, 0.015024282969534397, 0.030563300475478172, 0.051727715879678726, -0.006179234012961388, -0.05058353394269943, -0.03794114664196968, -0.019141560420393944, -0.026853282004594803, -0.019253524020314217, 0.004401031415909529, -0.008256745524704456], [0.1298249214887619, -0.04839848354458809, -0.08479350805282593, 0.13973955810070038, -0.06215185672044754, -0.06434637308120728, 0.003932031337171793, 0.04823148250579834, 0.008337791077792645, 0.1724046915769577, 0.06212128698825836, 0.002787984674796462, 0.06307761371135712, 0.025080682709813118, 0.09814832359552383, 0.00672328844666481, -0.0635012686252594, -0.05186091735959053, 0.1566683053970337, -0.021990688517689705, 0.13041101396083832], [0.005116252228617668, -0.006819099187850952, 0.00741858733817935, 0.01857323944568634, -0.04098233953118324, -0.030438126996159554, -0.029845740646123886, 0.017750227823853493, -0.009718053974211216, 0.0005303059006109834, -0.022867348045110703, 0.03878845274448395, 0.023516030982136726, 0.009308280423283577, 0.10002267360687256, -0.007619128096848726, 0.026457197964191437, 0.009824834764003754, 0.0001686535106273368, 0.03640682250261307, 0.001936888787895441], [-0.027623174712061882, -0.01826821267604828, 0.14076024293899536, -0.014926974661648273, -0.0293001439422369, 0.03409275412559509, -0.11830531060695648, 0.06295467168092728, -0.058509547263383865, 0.01571008749306202, -0.06485073268413544, 0.03805730119347572, -0.0025201968383044004, -0.020392656326293945, 0.05122704431414604, 0.004560884088277817, 0.15600371360778809, -0.002167133381590247, -0.025939252227544785, 0.0038898682687431574, -0.03555667772889137], [0.028746208176016808, -0.013294469565153122, -0.162019282579422, 0.11859695613384247, -0.02010652981698513, 0.06705648452043533, -0.08447866141796112, -0.0613291971385479, -0.033476218581199646, -0.0029904735274612904, -0.03791435807943344, -0.08108803629875183, -0.07711359858512878, 0.03782615065574646, -0.03864378109574318, -0.05362510308623314, -0.11421547830104828, 0.004872629418969154, -0.00367552088573575, 0.019538231194019318, -0.04744705557823181], [0.022967422381043434, 0.06822779029607773, 0.09166481345891953, -0.1251172423362732, -0.028721628710627556, 0.05831458419561386, -0.048208463937044144, -0.017938729375600815, 0.04853801801800728, 0.13001468777656555, -0.06389721482992172, -0.12664949893951416, 0.12751078605651855, 0.02054792270064354, 0.15299217402935028, -0.007299395743757486, 0.07220186293125153, 0.07410542666912079, 0.023404235020279884, 0.009472126141190529, -0.04681320860981941], [-0.150521919131279, 0.021329227834939957, 0.11426907032728195, 0.018722813576459885, -0.10563051700592041, 0.08769611269235611, 0.10351812839508057, -0.1429305076599121, 0.10561826080083847, -0.038136474788188934, 0.0376952700316906, 0.01349856797605753, -0.025382107123732567, 0.0039468687027692795, -0.03547906130552292, 0.054371729493141174, -0.22144438326358795, 0.02097119763493538, 0.06962286680936813, 0.032927654683589935, -0.044951848685741425], [0.07767584919929504, 0.0006973326089791954, 0.11834438145160675, -0.013197565451264381, 0.0649334192276001, 0.0815495178103447, 0.05509721860289574, -0.097519651055336, -0.03641342744231224, 0.09318064153194427, 0.011240304447710514, 0.0326705276966095, 0.025292951613664627, -0.003999047912657261, 0.00867666956037283, -0.045782722532749176, -0.04821450635790825, 0.02542917989194393, -0.042227230966091156, 0.0615125447511673, 0.08943083882331848], [0.0572039820253849, 0.030602579936385155, 0.010781664401292801, -0.06685042381286621, 0.01356161292642355, 0.03282655403017998, -0.03109789825975895, 0.05911659449338913, 0.02760404720902443, 0.07980095595121384, -0.04075680300593376, -0.037189699709415436, 0.03339536115527153, -0.058445535600185394, 0.0032367235980927944, -0.00452050007879734, 0.16323469579219818, 0.027484308928251266, 0.09464050829410553, -0.024019377306103706, -0.06333249062299728], [-0.001066187978722155, 0.1163906455039978, 0.09014704823493958, -0.026018992066383362, 0.05391643941402435, -0.03347444906830788, -0.010535764507949352, 0.05710553005337715, 0.029171694070100784, 0.05666256695985794, -0.025446027517318726, -0.18956075608730316, -0.056287169456481934, -0.027084529399871826, 0.21817661821842194, 0.0473245307803154, -0.07217638939619064, 0.0731494352221489, -0.08199135214090347, -0.02778513729572296, -0.0755283311009407], [-0.04489104077219963, -0.06472569704055786, -0.0994134247303009, 0.07103172689676285, -0.032750047743320465, 0.05071694031357765, -0.03049560636281967, -0.0608162097632885, -0.012355612590909004, -0.052192237228155136, 0.1172681376338005, -0.02342015504837036, 0.0035072409082204103, -0.014712998643517494, -0.11743642389774323, -0.014387819916009903, -0.12647885084152222, -0.058313529938459396, -0.041198376566171646, 0.018402056768536568, -0.05373264104127884], [-0.026443498209118843, -0.04920058324933052, 0.03739927336573601, -0.014473184011876583, -0.04018789157271385, -0.08303102105855942, 0.002951551927253604, 0.006586228031665087, 0.02621675468981266, 0.020658735185861588, -0.025676704943180084, -0.009771679528057575, -0.04128049686551094, -0.0649854838848114, 0.0063953218050301075, 0.01080433838069439, -0.24735769629478455, 0.061996281147003174, 0.02433803677558899, -0.03074064292013645, -0.03937821090221405], [0.05821119248867035, -0.04798315465450287, 0.0031225141137838364, 0.04014521837234497, -0.1139279454946518, -0.07702548801898956, -0.07976040989160538, 0.06388411670923233, 0.05582871660590172, 0.08423963934183121, 0.017956586554646492, -0.01044817827641964, -0.1324031800031662, -0.07004434615373611, -0.07757992297410965, 0.016745014116168022, 0.04901885613799095, -0.005253990180790424, 0.05929412320256233, -0.0540907084941864, 0.09702061116695404], [0.11055125296115875, -0.08809798955917358, 0.1276802122592926, 0.16467885673046112, -0.08988496661186218, 0.04568229615688324, 0.13825960457324982, 0.03538791090250015, 0.05578116327524185, 0.04472916200757027, 0.09923520684242249, -0.07655064016580582, -0.14262567460536957, 0.06544312089681625, 0.0028941300697624683, -0.026280853897333145, -0.12114997208118439, -0.04662427306175232, 0.0738961324095726, 0.1289299726486206, 0.03489372134208679], [-0.02748963236808777, -0.07731354236602783, -0.014511519111692905, -0.17356005311012268, 0.02828160487115383, -0.057177066802978516, 0.05556917190551758, 0.05940205603837967, -0.07008662074804306, 0.10168212652206421, 0.08638955652713776, -0.008499972522258759, -0.1278543323278427, 0.07290486991405487, -0.04739770665764809, 0.031013764441013336, 0.1028083935379982, 0.005612336099147797, -0.14479593932628632, 0.06812197715044022, -0.08049681037664413], [-0.18801812827587128, 0.04274483397603035, -0.04706083610653877, -0.20483118295669556, -0.07405396550893784, -0.04932420328259468, -0.06463044136762619, -0.02585012838244438, -0.08780219405889511, 0.195731058716774, -0.09864486008882523, 0.05246034264564514, -0.008695296943187714, 0.0007696549873799086, -0.01868201233446598, 0.03265045955777168, 0.05907921865582466, -0.027983546257019043, -0.18557222187519073, -0.05329412221908569, -0.1421782225370407], [-0.04618934914469719, 0.11519356817007065, 0.09135472774505615, 0.052885137498378754, 0.054834555834531784, -0.0545801967382431, 0.036484356969594955, -0.06529101729393005, -0.008008195087313652, -0.019426532089710236, -0.0291224867105484, -0.036948129534721375, 0.0215812586247921, 0.040110692381858826, 0.13804180920124054, -0.034602273255586624, 0.05841585993766785, 0.06515944749116898, 0.00376240280456841, -0.029991988092660904, -0.015032707713544369], [-0.15196527540683746, -0.07073140144348145, 0.005702037364244461, 0.026309771463274956, 0.002153343055397272, -0.08642053604125977, 0.03828851878643036, -0.02404898777604103, -0.11133351176977158, 0.16981478035449982, -0.08994477242231369, 0.10258524119853973, -0.13790790736675262, -0.020513731986284256, -0.018721438944339752, -0.023169495165348053, 0.03881535306572914, -0.05472256988286972, -0.1553870290517807, -0.07061444967985153, 0.009866521693766117], [-0.08437792211771011, -0.07460738718509674, -0.001688862917944789, 0.02617001347243786, 0.10366277396678925, -0.0501469150185585, 0.07638988643884659, 0.015527368523180485, -0.023829247802495956, 0.052377376705408096, -0.018059376627206802, -0.10485776513814926, -0.03715813159942627, 0.08507288247346878, 0.043082136660814285, -0.016754968091845512, -0.13849149644374847, 0.10662632435560226, 0.08366409689188004, -0.007254157681018114, -0.10632863640785217], [-0.002830631099641323, -0.017131542786955833, 0.014317083172500134, -0.010062585584819317, 0.009116564877331257, 0.01786612905561924, 0.03716591000556946, 0.0016508566914126277, -0.00796348974108696, -0.01786215975880623, 0.01826556585729122, 0.012058977037668228, 0.0051937405951321125, 0.018681630492210388, 0.0250331312417984, 0.0018309978768229485, -0.010023356415331364, 0.011006448417901993, 0.004873053636401892, -0.0047676097601652145, -0.0031617656350135803], [0.010828778147697449, -0.00833767931908369, 0.013411744497716427, 0.009872330352663994, -0.04376918822526932, 0.002259243978187442, 0.004509879741817713, 0.01570471189916134, -0.0067153749987483025, 0.0019399586599320173, -0.011969265528023243, 0.009023210033774376, 0.0008401364902965724, -0.01722853071987629, 0.04214958846569061, 0.008468231186270714, -0.08319918066263199, 0.006385038606822491, -0.004694856237620115, -0.006981533952057362, 0.005957610439509153], [0.19023975729942322, 0.1162370964884758, -0.13923420011997223, -0.006051606498658657, 0.05081017687916756, 0.045395590364933014, -0.09047620743513107, -0.01239525992423296, 0.12400472909212112, 0.1716061383485794, -0.004302759189158678, 0.07975578308105469, 0.05486570671200752, -0.020810620859265327, 0.008202602155506611, 0.0005442008259706199, 0.04060286656022072, -0.015127120539546013, 0.08355076611042023, 0.010357910767197609, 0.1356358677148819], [-0.027213962748646736, 0.03861774504184723, 0.12710052728652954, 0.056912053376436234, -0.040319085121154785, 0.05549970641732216, 0.09442459791898727, 0.13930463790893555, 0.01750796101987362, 0.004630418960005045, 0.019424673169851303, 0.031556032598018646, -0.01902622915804386, 0.022616036236286163, 0.06265265494585037, -0.17373214662075043, -0.08820200711488724, -0.06995072215795517, 0.05842414125800133, -0.02977760322391987, 0.0874544009566307], [0.04266592487692833, -0.05211947113275528, 0.07428257912397385, -0.11556129902601242, 0.09955938905477524, -0.029185237362980843, -0.0031632862519472837, 0.0003028563514817506, 0.045666974037885666, -0.10750270634889603, -0.020658180117607117, 0.02734980173408985, 0.05526857078075409, 0.02067326381802559, 0.07925835251808167, 0.047042861580848694, 0.02037656679749489, 0.09032940864562988, 0.1550333946943283, 0.05959821492433548, 0.09509386867284775], [0.008254338055849075, 0.12656302750110626, 0.20501436293125153, -0.04569234326481819, -0.0160212405025959, 0.031746067106723785, -0.07168776541948318, -0.12370886653661728, -0.038408584892749786, 0.1935366839170456, -0.10051001608371735, -0.030983002856373787, -0.02349221333861351, -0.035957735031843185, 0.06697049736976624, -0.1297241598367691, -0.02065208926796913, 0.1792674958705902, 0.10121504217386246, -0.03318343311548233, -0.07344081997871399], [-0.014943434856832027, -0.05074574425816536, 0.07261156290769577, -0.013253332115709782, 0.006914544850587845, -0.013584841974079609, 0.04276467114686966, -0.034595925360918045, -0.02558133378624916, -0.15987232327461243, -0.019350597634911537, -0.055107105523347855, -0.006044150795787573, 0.034318968653678894, 0.04928797110915184, 0.046652860939502716, -0.03491361066699028, -0.018762927502393723, -0.08712673932313919, -0.09602847695350647, -0.057517942041158676], [-0.016715090721845627, 0.02022344432771206, -0.14005520939826965, -0.05835912749171257, 0.06767162680625916, 0.03259370103478432, -0.05288288742303848, 0.05842982232570648, -0.04076443985104561, -0.03836037963628769, -0.10871776938438416, 0.11143083870410919, -0.09428965300321579, -0.073912613093853, -0.009309856221079826, 0.11540497839450836, -0.011425230652093887, -0.01874813809990883, 0.08214971423149109, -0.08586383610963821, -0.061643049120903015], [0.009801124222576618, 0.020039428025484085, -0.01702696643769741, -0.025987546890974045, 0.03767227753996849, -0.018843870609998703, -0.002997227944433689, -0.0019378955475986004, 0.03430420532822609, 0.06027618423104286, -0.016830258071422577, -0.026653438806533813, 0.030627304688096046, 0.0034237056970596313, 0.0118942866101861, -0.007763457018882036, -0.041251398622989655, -0.0017540311673656106, 0.04922322928905487, 0.024907033890485764, 0.008371086791157722], [-0.04595630615949631, -0.07681693881750107, 0.10068444907665253, -0.04097231477499008, 0.10582420229911804, 0.10350163280963898, 0.13654828071594238, -0.07495450228452682, -0.06269627064466476, 0.0789780393242836, 0.02110963501036167, -0.018070822581648827, -0.11034565418958664, -0.09914599359035492, -0.0814114511013031, 0.03414769098162651, -0.22573909163475037, 0.03721313551068306, 0.12593691051006317, 0.03869057074189186, 0.05525129660964012], [-0.052095677703619, -0.02824798785150051, -0.06480131298303604, 0.014780135825276375, -0.04478597268462181, -0.04735898971557617, -0.04871491715312004, 0.03735097870230675, -0.09343994408845901, 0.1744043231010437, -0.08549296855926514, 0.08144257962703705, -0.04819554090499878, 0.04324236139655113, -0.0494866743683815, -0.10641609877347946, 0.0003414106904529035, -0.0754363164305687, -0.11307553201913834, -0.12593039870262146, 0.07855769991874695], [0.048399459570646286, 0.03970123082399368, 0.10350843518972397, -0.07711073756217957, -0.026171522215008736, 0.1365150511264801, -0.075579933822155, -0.03674786910414696, 0.08058623969554901, -0.06554824113845825, 0.03247963264584541, 0.014707585796713829, 0.04214008152484894, 0.037302013486623764, -0.04161778837442398, 0.10940933972597122, 0.05794084444642067, -0.020447012037038803, -0.05224836245179176, -0.14581958949565887, 0.12821388244628906], [-0.015418415889143944, 0.002533324295654893, 1.7431480955565348e-05, -0.019127216190099716, -0.0416673943400383, 0.053582582622766495, 0.04732983186841011, -0.010310154408216476, -0.03246142342686653, 0.05056785047054291, 0.03524187579751015, -0.015167255885899067, 0.07079830765724182, 0.009715835563838482, 0.016131538897752762, -0.025182079523801804, 0.08763624727725983, 0.0238245390355587, -0.03560330346226692, -0.0017994882073253393, -0.0226847305893898], [0.07015711814165115, 0.018557807430624962, -0.06221671774983406, 0.10644244402647018, 0.04308813437819481, -0.014352299273014069, 0.013506359420716763, -0.006889164913445711, 0.048521000891923904, -0.028449559584259987, -0.02249671332538128, -0.009958971291780472, 0.08566231280565262, 0.006932748015969992, -0.004290974698960781, 0.01450307946652174, -0.05690460652112961, 0.001527245040051639, 0.06651456654071808, 0.08581938594579697, 0.015346962958574295], [0.009111659601330757, 0.027084827423095703, -0.02920696884393692, 0.045417480170726776, 0.02738684043288231, -0.039932459592819214, -0.026519207283854485, 0.04310455918312073, 0.016278518363833427, -0.03648599237203598, -0.034369535744190216, 0.01833922602236271, 0.05122973769903183, -0.06339515000581741, -0.025834225118160248, 0.019660815596580505, -0.11563944816589355, 0.05861406400799751, -0.03113054856657982, 0.09220724552869797, 0.0024710968136787415], [-0.04918723180890083, 0.036269791424274445, -0.05272996425628662, -0.16413889825344086, -0.04358070343732834, -0.11300579458475113, -0.027890486642718315, -0.05539821833372116, -0.0409129373729229, 0.11126906424760818, 0.049983736127614975, 0.12097901850938797, -0.08238377422094345, 0.041882406920194626, -0.07450705021619797, 0.006121569313108921, -0.0777004063129425, -0.05118299648165703, -0.08104436844587326, -0.03551805391907692, 0.0773678794503212], [-0.07708042114973068, 0.04034867882728577, -0.03525371104478836, -0.005434035789221525, -0.07818479090929031, 0.09316030144691467, 0.11246779561042786, -0.009332693181931973, 0.10371507704257965, 0.022355495020747185, 0.07051609456539154, -0.05023714527487755, 0.020688796415925026, -0.042355015873909, 0.13872697949409485, -0.02909153699874878, -0.07697876542806625, 0.0777571052312851, -0.06530783325433731, 0.0218220017850399, 0.051689788699150085], [-0.004931561183184385, 0.02145175077021122, 0.055453747510910034, 0.033769641071558, -0.020956099033355713, -0.030747737735509872, -0.015944933518767357, -0.0055927312932908535, -0.018352732062339783, 0.0069633056409657, 0.02364957146346569, 0.003095338586717844, -0.04541139677166939, -0.001750270021148026, -0.0020536468364298344, -0.0024575635325163603, -0.03920223191380501, 0.04974130541086197, -0.014303738251328468, 0.0018980655586346984, -0.01284085214138031], [-0.002581431530416012, 0.01260995864868164, -0.08730999380350113, 0.058616943657398224, 0.009422052651643753, 0.05989968776702881, 0.04273708537220955, 0.011416055262088776, 0.01266301330178976, -0.049757134169340134, 0.018064143136143684, 0.035920754075050354, -0.07791312783956528, 0.055836934596300125, -0.06627645343542099, -0.04954250529408455, -0.10091300308704376, -0.03536117821931839, -0.020820587873458862, -0.009620088152587414, 0.04685784876346588], [-0.09401320666074753, 0.041718147695064545, -0.015402947552502155, -0.035929542034864426, -0.00932030938565731, 0.09020788967609406, -0.0891675129532814, -0.0965084508061409, 0.01549016684293747, 0.025107141584157944, 0.08876916766166687, 0.0032081836834549904, 0.18361255526542664, 0.17221377789974213, -0.15092316269874573, -0.009807376191020012, -0.07594450563192368, -0.05958109721541405, 0.04313701391220093, 0.00015376742521766573, -0.12325885146856308], [-0.02299628034234047, -0.026566389948129654, 0.05317234247922897, -0.059525489807128906, -0.05505937710404396, 0.10982564836740494, 0.01585881970822811, -0.11168491095304489, -0.0941193476319313, 0.09377510100603104, -0.12306216359138489, 0.06845961511135101, -0.13714663684368134, 0.033447831869125366, 0.08388424664735794, 0.05330754071474075, 0.03829650208353996, -0.026728833094239235, 0.004603817127645016, 0.0026425065007060766, -0.12500989437103271], [-0.02714725397527218, 0.011353147216141224, -0.04133901372551918, -0.059437062591314316, 0.01589239574968815, 0.008574375882744789, -0.02443770319223404, 0.03251795098185539, -0.03988620266318321, 0.05915597826242447, -0.0354495607316494, -0.020242074504494667, 0.12505030632019043, -0.0006632870063185692, -0.01225225068628788, 0.018249178305268288, 0.030707556754350662, 0.01735902763903141, -0.026310304179787636, -0.014672636054456234, -0.03687852621078491], [0.04343240335583687, -0.010839865542948246, 0.04131432995200157, -0.13975174725055695, -0.02230239473283291, -0.052373453974723816, -0.13418743014335632, 0.09945672005414963, -0.018419193103909492, 0.0581325925886631, -0.07326120138168335, 0.06243274733424187, 0.026015527546405792, -0.08450596779584885, 0.05740831792354584, -0.051844317466020584, 0.20891417562961578, -0.04509613290429115, 0.005323999095708132, 0.011100072413682938, -0.044894155114889145], [0.14344483613967896, -0.04898389056324959, -0.15509267151355743, 0.10562524944543839, 0.1559465229511261, 0.0471932552754879, -0.1255161315202713, 0.08352205157279968, -0.04441113770008087, -0.09423535317182541, 0.07867689430713654, 0.10433046519756317, 0.11262322962284088, 0.008020395413041115, 0.09726967662572861, -0.09087889641523361, -0.006676726508885622, 0.10467300564050674, -0.0765923410654068, -0.06300817430019379, 0.018238471820950508], [0.01956845633685589, 0.03194713965058327, -0.06466419994831085, 0.03458066284656525, 0.08700847625732422, 0.04575984552502632, -0.042404282838106155, 0.10618238151073456, -0.024599319323897362, 0.036054085940122604, -0.028202911838889122, -0.08458107709884644, 0.009927614592015743, 0.03461397439241409, -0.0415189266204834, -0.08667497336864471, 0.21938149631023407, -0.03937031328678131, 0.08187404274940491, -0.05986373499035835, -0.03325187414884567], [-0.017196431756019592, -0.053592290729284286, -0.043081026524305344, -0.050469156354665756, 0.046831779181957245, -0.04405984282493591, -0.01887321099638939, 0.01127020362764597, -0.035833828151226044, 0.10790003091096878, -0.02661127969622612, 0.06311364471912384, -0.0017737563466653228, 0.007949726656079292, 0.016897600144147873, 0.025115815922617912, -0.024938561022281647, -0.03760140761733055, 0.003916184417903423, 0.04226060211658478, -0.03265780210494995], [0.059095703065395355, 0.11718534678220749, -0.274383008480072, -0.03340224549174309, 0.06370772421360016, 0.14100447297096252, 0.05253497511148453, -0.021866273134946823, 0.08718924224376678, 0.06708689779043198, -0.026847001165151596, 0.11717431247234344, -0.09461220353841782, -0.012810924090445042, 0.02131034806370735, -0.057123128324747086, -0.04115203768014908, 0.12099696695804596, 0.02468576282262802, 0.04022003710269928, -0.09106563776731491], [-0.021371517330408096, 0.01815842092037201, 0.027202818542718887, -0.008769502863287926, -0.030430123209953308, 0.0032192820217460394, -0.01871512457728386, -0.006771496497094631, -0.030898692086338997, -0.02749689482152462, -0.005496378522366285, -0.023288866505026817, 0.0729900598526001, 0.0332476831972599, 0.012336239218711853, 0.0028955440502613783, 0.07141823321580887, 0.017006399109959602, -0.03849213942885399, -0.01618567667901516, -0.00933022703975439], [-0.021560968831181526, -0.018795760348439217, 0.02933010272681713, -0.045989278703927994, -0.07262369990348816, 0.0015028349589556456, 0.03726847097277641, -0.021571742370724678, -0.07142224162817001, -0.01784878596663475, -0.02931734174489975, -0.1403508186340332, 0.008001459762454033, 0.06741208583116531, 0.06219382956624031, 0.010286102071404457, 0.1319475919008255, 0.07356110215187073, -0.051548514515161514, -0.04266814514994621, -0.0017756803426891565], [0.047898873686790466, 0.10379723459482193, 0.07241183519363403, -0.025618407875299454, -0.01936880685389042, -0.0739472508430481, -0.12501463294029236, 0.1585797518491745, -0.12621180713176727, 0.012263943441212177, -0.17609907686710358, 0.005792227573692799, 0.01947312243282795, -0.09466926008462906, -0.0146366897970438, 0.004995417781174183, 0.2953430116176605, 0.05970817059278488, 0.10979883372783661, -0.0770186260342598, -0.07550442963838577], [-0.028377428650856018, -0.011266772635281086, 0.025838075205683708, 0.08545764535665512, -0.02538348361849785, -0.01542880479246378, 0.05457265302538872, -0.03595660626888275, 0.009190965443849564, -0.047812268137931824, 0.0033830618485808372, 0.029552143067121506, -0.11042603850364685, 0.06891639530658722, -0.028325313702225685, -0.0007764772162772715, 0.06688995659351349, 0.007252996321767569, 0.014529840089380741, -0.00797731801867485, -0.006591004319489002], [0.03681566193699837, 0.023495839908719063, -0.021571284160017967, 0.07061518728733063, 0.05794432386755943, -0.025991184636950493, 0.03557968512177467, 0.032568782567977905, 0.015540597960352898, 0.025692541152238846, 0.015929020941257477, -0.01057446002960205, -0.031207730993628502, 0.013482335954904556, 0.06708897650241852, -0.05597062408924103, -0.06102524697780609, 0.015422238036990166, 0.04149915277957916, 0.056306276470422745, 0.028165556490421295], [0.0070608253590762615, 0.028062447905540466, 0.01570984162390232, -0.011977513320744038, 0.05643097683787346, -0.04826139286160469, 0.06693536788225174, 0.06172378733754158, 0.044906195253133774, 0.05337020382285118, 0.005298567470163107, 0.007863673381507397, 0.025002941489219666, -0.04206791892647743, 0.12214665114879608, -0.036456890404224396, -0.1814301759004593, 0.04348583146929741, 0.04585249722003937, 0.003446767572313547, -0.08279384672641754], [-0.05051872879266739, -0.010574772022664547, 0.06914268434047699, -0.08335136622190475, 0.03059508465230465, -0.03265530243515968, 0.08328605443239212, -0.04397008195519447, -0.06763572245836258, 0.01179796177893877, -0.02081443928182125, 0.008847902528941631, 0.15003719925880432, -0.036737244576215744, -0.019093424081802368, 0.03658759593963623, 0.15157632529735565, -0.014504319988191128, 0.020602893084287643, 0.01441993284970522, -0.08415853977203369]], "b1": [-0.11088890582323074, 0.19704951345920563, 0.04277878999710083, -0.14275521039962769, 0.18916484713554382, 0.10350937396287918, -0.038059987127780914, 0.001750375609844923, 0.09796706587076187, 0.06761617958545685, -0.12451721727848053, 0.0023996306117624044, 0.02961629629135132, -0.09711107611656189, -0.0895446389913559, 0.03482048958539963, -0.02762996405363083, 0.12400583922863007, 0.04096999391913414, 0.11219196021556854, -0.01848742738366127, 0.0561160072684288, 0.1411200612783432, 0.04700928181409836, 0.026796890422701836, -0.11049072444438934, 0.09434188902378082, 0.030919406563043594, -0.0202631913125515, 0.010801904834806919, -0.081534743309021, -0.019287485629320145, 0.05637849122285843, 0.10607951134443283, 0.13311827182769775, 0.0939674824476242, 0.20079405605793, 0.03752349689602852, -0.08207888156175613, 0.07362160086631775, -0.05838596820831299, -0.01419166661798954, -0.015781721100211143, 0.021828794851899147, -0.06276528537273407, 0.026628125458955765, -0.014871947467327118, -0.07127119600772858, -0.011489365249872208, 0.08230859041213989, 0.011801016516983509, 0.03982279449701309, -0.024196993559598923, 0.06822153180837631, 0.033376678824424744, -0.0214061476290226, 0.031914256513118744, -0.07276550680398941, 0.03407572954893112, 0.024410691112279892, -0.15095290541648865, 0.08167784661054611, 0.010439708828926086, 0.04585554450750351, -0.03785337507724762, -0.10623789578676224, -0.14728786051273346, 0.12246265262365341, -0.06522218883037567, -0.08206474781036377, -0.00119543366599828, -0.021620968356728554, -0.07908304035663605, -0.1418304145336151, -0.03750082850456238, 0.025282112881541252, 0.16397151350975037, 0.07681884616613388, 0.03537742421030998, -0.02922217734158039, -0.0034865352790802717, 0.15325719118118286, 0.04894870147109032, 0.008031248115003109, 0.08646269142627716, 0.06078193336725235, 0.1435837596654892, -0.05151674523949623, 0.031748682260513306, 0.03735790774226189, 0.08783905953168869, -0.013192489743232727, 0.011702180840075016, -0.041022226214408875, 0.09762897342443466, 0.12141884863376617], "W2": [[0.07493215799331665, 0.01830834336578846, 0.0038174756336957216, -0.03798144310712814, -0.01921718940138817, -0.022296249866485596, -0.055966123938560486, -0.007286609150469303, 0.037638794630765915, 0.05920402333140373, -0.08872482925653458, 0.04528772830963135, -0.027788376435637474, 0.01743047870695591, 0.01972794160246849, 0.006216995883733034, 0.007306279148906469, -0.08363401144742966, 0.07936929166316986, -0.00025676394579932094, 0.08886063098907471, -0.07020556926727295, 0.07217404991388321, 0.06401821225881577, -0.006504586897790432, 0.02082098461687565, 0.05245179682970047, 0.0663549154996872, -0.03834036737680435, 0.05375973507761955, -0.04490387812256813, -0.010066007263958454, -0.028869878500699997, 0.007914978079497814, 0.0573376826941967, 0.06274387240409851, 0.030257945880293846, 0.05173937976360321, -0.027177555486559868, -0.04217363893985748, 0.025129230692982674, 0.05878785252571106, -0.053824570029973984, 0.008519068360328674, -0.013106493279337883, 0.04623500630259514, -0.07435158640146255, -0.04379621520638466, 0.037306107580661774, 0.053084392100572586, 0.053056441247463226, 0.03789439797401428, 0.06989452242851257, -0.043602753430604935, -0.014030308462679386, 0.020460834726691246, -0.03156308829784393, 0.007049840409308672, 0.11254405975341797, -0.018781552091240883, -0.014268461614847183, -0.03467150405049324, -0.01923246681690216, -0.04035087302327156, -0.007908103056252003, -0.09794409573078156, -0.05463656783103943, 0.033070314675569534, -0.0186324380338192, -0.03959666192531586, 0.030396975576877594, 0.041682276874780655, -0.048941489309072495, -0.057455044239759445, -0.001412567333318293, 0.017419731244444847, -0.010098086670041084, 0.03589118272066116, -0.03772109001874924, -0.0001897142210509628, -0.037628304213285446, 0.07448750734329224, 0.01842251606285572, 0.01830502413213253, 0.06923869997262955, -0.009875509887933731, -0.01572893001139164, -0.007890160195529461, -0.04161437600851059, -0.006087597459554672, 0.05082140862941742, 0.09300829470157623, -0.021158786490559578, -0.052832089364528656, 0.04661817103624344, 0.03566604480147362], [0.02957668900489807, 0.009499972686171532, 0.01780482567846775, 0.02111028879880905, -0.017048891633749008, 0.006190018728375435, 0.0057752905413508415, -0.0028370325453579426, 0.001483000349253416, 0.009907259605824947, 0.03584599867463112, 0.005355535540729761, 0.012648395262658596, 0.028747212141752243, 0.0009648400009609759, 0.006912834011018276, 0.019262362271547318, 0.026012979447841644, -0.00955998059362173, 0.02493128925561905, 0.03135805204510689, 0.009732072241604328, -0.004862088710069656, -0.01649617590010166, 0.009144636802375317, 0.026670243591070175, -0.014787229709327221, 0.02100970782339573, 0.04459879919886589, -0.021547242999076843, -0.0026929222512990236, 0.038791224360466, 0.006863895803689957, 0.030293386429548264, -0.0005170986405573785, -0.030879279598593712, -0.021850984543561935, -0.004852183163166046, 0.01124369166791439, 0.024505186825990677, -0.0017204622272402048, 0.016169575974345207, 0.01318337582051754, -0.013697653077542782, 0.014680026099085808, -0.0019921520724892616, -0.005496078170835972, 0.022426417097449303, -0.009552300907671452, 0.006380737759172916, -0.04238502308726311, 0.004509863443672657, 0.00202470226213336, 0.0016951521392911673, 0.010643653571605682, 0.0011115826200693846, 0.020922766998410225, 0.015172409825026989, -0.010752183385193348, -0.006973618175834417, 0.033048469573259354, 0.013896692544221878, -0.0005699759931303561, -0.0015044412575662136, 0.0441930890083313, 0.035989947617053986, 0.0023392706643790007, -0.04301457852125168, 0.017753291875123978, 0.016611842438578606, 0.009937312453985214, 0.0068450006656348705, 0.029503392055630684, 0.0124089689925313, -0.005397147033363581, -0.00300424313172698, -0.0005068587488494813, 0.03360871598124504, 0.016255782917141914, 0.007176672108471394, 0.021200593560934067, -0.019092554226517677, 0.008509955368936062, 0.0041382089257240295, 0.032424017786979675, -0.004751974251121283, 0.011279357597231865, -0.00385457556694746, 0.05848385766148567, 0.001633182750083506, 0.008588168770074844, -0.013241718523204327, 0.00966609176248312, -0.014726550318300724, 0.021622123196721077, 0.008196666836738586], [0.05989730730652809, 0.05525524169206619, -0.0024098532740026712, -0.041824210435152054, 0.03927287459373474, -0.02519151382148266, -0.06420522928237915, 0.00345098739489913, 0.023387659341096878, -0.008077951148152351, -0.04718918353319168, 0.07794450223445892, 0.032460637390613556, -0.09018784761428833, 0.029406001791357994, 0.014576702378690243, 0.04594931751489639, -0.018915098160505295, -8.492642518831417e-05, -0.014574060216546059, 0.01701127365231514, -0.061867985874414444, 0.02662837877869606, 0.031416621059179306, 0.09644865244626999, 0.023551231250166893, 0.0636005848646164, 0.05276960879564285, -0.1478291153907776, -0.053953513503074646, -0.0641574114561081, -0.10050053894519806, -0.043232645839452744, 0.054782237857580185, -0.03488817811012268, 0.1179363876581192, -0.0038242898881435394, 0.09257367253303528, -0.0034415272530168295, -0.05380360409617424, 0.0032972777262330055, -0.053306836634874344, -0.04823088273406029, -0.0063101425766944885, -0.0516657829284668, 0.010089403949677944, 0.05643117055296898, -0.02821325697004795, 0.11779025197029114, -0.060644570738077164, -0.04584168270230293, 0.09301149845123291, 0.01965446211397648, 0.04882444068789482, -0.04931572824716568, 0.11095955967903137, -0.007112524006515741, 0.037896350026130676, 0.11542094498872757, -0.04473235830664635, 0.004851660691201687, 0.019299544394016266, -0.02995437942445278, -0.003203636035323143, 0.013978090137243271, -0.08135509490966797, -0.04332365840673447, 0.014250800013542175, -0.013054107315838337, -0.05494359880685806, 0.0721624344587326, -0.008602428250014782, -0.011571139097213745, -0.0834055244922638, 0.012795715592801571, 0.017739130184054375, 0.060027457773685455, -0.01838845945894718, -0.01185841578990221, -0.05329962074756622, 0.008784021250903606, 0.012712452560663223, 0.03985893726348877, 0.11020977050065994, 0.06425001472234726, 0.054220590740442276, 0.057430367916822433, -0.022089187055826187, -0.10081212967634201, 0.055707767605781555, 0.07193303853273392, 0.19326871633529663, 0.02792397513985634, 0.029986869543790817, 0.035960834473371506, 0.10800991952419281], [0.01787499338388443, 0.03758849576115608, 0.0025996684562414885, -0.018093885853886604, 0.00863015465438366, 0.011142710223793983, -0.012430572882294655, 0.0014054510975256562, 0.007643049582839012, 0.018889572471380234, -0.012199413031339645, -0.0067758746445178986, -0.009471824392676353, -0.006096801720559597, -0.001753972377628088, 0.012975478544831276, 0.00021953656687401235, -0.0072763715870678425, 0.010482163168489933, 0.0024631803389638662, -0.0018237215699627995, 0.01740497164428234, 0.003710999386385083, 0.0028631503228098154, -0.0024055757094174623, 0.0004992409376427531, 0.005926921963691711, 0.0186912938952446, -0.022457124665379524, -0.01700780540704727, -0.008964122273027897, -0.028431346639990807, 0.022236377000808716, -0.03093201294541359, 0.012369035743176937, 0.005462143570184708, -0.023461950942873955, 0.008838738314807415, -0.014417155645787716, -0.008183639496564865, -0.0019447169033810496, -0.006020385771989822, 0.022686541080474854, 0.009908264502882957, -0.02845928631722927, -0.0009204029338434339, -0.0020504884887486696, -0.019419405609369278, 0.0035054185427725315, -0.0028025915380567312, -0.02200384996831417, -0.0034250449389219284, 0.015036316588521004, 0.00199264008551836, 0.00172822002787143, 0.04317447170615196, 0.04327082633972168, -0.004779510200023651, 0.04239445552229881, 0.00142383249476552, -0.01382875069975853, -0.005176462698727846, 0.00036846648436039686, -0.00404213834553957, -0.02897779829800129, -0.0023047439754009247, -0.0019118633354082704, 0.023617981001734734, -0.015417680144309998, -0.0030313392635434866, -0.008503464981913567, -0.015757065266370773, -0.00541446078568697, -0.01671135239303112, -0.012479837983846664, 0.010510104708373547, -0.003307909471914172, 0.016832133755087852, -0.010130639187991619, -0.003272018628194928, -0.012462801299989223, 0.004867555573582649, 0.005008311942219734, 0.0023406934924423695, -0.015473438426852226, -0.001587821519933641, 0.015178976580500603, -0.005182058550417423, -0.02074248716235161, -1.6266810689558042e-06, 0.01673503965139389, 0.008014868944883347, -0.008084059692919254, 0.009450802579522133, -0.010127977468073368, 0.004309395793825388], [-0.039942145347595215, 0.0065901619382202625, 0.06679165363311768, -0.10360802710056305, 0.06682083010673523, -0.058743543922901154, -0.015780748799443245, 0.008224950172007084, -0.07023739069700241, -0.0060445405542850494, -0.09967564791440964, 0.012676283717155457, -0.04920518025755882, -0.054885488003492355, 0.0179382786154747, 0.05868583917617798, 0.03543783724308014, 0.03132489696145058, 0.07678665220737457, 0.0341145396232605, -0.05839335545897484, -0.020267516374588013, 0.1049017608165741, 0.06890038400888443, 0.06018861383199692, -0.08660726249217987, 0.06176353245973587, -0.03375445306301117, -0.06414458155632019, 0.07694725692272186, -0.03564021736383438, -0.03700043261051178, -0.043102703988552094, 0.056885238736867905, -0.0776086375117302, 0.10032793879508972, 0.046881332993507385, 0.08274544030427933, -0.044663067907094955, 0.05719352141022682, -0.007229909300804138, 0.06520792096853256, -0.016121039167046547, 0.007934403605759144, -0.07563821971416473, 0.022931303828954697, 0.08019321411848068, -0.030858317390084267, -0.023178480565547943, 0.0005329608102329075, -0.04366825148463249, 0.09033824503421783, -0.031158743426203728, 0.06243758276104927, -0.04742450267076492, 0.033409297466278076, 0.07000299543142319, 0.018964799121022224, 0.04592073708772659, 0.058049265295267105, 0.030299633741378784, -0.06444348394870758, -0.02506047859787941, -0.010079852305352688, -0.10112915933132172, -0.06048758700489998, -0.041436877101659775, 0.015822578221559525, -0.013842429965734482, -0.015187475830316544, -0.06775974482297897, 0.005238782148808241, -0.03832762688398361, -0.044534217566251755, 0.02477952465415001, -0.0567333847284317, -0.0625845193862915, 0.07135219871997833, 0.07151781767606735, 0.028750652447342873, 0.015532383695244789, 0.08337363600730896, 0.05765055492520332, -0.012311570346355438, 0.04876596853137016, -0.07302603125572205, 0.08880539983510971, -0.021494174376130104, 0.031070690602064133, -0.03637677803635597, -0.008848876692354679, 0.16088815033435822, -0.010305195115506649, -0.005738707259297371, 0.0014850232983008027, 0.025289593264460564], [0.08943454921245575, 0.004281750414520502, 0.00021494593238458037, 0.060195766389369965, -0.07239591330289841, 0.0022819051519036293, 0.020716501399874687, 0.00010219708201475441, 0.0034482015762478113, -0.017527250573039055, 0.03729885816574097, -0.011760181747376919, -0.0034034536220133305, 0.10275158286094666, 0.0027272249571979046, 0.00761447986587882, 0.0826629102230072, -0.014991814270615578, -0.007612043060362339, -0.009643296711146832, -0.019783983007073402, -0.05569411814212799, -0.01265730056911707, -0.0054750447161495686, 0.02942715585231781, 0.12376891821622849, 0.011260551400482655, -0.014980246312916279, 0.029393242672085762, -0.032524239271879196, 0.039055850356817245, 0.026143640279769897, -0.01916387304663658, 0.004778672009706497, -0.06599853187799454, -0.05053757131099701, -0.024025676771998405, 0.019996168091893196, 0.04621288552880287, 0.05904940888285637, -0.004301269073039293, 0.00819774903357029, 0.02348000928759575, -0.018269209191203117, 0.03267906978726387, -0.020779717713594437, 0.02769644372165203, 0.025128355249762535, -0.04484307020902634, 0.024708889424800873, -0.03687804192304611, -0.0007262179860845208, -0.0805874764919281, -0.02965799905359745, -0.008522647432982922, -0.005243580788373947, 0.021029584109783173, 0.08225928246974945, 0.061828140169382095, -0.0456051230430603, 0.12364360690116882, -0.0052439128048717976, 0.007746815215796232, -0.012863686308264732, 0.04154617339372635, 0.03589305281639099, 0.0027856510132551193, -0.059109173715114594, -0.002882945816963911, 0.0289089847356081, 0.00617595948278904, -0.016912100836634636, 0.07174238562583923, 0.015101551078259945, -0.004699797369539738, 0.018238883465528488, -0.01565583422780037, 0.08924911171197891, -0.01421076525002718, 0.0165410079061985, 0.019555889070034027, -0.02085093781352043, 0.027054009959101677, 0.021998072043061256, 0.02291872538626194, -0.005871940404176712, 0.0017144461162388325, 0.0294998399913311, 0.06792297214269638, -0.03841163590550423, 0.00571919372305274, -0.03289123252034187, 0.005683594383299351, 0.028360718861222267, -0.015612522140145302, -0.0029842453077435493], [-0.04215594753623009, -0.06363094598054886, -0.05165283754467964, 0.021477920934557915, -0.010319309309124947, -0.04895840585231781, 0.011687974445521832, 0.022252831608057022, -0.015570533461868763, -0.078544020652771, 0.013944675214588642, -0.01046016626060009, -0.01439348328858614, 0.011038246564567089, 0.030139468610286713, -0.013814853504300117, -0.03266984596848488, 0.06550386548042297, -0.015093520283699036, 0.025882096961140633, 0.03536653891205788, -0.009186306037008762, -0.031962648034095764, -0.010682495310902596, -0.018766384571790695, -0.06475842744112015, 0.02522006817162037, -0.013750673271715641, -0.024197936058044434, -0.08329488337039948, -0.010685368441045284, 0.011539721861481667, 0.020322933793067932, -0.0230585765093565, -0.05395015329122543, -0.0034593192394822836, 0.007324093021452427, -0.0781826302409172, -0.0632636696100235, -0.059942957013845444, -0.016314486041665077, -0.012256796471774578, -0.04282499477267265, 0.01889282651245594, -0.006036937702447176, -0.0013394731795415282, -0.03486580774188042, 0.014188011176884174, 0.00017775848391465843, -0.019581522792577744, -0.05770497769117355, -0.022019021213054657, 0.02385491319000721, -0.06620562821626663, 0.02200249768793583, -0.06011345610022545, -0.005184104200452566, -0.0376010499894619, -0.05104335770010948, 0.044034987688064575, -0.046413976699113846, -0.03372403234243393, 0.026318056508898735, -0.0006244193064048886, -0.04010099545121193, -0.003540361300110817, 0.016028713434934616, 0.006311388686299324, 0.02800004556775093, 0.012069053947925568, -0.014403228648006916, -0.008153943344950676, -0.04654708877205849, 0.020378585904836655, -0.027446942403912544, -0.06336213648319244, -0.03838410973548889, -0.029028818011283875, -0.055522751063108444, 0.02631259337067604, -0.029355689883232117, -0.05496281385421753, -0.051241908222436905, -0.029202468693256378, -0.038837142288684845, -0.05002367123961449, -0.02918301895260811, 0.0006449134671129286, -0.03161923587322235, 0.026470543816685677, 0.05059293285012245, -0.010030253790318966, 0.01718515157699585, -0.013011109083890915, 0.0289801973849535, 0.0017653974937275052], [-1.992333636735566e-06, -3.9461076084990054e-05, 2.0084078641957603e-05, 2.426744504191447e-05, -1.2679565770667978e-05, 7.296803232748061e-07, 8.352386794285849e-05, 7.224977252917597e-06, 3.722884866874665e-05, 6.013015809003264e-05, 8.624562178738415e-05, 1.9472052372293547e-05, 1.7912321709445678e-05, 2.731940185185522e-05, -2.4045737518463284e-05, 2.8529666451504454e-06, -1.3971053704153746e-05, 0.0001299594296142459, 1.1910554349015001e-05, 1.9726351183635416e-06, 5.836050695506856e-05, -2.4858494725776836e-05, -6.723950355080888e-05, -1.0272898180119228e-05, -4.861286288360134e-05, 2.2833008188172244e-05, 1.0868592653423548e-05, -2.2826154236099683e-05, 0.00016202476399485022, -2.2832790591564844e-07, 8.152236114256084e-05, 8.165265899151564e-05, 5.5604403314646333e-05, -2.8632653993554413e-05, 7.936143083497882e-05, 2.460433097439818e-05, 4.298306885175407e-05, 3.9978524000616744e-05, 4.1894440073519945e-05, 1.1197111007277272e-06, -1.8310754967387766e-05, 8.732481546758208e-06, 1.9087068722001277e-05, -1.1165891010023188e-05, 0.00010670939809642732, -1.80672559508821e-05, -8.345034984813537e-06, -1.5731990288259112e-06, -1.492964474891778e-05, 4.368544978206046e-05, -6.5964973146037664e-06, -1.8632874798640842e-06, -2.5363005988765508e-05, 8.638729923404753e-06, -2.7569847588893026e-05, -2.054061224043835e-05, 5.761614374932833e-05, 2.9179087505326606e-05, -3.2454066968057305e-05, 2.122770092682913e-05, 5.280450750433374e-06, 5.613818939309567e-05, 3.3807380077632843e-06, -1.357780774924322e-06, 0.00016319769201800227, 7.18163137207739e-05, 3.557513628038578e-05, -7.229053153423592e-05, 1.6130026779137552e-05, -2.5524066586513072e-05, 7.793512850184925e-06, -1.560906866870937e-06, 5.3791234677191824e-05, 6.636107718804851e-05, 2.0578341718646698e-05, -9.881427104119211e-06, -9.615726412448566e-06, 3.553164106051554e-06, 4.379407619126141e-05, 2.0624329408747144e-05, 3.5791632399195805e-05, -6.11356008448638e-05, -7.049068517517298e-05, -4.667469511332456e-06, 3.717487197718583e-05, -1.0589734301902354e-05, 1.7060865502571687e-05, -9.624595804780256e-06, 0.00012217287439852953, 1.3576312085206155e-05, 3.6149802326690406e-05, -7.13270201231353e-05, 4.8847396101336926e-05, 2.257540836581029e-05, 3.466676207608543e-05, 2.9025037292740308e-05], [0.12917713820934296, 0.06827010214328766, -0.00865283515304327, 0.03168134018778801, 0.018688645213842392, -0.036025650799274445, 0.01035058218985796, 0.029864754527807236, -0.008007203228771687, -0.042890701442956924, -0.09136547148227692, 0.06719756126403809, 0.08540266752243042, -0.0765620544552803, -0.05037922039628029, 0.04356619715690613, 0.08955654501914978, -0.17868852615356445, -0.04676227644085884, 0.08912932872772217, 0.0797383263707161, 0.01147426012903452, 0.059097595512866974, -0.05370287597179413, 0.034375324845314026, -0.02249196544289589, 0.05526113882660866, 0.12801654636859894, 0.0089624198153615, 0.08257526904344559, -0.020358627662062645, -0.017200632020831108, -0.06372232735157013, 0.0775635614991188, 0.08688338100910187, -0.007039181422442198, 0.06517308205366135, -0.042690545320510864, -0.028059490025043488, 0.004823893308639526, 0.04828514903783798, -0.042499952018260956, 0.014694721437990665, -0.013479421846568584, -0.053608935326337814, -0.06526466459035873, -0.12250557541847229, -0.0714273452758789, -0.0033066165633499622, 0.04711402207612991, 0.03616264835000038, -0.013473493047058582, 0.058072641491889954, 0.0010043499059975147, 0.11595471203327179, 0.045503560453653336, 0.0383116751909256, -0.07692443579435349, 0.04709406942129135, 0.005164978560060263, 0.0189388245344162, 0.0013014256255701184, 0.0540119968354702, 0.02232203632593155, -0.02528202347457409, -0.029804525896906853, -0.09095040708780289, 0.06824958324432373, -0.018023774027824402, -0.03345414251089096, -0.019471734762191772, 0.08364959806203842, -0.03152143582701683, -0.10252667963504791, -0.0028558834455907345, 0.03262500837445259, 0.0660293847322464, 0.041052043437957764, 0.025781800970435143, 0.009089283645153046, -0.08029013872146606, 0.005689646117389202, 0.06212073937058449, -0.08080317825078964, -0.13236208260059357, -0.11528270691633224, -0.04984649270772934, 0.03366127982735634, -0.09052078425884247, 0.014685951173305511, -0.06436922401189804, -0.1820894032716751, 0.007450854871422052, 0.035256050527095795, 0.0569017268717289, -0.07636673003435135], [0.11051255464553833, 0.06875931471586227, -0.0539751872420311, -0.026333274319767952, 0.0329984575510025, 0.010671309195458889, -0.07291513681411743, 0.006141068413853645, 0.016028033569455147, 0.02484036237001419, -0.05084749683737755, -0.003511881222948432, 0.037232328206300735, 0.018947439268231392, -0.020970148965716362, 0.001876870053820312, 0.04499443247914314, 0.01577633246779442, 0.03771819919347763, 0.012571790255606174, 0.033253833651542664, -0.05458647757768631, 0.03707505017518997, 0.04745133966207504, -0.024865619838237762, -0.09592510759830475, 0.01534563023597002, 0.037489645183086395, 0.04350220412015915, -0.054643336683511734, -0.07786383479833603, -0.07632826268672943, -0.048552729189395905, -0.04258352145552635, 0.07066987454891205, 0.04763322323560715, 0.06313569098711014, 0.055378153920173645, -0.02069210819900036, -0.013689997605979443, 0.04007896035909653, -0.05320541560649872, -0.014701013453304768, -0.011645004153251648, -0.115517258644104, -0.04078417643904686, -0.044507019221782684, -0.02965444326400757, 0.067733995616436, 0.023883577436208725, 0.042351752519607544, -0.03143566846847534, 0.04001427814364433, -0.03209742158651352, 0.04676882550120354, -0.028664974495768547, -0.05438115447759628, 0.03948361054062843, 0.07073432952165604, 0.012091822922229767, -0.0363488644361496, 0.05726335570216179, 0.04111211746931076, -0.03413284197449684, -0.09228765964508057, -0.09459124505519867, -0.05821406468749046, 0.020443331450223923, -0.05541239678859711, -0.012000491842627525, -0.0016956166364252567, 0.07398687303066254, -0.03869713097810745, -0.039100922644138336, 0.052095234394073486, -0.048692330718040466, -0.055734362453222275, 0.07240442931652069, 0.07479698210954666, -0.025410572066903114, 0.023590128868818283, 0.06143295764923096, 0.060944344848394394, 0.020199615508317947, -0.007467267569154501, -0.07926853746175766, -0.016648204997181892, -0.03579866513609886, -0.048112571239471436, 0.03493357449769974, -0.01484755240380764, -0.028613604605197906, -0.03242713212966919, -0.09231897443532944, -0.021350346505641937, -0.018553148955106735], [-0.01805919222533703, -0.018187031149864197, 0.027874242514371872, -0.026099156588315964, -0.026008397340774536, 0.00278931250795722, -0.027249975129961967, 0.008179539814591408, -0.026359805837273598, 0.03416032716631889, -0.016057414934039116, -0.011303726583719254, -0.02382643334567547, 0.0012281987583264709, -0.005777059122920036, -0.020609794184565544, -0.01252803485840559, 0.0330626480281353, 0.02444177307188511, -0.02041896991431713, -0.04233892634510994, -0.02553880587220192, 0.027780745178461075, 0.04310426115989685, 0.035560205578804016, -0.014110032469034195, 0.023451095446944237, 0.01286571566015482, 0.027314363047480583, -0.011554504744708538, -0.00586472125723958, -0.015635427087545395, 0.021990912035107613, 0.01897292770445347, 0.0064048259519040585, 0.06888352334499359, 0.03385263681411743, 0.02693379856646061, -0.010484802536666393, -0.0007752382662147284, 0.01784638687968254, 0.011927439831197262, -0.019041651859879494, -0.001405281131155789, -0.013045227155089378, 0.0186652522534132, -0.0004895016318187118, -0.012376400642096996, 0.005904485937207937, 0.03907904028892517, 0.0022784038446843624, 0.013071215711534023, -0.022363023832440376, -0.020352138206362724, -0.008168928325176239, -0.009436486288905144, 0.04048912227153778, -0.0038244968745857477, 0.01590385101735592, -0.024618277326226234, 0.0024062420707195997, 0.0011041219113394618, -0.017842940986156464, -0.006772493943572044, 0.0166818518191576, -0.039024680852890015, -0.01504729874432087, -0.01880185306072235, -0.026108654215931892, 0.0098018329590559, -0.006208647508174181, 0.01537449099123478, -0.003736112266778946, 0.00017748460231814533, 0.01879817433655262, 0.00962801929563284, 0.022825298830866814, 0.021838584914803505, -0.011081316508352757, 0.008028491400182247, -0.023349007591605186, -0.0077235945500433445, 0.02362522855401039, 0.004489016719162464, 0.02206561714410782, -0.03401845693588257, 0.013595486991107464, -0.015976930037140846, 0.026694800704717636, 0.010085695423185825, 0.052902258932590485, 0.02887936681509018, -0.019901880994439125, -0.01321110874414444, 0.015050629153847694, 0.02430128864943981], [0.11110294610261917, -0.07415112107992172, -0.01481661293655634, 0.06209605932235718, -0.02493346482515335, -0.010621300898492336, -0.02591891586780548, 0.04735489934682846, -0.010869394056499004, -0.05137038603425026, 0.06716902554035187, 0.016035087406635284, -0.035781316459178925, 0.12110980600118637, 0.01812192052602768, 0.015166806988418102, 0.07513681799173355, -0.06730817258358002, -0.05430930480360985, -0.00543958181515336, -0.09848760068416595, -0.08812189102172852, -0.06864689290523529, -0.045017220079898834, 0.009603317826986313, 0.04473556578159332, 0.03189895302057266, -0.027879182249307632, 0.005087601486593485, 0.054994359612464905, 0.035995982587337494, -0.05526188388466835, -0.054258041083812714, -0.07368180900812149, 0.017400281503796577, -0.01900467276573181, -0.055533718317747116, -0.007633778732270002, 0.0386846624314785, 0.01068394910544157, 0.008345737121999264, -0.0197877399623394, -0.04596567153930664, -0.012548710219562054, 0.036568254232406616, 0.018391674384474754, -0.028508108109235764, -0.010493620298802853, -0.044057492166757584, -0.08039233833551407, -0.03417789563536644, 0.0031757992692291737, -0.06829103082418442, -0.04185567796230316, -0.04162822663784027, 0.019919900223612785, -0.05079641193151474, 0.09556739777326584, 0.04153018072247505, -0.06104379892349243, 0.07229865342378616, -0.07080360502004623, -0.03420078009366989, 0.018268588930368423, 0.03739086166024208, 0.06197865307331085, -0.0059721460565924644, -0.011094955727458, -0.006035193335264921, 0.0815352201461792, -0.016365403309464455, -0.07699891924858093, 0.12235397845506668, 0.06920512020587921, 0.013166570104658604, 0.05094989016652107, -0.04049448296427727, -0.002664034953340888, -0.012637981213629246, 0.001988158794119954, -0.03990505263209343, -0.006453756708651781, 0.0768665298819542, 0.043047502636909485, 0.05732638016343117, 0.03562340885400772, -0.028076792135834694, 0.08380987495183945, -0.004094920586794615, -0.034247901290655136, -0.07692638784646988, -0.04373783990740776, -0.07399994879961014, -0.009099748916924, 0.02094602957367897, -0.02034701220691204], [-0.015823524445295334, -0.017857544124126434, -2.355739525228273e-06, 0.02083728462457657, 0.0030787927098572254, 0.0015179505571722984, 0.011262595653533936, 0.040038373321294785, -0.004377542529255152, 0.0505651980638504, 0.0028258590027689934, 0.04411495849490166, 0.01651621237397194, -0.006998969707638025, 0.003127299714833498, 0.004718262702226639, -0.028134984895586967, 0.005839321296662092, -0.0127290990203619, -0.00662250816822052, 0.01181963924318552, 0.017336862161755562, -0.00632754759863019, 0.003130609868094325, -0.012765038758516312, -0.02862243726849556, 0.009227648377418518, 0.015681931748986244, 0.035530559718608856, -0.02200961485505104, -0.0004716709372587502, 0.008790294639766216, -0.007966983132064342, 0.002027222653850913, 0.01602056808769703, 0.005970773287117481, -0.029994605109095573, -0.009934991598129272, -0.0003382741124369204, -0.017165975645184517, -0.006254378240555525, -0.004350097384303808, 0.03138478472828865, -0.004946902394294739, 0.03321727365255356, 0.0032221029978245497, -0.009628516621887684, 0.02256901189684868, 0.026294169947504997, 0.00819733738899231, 0.006565652787685394, 0.015719827264547348, -0.010252558626234531, 0.004773441236466169, 0.013509525917470455, 0.022080982103943825, 0.0009791977936401963, -0.027107147499918938, -0.03701838478446007, -0.020330367609858513, -0.028026653453707695, 0.027684014290571213, 0.0026112846098840237, 0.008245919831097126, 0.06902050971984863, 0.029759109020233154, 0.02700714021921158, 0.026823632419109344, 0.008045218884944916, -0.0006505753262899816, 0.018114665523171425, 0.020582148805260658, -0.006371837574988604, 0.023686140775680542, -0.005075271241366863, 0.0011423229007050395, -0.0023264128249138594, -0.022949177771806717, -0.011003365740180016, 0.002339357743039727, 0.017310911789536476, -0.02677512727677822, -0.02483518049120903, 0.0010082697262987494, 0.022229379042983055, 0.00073254230665043, 0.011211035773158073, 0.006424251012504101, 0.018831303343176842, -0.00031207146821543574, 0.008565115742385387, -0.0037777579855173826, -0.005451308563351631, 0.0016194417839869857, 0.02488899789750576, -0.0037171836011111736], [0.03670760244131088, 0.060106970369815826, 0.07097304612398148, 0.02587064914405346, 0.015372855588793755, 0.0647568330168724, -0.10499713569879532, 0.0885109007358551, 0.07051616162061691, 0.05000532045960426, -0.09636761993169785, 0.08198927342891693, 0.05949411168694496, 0.004384641535580158, 0.01147631835192442, 0.01863812282681465, 0.07816191017627716, -0.16328881680965424, 0.0131594929844141, 0.03626999258995056, 0.08761727064847946, -0.04603280872106552, -0.04298234358429909, -0.017563801258802414, -0.0026465340051800013, 0.004940433893352747, -0.02631312794983387, 0.09858443588018417, -0.010943418368697166, 0.03357573598623276, -0.06311647593975067, -0.043591562658548355, -0.015908028930425644, 0.09789013117551804, 0.12454137951135635, 0.03732636570930481, -0.05783089995384216, -0.05693133920431137, -0.07634882628917694, 0.06788899749517441, 0.007833351381123066, -0.006914489436894655, 0.04322279617190361, 0.054090242832899094, 0.03181764855980873, -0.015454350039362907, -0.08722738176584244, -0.0936630368232727, -0.08567894250154495, 0.040889039635658264, -0.03118574433028698, -0.11381639540195465, -0.025365406647324562, -0.053221095353364944, 0.09017740190029144, -0.04030075669288635, 0.08364458382129669, -0.029252732172608376, 0.06971631199121475, 0.05937289074063301, -0.035399723798036575, -0.0079515865072608, 0.0051157064735889435, 0.034757912158966064, -0.09039328247308731, 0.0545683354139328, 0.023369958624243736, 0.027009526267647743, 0.030178295448422432, -0.04549384117126465, 0.009760218672454357, -0.0006400837446562946, -0.04578075557947159, -0.02482028119266033, -0.039956167340278625, -0.04059767723083496, 0.055722255259752274, -0.0002762170333880931, 0.0007854658761061728, 0.0008727696258574724, -0.06400399655103683, 0.013167214579880238, -0.006554012652486563, 0.014431209303438663, -0.14414151012897491, -0.004488600417971611, -0.007717333268374205, 0.06272618472576141, 0.03327391669154167, -0.05597289279103279, -0.01762593537569046, -0.1340165138244629, -0.03149789199233055, -0.03179548308253288, 0.06017128378152847, -0.059490058571100235], [0.015329147689044476, 0.03296623378992081, -0.040860723704099655, -0.021399974822998047, 0.024052875116467476, -0.027396008372306824, -0.027700215578079224, 0.027907313778996468, -0.02659516967833042, -0.022417623549699783, -0.02229125425219536, -0.027754902839660645, 0.0022219144739210606, 0.035642631351947784, 0.0031205848790705204, 0.02497795596718788, 0.02635032869875431, -0.0358329638838768, 0.002445496153086424, 0.05159056559205055, 0.007192180026322603, -0.001657077926211059, 0.06903134286403656, 0.03988911956548691, 0.05130456015467644, -0.04105333983898163, 0.037072762846946716, 0.04348289221525192, 0.010761317797005177, 0.07519219070672989, 0.00930988509207964, -0.01089983619749546, -0.015289473347365856, 0.08009926229715347, 0.06707821041345596, 0.055873095989227295, 0.03265030309557915, -0.01404247060418129, -0.04358144849538803, 0.02566552348434925, 0.019367383792996407, -0.038907214999198914, 0.05611628666520119, 0.029531104490160942, -0.057728689163923264, 0.021499549970030785, 0.03480665013194084, -0.03959014266729355, 0.01968999207019806, 0.06155448406934738, -0.002716680057346821, 0.013493333011865616, 0.05354643613100052, 0.030619822442531586, 0.0022487014066427946, 0.03819851577281952, 0.013233145698904991, -0.014292842708528042, 0.09222696721553802, -0.004636114463210106, -0.02941405214369297, 0.029457489028573036, -0.01587204821407795, -0.016436254605650902, -0.03594648465514183, -0.018334057182073593, -0.04300559684634209, 0.07545844465494156, -0.030002666637301445, -0.010302284732460976, -0.03327872231602669, 0.009732366539537907, -0.0004893774748779833, -0.026843102648854256, 0.013946440070867538, -0.0012204617960378528, -0.04064037650823593, 0.006721082609146833, 0.015488829463720322, 0.03262802213430405, -0.005982249975204468, 0.03615240752696991, 0.0575484074652195, -0.007987603545188904, 0.04096655175089836, -0.007421065121889114, 0.010683514177799225, 0.02165910415351391, -0.05048717185854912, -0.008003206923604012, 0.0496663935482502, 0.004328671842813492, 0.011253315024077892, 0.04207340255379677, 0.0362008735537529, -0.02604677341878414], [0.02348717302083969, -0.024926578626036644, -0.01049044169485569, 0.024338414892554283, -0.03402571752667427, 0.0005693900748156011, 0.006130312569439411, -0.018496708944439888, 0.0425332747399807, -0.018747767433524132, 0.018850980326533318, -0.01790151186287403, 0.0025855174753814936, 0.028002047911286354, -0.0007426123484037817, -0.008463072590529919, 0.052989330142736435, 0.0015430728672072291, -0.008138515055179596, -0.012077669613063335, -0.005565375089645386, -0.018391504883766174, -0.021008457988500595, -0.005028789397329092, 0.010155638679862022, 0.03451389819383621, 0.0008178697898983955, -0.010885423980653286, 0.001022492884658277, -0.009006264619529247, 0.013071563094854355, 0.008498448878526688, -0.0031079871114343405, -0.011682160198688507, -0.0292050801217556, -0.02407732605934143, -0.01506299339234829, 0.004871147219091654, 0.015319283120334148, 0.01274288073182106, 0.004127634689211845, 0.003538611577823758, -0.012825357727706432, -0.004485207609832287, -0.004244223237037659, -0.007460991386324167, 0.009895318187773228, -0.001617481466382742, -0.032164182513952255, -0.015523799695074558, -0.025342877954244614, -0.011180415749549866, -0.03628084436058998, -0.02262819930911064, -0.011600067839026451, -0.010405723005533218, -0.011036773212254047, 0.060648322105407715, 0.027451811358332634, -0.022011451423168182, 0.05201135575771332, -0.010962632484734058, 0.00020810043497476727, -0.006375872064381838, -0.0022700491826981306, -0.004790442530065775, -0.005855835974216461, -0.052280157804489136, 0.002127890009433031, 0.01703810505568981, -0.00680569838732481, -0.017883194610476494, 0.02828230895102024, 0.005634184926748276, -0.001942161121405661, -0.0024321579840034246, -0.002546340227127075, 0.03217090293765068, -0.01788335107266903, -0.005186145659536123, -0.007175881881266832, -0.016291886568069458, 0.0356982946395874, 0.0037224243860691786, 0.00639050267636776, -0.004100482910871506, -0.014378123916685581, 0.009776415303349495, 0.03071492910385132, -0.010104025714099407, -0.016128001734614372, -0.021964628249406815, -0.007480209227651358, -0.00740219047293067, -0.02419361099600792, 0.013509219512343407], [0.0006302915280684829, -0.023152876645326614, 0.010448036715388298, 0.0036000777035951614, -0.022732513025403023, 0.00048428805894218385, -0.002703884383663535, -0.0026820821221917868, -0.005881220102310181, -0.009945027530193329, 0.004975912626832724, 0.003979372326284647, 0.002483098767697811, 0.00031650305027142167, 0.0015915653202682734, -0.0013169969897717237, -0.0038455615285784006, 0.004365743603557348, -0.006911988835781813, -0.0035632832441478968, 0.004777750466018915, 0.0011184512404724956, -0.001186876674182713, -0.003581369062885642, 0.007001219782978296, -0.006569986697286367, -0.00020724481146316975, 0.004289993084967136, 0.016947995871305466, 0.007276890799403191, 0.000940206868108362, 0.004385586827993393, -0.0011111709754914045, -0.003618424292653799, 0.004212968051433563, -0.0007041363278403878, -0.0007766333874315023, -0.004533950239419937, 0.000532622798345983, 0.0042614261619746685, -0.0010194744681939483, 0.0013380568707361817, -0.008867311291396618, -0.0017538282554596663, -0.0001132516044890508, 0.0048634931445121765, 0.008715751580893993, -0.0014622658491134644, -0.0046112677082419395, -0.018039414659142494, 0.009635734371840954, -0.004168097861111164, 0.010760826990008354, -0.008816746063530445, -0.004314541816711426, 0.0033572285901755095, -0.026463713496923447, -0.0002802262897603214, -0.014169213362038136, 0.007546606939285994, -0.005928597413003445, -0.005207051523029804, -0.004303108900785446, 0.004885677248239517, 0.017192687839269638, 0.00671563483774662, 0.003656355431303382, -0.019146395847201347, 0.00808499101549387, 0.0005741208442486823, 0.0017064879648387432, -0.010804064571857452, 0.0010838572634384036, 0.005182818975299597, -0.0014609333593398333, -0.002239060588181019, -0.018056049942970276, 0.0020572685170918703, -0.017203522846102715, -0.0012245253892615438, -7.23914781701751e-05, -0.003832684364169836, 0.002279167529195547, -0.0004332686075940728, 0.013466625474393368, 0.010569826699793339, -0.01305107120424509, -0.00011946131417062134, 0.016574250534176826, -0.00020352074352558702, 0.013629421591758728, -0.006914260797202587, -0.006301532965153456, -0.0022687152959406376, -0.010640650056302547, -0.007353338412940502], [0.019638409838080406, -0.0033346451818943024, -0.006643535103648901, -0.021512268111109734, 0.01978803239762783, 0.006192885339260101, -0.01380555983632803, -0.0172257199883461, 0.00014846122940070927, 0.006068493239581585, -0.01623547449707985, -0.002670794725418091, -0.002964905695989728, -0.017823338508605957, -0.00030757515924051404, -0.0026764001231640577, 0.007374468259513378, -0.04076244309544563, 0.003975989762693644, 0.0017350803827866912, 0.008794658817350864, -0.01256000716239214, 0.017750317230820656, -0.004753480199724436, -0.0065813674591481686, -0.005768279079347849, 0.004327502101659775, 0.0072154393419623375, -0.021999651566147804, -0.0022704689763486385, -0.01681152731180191, -0.010630903765559196, -0.003732754848897457, -0.005488074384629726, 0.0034700704272836447, -0.00858575664460659, -0.015466984361410141, 0.005914471112191677, -0.009812072850763798, 0.007631113287061453, -0.0007562655955553055, -0.005625495221465826, 0.006186910904943943, 0.0024673822335898876, -0.01943838782608509, -0.00025887193623930216, -0.007979010231792927, -0.008824528194963932, -0.0032328020315617323, 0.019744427874684334, -0.0029094796627759933, -0.0092703802511096, 0.012075087055563927, 0.01167238224297762, 0.002309413393959403, -0.0013007057132199407, -0.011691923253238201, 0.002961084945127368, 0.021096380427479744, 0.0038855543825775385, -0.00302559114061296, 0.0016620027599856257, -0.0027915919199585915, -0.0002479991235304624, -0.029285859316587448, -0.009783752262592316, -0.01359192281961441, 0.022205445915460587, -0.010145789943635464, -0.010244349017739296, -0.006206106394529343, -0.005156527739018202, -0.005331072025001049, -0.012182875536382198, -0.0022093849256634712, -0.0032142389100044966, 0.0018542023608461022, 0.012188058346509933, 0.0036721096839755774, 0.0018877709517255425, -0.003797852899879217, 0.02342904917895794, 0.007733060512691736, 0.0025826336350291967, -0.01076537650078535, 0.004012149292975664, -0.0007496323087252676, -0.0018641892820596695, -0.03234756737947464, 0.004496986977756023, -0.007911797612905502, 0.0014848842984065413, -0.002075720578432083, -0.009367043152451515, -0.004033140372484922, 0.0017673533875495195], [0.0026274719275534153, 0.0018713269382715225, -0.0009972763946279883, -0.0038836721796542406, 0.003009828506037593, 0.0015320717357099056, -0.0015735500492155552, -0.0013434127904474735, -0.0009138004970736802, -0.003633447689935565, -0.00287247309461236, -0.0015300264349207282, -0.0016775487456470728, -0.00146204954944551, -0.00021467325859703124, -0.000500061025377363, 0.001256108982488513, -0.007086410187184811, 0.0015430585481226444, 0.0020371421705931425, -0.003536809701472521, 0.0012360686669126153, 0.005170505493879318, 0.0016027389792725444, -0.001653309678658843, -0.00030414111097343266, 0.0002939832047559321, 0.0014599020360037684, -0.009294775314629078, -0.0014387945411726832, -0.0022735516540706158, -0.0030421100091189146, -0.0008091379422694445, 0.003637850284576416, -0.005807600449770689, -0.00037350045749917626, 0.001315259956754744, 0.0010497799376025796, -0.0012265790719538927, 0.0003441624576225877, 0.0008204604382626712, 0.00022406921198125929, -0.0018813947681337595, 0.001127102063037455, -0.005007885862141848, 0.0005952693754807115, -0.0016029634280130267, -0.0013377065770328045, 0.001907719299197197, 0.005077899433672428, 0.001462083775550127, -0.0008280001929961145, 0.0032416381873190403, 0.0017515693325549364, -0.000255333463428542, -0.00209883158095181, -0.007547424640506506, -0.0009592378628440201, 0.005827784072607756, 0.0010216273367404938, 0.0002237428998341784, -0.001187753165140748, -0.00011992861254839227, -0.0007826380897313356, -0.005382653791457415, -0.00456585455685854, -0.002011915436014533, 0.003981428686529398, -0.0016828743973746896, -0.001364325638860464, -0.0009680137154646218, 0.0018223416991531849, -0.0013770132791250944, -0.0023374734446406364, 0.0003304222191218287, 0.00046934245619922876, -0.00027619258617050946, 0.001092557329684496, 0.0037927990779280663, -0.0005152026424184442, -0.0010741871083155274, 0.005414981860667467, 0.0014806243125349283, 0.0001721354783512652, -0.003393796971067786, 0.0003221695660613477, -3.811433998635039e-05, -0.00035654392559081316, -0.007038417737931013, -0.0002821899252012372, -0.0004101299855392426, 0.0010115710319951177, 5.719758337363601e-05, -0.00015567528316751122, 0.00027827260782942176, -0.00024167871742974967], [0.0949171856045723, 0.07156132161617279, -0.035742584615945816, -0.025136955082416534, 7.200771506177261e-05, 0.05181486904621124, -0.029423628002405167, -0.026181919500231743, 0.037528470158576965, 0.00011975493544014171, -0.021243194118142128, -0.008140095509588718, -0.004301905166357756, -0.023960262537002563, 0.004115100018680096, 0.03306007757782936, 0.008098235353827477, 0.012371518649160862, -0.007700004614889622, 0.005889704916626215, 0.0702546238899231, -0.042227212339639664, -0.03593944013118744, -0.029189465567469597, 0.023734381422400475, 0.013676273636519909, -0.0059876758605241776, 0.05655834451317787, -0.0765550285577774, -0.025422662496566772, -0.011753682978451252, -0.06525199115276337, 0.020217888057231903, -0.046592433005571365, 0.01691790111362934, 0.03893814980983734, 0.053618304431438446, -0.025177698582410812, -0.045713264495134354, 0.0467379093170166, 0.00734880892559886, -0.046054739505052567, 0.04775163158774376, 0.02967498078942299, -0.02234986051917076, -0.004128691274672747, 0.03365812450647354, -0.0004663133586291224, -0.023566273972392082, 0.0335867702960968, -0.00631408765912056, -0.040986742824316025, 0.0650520771741867, -0.0016099420608952641, -0.01163856964558363, 0.04027659446001053, -0.06866279989480972, 0.014784742146730423, 0.04465494677424431, -0.02032351680099964, -0.002762619173154235, 0.040640123188495636, -0.018517309799790382, 0.039716046303510666, -0.09049904346466064, -0.05150477588176727, -0.07520322501659393, 0.09340550005435944, 0.00430847704410553, -0.055821742862463, -0.017130719497799873, 0.04536096751689911, -0.050749748945236206, -0.020143287256360054, -0.024204907938838005, -0.062259215861558914, 0.02148834429681301, 0.02742597460746765, -0.03514452278614044, -0.03487176075577736, -0.03270197659730911, 0.0021048509515821934, 0.04376642778515816, -0.000219793088035658, 0.009100168012082577, -0.05173439159989357, -0.04168444499373436, -0.02879384718835354, -0.09095153957605362, -0.017855845391750336, -0.0011110490886494517, -0.00065844994969666, -0.03472619131207466, -0.02278461866080761, 0.05131286755204201, -0.025130262598395348], [-0.08738914132118225, -0.0066022505052387714, -0.021269416436553, 0.043086692690849304, -0.10837740451097488, 0.05048871040344238, -0.025877483189105988, -0.01648508757352829, -0.01864016242325306, 0.016154920682311058, -0.03925031051039696, -0.007613691966980696, 0.03731294348835945, -0.027068978175520897, 0.01161988452076912, 0.01089015044271946, 0.028449390083551407, 0.07107099145650864, -0.045032061636447906, 0.014871849678456783, 0.07334990799427032, -0.0160130113363266, -0.005045812577009201, 0.0008512195199728012, -0.011994762346148491, -0.07674051076173782, -0.010765139013528824, -0.01863710954785347, -0.049164894968271255, -0.04810876026749611, 0.028641104698181152, -0.0215937290340662, 0.00984887219965458, 0.015046875923871994, 0.012458291836082935, -0.0525926798582077, 0.05329132452607155, -0.07399263232946396, -0.02939310111105442, 0.02051827684044838, 0.04073374345898628, -0.014486948028206825, -0.08739738911390305, -0.025931887328624725, 0.040376972407102585, -0.0447884164750576, -0.039603911340236664, 0.009385200217366219, 0.004826476331800222, -0.034030940383672714, -0.031150875613093376, 0.02691892348229885, 0.021325312554836273, -0.08066468685865402, -0.051264114677906036, -0.05205840989947319, -0.055861327797174454, -0.009162131696939468, -0.021077536046504974, -0.051510803401470184, -0.07711504399776459, -0.01595265232026577, 0.0005482392734847963, -0.010663378983736038, -0.00863715447485447, 0.01843762956559658, 0.007454822771251202, -0.04430621489882469, 0.05145777016878128, 0.04026972874999046, 0.04941745474934578, -0.04708860442042351, -0.05709042400121689, 0.05500960350036621, -0.038001127541065216, -0.04831987991929054, -0.016098687425255775, -0.08237447589635849, 0.007402556482702494, -0.01871599443256855, 0.012444929219782352, -0.029681799933314323, -0.05478711798787117, 0.01811930723488331, -0.013089715503156185, -0.01400045771151781, -0.08756260573863983, 0.010033964179456234, 0.08405330777168274, -0.05281778797507286, -0.036767490208148956, -0.017146514728665352, -0.09017474204301834, -0.012822017073631287, -0.056301407516002655, -0.008899856358766556], [-0.07177913933992386, -0.11850276589393616, -0.022571193054318428, -0.009909983724355698, -0.043300747871398926, -0.03352765366435051, -0.001551931258291006, 0.01744958944618702, 0.009599081240594387, -0.047123346477746964, -0.022902974858880043, 0.007091199979186058, 0.011966892518103123, -0.030767835676670074, 0.010370994918048382, -0.016071248799562454, -0.04408229887485504, 0.06011287868022919, -0.03170233964920044, -0.019585436210036278, 0.032960422337055206, -0.0010328511707484722, -0.04395101219415665, -0.0016903659561648965, 0.0011345385573804379, -0.1256038397550583, 0.02015875093638897, 0.0027833792846649885, -0.038785237818956375, -0.061115045100450516, -0.01601061224937439, 0.0066954512149095535, 0.014849243685603142, -0.003200084436684847, 0.05163971707224846, -0.02709331549704075, -0.03423183411359787, -0.057060662657022476, -0.058327898383140564, -0.05215093120932579, -0.004025537054985762, -0.009147988632321358, -0.050955794751644135, -0.004955294542014599, -0.017298700287938118, 0.0007482145447283983, 0.0057372585870325565, -0.014632299542427063, 0.013312596827745438, -0.05362271890044212, -0.02164372242987156, -0.014627580530941486, 0.0591856874525547, -0.03571276739239693, 0.009841409511864185, -0.021281227469444275, -0.07193142920732498, -0.0488772876560688, -0.08200815320014954, 0.007482900284230709, -0.09197136759757996, 0.002776782028377056, 0.0014472828479483724, -0.00068047852255404, -0.018290365114808083, -0.006796157918870449, 0.0017381098587065935, -0.00897100381553173, 0.034523915499448776, -0.010017532855272293, -0.003599417395889759, -0.03229842707514763, -0.03637753799557686, 0.003630873281508684, -0.013670087791979313, -0.05172716826200485, -0.011211580596864223, -0.06074797362089157, -0.022879399359226227, -0.004763862583786249, -0.037051644176244736, -0.028247622773051262, -0.0314323864877224, -0.02342284843325615, -0.005916370078921318, -0.027686526998877525, -0.02208007499575615, -0.004523461684584618, -0.008156335912644863, 0.0009383484139107168, -0.010714883916079998, -0.03389624506235123, -0.03925882279872894, -0.017038660123944283, 0.008327909745275974, -0.021558666601777077], [0.1171817034482956, -0.12398863583803177, -0.03598027303814888, 0.026034999638795853, -0.06388437002897263, 0.03187760338187218, -0.06593475490808487, -0.036665402352809906, -0.021140705794095993, -0.09648030251264572, 0.061925821006298065, -0.04751050844788551, 0.027863210067152977, 0.07524167746305466, 0.03339272737503052, -0.04041096568107605, 0.061027832329273224, -0.04650024324655533, -0.028938235715031624, -0.020058805122971535, -0.011755525134503841, -0.06316155195236206, -0.006031927652657032, 0.012911838479340076, 0.06106799840927124, 0.05321265012025833, -0.001985454699024558, -0.01586589775979519, -0.05094931647181511, -0.0818166583776474, -0.01492829155176878, -0.003308139508590102, 0.06391721963882446, -0.003510858165100217, -0.057949986308813095, -0.007599098142236471, -0.04549843445420265, -0.018313001841306686, 0.053205009549856186, 0.08541747182607651, 0.05298575013875961, -0.029443763196468353, -0.019069788977503777, -0.02306438609957695, -0.028276558965444565, 0.017324304208159447, 0.00915426854044199, -0.05171649530529976, -0.14162981510162354, -0.05140106752514839, -0.036824408918619156, -0.0704631432890892, -0.0954953208565712, -0.06964454799890518, -0.033871669322252274, -0.007267313078045845, -0.03237583860754967, 0.1073492243885994, 0.07687646895647049, -0.04061061888933182, 0.1477048099040985, -0.09461135417222977, 0.020622769370675087, 0.011637172661721706, -0.05131448805332184, 0.004377918783575296, 0.05091771483421326, -0.13022658228874207, -0.039090514183044434, 0.07332570850849152, -0.027325157076120377, -0.04426976665854454, 0.1052074059844017, 0.007123254705220461, 0.03436128795146942, 0.02707829140126705, -0.07095091789960861, 0.08756715804338455, -0.012431726790964603, 0.02103806845843792, -0.05529056116938591, -0.07829594612121582, -0.04302418604493141, 0.0021426330786198378, 0.03448454290628433, -0.007986309938132763, -0.07321162521839142, 0.03397500887513161, -0.0441836379468441, -0.018677441403269768, -0.07924847304821014, -0.08080250024795532, 0.015476251021027565, -0.04102671891450882, -0.002730731153860688, 0.03428919240832329], [0.07479957491159439, -0.0016249784966930747, 0.007148389238864183, 0.060230519622564316, -0.08998855948448181, 0.028014345094561577, 0.033172495663166046, 0.026906467974185944, 0.05431123450398445, -0.05056602135300636, 0.03802178055047989, -0.014124519191682339, -0.005219515413045883, 0.06356385350227356, 0.014155566692352295, -0.007711924612522125, 0.10299781709909439, -0.015920719131827354, -0.018575076013803482, -0.039683591574430466, -0.012863433919847012, -0.03740374371409416, -0.00245597749017179, -0.01868460886180401, -0.04227220267057419, 0.11041812598705292, -0.019062383100390434, -0.010766899213194847, -0.014178036712110043, 0.045694734901189804, 0.03240754082798958, -0.012624131515622139, 0.009489313699305058, -0.046104319393634796, -0.0204533152282238, -0.034281689673662186, -0.06455281376838684, 0.003839312121272087, 0.05837912857532501, 0.015376800671219826, 0.0013608908047899604, -0.013460207730531693, 0.011539422906935215, -0.033246248960494995, 0.02280418574810028, -0.0016324002062901855, -0.017306677997112274, 0.0036857984960079193, -0.03507442772388458, -0.05108567327260971, -0.006594762671738863, 0.0033260085619986057, -0.06749892234802246, -0.056156937032938004, -0.03526001796126366, -0.0004978280048817396, -0.05894652009010315, 0.06573247164487839, 0.0158317182213068, -0.017024431377649307, 0.10349451750516891, -0.052726734429597855, 0.005369148682802916, -0.0031742420978844166, 0.01567452773451805, 0.020647898316383362, -0.003530014306306839, -0.0231173038482666, -0.005650038830935955, 0.012499963864684105, -0.008197478018701077, -0.03798084333539009, 0.049549829214811325, 0.03044244647026062, -0.004767254926264286, -0.017221279442310333, -0.04616270214319229, 0.04024778679013252, -0.04019393399357796, -0.01822575181722641, -0.02416968159377575, -0.015079200267791748, 0.023471180349588394, 0.010412930510938168, 0.049954432994127274, -0.018934281542897224, 0.00200789631344378, 0.04487327113747597, -0.01675361581146717, -0.0471285842359066, -0.03513026610016823, -0.019879236817359924, -0.0414508655667305, -0.00825347937643528, -0.016315236687660217, -0.021003080531954765], [-0.007915052585303783, 0.01174409780651331, -0.012020869180560112, -0.021064531058073044, 0.018253380432724953, 0.009871156886219978, -0.015412900596857071, -0.0132832583039999, -0.02507241815328598, -0.003991089761257172, -0.01594739593565464, -0.012245774269104004, -0.009396166540682316, -0.011608248576521873, -0.0014402640517801046, -0.0019249299075454473, 0.00018715803162194788, -0.00186726835090667, 0.015321915037930012, 0.018505144864320755, -0.015411999076604843, 0.0017535289516672492, 0.015814106911420822, 0.010135386139154434, 0.01130637340247631, 0.00042363739339634776, -0.0006628171540796757, -0.0023481722455471754, -0.01843491941690445, 0.003269529202952981, -0.005896416027098894, -0.019519958645105362, -0.008999683894217014, -0.0034316321834921837, -0.0020582852885127068, 0.02842080406844616, -0.0005262922495603561, 0.008131735026836395, -0.0017305311048403382, 0.016873199492692947, 0.001790344249457121, -0.006875365972518921, -0.007231740280985832, 0.005980479996651411, -0.0246118176728487, 0.0021644318476319313, 0.001444868859834969, -0.005552490707486868, 0.0013972403248772025, 0.010908822529017925, 0.017683973535895348, 0.0020858068019151688, 0.0034848826471716166, 0.011269527487456799, -0.002599212573841214, 0.002143962075933814, -0.0054174368269741535, 0.004288611467927694, 0.01905103214085102, -0.0014584000455215573, -0.002078455174341798, -0.008966916240751743, 0.0005540791316889226, -0.003163385670632124, -0.028664711862802505, -0.008810766972601414, -0.017545372247695923, 0.03322373703122139, -0.012780505232512951, -0.00019530147255863994, -0.009446141310036182, 0.0014018170768395066, -0.0030456536915153265, -0.012675512582063675, 0.0024133878760039806, -0.001226050779223442, -0.00770779000595212, 0.0012551040854305029, 0.004915642086416483, 0.0019727589096874, 0.0005071184132248163, 0.02782517857849598, 0.02889314666390419, 0.001055572647601366, 0.004745554644614458, -0.004675895441323519, 0.019903285428881645, -0.00418776273727417, -0.010606733150780201, -0.007891892455518246, -0.01148240640759468, 0.026692884042859077, -0.007543474901467562, -0.0059690107591450214, -0.004298292566090822, 0.020274123176932335], [-0.10900145024061203, 0.058687370270490646, 0.04511973261833191, 0.0616220198571682, 0.03835613653063774, -0.041304487735033035, 0.06885445863008499, 0.08178918063640594, 0.020388204604387283, 0.06776036322116852, -0.018626734614372253, 0.06332377344369888, 0.047884877771139145, 0.029845014214515686, 0.020688001066446304, 0.06661170721054077, -0.11915575712919235, 0.0649406686425209, -0.01859404146671295, -0.06923457980155945, 0.08372295647859573, -0.014282958582043648, -0.05654451251029968, 0.012920653447508812, -0.03684446960687637, -0.10094983875751495, -0.039694130420684814, -0.014842926524579525, 0.0005268604145385325, -0.06402746587991714, -0.04582291468977928, 0.057074617594480515, 0.0036776575725525618, -0.06058796867728233, 0.07077240943908691, 0.004401990212500095, -0.07392649352550507, -0.02246882952749729, -0.002297665923833847, -0.08169268816709518, -0.0058378963731229305, 0.04942939803004265, 0.07643429189920425, 0.00010005463991547003, 0.007045688107609749, 0.06699410080909729, 0.015163389034569263, 0.05610990151762962, 0.009074117057025433, -0.048016708344221115, 0.08515045791864395, 0.0221543125808239, 0.08031723648309708, 0.02587589994072914, 0.035519078373909, -0.08005198836326599, 0.03313479945063591, -0.0133031802251935, -0.06603138148784637, 0.07639166712760925, -0.11877791583538055, -0.01317732036113739, -0.009613647125661373, 0.039878010749816895, 0.00374514888972044, -0.011892648413777351, 0.06504299491643906, -0.027146875858306885, 0.06637389957904816, 0.018577415496110916, 0.02035175822675228, 0.059457145631313324, -0.014662460424005985, -0.01118866540491581, 0.027209322899580002, -0.06963913142681122, -0.026194801554083824, -0.05967506021261215, -0.04134215787053108, -0.015537728555500507, 0.036694109439849854, -0.09600280225276947, -0.07272837311029434, -0.03619653731584549, -0.025919079780578613, 0.01785116083920002, 0.02295130491256714, 0.016007624566555023, 0.016174351796507835, 0.060345619916915894, 0.04166518151760101, 0.03165939822793007, -0.013260374777019024, -0.003830943489447236, 0.07259456813335419, -0.002338637365028262], [0.055075228214263916, 0.07228226214647293, -0.06159307435154915, -0.10564720630645752, -0.011957157403230667, 0.003615020774304867, -0.051480405032634735, 0.062150739133358, 0.043068788945674896, 0.02074366994202137, -0.03417281061410904, -0.04301352798938751, 0.07160218060016632, -0.031230824068188667, -0.011844614520668983, 0.06159915775060654, 0.00905225146561861, -0.11074930429458618, 0.05890907347202301, 0.05385097116231918, -0.020863717421889305, -0.010472149588167667, 0.014746990986168385, -0.002596022328361869, -0.00824328139424324, -0.0818951278924942, 0.004030801355838776, 0.05549781396985054, -0.06529197841882706, 0.0847809836268425, -0.07990597933530807, 0.0009879372082650661, -0.06416693329811096, 0.016273310407996178, -0.012349775061011314, -0.007119144778698683, 0.06553158164024353, 0.04856419190764427, -0.04154694452881813, 0.08143095672130585, 0.0472305566072464, 0.05442816764116287, 0.010301514528691769, -0.018004540354013443, -0.0133591303601861, -0.02925380878150463, -0.043219342827796936, -0.00033922941656783223, -0.06593380123376846, 0.0005123352748341858, -0.007690485566854477, -0.04509502649307251, 0.04203612357378006, -0.01795336790382862, 0.08921626955270767, 0.010805468074977398, 0.034192878752946854, -0.04130689799785614, 0.01380061637610197, 0.06438007205724716, -0.05826202780008316, -0.02304750122129917, 0.03545092046260834, 0.07388852536678314, -0.03886416554450989, 0.023555027320981026, -0.032804347574710846, 0.003950266167521477, -0.07234451919794083, 0.010081538930535316, 0.024826055392622948, 0.0513494573533535, 0.005275674629956484, -0.037986986339092255, 0.002157453680410981, -0.0771956667304039, -0.020216412842273712, 0.05636211484670639, 0.08265561610460281, -0.005821129307150841, -0.02124546468257904, 0.018846096470952034, -0.039670079946517944, -0.00974657479673624, 0.012768089771270752, -0.07178296148777008, -0.03860783204436302, 0.0012645088136196136, -0.022090304642915726, -0.01666238345205784, -0.05421583727002144, -0.04660139977931976, 0.05058146268129349, -0.07816362380981445, -0.0425078421831131, 0.07026540488004684], [-0.003965661861002445, -0.0346347875893116, -0.048127807676792145, -0.005316353403031826, -0.017446300014853477, -0.010148354806005955, -0.03597196564078331, 0.047751981765031815, 0.0601535439491272, 0.033916328102350235, -0.021396027877926826, -0.024431563913822174, -0.03063858300447464, 0.028404600918293, -0.02539428137242794, -0.025863392278552055, -0.007104387506842613, -0.024941621348261833, 0.0039494638331234455, -0.03779571130871773, 0.021868642419576645, -0.06023257598280907, 0.00798259861767292, 0.004575780592858791, -0.015944477170705795, 0.016357284039258957, 0.07264174520969391, 0.052736859768629074, 0.03785610944032669, -0.022665495052933693, -0.02884717844426632, 0.005743126384913921, -0.02519221231341362, 0.07472609728574753, 0.029401808977127075, 0.010052870027720928, 0.08085115253925323, 0.011284803971648216, -0.02736925520002842, 0.025531625375151634, 0.017446961253881454, 0.016435354948043823, 0.03549404442310333, -0.004166472237557173, -0.09779553860425949, 0.03570298105478287, 0.03302527219057083, -0.019406381994485855, -0.038840994238853455, -0.013585804961621761, 0.013643341138958931, -0.041442833840847015, 0.07356753200292587, -0.02943521738052368, -0.02259715273976326, 0.036868952214717865, 0.035115353763103485, -0.030463188886642456, 0.047135721892118454, 0.03532525151968002, -0.033609382808208466, 0.023940585553646088, -0.03421681746840477, 0.0468977726995945, -0.09494522958993912, -0.06904617697000504, -0.08879425376653671, 0.02324564941227436, -0.04689580574631691, -0.011720097623765469, -0.02993030659854412, 0.05827493220567703, 0.017962640151381493, 0.01452480349689722, 0.056404173374176025, -0.05283486098051071, -0.023444764316082, -0.033472996205091476, 0.023149223998188972, 0.04259750619530678, -0.019064970314502716, 0.044501978904008865, 0.10278698801994324, 0.025477292016148567, -0.011416686698794365, -0.0668821707367897, 0.03579440340399742, 0.025514008477330208, -0.015401650220155716, 0.008235342800617218, 0.004398182034492493, 0.09001678228378296, 0.01033760979771614, -0.03607765957713127, 0.014785699546337128, 0.053996164351701736], [0.060987189412117004, -0.10560735315084457, 0.0031729231122881174, 0.031092969700694084, -0.04996543005108833, 0.023987464606761932, 0.012543183751404285, -0.020254794508218765, 0.02848026528954506, -0.061649471521377563, 0.03863831236958504, -0.04488024115562439, -0.007531264331191778, 0.034406982362270355, 0.004661917220801115, -0.018770625814795494, 0.0968499407172203, -0.04336341470479965, -0.018670588731765747, -0.018006443977355957, 0.01471657119691372, -0.026842473074793816, -0.011804855428636074, -0.029141157865524292, -0.021510528400540352, 0.02175595983862877, -0.014737945981323719, -0.004041018430143595, -0.013588262721896172, -0.007340074051171541, 0.044867679476737976, -0.010579285211861134, -0.04418649151921272, -0.039009395986795425, -0.002031747018918395, -0.00765771372243762, -0.030384423211216927, 0.001316762762144208, 0.03266081586480141, -0.026142699643969536, 0.012408887036144733, 0.018189305439591408, -0.053035903722047806, 0.0010701748542487621, -0.022827085107564926, -0.00918256863951683, 0.0029689630027860403, -0.019286271184682846, -0.059928275644779205, -0.03517314791679382, -0.05920679494738579, -0.024186132475733757, -0.08645914494991302, -0.05443223938345909, -0.022557077929377556, -0.015338470228016376, 0.0040405686013400555, 0.09809818863868713, 0.062227942049503326, -0.024744562804698944, 0.09665005654096603, -0.04048430919647217, 0.010058116167783737, -0.02573844976723194, -0.017905287444591522, -0.0075683994218707085, -0.0006415464449673891, 0.00505997845903039, -0.013724453747272491, 0.03678445518016815, -0.02148458920419216, -0.017488420009613037, 0.0202534981071949, 0.034267768263816833, -0.031208615750074387, 0.013610205613076687, -0.02430102974176407, 0.00995808094739914, -0.059905972331762314, -0.0029490445740520954, -0.03858087584376335, -0.0396261103451252, -0.0018378372769802809, 0.019710537046194077, -0.03250740468502045, 0.023228053003549576, -0.017093343660235405, 0.04207298159599304, 0.02336902916431427, 0.024196404963731766, -0.07115488499403, -0.030058041214942932, -0.030734436586499214, -0.019938940182328224, -0.0613098070025444, 0.0024472735822200775], [0.09237147122621536, 0.005513107869774103, -0.017343658953905106, 0.0542316697537899, -0.022695347666740417, -0.04452993720769882, 0.020289719104766846, 0.02146104909479618, -0.011283556930720806, -0.047697242349386215, 0.04129713401198387, 0.0055312360636889935, -0.0015703430399298668, 0.039817649871110916, 0.00425577349960804, 0.0011955092195421457, 0.05673859268426895, 0.026583591476082802, -0.011695937253534794, -0.00015404209261760116, -0.029682088643312454, 0.007778987288475037, -0.0605093389749527, -0.02691727690398693, 0.0015460909344255924, 0.11095617711544037, 0.01195458322763443, -0.03341469168663025, 0.023714151233434677, -0.028993412852287292, 0.025135070085525513, 0.018662264570593834, -0.03343811258673668, -0.0322737991809845, -0.02856890670955181, -0.05107586085796356, 0.024447711184620857, 0.01187068223953247, 0.03701349347829819, 0.014204813167452812, 0.0005054338253103197, 0.0020354376174509525, 0.025242749601602554, -0.027932295575737953, 0.033520765602588654, -0.029128260910511017, -0.009997135028243065, -0.0034868833608925343, -0.04298212006688118, 0.010960981249809265, 0.020370351150631905, 0.0128196245059371, -0.06866583228111267, -0.04835638776421547, -0.02674132212996483, 0.01113719493150711, -0.010836051777005196, 0.06855408102273941, 0.05253104120492935, -0.04017183557152748, 0.08331041038036346, -0.026365822181105614, -0.006507066544145346, -0.019052691757678986, -0.0033249808475375175, 0.024456173181533813, 0.010759904980659485, 0.027079889550805092, -0.012498414143919945, 0.013773074373602867, -0.00036852547782473266, -0.02313333749771118, 0.0705689787864685, 0.017136842012405396, 0.013231035321950912, 0.025498535484075546, 0.003588309045881033, 0.03794180229306221, -0.024139203131198883, -0.01187165267765522, -0.010141248814761639, -0.03236611932516098, 0.04044724628329277, 0.02479206956923008, 0.03977558761835098, 0.00010749381908681244, -0.053134895861148834, 0.022316133603453636, 0.07007638365030289, 0.007908690720796585, -0.0073137893341481686, -0.03303363919258118, 0.006221359595656395, 0.0045440527610480785, -0.028609445318579674, -0.02843455784022808], [-0.06763166189193726, -0.07481653988361359, -0.05119997262954712, 0.03457094356417656, -0.07995884120464325, 0.02037341147661209, -0.024204643443226814, -0.0020566838793456554, -0.02652536891400814, -0.04240122437477112, -0.018270770087838173, 0.0017596501857042313, 0.015484173782169819, -0.03144380450248718, 0.011907437816262245, -0.011460200883448124, -0.01740838959813118, 0.00045612422400154173, -0.011085287667810917, -0.01486372109502554, 0.05519219487905502, -0.0012581389164552093, 0.03348179906606674, 0.026932168751955032, -0.013351996429264545, -0.08578963577747345, 0.006420866586267948, -0.03975092992186546, -0.03881818428635597, -0.05486469715833664, 0.003905855119228363, -0.010667439550161362, -0.024531006813049316, 0.003791945753619075, -0.0012062620371580124, 0.016399843618273735, -0.025454571470618248, -0.020038701593875885, -0.042865172028541565, -0.02780352346599102, -0.01941388100385666, 0.004195039160549641, -0.048185866326093674, 0.00014262026525102556, -0.013912336900830269, 0.003911815118044615, 0.00791132915765047, -0.0032167343888431787, -0.02027519978582859, 0.017074797302484512, -0.02270367741584778, -0.009270097129046917, 0.02755569852888584, -0.02026359736919403, -0.01723787747323513, -0.009156742133200169, -0.0753396600484848, 0.006974252872169018, -0.0611359104514122, -0.01310656312853098, -0.06176506355404854, 0.03557907044887543, 0.004791948013007641, -0.01000879891216755, -0.021536201238632202, -0.0285390205681324, -0.020857272669672966, -0.05604727193713188, 0.053313180804252625, -0.012243020348250866, -0.01886606030166149, -0.008669255301356316, -0.03182617202401161, -0.011140426620841026, 0.022216374054551125, 0.0011855948250740767, -0.011128883808851242, -0.03535311669111252, 0.03045635297894478, -0.003721938468515873, -0.02616465650498867, -0.03856818005442619, -0.006409416440874338, -0.00042031367775052786, -0.027387866750359535, -0.0053884549997746944, 0.006619043182581663, -8.13951701275073e-05, 0.07088584452867508, -0.039954774081707, 0.028197618201375008, -0.007850742898881435, -0.002396493684500456, -0.004188713151961565, -0.003991992212831974, 0.0031701272819191217], [-0.10443095117807388, -0.020597338676452637, 0.002967103384435177, -0.012917540967464447, -0.07867713272571564, 0.021396471187472343, -0.02361864224076271, 0.002003851579502225, -0.015375661663711071, 0.009759697131812572, -0.03036753460764885, 0.002151606371626258, 0.010511896573007107, -0.02355947531759739, 0.00679672509431839, 0.014163471758365631, -0.04819750785827637, -0.030546430498361588, -0.032790832221508026, -0.022649023681879044, 0.015905380249023438, 0.05463969334959984, 0.010874885134398937, -0.0354384109377861, -0.032545413821935654, -0.11754784733057022, 0.05674191936850548, 0.04842137172818184, -0.006874418817460537, 0.043857965618371964, 0.002248522127047181, -0.04244234040379524, -0.03039470501244068, 0.0023454942274838686, 0.04241349175572395, 0.024520497769117355, -0.028016407042741776, -0.024360626935958862, -0.009063034318387508, -0.035765014588832855, -0.001453796518035233, -0.007150741759687662, -0.08919437974691391, -0.032560572028160095, 0.05854974314570427, 0.030887490138411522, -0.02955622598528862, -0.03219342976808548, 0.06743519008159637, -0.029857786372303963, -0.020034125074744225, -0.027759596705436707, 0.039089370518922806, -0.017132656648755074, -0.04652993381023407, 0.02564430609345436, -0.09482572972774506, -0.07147111743688583, -0.056234732270240784, 0.009094396606087685, -0.09521445631980896, -0.010244891047477722, -0.011382742784917355, 0.047862082719802856, -0.041030313819646835, -0.023904142901301384, -0.016753168776631355, -0.020996032282710075, 0.03742154687643051, 0.004697541706264019, -0.03897580876946449, -0.036462485790252686, -0.03638722375035286, 0.0514853373169899, -0.031308792531490326, -0.07500523328781128, 0.036571353673934937, -0.05876076593995094, 0.024367893114686012, -0.04062942415475845, -0.03422290086746216, -0.014504256658256054, 0.022189687937498093, -0.008719670586287975, 0.05886789411306381, 0.055851954966783524, -0.007471457589417696, -0.010274617001414299, 0.0752331092953682, -0.02269882895052433, -0.03615717962384224, -0.014287615194916725, -0.06795316934585571, 0.015440992079675198, 0.020590759813785553, -0.036227479577064514]], "b2": [0.029149813577532768, -0.05965036526322365, 0.03228900209069252, 0.05037635192275047, 0.05288948118686676, -0.13031382858753204, -0.06036384776234627, -0.0026994457002729177, 0.040147747844457626, 0.03827814757823944, 0.015472836792469025, -0.05277194082736969, -0.055232420563697815, -0.01329081691801548, 0.10530643165111542, -0.07970324903726578, 0.023393219336867332, 0.07961314171552658, -0.005071947351098061, 0.10340005904436111, 0.0601092129945755, 0.007719236426055431, -0.03245314583182335, -0.04882192611694336, 0.061054445803165436, -0.013590543530881405, 0.004423338454216719, 0.12271322309970856, -0.03464370220899582, -0.09560725092887878, -0.06600678712129593, -0.041125670075416565], "W3": [[-0.15966671705245972, 0.06348562985658646, -0.21262861788272858, -0.0447852797806263, -0.251470148563385, 0.1316658854484558, 0.16275347769260406, 0.0002107564068865031, -0.20139004290103912, -0.1660221368074417, -0.056110408157110214, 0.19200266897678375, 0.06067465618252754, -0.19541998207569122, -0.07383225858211517, 0.07356142997741699, 0.02359258197247982, -0.05007914826273918, -0.01133363414555788, -0.10132943838834763, 0.17963531613349915, 0.13834939897060394, 0.23543474078178406, 0.1379873901605606, -0.05419720336794853, 0.22056306898593903, -0.1681033819913864, -0.11893583834171295, 0.15423212945461273, 0.12425930052995682, 0.1447770595550537, 0.1417057067155838]], "b3": [0.13643164932727814]}}''')
MODEL_FEATS = ['deals_tail1_share', 'dsize_t3', 'dsize_t3_max', 'pre_vol_share', 'ret_1455', 'ret_t3_std', 'ret_tail1', 'ret_tail3', 't3_amt_share', 't3_dn_vs_pre_dn', 't3_dsize_vs_day', 't3_imb1_mean', 't3_ret_vs_day', 't3_sgnvol', 't3_spread_mean', 't3_vol_conc', 't3_vs_pre_ret', 'tail1_deal_size', 'tail3_deals_share', 'vol_1455_share', 'vol_tail1_share']


def _erf_np(x):
    sign = np.sign(x)
    ax = np.abs(x)
    t = 1.0 / (1.0 + 0.3275911 * ax)
    y = 1.0 - (((((1.061405429 * t - 1.453152027) * t) + 1.421413741) * t
                - 0.284496736) * t + 0.254829592) * t * np.exp(-ax * ax)
    return sign * y


def _gelu_np(x):
    return 0.5 * x * (1.0 + _erf_np(x / np.sqrt(2.0)))


def _predict_year(year, values):
    w = MLP_WEIGHTS[str(_model_year(year))]
    W1 = np.asarray(w["W1"], dtype=np.float64)
    b1 = np.asarray(w["b1"], dtype=np.float64)
    W2 = np.asarray(w["W2"], dtype=np.float64)
    b2 = np.asarray(w["b2"], dtype=np.float64)
    W3 = np.asarray(w["W3"], dtype=np.float64)
    b3 = np.asarray(w["b3"], dtype=np.float64)
    X = values.astype(np.float64)  # main 已按 MODEL_FEATS 取子集
    h = _gelu_np(X @ W1.T + b1)
    h = _gelu_np(h @ W2.T + b2)
    return (h @ W3.T + b3).ravel()


# ---------------------------------------------------------------- 主入口

def main(datasources, start_date, end_date):
    import dai  # noqa: F401  (平台环境)
    df1m = _load_bar1m(datasources, start_date, end_date, buf_days=100)
    fl = _load_factorlib(start_date, end_date, buf_days=100)
    fin = _load_fin(datasources, start_date, end_date)
    stk_grid = _stk("2019-01-01", _fin_grid_end(end_date))

    feats = _features(df1m, fl, fin, stk_grid)

    panel = None
    for name in FEATURE_COLS:
        f = feats[name].rename(columns={"factor": name})
        panel = f if panel is None else panel.merge(
            f, on=["date", "instrument"], how="inner")
    panel = panel.sort_values(["date", "instrument"]).reset_index(drop=True)
    panel = panel[(panel["date"] >= pd.to_datetime(start_date))
                  & (panel["date"] <= pd.to_datetime(end_date))]
    # 先收敛到股票池再 z: 训练面板的特征缓存本身就只含池内股票,
    # 若 z 在全市场(含池外 ~2000 只)上算, 均值/std 不同导致输出偏移
    stk = _stk(start_date, end_date)
    panel = panel.merge(stk, on=["date", "instrument"], how="inner")
    for name in FEATURE_COLS:
        g = panel.groupby("date", group_keys=False)[name]
        panel[name] = (panel[name] - g.transform("mean")) \
            / g.transform("std").replace(0, np.nan)

    panel["year"] = panel["date"].dt.year
    pred = np.full(len(panel), np.nan)
    for yr in sorted(panel["year"].unique()):
        idx = np.flatnonzero((panel["year"] == yr).to_numpy())
        values = panel.iloc[idx][MODEL_FEATS].fillna(0.0) \
            .astype("float32").to_numpy()
        pred[idx] = _predict_year(int(yr), values)
    panel["factor"] = pred * SIGN

    out = panel
    out["factor"] = out["factor"].replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=["factor"])

    # ---- 缺日补齐 (平台缺日检查: 整日全缺的交易日用最近有值日填充) ----
    _cal = sorted(pd.to_datetime(stk["date"]).dt.normalize().unique())
    _have = set(pd.to_datetime(out["date"]).dt.normalize().unique())
    _miss = [d for d in _cal if d not in _have]
    if _miss:
        _days = sorted(pd.to_datetime(out["date"]).dt.normalize().unique())
        _od = out.assign(_d=pd.to_datetime(out["date"]).dt.normalize())
        _fill = []
        for d in _miss:
            _prev = [x for x in _days if x < d]
            if _prev:
                _src = _prev[-1]
            else:
                _src = min(_days, key=lambda x: abs((x - d).days))
            _f = _od[_od["_d"] == _src].drop(columns=["_d"]).copy()
            _f["date"] = d
            _fill.append(_f)
        out = pd.concat([_od.drop(columns=["_d"])] + _fill,
                        ignore_index=True)
        out = out.sort_values(["date", "instrument"]).reset_index(drop=True)

    return out[["date", "instrument", "factor"]]
